# 05. 소둔로(CAL01) 전체 데이터 시계열 분석

**대상:** `P1-CR1-CAL01` 연속 소둔로 · **분석 단위:** LOT 안의 연속 관측 구간 · **작성일:** 2026-10-04

온도·제어출력의 시간 흐름을 조업 이벤트, 정비, 정기수리, 부품교체와 함께 살펴봅니다. 먼저 분석 가능한 관측 범위를 정하고, 수집 품질과 조업조건 차이를 확인한 다음 점검할 변화 후보를 좁힙니다. 01~04의 전처리 기준을 재사용하되 원본 CSV에서 수치를 다시 계산했습니다. **질문 → 시각화 → 실제 결과 → 해석 → 활용과 한계** 순서로 정리했습니다.

> 그래프와 집계 출력은 이 노트북 안에 포함합니다. 마크다운의 수치는 아래 CSV 해시로 식별한 현재 자료의 분석 기록입니다. CSV를 바꿔 재실행하면 집계표·그래프를 확인하고 기록된 해석도 갱신해야 합니다.

<a id="toc"></a>
## 목차와 분석의 큰 흐름

| 단계 | 분석 내용 | 답하려는 질문 |
|---|---|---|
| [1. 연결과 전처리](#prepare) | 7개 자료의 역할, 01~04 근거, 조인 검증 | 같은 설비·시간을 제대로 연결했는가? |
| [2. 관측 타임라인](#timeline) | LOT·정비·강종변경·정기수리 | 어느 시간에 센서로 설명할 수 있는가? |
| [3. 수집 품질](#quality) | 측정 간격·공백·결측 위치 | 공정 변화와 수집 문제를 구분할 수 있는가? |
| [4. 22개 신호의 흐름](#signals) | 공정별 모든 신호를 LOT 시작에 맞춰 비교 | 어느 공정·신호에 차이가 집중되는가? |
| [5. LOT 비교와 사례 선정](#lots) | 14 LOT의 온도·출력, 동일 강종 반복 | 상세히 볼 LOT를 어떤 근거로 고르는가? |
| [6. 강종별 LOT 비교 — 다중신호 후보 좁히기](#trend) | 전체 후보 → 강종 효과 통제 → LOT 특이성 → 시간 지속성 → 계획수리 | 같은 강종에서 남는 차이는 무엇이며 얼마나 지속되는가? |
| [7. 사례 LOT 공정 흐름](#case) | 공정별 온도·출력·CP, 초반과 후반 | 온도가 오르는 구간이 어느 공정에 있는가? |
| [8. 반복 주기](#period) | 자기상관·주파수 분석 | 반복 진동을 일회성 변화와 구분할 수 있는가? |
| [9. 온도와 출력의 시차](#lag) | 수준·차분 시차상관 | 어느 시간 차이에서 함께 움직이는가? |
| [10. 급변과 상대 이탈](#change) | 변화율·존 간 차이·탐색용 후보 | 공통 승온과 한 존의 이탈이 어떻게 다른가? |
| [11. 정비와 실제 관측](#maintenance) | 정비 빈도·정지시간·감지 주변 센서 | 정비 전후를 실제로 비교할 데이터가 있는가? |
| [12. 부품교체와 보전](#parts) | 교체 시점·사유·표준수명·리드타임 | 센서 분석을 어떤 부품 점검과 연결할 수 있는가? |
| [13. 종합 해석·LOT 채택](#conclusion) | 결과·점검 우선순위·[LOT별 채택 이유](#lot-selection-final) | 무엇을 확인했고 어떤 LOT를 왜 채택했는가? |
| [확대 구간과 근거 점검](#interval-review) | 실제 분·시각, 비교군, 피크 제거·정비 창 민감도 | 어느 구간을 왜 보며 그 해석은 유지되는가? |

분석에서 사용하는 TC·OP·CP의 공정 배정은 기존 노트북을 따릅니다. 단위는 사용자 확인에 따라 **TC·CLN-IN: ℃, OP: %, CP: 무차원**으로 표시합니다. 목표값·센서 허용범위는 자료에 없으므로 탐색 기준을 정상 운전 한계로 해석하지 않습니다. 정비유형·고장모드는 기록된 분류이며 모든 정비를 고장으로 간주하지 않습니다.

<a id="prepare"></a>
## 1. 데이터 연결과 전처리 기준

**분석 질문:** 7개 자료를 어떤 키와 시간 기준으로 묶는가?

**계산 방법:** 설비마스터의 정확한 EQP_TAG를 확인하고, 교체이력→정비이력은 WO_NO, 교체이력→부품마스터는 PART_CD로 연결합니다. 강종 후보는 CR1 강종변경 시각과 LOT 시작 시각의 정확한 일치로 연결합니다.

**그림 읽는 법:** 아래 자료 역할표와 검증표를 먼저 봅니다. 정비 작업 수와 교체 내역 행 수의 분모를 구분합니다.

| CSV | 역할 | 연결 기준 |
|---|---|---|
| T-CR1-CAL01_온도 | 22개 신호의 측정값·관측 구간 | MEAS_DT·LOT_NO, 파일명의 CAL01 |
| G-02_조업이벤트 | 강종 후보와 변경 시각 | CR1·강종변경·EVT_DT |
| G-02_정비이력 | 감지·착수·완료 시각과 정비 유형 | 정확한 EQP_TAG·WO_NO |
| G-01_정기수리캘린더 | 계획 수리 구간 | CR1·STRT_DT·END_DT |
| G-02_부품교체이력 | 교체부품·수량·사유 | WO_NO·PART_CD |
| G-01_설비마스터 | 대상 설비와 설치일 | EQP_TAG |
| G-01_부품마스터 | 부품명·표준수명·리드타임 | PART_CD |

### 01~04에서 이어받은 근거

| 선행 노트북 | 이번 분석에 반영한 기준 |
|---|---|
| [01_eda_1_sampling_interval](01_eda_1_sampling_interval.ipynb), [01_eda_2_time_range_gaps](01_eda_2_time_range_gaps.ipynb) | LOT 내부 간격과 LOT 사이 공백을 분리. 10초 초과는 계산 구간을 끊는 탐색 기준 |
| [01_eda_3_lot_segments](01_eda_3_lot_segments.ipynb), [01_eda_4_event_overlap](01_eda_4_event_overlap.ipynb) | LOT 시작·끝으로 실제 중첩을 판정. 연간 최소·최대 시각만으로 관측 가능하다고 판단하지 않음 |
| [01_eda_5_missing_constant](01_eda_5_missing_constant.ipynb), [03_eda_7_temp_missing_events](03_eda_7_temp_missing_events_IWeasdwe1234.ipynb) | 결측률과 발생 위치를 함께 확인. 원본 결측·0을 임의로 대체하지 않음 |
| [02_eda_6_event_contents](02_eda_6_event_contents_sayletskin.ipynb), [02_eda_lot_grade_event_alignment](02_eda_lot_grade_event_alignment.ipynb), [02_eda_7_steel_grade_temperature](02_eda_7_steel_grade_temperature_Jinny.ipynb) | CR1 이벤트만 연결. 강종 후보와 LOT를 구분하고 후보별·LOT별 통계를 함께 확인 |
| [03_parts_replacement_history](03_parts_replacement_history_jinny.ipynb) | 교체 행 수·작업 수·수량을 구분하고 반복 행은 근거 없이 삭제하지 않음 |
| [04_equipment_master_timeseries_join](04_equipment_master_timeseries_join_namelesspark.ipynb) | 온도 파일명을 근거로 P1-CR1-CAL01을 지정. 설비마스터는 고정 속성 |
| [04_eda_g02_maintenance_history](04_eda_g02_maintenance_history_IWEasdwe1234.ipynb), [04_parts_master](04_parts_master_yujin.ipynb) | 정비 감지·착수·완료를 분리. STD_LIFE_H는 실제 사용시간이나 잔여수명이 아님 |


**시간 처리:** 원본 역순·중복을 정렬 전에 검사 → 시간순 정렬 → LOT 또는 LOT 내부 10초 초과 공백에서 SEG 분리 → SEG 안에서만 평균·차분·시차 계산. 30초 평균은 15개 이상, 1분 평균은 45개 이상 유효 관측이 있는 칸만 사용하며 보간하지 않습니다. 임곗값은 데이터 사양이 확인되지 않은 탐색 설정입니다.

In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
# 이걸 기준으로 코드 작성할 것
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
FIG_DIR = os.path.join("..", "figures")        # 그래프 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

# 기존 3개 (T- 센서 계측, G-02 사건 기록, G-01 기준 정보)
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")          # 소둔로 온도·제어출력, 1초 주기
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")           # 강종변경 등, PLANT_CD 단위
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")       # 계획 정지, PLANT_CD 단위

# 2026-09-28 추가된 4개 (상위 이슈 #43)
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")             # 설비 사전, 키 EQP_TAG
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")            # 부품 사전, 키 PART_CD
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")             # 정비·고장 기록, 키 WO_NO + EQP_TAG
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")        # 부품 교체, 키 WO_NO + PART_CD

ENC = "utf-8-sig"                              # G- 로 시작하는 마스터 CSV는 BOM 있음

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 그래프 한글 깨짐 방지 - 설치된 폰트 중 있는 것을 골라 rcParams에 넣습니다
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False         # 음수 기호가 □ 로 나오는 것 방지
rcParams["figure.dpi"] = 110

In [ ]:
import matplotlib.dates as mdates
FIG_OUT = os.path.join(FIG_DIR, "05_all_data_timeseries_jinny")   # 이 노트북 그림 저장 폴더 (fig.savefig 대상)
os.makedirs(FIG_OUT, exist_ok=True)
TARGET = "P1-CR1-CAL01"             # 온도 파일의 설비 (연속 소둔로)
X_MIN = "LOT 수집 시작 후 경과 시간 [분]"

# ---- 1) 불러오기
temp    = pd.read_csv(TEMP_CSV, encoding=ENC, parse_dates=["MEAS_DT"])
event   = pd.read_csv(EVENT_CSV, encoding=ENC, parse_dates=["EVT_DT"])
repair  = pd.read_csv(REPAIR_CSV, encoding=ENC, parse_dates=["STRT_DT", "END_DT"])
eqp     = pd.read_csv(EQP_CSV, encoding=ENC)
part    = pd.read_csv(PART_CSV, encoding=ENC)
maint   = pd.read_csv(MAINT_CSV, encoding=ENC, parse_dates=["DETC_DT", "STRT_DT", "END_DT"])
partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

# ---- 2) 온도: 정렬 + 연속구간(10초 넘게 끊기면 새 구간) + LOT 수집 시작 후 경과 초
temp = temp.sort_values(["LOT_NO", "MEAS_DT"]).reset_index(drop=True)
gap_s = temp.groupby("LOT_NO").MEAS_DT.diff().dt.total_seconds()
temp["seg"] = (temp.LOT_NO.ne(temp.LOT_NO.shift()) | gap_s.gt(10)).cumsum()
temp["sec"] = (temp.MEAS_DT - temp.groupby("LOT_NO").MEAS_DT.transform("min")).dt.total_seconds()
SIG = [c for c in temp.columns if c not in ["MEAS_DT", "LOT_NO", "seg", "sec"]]

# ---- 3) 그림용 30초 평균 (연속구간 안에서만). x축 "min" = LOT 수집 시작 후 경과 분
ts = temp.set_index("MEAS_DT").groupby(["LOT_NO", "seg"])[SIG].resample("30s").mean().reset_index()
ts["CP-Z4M"] = ts["CP-Z4M"] * 1e10                                   # 1e-10 단위로 표시
ts["min"] = (ts.MEAS_DT - ts.groupby("LOT_NO").MEAS_DT.transform("min")).dt.total_seconds() / 60
dif = ts.groupby("seg")[SIG].diff()                                  # 30초 변화량 (연속구간 안)

# ---- 4) LOT 표 + 강종 후보 (LOT 시작 시각 = CR1 강종변경 시각)
cr1_grade = event[(event.PLANT_CD == "CR1") & (event.EVT_TYPE == "강종변경")]
lots = temp.groupby("LOT_NO").agg(start=("MEAS_DT", "min"), end=("MEAS_DT", "max"), rows=("MEAS_DT", "size"))
lots["minutes"] = (lots.end - lots.start).dt.total_seconds() / 60
lots["grade"] = lots.start.map(cr1_grade.set_index("EVT_DT").AFT_VAL).fillna("미확인")

# ---- 5) CAL01 정비·부품교체, CR1 정기수리(종료일 당일 포함)
cal_maint = maint[maint.EQP_TAG == TARGET].sort_values("DETC_DT").reset_index(drop=True)
cal_chg = (partchg.merge(cal_maint[["WO_NO", "END_DT", "MNT_TYPE"]], on="WO_NO")
                  .merge(part[["PART_CD", "PART_NM", "STD_LIFE_H"]], on="PART_CD"))
cr1_repair = repair[repair.PLANT_CD == "CR1"].assign(END_INCL=lambda d: d.END_DT + pd.Timedelta(days=1))

# ---- 6) 컬럼 사전 (데이터사전 기준)
COL_INFO = pd.DataFrame([
    ["CLN-IN", "세정조 입측 온도", "세정조"], ["TC-Z1", "건조존 1존 온도", "건조존"], ["TC-Z2", "건조존 2존 온도", "건조존"],
    ["OP-Z1", "건조존 1존 제어출력", "건조존"], ["OP-Z2", "건조존 2존 제어출력", "건조존"],
    ["TC-Z3", "가열·균열존 1존 온도", "가열·균열존"], ["TC-Z4", "가열·균열존 2존 온도", "가열·균열존"],
    ["TC-Z5", "가열·균열존 3존 온도", "가열·균열존"], ["TC-Z6", "가열·균열존 4존 온도", "가열·균열존"],
    ["OP-Z3", "가열·균열존 1존 제어출력", "가열·균열존"], ["OP-Z4", "가열·균열존 2존 제어출력", "가열·균열존"],
    ["OP-Z5", "가열·균열존 3존 제어출력", "가열·균열존"], ["OP-Z6", "가열·균열존 4존 제어출력", "가열·균열존"],
    ["CP-Z4", "카본포텐셜", "가열·균열존"], ["CP-Z4M", "카본포텐셜 모니터", "가열·균열존"],
    ["TC-Z7", "과시효대 1존 온도", "과시효대"], ["TC-Z8", "과시효대 2존 온도", "과시효대"],
    ["TC-Z9", "과시효대 슬러지 제거부 온도", "과시효대"], ["OP-Z7", "과시효대 1존 제어출력", "과시효대"],
    ["OP-Z8", "과시효대 2존 제어출력", "과시효대"], ["TC-OUT1", "냉각대 1존 온도", "냉각대"], ["TC-OUT2", "냉각대 2존 온도", "냉각대"],
], columns=["컬럼", "한글명", "공정"]).set_index("컬럼")
UNIT = {c: "온도 [℃]" if c.startswith("TC") or c == "CLN-IN" else "제어출력 [%]" if c.startswith("OP") else
        "CP-Z4 [무차원]" if c == "CP-Z4" else "CP-Z4M [무차원, ×10⁻¹⁰]" for c in SIG}

# ---- 7) 색 규칙: 같은 공정 = 같은 색 계열, 같은 존 TC·OP = 같은 색
ZONE_COLOR = {"CLN-IN": "#6B7280", #세정조 회색
              "Z1": "#0072B2", "Z2": "#56B4E9",                                   # 건조존: 파랑
              "Z3": "#E69F00", "Z4": "#D55E00", "Z5": "#CC3311", "Z6": "#8C2D04",  # 가열·균열존: 주황~적갈
              "Z7": "#00694D", "Z8": "#00A6A6", "Z9": "#2E7D32",                   # 과시효대: 초록~청록
              "OUT1": "#003F5C", "OUT2": "#2F9EBC",                                # 냉각대: 진한 파랑~청록
              "CP-Z4": "#7B2CBF", "CP-Z4M": "#C77DFF"}        # , CP 보라
ZONE_LS = {"Z1": "-", "Z2": "--", "Z3": "-", "Z4": "--", "Z5": "-.", "Z6": ":", "Z7": "-", "Z8": "--", "Z9": "-.",
           "OUT1": "-", "OUT2": "--", "CLN-IN": "-", "CP-Z4": "-", "CP-Z4M": "-"}
# 강종별 고유 색: 짙은 회색·파랑·빨강·초록·보라·주황·어두운 황토색
GRADE_COLOR = {'SGCC': '#263238', 'SPFH590': '#0072B2', 'SPCE': '#D62728', 'SPCEN': '#009E73', 'SPHC': '#7B2CBF', 'DP590': '#E69F00', 'SPCD': '#8F8301', '미확인': '#9AA0A6'}
MNT_STYLE = {"BM": ("X", "#D62728", "BM 사후정비"), "CBM": ("o", "#1F77B4", "CBM 상태기반"), "TBM": ("s", "#7F7F7F", "TBM 시간기준")}
CANDIDATE = "#FFD54F"        # 노랑 = 후보 구간
MISSING = "0.45"             # 회색 빗금 = 결측 구간
TRANSITION = "#E3E3E3"       # 연회색 = 운전 전환 구간

TC_GROUPS = {"세정조 입측 (CLN-IN)": ["CLN-IN"], "건조존 (TC-Z1, TC-Z2)": ["TC-Z1", "TC-Z2"],
             "가열·균열존 (TC-Z3~TC-Z6)": ["TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6"],
             "과시효대 (TC-Z7, TC-Z8)": ["TC-Z7", "TC-Z8"], "냉각대 (TC-OUT1, TC-OUT2)": ["TC-OUT1", "TC-OUT2"]}
OP_GROUPS = {"건조존 (OP-Z1, OP-Z2)": ["OP-Z1", "OP-Z2"], "가열·균열존 (OP-Z3~OP-Z6)": ["OP-Z3", "OP-Z4", "OP-Z5", "OP-Z6"],
             "과시효대 (OP-Z7, OP-Z8 · OP-Z9 없음)": ["OP-Z7", "OP-Z8"]}
HEAT_TC, HEAT_OP = TC_GROUPS["가열·균열존 (TC-Z3~TC-Z6)"], OP_GROUPS["가열·균열존 (OP-Z3~OP-Z6)"]


def zone(s):
    return s if s in ZONE_COLOR else s.split("-")[1]


def line(s, many=True):
    """센서 선 모양: TC=굵은 선, OP=얇고 연한 선. 한 축에 여러 개면 선 모양으로도 구분."""
    kw = {"color": ZONE_COLOR[zone(s)], "ls": ZONE_LS[zone(s)] if many else "-"}
    kw.update({"lw": 1.2, "alpha": .8} if s.startswith("OP-") else {"lw": 1.8})
    return kw


def plot_segments(ax, df, col, x="min", label=None, **kw):
    """연속구간마다 따로 그려 계측 공백을 선으로 잇지 않는다."""
    for i, (_, g) in enumerate(df.groupby("seg")):
        ax.plot(g[x], g[col], label=label if i == 0 else None, **kw)


def shade(ax, x, mask, color=CANDIDATE, alpha=.35, label=None):
    """mask가 True인 연속 구간을 세로 음영으로 표시 (x = 경과 분, 30초 칸)."""
    x, mask = np.asarray(x), np.asarray(mask, bool)
    edges = np.diff(np.r_[0, mask.astype(int), 0])
    for i, (a, b) in enumerate(zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1) - 1)):
        ax.axvspan(x[a] - .25, x[b] + .25, color=color, alpha=alpha, lw=0, zorder=0, label=label if i == 0 else None)


def missing_runs(lot, col):
    """1초 원본에서 col이 비어 있는 연속 구간 → [(시작 초, 끝 초, 행 수)]."""
    r = temp[temp.LOT_NO == lot]
    m = r[col].isna().to_numpy()
    sec = r.sec.to_numpy()
    edges = np.diff(np.r_[0, m.astype(int), 0])
    return [(sec[a], sec[b], b - a + 1) for a, b in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1) - 1)]


def mark_missing(ax, lot, col, unit="min", text=True):
    """결측 구간을 회색 빗금으로 칠하고(최소 폭 1분/1초로 넓혀 보이게) 행 수·위치를 적는다."""
    runs = missing_runs(lot, col)
    k = 60 if unit == "min" else 1
    width = 1 if unit == "min" else 1
    for i, (a, b, n) in enumerate(runs):
        ax.axvspan(a / k, max(b / k, a / k + width), facecolor="none", edgecolor=MISSING, hatch="////", lw=0,
                   zorder=1, label=f"{col} 결측" if i == 0 else None)
    if runs and text:
        n = sum(r[2] for r in runs)
        ax.annotate(f"{col} 결측 {n}행\n({runs[0][0] / k:.1f}~{runs[-1][1] / k:.1f}{'분' if unit == 'min' else '초'})",
                    xy=(runs[0][0] / k, .5), xycoords=("data", "axes fraction"), xytext=(18, 0), textcoords="offset points",
                    fontsize=8.5, color="0.2", va="center", arrowprops=dict(arrowstyle="->", color="0.3"),
                    bbox=dict(fc="white", ec="0.6", alpha=.9))
    return runs


def note(ax, text, loc="upper left"):
    """패널 안 숫자 메모."""
    x, ha = (.01, "left") if loc.endswith("left") else (.99, "right")
    y, va = (.96, "top") if loc.startswith("upper") else (.04, "bottom")
    ax.text(x, y, text, transform=ax.transAxes, ha=ha, va=va, fontsize=8.5, bbox=dict(fc="white", ec="0.8", alpha=.85))


def legend_in(ax, title=None, extra=(), loc="upper right", ncol=None):
    """그래프 안 범례 (단위는 y축, 선·음영 설명은 범례)."""
    handles, _ = ax.get_legend_handles_labels()
    handles = list(handles) + list(extra)
    ax.legend(handles=handles, title=title, loc=loc, ncol=ncol or min(len(handles), 4), fontsize=8, title_fontsize=8,
              framealpha=.9, edgecolor="0.8")


def titles(fig, title, subtitle):
    """제목(제목만) + 부제(설명). 높이에 맞춰 여백 확보."""
    h = fig.get_figheight()
    fig.suptitle(title, x=.01, y=1 - .06 / h, ha="left", va="top", fontsize=15, fontweight="bold")
    fig.text(.01, 1 - .40 / h, subtitle, ha="left", va="top", fontsize=10.5, color="0.3")
    fig.tight_layout(rect=(0, 0, 1, 1 - .72 / h))


def lot_info(lot):
    r = lots.loc[lot]
    return f"LOT {lot} · 강종 후보 {r.grade} · {r.start:%Y-%m-%d %H:%M} ~ {r.end:%H:%M} · {r.minutes:.0f}분"


print(f"센서 {len(temp):,}행 · LOT {len(lots)}개 · 신호 {len(SIG)}개 · 30초 평균 {len(ts):,}행")
print(f"강종 후보 연결 {int((lots.grade != '미확인').sum())}/{len(lots)} LOT · CAL01 정비 {len(cal_maint)}건 · CAL01 부품교체 {len(cal_chg)}행")

#### 그림 저장

각 그림은 코드로 생성한 뒤 **figures/05_all_data_timeseries_jinny**에 **번호_설명.png**로 저장합니다. 제목·축·단위·범례를 포함하며, 같은 폴더의 README에 그림 목록·사용 데이터·생성 코드 경로를 기록합니다.

아래 저장 설정을 먼저 실행하세요. 이후 그림 셀을 다시 실행하면 같은 PNG 파일을 갱신합니다.


In [ ]:
# 그림을 직접 그린 뒤 PNG로 저장하고 노트북에도 표시합니다.
# figures/README.md 형식: 번호_설명.png, 제목·축·단위·범례 유지.
import re
from pathlib import Path

FIGURE_DPI = 200
FIGURE_EXPORTS = {}
FIGURE_DATA_FILES = [
    TEMP_CSV, EVENT_CSV, REPAIR_CSV, EQP_CSV, PART_CSV, MAINT_CSV, PARTCHG_CSV
]


def figure_export_title(fig):
    """suptitle 또는 그림 번호가 있는 상단 텍스트에서 제목을 읽습니다."""
    if fig._suptitle is not None and fig._suptitle.get_text().strip():
        return fig._suptitle.get_text().strip()
    for artist in fig.texts:
        title = artist.get_text().strip()
        if re.match(r"그림\s+\d+", title):
            return title
    raise ValueError("저장할 그림에 '그림 번호. 설명' 제목이 필요합니다.")


def figure_export_name(title):
    """예: 그림 18. 표준수명·조달 리드타임 → 18_표준수명·조달_리드타임.png."""
    match = re.match(r"그림\s+(\d+)([^.]*)\.\s*(.+)", title, flags=re.S)
    if match is None:
        raise ValueError(f"그림 번호와 설명을 확인하세요: {title}")
    number, suffix, description = match.groups()
    # Windows 파일명에 쓸 수 없는 기호와 연속 공백을 제거합니다.
    def clean(value):
        value = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", value)
        value = re.sub(r"\s+", "_", value.strip())
        return re.sub(r"_+", "_", value).strip("._-")
    prefix = f"{int(number):02d}"
    suffix = clean(suffix)
    if suffix:
        prefix += "_" + suffix
    return f"{prefix}_{clean(description)[:90]}.png"


def write_figure_export_index():
    """그림 목록과 데이터·생성 코드 경로를 같은 폴더 README에 기록합니다."""
    lines = [
        "# 05_all_data_timeseries_jinny 결과 그림",
        "",
        "figures/README.md의 `번호_설명.png` 형식으로 저장한 시각화입니다.",
        "노트북에서 Matplotlib으로 직접 생성하며 제목·축 이름·단위·범례를 포함합니다.",
        f"PNG 해상도는 {FIGURE_DPI} dpi이며, 바깥 범례와 설명까지 저장합니다.",
        "",
        "생성 코드: [05_all_data_timeseries_jinny.ipynb](../../notebooks/05_all_data_timeseries_jinny.ipynb)",
        "",
        "노트북의 데이터 설정·그림 저장 셀부터 순서대로 실행하면 각 그림 셀에서 PNG와 이 목록을 갱신합니다.",
        "동일한 그림은 같은 파일명으로 다시 저장합니다. 이미지를 불러와 그래프를 대체하지 않습니다.",
        "",
        "## 사용 데이터",
        "",
    ]
    for source in FIGURE_DATA_FILES:
        name = Path(source).name
        lines.append(f"- [{name}](../../data/{name})")
    lines += [
        "",
        "정비 완료일은 실제 부품 교체 시각의 대리값입니다. 결측·관측 공백은 보간하지 않습니다.",
        "그림별 비교 기준과 관측 결과·후속 작업은 노트북의 해당 그림 아래 마크다운에 있습니다.",
        "",
        f"## 그림 목록 ({len(FIGURE_EXPORTS)}개)",
        "",
        "| 파일 | 그림 제목 |",
        "|---|---|",
    ]
    for name, title in FIGURE_EXPORTS.items():
        caption = " ".join(title.split()).replace("|", "·")
        lines.append(f"| [{name}](./{name}) | {caption} |")
    Path(FIG_OUT, "README.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def save_and_show_figures():
    """현재 열린 그림을 저장 → 표시 → 닫기. 데이터와 계산에는 영향을 주지 않습니다."""
    figures = [plt.figure(number) for number in plt.get_fignums()]
    for fig in figures:
        title = figure_export_title(fig)
        name = figure_export_name(title)
        if name in FIGURE_EXPORTS and FIGURE_EXPORTS[name] != title:
            raise ValueError(f"그림 파일명 중복: {name}")
        fig.savefig(
            Path(FIG_OUT, name), dpi=FIGURE_DPI, bbox_inches="tight",
            pad_inches=0.16, facecolor="white", transparent=False
        )
        FIGURE_EXPORTS[name] = title
    if figures:
        write_figure_export_index()
    plt.show()
    for fig in figures:
        plt.close(fig)


In [ ]:
# 앞의 두 설정 셀을 실행한 뒤 실행합니다.
# 원본 품질·키 조인 검증과 강종변경↔LOT 시작 시각의 연결을 함께 확인합니다.
from IPython.display import display, Markdown

# 1. 원본 파일에서 정렬 전 품질 확인
raw_temp_check = pd.read_csv(TEMP_CSV, encoding=ENC, parse_dates=["MEAS_DT"])
raw_reverse = int(raw_temp_check["MEAS_DT"].diff().dt.total_seconds().lt(0).sum())
raw_duplicates = int(raw_temp_check.duplicated().sum())
del raw_temp_check

for frame, key in [(maint,"WO_NO"), (part,"PART_CD"), (eqp,"EQP_TAG")]:
    assert frame[key].notna().all() and frame[key].is_unique, f"{key}: 결측·중복 키 확인 필요"

# inner 조인에서 사라질 수 있는 미매칭까지 확인하기 위해 left 조인합니다.
connection_check = partchg.merge(
    maint[["WO_NO","EQP_TAG"]], on="WO_NO", how="left",
    validate="many_to_one", indicator="work_match"
).merge(
    part[["PART_CD","PART_NM"]], on="PART_CD", how="left",
    validate="many_to_one", indicator="part_match"
)
assert len(connection_check) == len(partchg)
target_eqp = eqp[eqp["EQP_TAG"].eq(TARGET)]
assert len(target_eqp) == 1
checked_cal_chg = connection_check[connection_check["EQP_TAG"].eq(TARGET)]
assert len(cal_chg) == len(checked_cal_chg)

validation = pd.DataFrame([
    ("원본 역순 시각", raw_reverse),
    ("완전 중복 행", raw_duplicates),
    ("연속구간 수", int(temp["seg"].nunique())),
    ("정비키 미매칭 교체행", int(connection_check["work_match"].ne("both").sum())),
    ("부품키 미매칭 교체행", int(connection_check["part_match"].ne("both").sum())),
    ("교체 조인 전/후 행", f"{len(partchg)} / {len(connection_check)}"),
    ("대상 설비마스터 행", len(target_eqp)),
    ("CAL01 정비 작업", int(cal_maint["WO_NO"].nunique())),
    ("CAL01 교체행 / 작업 / 부품종류",
     f"{len(cal_chg)} / {cal_chg['WO_NO'].nunique()} / {cal_chg['PART_CD'].nunique()}"),
], columns=["확인","결과"])

# 2. 강종변경과 LOT의 연결을 독립적인 시간 키로 검증
assert cr1_grade["EVT_DT"].notna().all() and cr1_grade["EVT_DT"].is_unique
lot_event_link = lots.reset_index().merge(
    cr1_grade[["EVT_ID","EVT_DT","BEF_VAL","AFT_VAL"]],
    left_on="start", right_on="EVT_DT", how="left", validate="one_to_one"
).sort_values("start").reset_index(drop=True)
lot_event_link["start_offset_s"] = (
    lot_event_link["start"] - lot_event_link["EVT_DT"]
).dt.total_seconds()
matched_mask = lot_event_link["EVT_ID"].notna()
assert lot_event_link.loc[matched_mask,"grade"].eq(lot_event_link.loc[matched_mask,"AFT_VAL"]).all()
matched_lots = int(matched_mask.sum())
grade_count = int(lot_event_link.loc[matched_mask,"AFT_VAL"].nunique())

def in_lot_windows(value):
    return False if pd.isna(value) else bool(((lots["start"]<=value)&(lots["end"]>=value)).any())

grade_event_status = cr1_grade.sort_values("EVT_DT").copy()
grade_event_status["inside_observed_lot"] = grade_event_status["EVT_DT"].map(in_lot_windows)
grade_event_status["starts_observed_lot"] = grade_event_status["EVT_ID"].isin(lot_event_link["EVT_ID"].dropna())
lot_event_link["extra_changes_inside"] = [
    int(((cr1_grade["EVT_DT"]>r.start)&(cr1_grade["EVT_DT"]<=r.end)).sum())
    for r in lot_event_link.itertuples()
]
lot_event_link["pre10_sensor_rows"] = [
    int(((temp["MEAS_DT"]>=r.start-pd.Timedelta(minutes=10))&(temp["MEAS_DT"]<r.start)).sum())
    for r in lot_event_link.itertuples()
]
grade_event_codes = sorted(set(cr1_grade["BEF_VAL"].dropna()) | set(cr1_grade["AFT_VAL"].dropna()))
unobserved_codes = sorted(set(grade_event_codes)-set(lot_event_link.loc[matched_mask,"AFT_VAL"]))
linked_grade_counts = (
    lot_event_link.loc[matched_mask].groupby("AFT_VAL").size()
    .sort_values(ascending=False, kind="stable")
)
relation_summary = pd.DataFrame([
    ("LOT 시작과 강종변경 시각 정확히 일치", f"{matched_lots} / {len(lots)} LOT"),
    ("연결된 LOT의 시작 시각 차이", f"{lot_event_link.loc[matched_mask,'start_offset_s'].abs().max():.0f}초" if matched_lots else "미연결"),
    ("LOT 시작 이후 관측 중 추가 강종변경", f"{lot_event_link['extra_changes_inside'].sum()}건"),
    ("센서 관측 창 밖 CR1 강종변경", f"{(~grade_event_status['inside_observed_lot']).sum()} / {len(cr1_grade)}건"),
    ("변경 후 센서 LOT에서 관측된 강종 후보", f"{grade_count}종 / {len(lots)} LOT"),
    ("연결 LOT 시작 직전 10분의 센서 관측", f"{lot_event_link['pre10_sensor_rows'].sum():,}행"),
], columns=["강종변경과 LOT의 관계","실제 결과"])

def connection_table_md(frame):
    header = "| " + " | ".join(map(str,frame.columns)) + " |"
    divider = "| " + " | ".join(["---"]*len(frame.columns)) + " |"
    rows = ["| " + " | ".join(str(v) for v in row) + " |"
            for row in frame.itertuples(index=False,name=None)]
    return "\n".join([header,divider]+rows)

display(Markdown(f"""### 연결 결과와 의미

**핵심 결과:** 제공된 자료에서 **{matched_lots}/{len(lots)} LOT의 관측 시작이 CR1 강종변경 시각과 정확히 일치**합니다. 해당 이벤트의 변경 후 값 `AFT_VAL`을 LOT의 강종 후보로 연결할 수 있습니다. 같은 강종이 여러 LOT에 반복되므로 강종 수와 LOT 수는 다릅니다.

#### 강종변경과 센서 LOT는 어떻게 연결되는가?

{connection_table_md(relation_summary)}

#### 원본 품질과 키 조인

{connection_table_md(validation)}

**확인한 사실:** 온도 {len(temp):,}행·{len(SIG)}개 신호, {len(lots)} LOT·{grade_count}개 강종 후보입니다. CR1 이벤트에는 {len(grade_event_codes)}개 강종 코드가 나타납니다. 변경 후 센서 LOT에서 미관측된 코드: {', '.join(unobserved_codes) if unobserved_codes else '없음'}. CAL01 정비는 {cal_maint['WO_NO'].nunique()}작업, 교체는 {len(cal_chg)}내역·{cal_chg['WO_NO'].nunique()}작업입니다.

**해석:** 관측된 LOT는 강종변경 시점과 연결되지만, **모든 강종변경에 센서 LOT가 제공된 것은 아닙니다.** 관측 창 밖 이벤트가 있다는 사실은 제공 자료의 관측 범위를 뜻하며 생산 부재·기록 오류를 뜻하지 않습니다. LOT 직전 데이터가 없으므로 변경 전후 온도 효과까지 비교할 수는 없습니다.

**집계·처리 원칙:** 정비는 교체와 병합하기 전 `cal_maint`의 작업번호로 집계합니다. 원본 행은 삭제·대체하지 않습니다. 계획수리는 종료일 다음 날 00:00 미만까지 포함하며 CAL01 본체 기록만 선택합니다.

[↑ 목차](#toc)
"""))


<a id="timeline"></a>
## 2. 연간 타임라인과 강종변경–LOT 연결

**분석 질문:** 센서 LOT는 강종변경과 언제 연결되며, 같은 강종에 여러 LOT가 반복되는가?

**계산 방법:** CR1 강종변경 시각과 원본 LOT 시작 시각을 정확히 비교합니다. 연간 그림에서는 실제 날짜를 유지하고, 두 번째 그림에서는 각 LOT의 관측 시작을 0분에 맞춰 길이와 변경 전·후 강종을 비교합니다. 정비 감지·착수·완료의 관측 중첩은 각각 계산합니다.

**그림 1 읽는 법:** 회색 눈금은 관측 LOT와 연결되지 않은 강종변경, 색이 있는 점과 세로 연결선은 같은 시각에 시작한 센서 LOT입니다. LOT 막대는 실제 시작~끝, 짧은 세로 눈금은 시작 표시입니다. 옅은 파랑 배경은 CR1 계획수리입니다.

**그림 1-보조 읽는 법:** 왼쪽에서 0분은 강종변경과 관측 시작이 일치하는 시점입니다. 행 이름은 LOT·날짜·변경 전→후 강종, 막대 길이는 관측 지속시간입니다. 오른쪽은 변경 후 강종별 LOT 수와 LOT 번호 끝 세 자리입니다.


In [ ]:
# 현재 준비 셀의 변수명을 사용합니다: cal_maint, cal_chg, cr1_grade, cr1_repair, lots.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.colors import to_rgb

# 새 커널에서 앞 셀부터 실행해도 기존 변수명에 의존하지 않습니다.
cr_event = grade_event_status.copy()
cr_event["observed"] = cr_event["inside_observed_lot"]
cal = cal_maint.copy()
plans = cr1_repair.rename(columns={"END_INCL":"end_exclusive"}).copy()
colors = dict(GRADE_COLOR)
for grade in lots["grade"].unique():
    colors.setdefault(grade, "#687685")

for col in ["DETC_DT","STRT_DT","END_DT"]:
    cal[col+"_observed"] = cal[col].map(in_lot_windows)
cal["interval_observed"] = [
    bool(((lots["start"]<=r.END_DT)&(lots["end"]>=r.STRT_DT)).any())
    for r in cal.itertuples()
]
part_event = cal_chg[["WO_NO","END_DT"]].drop_duplicates()
coverage = ((lots["end"]-lots["start"]).dt.total_seconds()+1).sum()/3600
year_hours = (pd.Timestamp("2025-01-01")-pd.Timestamp("2024-01-01")).total_seconds()/3600
plan_sensor_rows = sum(
    int(((temp["MEAS_DT"]>=r.STRT_DT)&(temp["MEAS_DT"]<r.end_exclusive)).sum())
    for r in plans.itertuples()
)
overlap_summary = pd.DataFrame({
    "대조 기준": ["CR1 강종변경 시각","CAL01 정비 감지","CAL01 정비 착수", "CAL01 정비 완료","CAL01 작업 구간 중첩","계획수리 기간 센서 행"],
    "전체": [len(cr_event),len(cal),len(cal),len(cal),len(cal),len(temp)],
    "관측 창 포함/중첩": [int(cr_event["observed"].sum()),int(cal["DETC_DT_observed"].sum()),
        int(cal["STRT_DT_observed"].sum()),int(cal["END_DT_observed"].sum()),int(cal["interval_observed"].sum()),plan_sensor_rows]
})

# ── 그림 1: 실제 연간 날짜에서 강종변경→LOT 연결을 강조 ──
fig, ax = plt.subplots(figsize=(16,6.4))
Y = {"maint":4.2,"part":3.25,"event":2.35,"sensor":1.3,"repair":.2}
for r in plans.itertuples():
    ax.axvspan(r.STRT_DT,r.end_exclusive,color="#dbeaf4",alpha=.8,lw=0,zorder=0)
    ax.plot([r.STRT_DT,r.end_exclusive],[Y["repair"]]*2,color="#4f7c99",lw=7,solid_capstyle="butt")
    mid = r.STRT_DT+(r.end_exclusive-r.STRT_DT)/2
    ax.text(mid,Y["repair"]-.22,f"{r.STRT_DT:%m/%d}~{r.END_DT:%m/%d}",ha="center",va="top",fontsize=8)
for kind,(marker,color,label) in MNT_STYLE.items():
    d = cal[cal["MNT_TYPE"].eq(kind)]
    ax.scatter(d["DETC_DT"],np.full(len(d),Y["maint"]),marker=marker,s=48,color=color,zorder=4)
ax.scatter(part_event["END_DT"],np.full(len(part_event),Y["part"]),marker="D",s=36,color="#6A3D9A",zorder=4)
outside = cr_event[~cr_event["starts_observed_lot"]]
ax.scatter(outside["EVT_DT"],np.full(len(outside),Y["event"]),marker="|",s=85,lw=.9,color="#b5bec6",zorder=2)
for i,r in enumerate(lot_event_link.itertuples()):
    color = colors[r.grade]
    if pd.notna(r.EVT_ID):
        ax.plot([r.EVT_DT,r.start],[Y["event"],Y["sensor"]],color=color,lw=1.5,alpha=.8,zorder=3)
        ax.scatter([r.EVT_DT],[Y["event"]],color=color,s=42,zorder=5)
    ax.plot([r.start,r.end],[Y["sensor"]]*2,color=color,lw=7,solid_capstyle="butt",zorder=5)
    ax.vlines(r.start,Y["sensor"]-.17,Y["sensor"]+.17,color=color,lw=2,zorder=6)
    # 인접 날짜 LOT의 글자가 겹치지 않도록 위·아래로 번갈아 배치합니다.
    dy = -19 if i%2 else 10
    ax.annotate(str(r.LOT_NO)[-3:],(r.start,Y["sensor"]),xytext=(0,dy),textcoords="offset points",
                ha="center",va="top" if dy<0 else "bottom",fontsize=8,fontweight="bold",color=color)
ax.text(pd.Timestamp("2024-01-08"),Y["event"]+.27,
        f"색 점 {matched_lots}건 ↔ 같은 시각 LOT 시작 / 회색 눈금 {len(outside)}건: 연결된 관측 LOT 없음",
        fontsize=9,color="#425466")
ax.set_yticks([Y[k] for k in ["maint","part","event","sensor","repair"]],
              ["CAL01 정비 감지","CAL01 교체 작업\n(정비 완료 시각)","CR1 강종변경","센서 LOT 관측 창","CR1 계획수리"])
ax.set_ylim(-.5,4.7)
ax.set_xlim(pd.Timestamp("2024-01-01"),pd.Timestamp("2024-12-31 23:59:59"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m월"))
ax.set_xlabel("실제 관측 날짜 [2024년]")
ax.grid(axis="x",color="#e6e9ec",lw=.7); ax.grid(axis="y",visible=False)
ax.spines[["top","right"]].set_visible(False)
handles = [Line2D([],[],marker=marker,ls="",color=color,label=f"{label} {cal.MNT_TYPE.eq(kind).sum()}건")
           for kind,(marker,color,label) in MNT_STYLE.items()]
handles += [Line2D([],[],marker="D",ls="",color="#6A3D9A",label=f"교체 {len(part_event)}작업"),
            Line2D([],[],marker="|",ls="",color="#b5bec6",markersize=11,label="연결된 관측 LOT 없는 강종변경"),
            Line2D([],[],marker="o",color="#425466",lw=1.5,label="색 점–선: 강종변경↔LOT 시작"),
            Patch(facecolor="#dbeaf4",label="CR1 계획수리")]
ax.legend(handles=handles,loc="upper left",bbox_to_anchor=(0,-.19),ncol=4,fontsize=8,frameon=False)
fig.suptitle(f"그림 1. {matched_lots}개 센서 LOT는 강종변경 시각에서 관측이 시작됨",
             x=.13,y=.97,ha="left",fontsize=16,fontweight="bold")
fig.text(.13,.91,f"CR1 변경 {len(cr_event)}건 중 {matched_lots}건과 LOT 시작 연결 · 센서 {coverage:.1f}시간 ({coverage/year_hours*100:.2f}%) · 정비 감지 중 센서 있음 {cal.DETC_DT_observed.sum()}/{len(cal)}건",
         ha="left",fontsize=10.5,color="#425466")
fig.subplots_adjust(left=.13,right=.99,top=.81,bottom=.25)
save_and_show_figures()

# ── 그림 1-보조: 실제 날짜의 공백을 빼고 시작 시점·강종·LOT 길이를 비교 ──
link = lot_event_link.sort_values("start").copy()
fig,(ax_lot,ax_count) = plt.subplots(1,2,figsize=(16,10.2),gridspec_kw={"width_ratios":[2.05,1]})
for i,r in enumerate(link.itertuples()):
    color = colors[r.grade]
    ax_lot.barh(i,r.minutes,left=0,height=.52,color=color,alpha=.84)
    ax_lot.scatter([0],[i],marker="D",s=33,color=color,zorder=4)
    ax_lot.text(r.minutes+8,i,f"{r.minutes:.1f}분",va="center",fontsize=9,color="#425466")
labels = [f"LOT {r.LOT_NO} · {r.start:%m/%d %H:%M}\n{r.BEF_VAL if pd.notna(r.BEF_VAL) else '미연결'} → {r.AFT_VAL if pd.notna(r.AFT_VAL) else r.grade}"
          for r in link.itertuples()]
ax_lot.set_yticks(np.arange(len(link)),labels,fontsize=9)
ax_lot.set_ylim(len(link)-.35,-.8)
ax_lot.set_xlim(-30,max(60,link.minutes.max()*1.13))
ax_lot.axvspan(-30,0,color="#eef1f4",zorder=0)
ax_lot.axvline(0,color="#425466",lw=1.1,ls="--")
ax_lot.set_title("강종변경과 관측 시작을 0분에 맞춘 LOT 비교",loc="left",fontsize=12,pad=16)
ax_lot.set_xlabel("변경 시각 = 관측 시작 0분 → 경과 시간 [분]")
ax_lot.grid(axis="x",alpha=.2); ax_lot.grid(axis="y",visible=False)
ax_lot.spines[["top","right"]].set_visible(False)
for tick in ax_lot.get_yticklabels(): tick.set_color("#263238")
for i,(grade,count) in enumerate(linked_grade_counts.items()):
    ax_count.barh(i,count,height=.55,color=colors[grade],alpha=.84)
    rgb = to_rgb(colors[grade])
    luminance = .2126*rgb[0]+.7152*rgb[1]+.0722*rgb[2]
    text_color = "#263238" if luminance>.55 else "white"
    ax_count.text(count/2,i,str(count),va="center",ha="center",color=text_color,fontweight="bold",fontsize=11)
    codes = link.loc[link.AFT_VAL.eq(grade),"LOT_NO"].astype(str).str[-3:].tolist()
    ax_count.text(count+.1,i," · ".join(codes),va="center",fontsize=9,color="#425466")
ax_count.set_yticks(np.arange(len(linked_grade_counts)),linked_grade_counts.index,fontsize=10)
ax_count.set_ylim(len(linked_grade_counts)-.5,-.8)
ax_count.set_xlim(0,linked_grade_counts.max()+2.9)
ax_count.set_xticks(range(int(linked_grade_counts.max())+1))
ax_count.set_title("같은 강종에 여러 LOT가 반복됨",loc="left",fontsize=12,pad=16)
ax_count.set_xlabel("관측 LOT 수 [개] · 막대 옆 숫자 = LOT 끝 3자리")
ax_count.grid(axis="x",alpha=.2); ax_count.grid(axis="y",visible=False)
ax_count.spines[["top","right"]].set_visible(False)
fig.suptitle(f"그림 1-보조. 시작 시각은 {matched_lots}/{len(lots)} 일치 · {grade_count}개 강종에 {len(lots)} LOT가 반복",
             x=.16,y=.97,ha="left",fontsize=16,fontweight="bold")
fig.text(.16,.916,"왼쪽: 변경 전→후 강종과 관측 길이 / 오른쪽: 변경 후 강종별 LOT 수. 두 그림에서 같은 강종은 같은 색입니다.",
         fontsize=10.5,color="#425466")
fig.text(.16,.035,f"해석 범위: 연결 LOT 시작 직전 10분 관측은 {link.pre10_sensor_rows.sum():,}행. 시작 시각의 연결은 확인되지만 강종변경 전후 온도 효과는 비교할 수 없습니다.",
         fontsize=10,color="#425466")
fig.subplots_adjust(left=.16,right=.99,top=.84,bottom=.115,wspace=.42)
save_and_show_figures()


#### 그림 1·보조 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 날짜축·LOT 막대 | 막대는 실제 관측 시작~끝, 색은 강종 후보. 회색 강종변경 눈금과 관측 LOT를 구분. |
| 보조 그림 | LOT 시작을 0분으로 맞춘 길이와 강종별 반복 LOT 수를 확인. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| 강종변경 ↔ LOT | **14/14 LOT**의 시작이 CR1 변경 시각과 일치; LOT 내부 추가 변경 **0건** | LOT별 변경 후 강종 후보를 연결할 수 있음. |
| 강종별 표본 | SPHC **4**, SPCE **3**, SPFH590·SPCEN 각 **2**, SGCC·DP590·SPCD 각 **1** | 같은 강종 비교의 표본 수가 다름. |
| 관측 범위 | **98.9시간 / 2024년 1.13%**; CR1 변경 **134건 중 120건**은 관측 창 밖 | 연속 연간 기록이 아니라 일부 LOT의 관측. |
| 변경 직전·수리 중 센서 | 각 LOT 시작 직전 10분 합계 **0행**, 계획수리 기간 **0행** | 강종변경 전후와 수리 중의 연속 센서 비교는 불가능. |
| 정비 관측 | 감지·착수·완료 각각 관측 창 포함 **2/44건** | 각 시각에 포함된 작업이 같은 작업인지 별도 대조 필요. |

**결론:** 분석 단위는 **관측된 LOT 안의 흐름과 반복 강종 LOT 비교**입니다. 강종변경과 시작이 일치해도 변경의 온도 효과를 입증한 것은 아닙니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| SPHC·SPCE·SPFH590·SPCEN | 동일 강종의 비교 LOT를 확보하고 수집 품질부터 확인 | 그림 2·3 → 10-A·B |
| WO-2024-01859·WO-2024-02105 | 감지·착수·완료 중 어느 시각을 실제 센서로 볼 수 있는지 확인 | 그림 16·17의 작업별 시간 대조 |
| LOT 사이 공백 | 가동 중 미수집인지 실제 비가동인지 확인 | 가동·수집 로그; 공백을 보간하지 않음 |

[↑ 목차](#toc)


In [ ]:
# 후속 그래프도 같은 강종 팔레트로 다시 실행하기 위한 공통 분석 설정입니다.
# 앞에서 만든 표시용 ts와 사용자가 정한 TC_GROUPS는 유지합니다.
from scipy.signal import detrend, periodogram

temp["gap_s"] = temp.groupby("LOT_NO")["MEAS_DT"].diff().dt.total_seconds()
temp["SEG"] = temp["seg"]
temp["elapsed_min"] = temp["sec"]/60
lots["LOT_NO"] = lots.index
lots["EVT_ID"] = lots.index.map(lot_event_link.set_index("LOT_NO")["EVT_ID"])
lots["span_hours"] = ((lots["end"]-lots["start"]).dt.total_seconds()+1)/3600
lot_map = lots.set_index("LOT_NO")
replacement = partchg
joined = partchg.merge(
    maint[["WO_NO","EQP_TAG","DETC_DT","STRT_DT","END_DT","MNT_TYPE"]],
    on="WO_NO",how="left",validate="many_to_one",indicator="work_match"
).merge(part,on="PART_CD",how="left",validate="many_to_one",indicator="part_match")
cal_parts = joined[joined["EQP_TAG"].eq(TARGET)].copy()

def analysis_aggregate(freq,minimum):
    grouped = temp.set_index("MEAS_DT").groupby(["LOT_NO","SEG"])[SIG].resample(freq)
    result = grouped.mean().where(grouped.count().ge(minimum)).reset_index()
    result["elapsed_min"] = (result["MEAS_DT"]-result["LOT_NO"].map(lot_map["start"])).dt.total_seconds()/60
    return result

analysis_ts = analysis_aggregate("30s",15)
analysis_ts1 = analysis_aggregate("1min",45)
analysis_groups = {
    "세정조":["CLN-IN"],"건조존":["TC-Z1","TC-Z2"],
    "가열·균열존":["TC-Z3","TC-Z4","TC-Z5","TC-Z6"],
    "과시효대":["TC-Z7","TC-Z8"],"냉각대":["TC-OUT1","TC-OUT2"]}
for name,cols in {**analysis_groups,**TC_GROUPS}.items():
    analysis_ts[name] = analysis_ts[cols].mean(axis=1).where(analysis_ts[cols].notna().all(axis=1))
analysis_ts["Z3_relative"] = analysis_ts["TC-Z3"]-analysis_ts[["TC-Z4","TC-Z5","TC-Z6"]].mean(axis=1).where(analysis_ts[["TC-Z4","TC-Z5","TC-Z6"]].notna().all(axis=1))
mnt_colors = {kind:style[1] for kind,style in MNT_STYLE.items()}
colors = dict(GRADE_COLOR)
# 한글 폰트에 없는 위첨자 대신 읽을 수 있는 척도 표기를 사용합니다.
UNIT = {col: label.replace("×10⁻¹⁰", "×10^-10") for col, label in UNIT.items()}

def finish(fig,title,subtitle=""):
    fig.suptitle(title,fontsize=14,fontweight="bold")
    if subtitle:
        fig.text(.5,.015,subtitle.replace("−","-"),ha="center",fontsize=10,color="#34495e")
    fig.tight_layout(rect=(0,.05 if subtitle else 0,1,.94))
    save_and_show_figures()

assert len(SIG)==22 and len(joined)==len(partchg)


<a id="quality"></a>
## 3. 수집 품질 — 기록 간격과 채널별 결측

**분석 질문:** 1초보다 긴 기록 간격은 어느 LOT에 많고, 결측은 어느 신호와 시점에 집중되는가?

**계산 방법:** 같은 LOT의 인접 기록 사이 간격을 계산합니다. 그림 2의 첫째·둘째 패널은 모두 **건수**로 비교합니다. LOT 내부 간격 수가 다른 점을 고려해 결과 표에는 비율도 함께 제시합니다. 결측은 신호별·LOT별 실제 결측 행 수이며, 값이 0인 정상 기록과 구분합니다.

**그림 읽는 법:** 왼쪽은 간격별 건수(로그 축), 가운데는 1초 초과 간격의 LOT별 건수, 오른쪽은 결측 행 수를 구간별 색으로 표시합니다. 결측 0은 밝은 회색이고, OP-Z3와 LOT 240044·240061 및 해당 두 셀은 빨간색으로 강조합니다. 그림 3은 LOT마다 **왼쪽 축에 TC-Z3 온도 [℃], 오른쪽 축에 OP-Z3 출력 [%]**을 함께 표시하며, 회색 음영은 실제 OP-Z3 결측 구간입니다.

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.ticker import StrMethodFormatter
from matplotlib.patches import Patch, Rectangle
from matplotlib.lines import Line2D

# 같은 LOT 안의 인접 기록만 집계합니다. 첫째·둘째 패널은 모두 건수입니다.
gaps = temp["gap_s"].dropna()
gap_counts = gaps.value_counts().sort_index()
gap_by_lot = temp.groupby("LOT_NO")["gap_s"].agg(
    intervals="count", max_gap="max", irregular=lambda s: int(s.gt(1).sum())
)
gap_by_lot["irregular_pct"] = 100 * gap_by_lot["irregular"] / gap_by_lot["intervals"]
missing = temp.groupby("LOT_NO")[SIG].agg(lambda s: int(s.isna().sum())).reindex(lots.LOT_NO)
missing_op_lots = missing.index[missing["OP-Z3"].gt(0)].tolist()
focus_lots = set(missing_op_lots)
BLUE, RED = "#2874A6", "#C0392B"
ordered_gaps = gap_by_lot.sort_values("irregular", ascending=False)
one_second_pct = 100 * gap_counts.get(1, 0) / len(gaps)
op_missing_total = int(missing["OP-Z3"].sum())
op_lot_text = " · ".join(f"{lot}({missing.loc[lot, 'OP-Z3']:,}행)" for lot in missing_op_lots)

fig, axes = plt.subplots(1, 3, figsize=(19, 7.2), gridspec_kw={"width_ratios": [1.05, 1.4, 2.6]})
positions = np.arange(len(gap_counts))
axes[0].bar(positions, gap_counts.values, color=BLUE, width=.65)
for x, count in zip(positions, gap_counts.values):
    axes[0].text(x, count * 1.15, f"{count:,}", ha="center", va="bottom", fontsize=8)
axes[0].set_xticks(positions, [f"{value:g}" for value in gap_counts.index])
axes[0].set_yscale("log")
axes[0].set_ylim(.5, gap_counts.max() * 3)
axes[0].set(xlabel="LOT 내부 기록 간격 [초]", ylabel="간격 건수 [건, 로그 축]")
axes[0].grid(axis="y", alpha=.18)
axes[0].set_axisbelow(True)

max_count = int(ordered_gaps["irregular"].max())
axes[1].barh(np.arange(len(ordered_gaps)), ordered_gaps["irregular"], height=.62,
             color=[RED if i == 0 else "#A8BCCB" for i in range(len(ordered_gaps))])
axes[1].set_yticks(np.arange(len(ordered_gaps)), ordered_gaps.index.astype(str), fontsize=9)
axes[1].invert_yaxis()
for y, count in enumerate(ordered_gaps["irregular"]):
    axes[1].text(count + max_count * .025, y, f"{count:,}", va="center", fontsize=8.5)
axes[1].set_xlim(0, max_count * 1.23)
axes[1].xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
axes[1].set_xlabel("1초 초과 간격 [건]")
axes[1].grid(axis="x", alpha=.18)
axes[1].set_axisbelow(True)
axes[1].get_yticklabels()[0].set_color(RED)
axes[1].get_yticklabels()[0].set_fontweight("bold")

# 0을 별도 색으로 구분합니다. 원래 결측 건수에 정수 구간을 직접 적용합니다.
heat_colors = ["#F5F7FA", "#DCE7F0", "#8AAEC8", "#F4C1AD", "#DC7355", "#A62639"]
heat_bounds = [-.5, .5, 4.5, 9.5, 49.5, 99.5, max(199.5, float(missing.to_numpy().max()) + .5)]
heat_cmap = ListedColormap(heat_colors)
heat_norm = BoundaryNorm(heat_bounds, heat_cmap.N)

# 중요: transpose 제거
im = axes[2].imshow(
    missing,
    aspect="auto",
    cmap=heat_cmap,
    norm=heat_norm,
    interpolation="nearest"
)

# x축 = 신호
axes[2].set_xticks(range(len(SIG)))
axes[2].set_xticklabels(SIG, rotation=60, ha="right", fontsize=9)

# y축 = LOT
axes[2].set_yticks(range(len(missing.index)))
axes[2].set_yticklabels(missing.index.astype(str), fontsize=9)

# 셀 경계
axes[2].set_xticks(np.arange(-.5, len(SIG), 1), minor=True)
axes[2].set_yticks(np.arange(-.5, len(missing.index), 1), minor=True)
axes[2].grid(which="minor", color="white", linewidth=.65)
axes[2].grid(which="major", visible=False)
axes[2].tick_params(which="minor", bottom=False, left=False)

# x축에서 OP-Z3 강조
for label in axes[2].get_xticklabels():
    if label.get_text() == "OP-Z3":
        label.set_color(RED)
        label.set_fontweight("bold")

# y축에서 OP-Z3 결측 LOT 강조
for label in axes[2].get_yticklabels():
    if label.get_text() in {str(lot) for lot in focus_lots}:
        label.set_color(RED)
        label.set_fontweight("bold")

# 셀 값 표시
for row, lot in enumerate(missing.index):
    for col, signal in enumerate(SIG):
        count = int(missing.loc[lot, signal])
        if count:
            important = (signal == "OP-Z3") and (lot in focus_lots)
            axes[2].text(
                col, row, str(count),
                ha="center", va="center",
                color="white" if count >= 50 else "#263238",
                fontsize=8,
                fontweight="bold" if important else "normal"
            )
            if important:
                axes[2].add_patch(
                    Rectangle(
                        (col - .5, row - .5), 1, 1,
                        fill=False, edgecolor=RED, lw=1.8
                    )
                )

for spine in axes[2].spines.values():
    spine.set_visible(False)

# ------------------------------------------------------------
# 히트맵 패널 제목
# ------------------------------------------------------------
axes[2].set_title(
    "LOT × 신호 결측 행 수",
    fontsize=11,
    pad=10
)

# ------------------------------------------------------------
# 왼쪽 두 패널 스타일
# ------------------------------------------------------------
for ax in axes[:2]:
    ax.spines[["top", "right"]].set_visible(False)

# ------------------------------------------------------------
# 전체 그림 제목 / 설명
# ------------------------------------------------------------
fig.suptitle(
    "그림 2. 수집 품질 점검 — 기록 간격 분포와 결측 집중 위치",
    x=.065,
    y=.975,
    ha="left",
    fontsize=16,
    fontweight="bold"
)

fig.text(
    .065,
    .918,
    (
        f"전체 간격의 {one_second_pct:.2f}%가 1초 · "
        f"OP-Z3 결측 {op_missing_total:,}행: {op_lot_text}"
    ),
    ha="left",
    fontsize=11,
    color="#425466"
)

# ------------------------------------------------------------
# 레이아웃
# ------------------------------------------------------------
fig.subplots_adjust(
    left=.065,
    right=.985,
    top=.80,
    bottom=.23,
    wspace=.40
)

# ------------------------------------------------------------
# 히트맵 colorbar
# 0 / 1~4 / 5~9 / 10~49 / 50~99 / 100 이상
# ------------------------------------------------------------
heat_position = axes[2].get_position()

colorbar_ax = fig.add_axes([
    heat_position.x0,
    .055,
    heat_position.width,
    .027
])

colorbar = fig.colorbar(
    im,
    cax=colorbar_ax,
    orientation="horizontal",
    spacing="uniform"
)

colorbar.set_ticks(
    [
        (left + right) / 2
        for left, right
        in zip(
            heat_bounds[:-1],
            heat_bounds[1:]
        )
    ],
    labels=[
        "0",
        "1~4",
        "5~9",
        "10~49",
        "50~99",
        "100 이상"
    ]
)

colorbar.set_label(
    "결측 행 수 [행] · 구간별 색",
    fontsize=9
)

colorbar.ax.tick_params(
    labelsize=8
)

colorbar.outline.set_visible(False)

save_and_show_figures()
save_and_show_figures()

# LOT마다 두 세로축을 사용해 같은 시간의 온도·출력을 한 패널에서 비교합니다.
fig, axes = plt.subplots(1, len(missing_op_lots), figsize=(16.8, 5.8), sharex=True, squeeze=False)
missing_detail_rows = []
for ax, lot in zip(axes.flat, missing_op_lots):
    d = temp[temp.LOT_NO.eq(lot) & temp.elapsed_min.lt(10)].sort_values("MEAS_DT")
    right = ax.twinx()
    ax.plot(d.elapsed_min, d["TC-Z3"], color=RED, lw=1.5)
    right.plot(d.elapsed_min, d["OP-Z3"], color=BLUE, lw=1.2, ls="--")
    absent_mask = d["OP-Z3"].isna().to_numpy()
    edges = np.diff(np.r_[0, absent_mask.astype(int), 0])
    for first, last in zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1) - 1):
        start = d.elapsed_min.iloc[first]
        end = max(d.elapsed_min.iloc[last], start + 1/60)
        ax.axvspan(start, end, color="#DDE1E5", alpha=.75, lw=0, zorder=0)
    absent = d.loc[absent_mask]
    tc_observed = int(absent["TC-Z3"].notna().sum())
    if len(absent):
        start_sec, end_sec = absent.elapsed_min.min() * 60, absent.elapsed_min.max() * 60
        ax.text(0, 1.025, f"회색 구간에서 TC-Z3 온도는 {tc_observed:,}행 모두 관측됨",
                transform=ax.transAxes, va="bottom", fontsize=9.5, color=RED)
        missing_detail_rows.append({"LOT_NO": lot, "OP-Z3 결측 [행]": len(absent),
                                   "첫 결측 [초]": start_sec, "마지막 결측 [초]": end_sec,
                                   "동시 TC-Z3 관측 [행]": tc_observed})
    ax.set_title(f"LOT {lot} · OP-Z3 결측 {len(absent):,}행 ({start_sec:.0f}~{end_sec:.0f}초)",
                 loc="left", fontsize=11.5, fontweight="bold", pad=33)
    ax.set_xlim(0, 10)
    ax.set_xlabel("LOT 시작 후 경과 시간 [분]")
    ax.set_ylabel("TC-Z3 온도 [℃]", color=RED)
    ax.tick_params(axis="y", colors=RED)
    ax.spines["left"].set_color(RED)
    ax.spines["top"].set_visible(False)
    right.set_ylabel("OP-Z3 제어출력 [%]", color=BLUE)
    right.set_ylim(0, 105)
    right.tick_params(axis="y", colors=BLUE)
    right.spines["right"].set_color(BLUE)
    right.spines[["top", "left"]].set_visible(False)
    ax.grid(axis="both", alpha=.15)
    right.grid(visible=False)
fig.suptitle("그림 3. OP-Z3 초반 결측 구간의 온도·출력 비교",
             x=.065, y=.975, ha="left", fontsize=16, fontweight="bold")
fig.text(.065, .895, "회색 구간에서도 TC-Z3 온도는 관측됨 · 두 LOT의 시작 후 10분 원자료",
         fontsize=11, color="#425466")
fig.legend(handles=[Line2D([], [], color=RED, lw=1.5, label="TC-Z3 온도 · 왼쪽 축 [℃]"),
                    Line2D([], [], color=BLUE, lw=1.2, ls="--", label="OP-Z3 출력 · 오른쪽 축 [%]"),
                    Patch(facecolor="#DDE1E5", label="OP-Z3 결측 구간")],
           loc="upper left", bbox_to_anchor=(.06, .85), ncol=3, fontsize=9, frameon=False)
fig.subplots_adjust(left=.065, right=.94, top=.70, bottom=.15, wspace=.35)
save_and_show_figures()

quality_rows = []
for col in SIG:
    d = temp[temp[col].isna()]
    if len(d):
        quality_rows.append({"신호": col, "결측행": len(d), "비율 [%]": 100*len(d)/len(temp),
                             "발생 LOT 수": d.LOT_NO.nunique(), "최초 [초]": d.elapsed_min.min()*60,
                             "최후 [초]": d.elapsed_min.max()*60})
quality_table = pd.DataFrame(quality_rows).sort_values("결측행", ascending=False)
display(quality_table)
boundary_gap = temp.MEAS_DT.diff().dt.total_seconds().where(temp.LOT_NO.ne(temp.LOT_NO.shift())).dropna()


#### 그림 2·3 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 그림 2 | 간격별·LOT별 건수와 채널별 결측 지도. 1초 초과 건수는 관측 길이도 함께 고려. |
| 그림 3 | TC-Z3(왼쪽 ℃)와 OP-Z3(오른쪽 %)의 같은 시각 기록 유무. 두 축의 선 높이를 직접 비교하지 않음. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| 기록 간격 | LOT 내부 **345,376간격** 중 1초 **334,966건(96.99%)**; 1초 초과 **10,410건** | 1초가 대표 간격. 2초 간격을 곧바로 설비 이상으로 해석하지 않음. |
| 240069의 긴 간격 | 1초 초과 **1,894건**, 전체 LOT 중 최다 건수 | 긴 LOT의 관측량 효과가 있어 비율과 공백 위치도 봐야 함. |
| **240044·240061 / OP-Z3** | 초반 결측 각각 **75행(0~81초)**, **179행(0~183초)** | 결측이 특정 LOT·채널·초반에 집중. |
| 같은 시각 TC-Z3 | OP-Z3 결측인 모든 행에서 TC-Z3 관측 | 전체 수집 공백보다 OP-Z3 채널 점검을 우선할 근거. |

**결론:** **값이 비어 있음과 출력 0%는 다릅니다.** 결측은 보간하지 않고 같은 LOT·연속구간의 유효 관측만 비교합니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240044 ↔ 240061; 대조 240069·240091 | OP-Z3 최초 유효값과 TC-Z3·OP-Z4·Z5·Z6 기록을 대조 | 그림 10-A 뒤 결측 확대; 태그·통신·저장 로그 |
| 240069 및 10초 초과 공백 LOT | 긴 간격의 위치와 채널 동시성·수집 규칙 확인 | 원본 시각·수집기 사양; SEG 경계 보호 |

[↑ 목차](#toc)


<a id="signals"></a>
## 4. 전체 22개 신호 — 존별 온도·출력의 LOT 시작 정렬

**분석 질문:** 같은 존의 온도와 제어출력을 위아래로 비교하면 어떤 LOT의 상승·하강·급변이 눈에 띄는가?

**계산 방법:** LOT·SEG별 유효 30초 평균을 LOT 원본 시작 시각에 맞춥니다. 모든 신호를 **한 열**에 넓게 배치하며, 같은 존의 **TC 바로 아래에 OP**를 놓습니다. Z4의 CP·CP 모니터는 Z4 온도·출력 다음에 표시합니다. 결측 칸과 연속구간 사이 공백은 선으로 잇지 않습니다.

**그림 읽는 법:** 같은 색은 같은 강종이고, 한 LOT의 관측이 여러 연속구간으로 나뉠 수 있습니다. 소제목은 모두 **왼쪽 정렬·검정색 볼드체**입니다. 소제목 아래의 **붉은 특이사항 설명**에 LOT 번호·경과 시간·값을 적고, 통계는 패널 위 오른쪽에 표시합니다. 그래프 안의 붉은 원 옆에도 해당 LOT 번호를 짧게 붙입니다. 온도·출력의 단위와 세로축 범위는 다르므로 같은 시간의 변화 방향을 비교합니다.

**표식의 기준:** 각 신호의 유효 30초 평균 전체에서 `Q1-3×IQR`보다 작거나 `Q3+3×IQR`보다 큰 값을 참고 지점으로 고릅니다. 상·하한 양쪽의 대표점 또는 한쪽의 최대 두 LOT를 표시합니다. OP-Z3는 출력값의 크기보다 변동이 특징인 신호이므로 차분 표준편차가 큰 두 LOT의 최대 30초 변화 지점을 표시합니다. TC-Z3도 범위 밖 값이 없으면 같은 변동 기준을 사용합니다. 시작 승온·끝 하강도 포함되는 시각적 점검 기준입니다.

**후반 하강의 별도 기준:** 극값 두 개만 고르면 큰 하강이 빠질 수 있어, 마지막 연속구간의 끝 60~45분 전과 끝 5분을 별도로 비교합니다. 가열 평균과 4개 가열 온도 중 3개 이상이 20℃ 이상 내려간 LOT는 **후반 동반 하강**으로 추가 표시합니다. 하강 구간 시작은 마지막 45분 안에서 가열 평균의 5분 이동 중앙값이 기준보다 5℃ 낮아진 첫 시각입니다. TC는 20℃ 이상, OP-Z1~Z8 전체에서 10%p 이상 하락한 신호에 표식을 붙입니다. 해당 LOT의 기존 극값 표식 수와 관계없이 추가합니다. 관측 마지막이 실제 조업 정지 시각이라는 뜻은 아닙니다.

In [ ]:
from matplotlib.lines import Line2D

# 한 열에서 TC → 같은 존 OP 순서로 읽습니다.
all_groups = {
    "세정조": ["CLN-IN"],
    "건조존": ["TC-Z1", "OP-Z1", "TC-Z2", "OP-Z2"],
    "가열·균열존": ["TC-Z3", "OP-Z3", "TC-Z4", "OP-Z4", "CP-Z4", "CP-Z4M",
                    "TC-Z5", "OP-Z5", "TC-Z6", "OP-Z6"],
    "과시효대": ["TC-Z7", "OP-Z7", "TC-Z8", "OP-Z8", "TC-Z9"],
    "냉각대": ["TC-OUT1", "TC-OUT2"],
}
assert sorted(sum(all_groups.values(), [])) == sorted(SIG)

# 30초 차분은 같은 LOT·SEG의 인접한 유효 칸에서만 계산합니다.
adjacent_30s = analysis_ts.groupby(["LOT_NO", "SEG"]).MEAS_DT.diff().dt.total_seconds().eq(30)
diffs = analysis_ts.groupby(["LOT_NO", "SEG"])[SIG].diff().where(adjacent_30s, axis=0)
short_std = diffs.groupby(analysis_ts.LOT_NO).std()
tc_cols = [c for c in SIG if c.startswith("TC") or c == "CLN-IN"]
op_cols = [c for c in SIG if c.startswith("OP")]
tc_top = short_std[["TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6"]].idxmax(axis=1)
op_top = short_std[op_cols].idxmax(axis=1)
signal_names = {
    "CLN-IN": "세정조 입측 온도", "CP-Z4": "카본포텐셜", "CP-Z4M": "카본포텐셜 모니터",
    "TC-OUT1": "냉각대 1 온도", "TC-OUT2": "냉각대 2 온도", "TC-Z9": "슬러지 제거부 온도",
}
for z in range(1, 9):
    signal_names[f"TC-Z{z}"] = f"{z}존 온도"
    signal_names[f"OP-Z{z}"] = f"{z}존 제어출력"
group_findings = {
    "세정조": "완만한 오르내림과 LOT별 온도 수준 비교",
    "건조존": "Z1·Z2 온도 바로 아래에서 같은 존의 출력 확인",
    "가열·균열존": f"TC-Z3 차분 변동이 온도 4개 존 중 {tc_top.eq('TC-Z3').sum()}/{len(lots)} LOT에서 가장 큼",
    "과시효대": "Z7·Z8 온도·출력 비교, Z9는 별도 반복 흐름",
    "냉각대": "두 냉각 채널의 반복적인 오르내림 비교",
}
RED = "#B52932"

def displayed_unit(col):
    if col.startswith("TC") or col == "CLN-IN":
        return "℃"
    if col.startswith("OP"):
        return "%"
    return "×10^-10" if col == "CP-Z4M" else "무차원"

def signal_points(col):
    """상·하한 밖 대표 지점 최대 2개. LOT·시점은 실제 관측 칸에서 가져옵니다."""
    valid = analysis_ts[["LOT_NO", "SEG", "MEAS_DT", "elapsed_min", col]].dropna(subset=[col]).copy()
    q1, q3 = valid[col].quantile([.25, .75])
    spread = q3 - q1
    points = []
    if spread > 0:
        lower, upper = q1 - 3*spread, q3 + 3*spread
        for mask, mode in [(valid[col].lt(lower), "낮은 값"), (valid[col].gt(upper), "높은 값")]:
            subset = valid.loc[mask]
            if len(subset):
                idx = subset[col].idxmin() if mode == "낮은 값" else subset[col].idxmax()
                row = valid.loc[idx].copy()
                row["구분"] = mode
                points.append(row)
        if len(points) == 1:
            mask = valid[col].lt(lower) if points[0]["구분"] == "낮은 값" else valid[col].gt(upper)
            other = valid.loc[mask & valid.LOT_NO.ne(points[0].LOT_NO)]
            if len(other):
                idx = other[col].idxmin() if points[0]["구분"] == "낮은 값" else other[col].idxmax()
                row = valid.loc[idx].copy()
                row["구분"] = points[0]["구분"]
                points.append(row)
    if col == "OP-Z3" or (not points and col == "TC-Z3"):
        points = []
        for lot in short_std[col].nlargest(2).index:
            indexes = valid.index[valid.LOT_NO.eq(lot)]
            changes = diffs.loc[indexes, col].abs().dropna()
            if len(changes):
                row = valid.loc[changes.idxmax()].copy()
                row["구분"] = "큰 30초 변화"
                points.append(row)
    return points

# 극값 표식과 별도로, 관측 후반의 여러 존 동반 하강을 찾습니다.
# 마지막 SEG에서 끝 60~45분 전 중앙값과 마지막 5분 중앙값을 비교합니다.
# 가열 존 평균과 4개 온도 중 3개 이상이 20℃ 이상 내려간 LOT를 선택합니다.
heat_columns = ["TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6"]
terminal_fall_records = []
terminal_fall_map = {}
for lot, full in analysis_ts.groupby("LOT_NO"):
    d = full[full.SEG.eq(full.SEG.max())].sort_values("MEAS_DT")
    end_time = d.MEAS_DT.max()
    before = d[d.MEAS_DT.ge(end_time-pd.Timedelta(minutes=60)) & d.MEAS_DT.lt(end_time-pd.Timedelta(minutes=45))]
    after = d[d.MEAS_DT.ge(end_time-pd.Timedelta(minutes=5))]
    if before[heat_columns].notna().all(axis=1).sum() < 15 or after[heat_columns].notna().all(axis=1).sum() < 8:
        continue
    before_values = before[heat_columns].median()
    after_values = after[heat_columns].median()
    differences = after_values-before_values
    mean_before = before["가열·균열존"].median()
    mean_after = after["가열·균열존"].median()
    if mean_after-mean_before > -20 or differences.le(-20).sum() < 3:
        continue
    rolling_heat = d.set_index("MEAS_DT")["가열·균열존"].rolling("5min", min_periods=8).median()
    crossed = rolling_heat[(rolling_heat.index >= end_time-pd.Timedelta(minutes=45)) & rolling_heat.le(mean_before-5)]
    if crossed.empty:
        continue
    onset_time = crossed.index[0]
    onset_minute = (onset_time-lot_map.loc[lot, "start"]).total_seconds()/60
    end_minute = (end_time-lot_map.loc[lot, "start"]).total_seconds()/60
    last_tc3 = float(d.dropna(subset=["TC-Z3"]).iloc[-1]["TC-Z3"])
    record = {"LOT": int(lot), "강종": lot_map.loc[lot, "grade"],
              "하강 구간 시작 [분]": onset_minute, "마지막 관측 [분]": end_minute,
              "TC-Z3 기준 [℃]": float(before_values["TC-Z3"]), "TC-Z3 마지막 [℃]": last_tc3,
              "TC-Z3 차이 [℃]": last_tc3-float(before_values["TC-Z3"]),
              "가열 평균 차이 [℃]": float(mean_after-mean_before),
              "20℃ 이상 하강 존 수": int(differences.le(-20).sum())}
    for col in op_cols:
        record[col+" 기준 [%]"] = float(before[col].median())
        record[col+" 끝5분 [%]"] = float(after[col].median())
        record[col+" 마지막 [%]"] = float(d.dropna(subset=[col]).iloc[-1][col])
        record[col+" 차이 [%p]"] = float(after[col].median()-before[col].median())
    terminal_fall_records.append(record)
    terminal_fall_map[int(lot)] = {"frame": d, "before": before, "after": after,
                                  "onset_time": onset_time, "record": record}
terminal_falls_table = pd.DataFrame(terminal_fall_records)

point_map = {col: signal_points(col) for col in SIG}
# 동반 하강은 기존 최저·최고 표식의 개수 제한과 별도로 표시합니다.
for lot, fall in terminal_fall_map.items():
    interval = fall["frame"][fall["frame"].MEAS_DT.ge(fall["onset_time"])]
    for col in heat_columns+op_cols:
        baseline = fall["before"][col].median()
        final = fall["after"][col].median()
        threshold = 20 if col.startswith("TC") else 10
        if not np.isfinite(baseline) or not np.isfinite(final) or baseline-final < threshold:
            continue
        available = interval.dropna(subset=[col])
        halfway = available[available[col].le((baseline+final)/2)]
        if halfway.empty:
            continue
        point = halfway.iloc[0].copy()
        point["구분"] = "후반 동반 하강"
        # 같은 LOT의 낮은 값 표식은 하강 표식으로 구체화합니다.
        point_map[col] = [p for p in point_map[col] if not (int(p.LOT_NO) == lot and p["구분"] == "낮은 값")]
        point_map[col].append(point)
signal_point_rows = []
for col, points in point_map.items():
    scale = 1e10 if col == "CP-Z4M" else 1
    for point in points:
        signal_point_rows.append({"신호": col, "LOT_NO": int(point.LOT_NO), "강종": lot_map.loc[point.LOT_NO, "grade"],
                                  "경과 [분]": float(point.elapsed_min), "표시값": float(point[col]*scale),
                                  "단위": displayed_unit(col), "구분": point["구분"]})
signal_point_table = pd.DataFrame(signal_point_rows)
present_grades = [grade for grade in colors if grade in set(lot_map["grade"])]
signal_plot_audit = []



#### 그림 4~8 공통 읽기

각 LOT의 시작을 0분으로 맞춘 30초 평균입니다. **같은 색은 같은 강종**, 붉은 원 옆 숫자는 **전체 LOT 번호**, 붉은 설명은 확인 지점입니다. 선이 끊긴 곳은 결측·공백을 연결하지 않은 것이며, 선이 짧은 것은 관측 길이가 짧기 때문입니다. 각 그림 바로 아래에서 관측 결과와 후속 비교 대상을 확인합니다.


In [ ]:
# 이 셀에서 해당 공정의 matplotlib 그래프를 직접 생성합니다.
figure_number = 4
group = '세정조'
cols = all_groups[group]
height = 3.8*len(cols) + 2.45
fig, axes = plt.subplots(len(cols), 1, figsize=(17.5, height), squeeze=False, sharex=True)
fig.subplots_adjust(left=.07, right=.985, top=1-2.0/height, bottom=.45/height, hspace=.52)
for ax, col in zip(axes.flat, cols):
    scale = 1e10 if col == "CP-Z4M" else 1
    points = point_map[col]
    highlighted_lots = {int(point.LOT_NO) for point in points}
    for (lot, seg), d in analysis_ts.groupby(["LOT_NO", "SEG"]):
        grade = lot_map.loc[lot, "grade"]
        ax.plot(d.elapsed_min, d[col]*scale, color=colors[grade],
                alpha=.95 if lot in highlighted_lots else .70,
                lw=1.3 if lot in highlighted_lots else 1.0)
    ax.set_title(f"{col} · {signal_names[col]}", loc="left", pad=35, fontsize=12,
                 color="black", fontweight="bold")
    ax.set_ylabel(UNIT[col], fontsize=10)
    ax.set_xlabel("LOT 시작 후 경과 시간 [분]", fontsize=10)
    ax.tick_params(axis="x", labelbottom=True)
    ax.grid(alpha=.13)
    ax.spines[["top", "right"]].set_visible(False)
    # 긴 설명은 패널 위에, 그래프 안에는 원형 표식과 짧은 LOT 번호를 둡니다.
    ax.text(1, 1.15, f"30초 차분 SD 중앙값: {short_std[col].median()*scale:.3f} {displayed_unit(col)}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="#425466", clip_on=False)
    notes = []
    for point in points:
        ax.scatter([point.elapsed_min], [point[col]*scale], s=53, facecolors="none", edgecolors=RED, lw=1.5, zorder=6)
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        near_right = point.elapsed_min > x_min + .82*(x_max-x_min)
        dx = -12 if near_right else 12
        dy = -15 if point[col]*scale > (y_min+y_max)/2 else 13
        ax.annotate(str(int(point.LOT_NO)), (point.elapsed_min, point[col]*scale),
                    xytext=(dx, dy), textcoords="offset points", color=RED, fontsize=9,
                    fontweight="bold", ha="right" if near_right else "left",
                    va="top" if dy < 0 else "bottom", zorder=7,
                    bbox=dict(facecolor="white", edgecolor="none", alpha=.82, pad=1.2),
                    arrowprops=dict(arrowstyle="-", color=RED, lw=.7))
        notes.append(f"○ LOT {int(point.LOT_NO)} · {point.elapsed_min:.1f}분 · {point[col]*scale:.3f} {displayed_unit(col)} ({point['구분']})")
    if notes:
        note_lines = ["   |   ".join(notes[i:i+2]) for i in range(0, len(notes), 2)]
        ax.text(0, 1.025, "\n".join(note_lines), transform=ax.transAxes, ha="left", va="bottom",
                fontsize=9.5, color=RED, clip_on=False)
    signal_plot_audit.append({"신호": col, "열": 1, "설명위치": "패널 밖", "표식수": len(points)})
fig.text(.07, 1-.12/height, f"그림 {figure_number}. {group} — {group_findings[group]}",
         ha="left", va="top", fontsize=15, fontweight="bold")
handles = [Line2D([], [], color=colors[grade], lw=2, label=grade) for grade in present_grades]
handles.append(Line2D([], [], marker="o", markerfacecolor="none", color=RED, ls="", label="LOT 확인 지점"))
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.065, 1-.48/height),
           ncol=len(handles), fontsize=9, frameon=False)
fig.text(.07, 1-.89/height, "같은 색 = 같은 강종 · TC 바로 아래에 같은 존 OP · 붉은 원형 표식과 LOT 번호 = 확인 지점",
         fontsize=10, color="#425466", ha="left", va="top")
save_and_show_figures()


#### 그림 4 · CLN-IN — 그림 읽기와결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 선·시간 | 한 선은 한 LOT의 연속구간. 같은 색은 같은 강종. 각 LOT 시작을 0분으로 정렬. |
| 수준·모양 | 선이 층으로 갈리는지, LOT 안에서 느리게 오르내리는지 확인. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| CLN-IN / 14 LOT | 선들이 서로 다른 온도 수준에서 완만하게 변화 | LOT 간 수준 차이와 LOT 안의 느린 흐름이 보임. |

**결론:** CLN-IN은 **완만한 수준 변화**를 볼 보조 채널입니다. 이 그림만으로 강종 효과나 설비 이상을 정하지 않습니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 같은 강종 LOT의 CLN-IN | 온도 중앙값과 초반·중간·끝의 흐름 비교 | 세정조 목표값·가동 기록; LOT별 운전조건 |
| CLN-IN ↔ TC-Z1·TC-Z2 | 세정조 변화와 건조존 변화가 같은 시각에 겹치는지 확인 | 다음 그림 5; 단위·목표값 구분 |


In [ ]:
# 이 셀에서 해당 공정의 matplotlib 그래프를 직접 생성합니다.
figure_number = 5
group = '건조존'
cols = all_groups[group]
height = 3.8*len(cols) + 2.45
fig, axes = plt.subplots(len(cols), 1, figsize=(17.5, height), squeeze=False, sharex=True)
fig.subplots_adjust(left=.07, right=.985, top=1-2.0/height, bottom=.45/height, hspace=.52)
for ax, col in zip(axes.flat, cols):
    scale = 1e10 if col == "CP-Z4M" else 1
    points = point_map[col]
    highlighted_lots = {int(point.LOT_NO) for point in points}
    for (lot, seg), d in analysis_ts.groupby(["LOT_NO", "SEG"]):
        grade = lot_map.loc[lot, "grade"]
        ax.plot(d.elapsed_min, d[col]*scale, color=colors[grade],
                alpha=.95 if lot in highlighted_lots else .70,
                lw=1.3 if lot in highlighted_lots else 1.0)
    ax.set_title(f"{col} · {signal_names[col]}", loc="left", pad=35, fontsize=12,
                 color="black", fontweight="bold")
    ax.set_ylabel(UNIT[col], fontsize=10)
    ax.set_xlabel("LOT 시작 후 경과 시간 [분]", fontsize=10)
    ax.tick_params(axis="x", labelbottom=True)
    ax.grid(alpha=.13)
    ax.spines[["top", "right"]].set_visible(False)
    # 긴 설명은 패널 위에, 그래프 안에는 원형 표식과 짧은 LOT 번호를 둡니다.
    ax.text(1, 1.15, f"30초 차분 SD 중앙값: {short_std[col].median()*scale:.3f} {displayed_unit(col)}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="#425466", clip_on=False)
    notes = []
    for point in points:
        ax.scatter([point.elapsed_min], [point[col]*scale], s=53, facecolors="none", edgecolors=RED, lw=1.5, zorder=6)
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        near_right = point.elapsed_min > x_min + .82*(x_max-x_min)
        dx = -12 if near_right else 12
        dy = -15 if point[col]*scale > (y_min+y_max)/2 else 13
        ax.annotate(str(int(point.LOT_NO)), (point.elapsed_min, point[col]*scale),
                    xytext=(dx, dy), textcoords="offset points", color=RED, fontsize=9,
                    fontweight="bold", ha="right" if near_right else "left",
                    va="top" if dy < 0 else "bottom", zorder=7,
                    bbox=dict(facecolor="white", edgecolor="none", alpha=.82, pad=1.2),
                    arrowprops=dict(arrowstyle="-", color=RED, lw=.7))
        notes.append(f"○ LOT {int(point.LOT_NO)} · {point.elapsed_min:.1f}분 · {point[col]*scale:.3f} {displayed_unit(col)} ({point['구분']})")
    if notes:
        note_lines = ["   |   ".join(notes[i:i+2]) for i in range(0, len(notes), 2)]
        ax.text(0, 1.025, "\n".join(note_lines), transform=ax.transAxes, ha="left", va="bottom",
                fontsize=9.5, color=RED, clip_on=False)
    signal_plot_audit.append({"신호": col, "열": 1, "설명위치": "패널 밖", "표식수": len(points)})
fig.text(.07, 1-.12/height, f"그림 {figure_number}. {group} — {group_findings[group]}",
         ha="left", va="top", fontsize=15, fontweight="bold")
handles = [Line2D([], [], color=colors[grade], lw=2, label=grade) for grade in present_grades]
handles.append(Line2D([], [], marker="o", markerfacecolor="none", color=RED, ls="", label="LOT 확인 지점"))
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.065, 1-.48/height),
           ncol=len(handles), fontsize=9, frameon=False)
fig.text(.07, 1-.89/height, "같은 색 = 같은 강종 · TC 바로 아래에 같은 존 OP · 붉은 원형 표식과 LOT 번호 = 확인 지점",
         fontsize=10, color="#425466", ha="left", va="top")
save_and_show_figures()


#### 그림 5 · TC-Z1·OP-Z1 / TC-Z2·OP-Z2 — 그림 읽기와결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 각 TC 바로 아래 OP | Z1과 Z2를 각각 한 쌍으로 같은 경과 시각에서 비교. ℃와 %는 다른 단위. |
| 붉은 원·LOT 번호 | 특정 LOT의 극값·후반 하강 확인 지점. 색 자체는 강종 식별. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| TC-Z1·TC-Z2 | 대부분 관측이 약 **100℃** 부근에 모임 | 중간의 공통 수준과 시작·끝 변화는 분리해 봐야 함. |
| **240058 · DP590 / 240114 · SPCEN** | 종료부에 TC-Z1·TC-Z2와 OP-Z1·OP-Z2가 함께 하강 | 온도 한 센서의 단독 급락보다 운전 전환을 우선 확인. |
| 240061 · SPHC / 240117 · SPCD | 초반 낮은 온도에서 상승하는 흐름 | 시작 가동 상태와 출력 변화의 대응을 확인할 대상. |

**결론:** 건조존에서는 **같은 존의 온도와 출력이 언제 함께 바뀌는가**를 확인했습니다. 출력 비율만으로 존별 열량을 직접 비교하지 않습니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240058 ↔ 240114 | 종료부 TC-Z1·TC-Z2와 OP-Z1·OP-Z2 하강 시각 대조 | 정지계획·가동·출력 목표 기록 |
| 240061 ↔ 240117 | 초반 TC·OP 변화와 다른 가열 존의 승온 동반 여부 확인 | 다음 그림 6; 강종이 달라 전환 형태 참고만 가능 |


In [ ]:
# 이 셀에서 해당 공정의 matplotlib 그래프를 직접 생성합니다.
figure_number = 6
group = '가열·균열존'
cols = all_groups[group]
height = 3.8*len(cols) + 2.45
fig, axes = plt.subplots(len(cols), 1, figsize=(17.5, height), squeeze=False, sharex=True)
fig.subplots_adjust(left=.07, right=.985, top=1-2.0/height, bottom=.45/height, hspace=.52)
for ax, col in zip(axes.flat, cols):
    scale = 1e10 if col == "CP-Z4M" else 1
    points = point_map[col]
    highlighted_lots = {int(point.LOT_NO) for point in points}
    for (lot, seg), d in analysis_ts.groupby(["LOT_NO", "SEG"]):
        grade = lot_map.loc[lot, "grade"]
        ax.plot(d.elapsed_min, d[col]*scale, color=colors[grade],
                alpha=.95 if lot in highlighted_lots else .70,
                lw=1.3 if lot in highlighted_lots else 1.0)
    ax.set_title(f"{col} · {signal_names[col]}", loc="left", pad=35, fontsize=12,
                 color="black", fontweight="bold")
    ax.set_ylabel(UNIT[col], fontsize=10)
    ax.set_xlabel("LOT 시작 후 경과 시간 [분]", fontsize=10)
    ax.tick_params(axis="x", labelbottom=True)
    ax.grid(alpha=.13)
    ax.spines[["top", "right"]].set_visible(False)
    # 긴 설명은 패널 위에, 그래프 안에는 원형 표식과 짧은 LOT 번호를 둡니다.
    ax.text(1, 1.15, f"30초 차분 SD 중앙값: {short_std[col].median()*scale:.3f} {displayed_unit(col)}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="#425466", clip_on=False)
    notes = []
    for point in points:
        ax.scatter([point.elapsed_min], [point[col]*scale], s=53, facecolors="none", edgecolors=RED, lw=1.5, zorder=6)
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        near_right = point.elapsed_min > x_min + .82*(x_max-x_min)
        dx = -12 if near_right else 12
        dy = -15 if point[col]*scale > (y_min+y_max)/2 else 13
        ax.annotate(str(int(point.LOT_NO)), (point.elapsed_min, point[col]*scale),
                    xytext=(dx, dy), textcoords="offset points", color=RED, fontsize=9,
                    fontweight="bold", ha="right" if near_right else "left",
                    va="top" if dy < 0 else "bottom", zorder=7,
                    bbox=dict(facecolor="white", edgecolor="none", alpha=.82, pad=1.2),
                    arrowprops=dict(arrowstyle="-", color=RED, lw=.7))
        notes.append(f"○ LOT {int(point.LOT_NO)} · {point.elapsed_min:.1f}분 · {point[col]*scale:.3f} {displayed_unit(col)} ({point['구분']})")
    if notes:
        note_lines = ["   |   ".join(notes[i:i+2]) for i in range(0, len(notes), 2)]
        ax.text(0, 1.025, "\n".join(note_lines), transform=ax.transAxes, ha="left", va="bottom",
                fontsize=9.5, color=RED, clip_on=False)
    signal_plot_audit.append({"신호": col, "열": 1, "설명위치": "패널 밖", "표식수": len(points)})
fig.text(.07, 1-.12/height, f"그림 {figure_number}. {group} — {group_findings[group]}",
         ha="left", va="top", fontsize=15, fontweight="bold")
handles = [Line2D([], [], color=colors[grade], lw=2, label=grade) for grade in present_grades]
handles.append(Line2D([], [], marker="o", markerfacecolor="none", color=RED, ls="", label="LOT 확인 지점"))
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.065, 1-.48/height),
           ncol=len(handles), fontsize=9, frameon=False)
fig.text(.07, 1-.89/height, "같은 색 = 같은 강종 · TC 바로 아래에 같은 존 OP · 붉은 원형 표식과 LOT 번호 = 확인 지점",
         fontsize=10, color="#425466", ha="left", va="top")
save_and_show_figures()


#### 그림 6 · TC-Z3~TC-Z6·OP-Z3~OP-Z6·CP-Z4 — 그림 읽기와결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 각 TC → 같은 존 OP | TC-Z3/OP-Z3, TC-Z4/OP-Z4, TC-Z5/OP-Z5, TC-Z6/OP-Z6의 같은 시점 흐름. |
| CP-Z4·CP-Z4M | Z4 뒤의 별도 패널. CP-Z4M은 ×10⁻¹⁰ 표시 척도이며 온도·출력과 직접 크기 비교하지 않음. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| **TC-Z3·OP-Z3 / 14 LOT** | 각각 가열 온도 4개·출력 8개 중 차분 std 최대 **14/14 LOT** | 단기 변동을 상세 비교할 핵심 짝. |
| **240058 · DP590 / 240114 · SPCEN** | TC-Z3·Z4·Z5·Z6 동반 하강: **177.5~216.5 / 280.0~319.0분** | 대표 극값 외에도 놓치지 않고 볼 후반 전환. |
| 240061 · SPHC / 240117 · SPCD | 네 TC가 낮은 초반에서 함께 상승 | TC-Z3 단독 센서 현상과 공통 승온을 구분. |
| **240069 · SPHC / CP-Z4** | 시작 후 **574.5분**, 30초 평균 약 **0.894** | 다른 TC·OP의 큰 변동과 별개로 CP 피크를 후속 확인. |

**결론:** **공통 승온·하강, TC-Z3·OP-Z3 단기 변동, CP 피크는 서로 다른 확인 항목**입니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240061 ↔ 240044·240069·240091 | 초반 결측·승온을 분리한 TC-Z3·OP-Z3 비교 | 그림 10-A 결측 확대·10-B·10-D·11·12 |
| 240069 ↔ 같은 SPHC 세 LOT | CP-Z4 중앙값·std와 피크의 시간 지속성 확인 | 그림 10-B-CP·10-C; 분위기 제어 기록 |
| 240058 ↔ 240114 | TC-Z3·Z4·Z5·Z6와 OP-Z3·Z4·Z5·Z6의 하강 순서 대조 | 종료·가동상태 기록; 각자의 동일 강종 조건 확인 |


In [ ]:
# 이 셀에서 해당 공정의 matplotlib 그래프를 직접 생성합니다.
figure_number = 7
group = '과시효대'
cols = all_groups[group]
height = 3.8*len(cols) + 2.45
fig, axes = plt.subplots(len(cols), 1, figsize=(17.5, height), squeeze=False, sharex=True)
fig.subplots_adjust(left=.07, right=.985, top=1-2.0/height, bottom=.45/height, hspace=.52)
for ax, col in zip(axes.flat, cols):
    scale = 1e10 if col == "CP-Z4M" else 1
    points = point_map[col]
    highlighted_lots = {int(point.LOT_NO) for point in points}
    for (lot, seg), d in analysis_ts.groupby(["LOT_NO", "SEG"]):
        grade = lot_map.loc[lot, "grade"]
        ax.plot(d.elapsed_min, d[col]*scale, color=colors[grade],
                alpha=.95 if lot in highlighted_lots else .70,
                lw=1.3 if lot in highlighted_lots else 1.0)
    ax.set_title(f"{col} · {signal_names[col]}", loc="left", pad=35, fontsize=12,
                 color="black", fontweight="bold")
    ax.set_ylabel(UNIT[col], fontsize=10)
    ax.set_xlabel("LOT 시작 후 경과 시간 [분]", fontsize=10)
    ax.tick_params(axis="x", labelbottom=True)
    ax.grid(alpha=.13)
    ax.spines[["top", "right"]].set_visible(False)
    # 긴 설명은 패널 위에, 그래프 안에는 원형 표식과 짧은 LOT 번호를 둡니다.
    ax.text(1, 1.15, f"30초 차분 SD 중앙값: {short_std[col].median()*scale:.3f} {displayed_unit(col)}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="#425466", clip_on=False)
    notes = []
    for point in points:
        ax.scatter([point.elapsed_min], [point[col]*scale], s=53, facecolors="none", edgecolors=RED, lw=1.5, zorder=6)
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        near_right = point.elapsed_min > x_min + .82*(x_max-x_min)
        dx = -12 if near_right else 12
        dy = -15 if point[col]*scale > (y_min+y_max)/2 else 13
        ax.annotate(str(int(point.LOT_NO)), (point.elapsed_min, point[col]*scale),
                    xytext=(dx, dy), textcoords="offset points", color=RED, fontsize=9,
                    fontweight="bold", ha="right" if near_right else "left",
                    va="top" if dy < 0 else "bottom", zorder=7,
                    bbox=dict(facecolor="white", edgecolor="none", alpha=.82, pad=1.2),
                    arrowprops=dict(arrowstyle="-", color=RED, lw=.7))
        notes.append(f"○ LOT {int(point.LOT_NO)} · {point.elapsed_min:.1f}분 · {point[col]*scale:.3f} {displayed_unit(col)} ({point['구분']})")
    if notes:
        note_lines = ["   |   ".join(notes[i:i+2]) for i in range(0, len(notes), 2)]
        ax.text(0, 1.025, "\n".join(note_lines), transform=ax.transAxes, ha="left", va="bottom",
                fontsize=9.5, color=RED, clip_on=False)
    signal_plot_audit.append({"신호": col, "열": 1, "설명위치": "패널 밖", "표식수": len(points)})
fig.text(.07, 1-.12/height, f"그림 {figure_number}. {group} — {group_findings[group]}",
         ha="left", va="top", fontsize=15, fontweight="bold")
handles = [Line2D([], [], color=colors[grade], lw=2, label=grade) for grade in present_grades]
handles.append(Line2D([], [], marker="o", markerfacecolor="none", color=RED, ls="", label="LOT 확인 지점"))
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.065, 1-.48/height),
           ncol=len(handles), fontsize=9, frameon=False)
fig.text(.07, 1-.89/height, "같은 색 = 같은 강종 · TC 바로 아래에 같은 존 OP · 붉은 원형 표식과 LOT 번호 = 확인 지점",
         fontsize=10, color="#425466", ha="left", va="top")
save_and_show_figures()


#### 그림 7 · TC-Z7·OP-Z7 / TC-Z8·OP-Z8 / TC-Z9 — 그림 읽기와결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| TC-Z7 → OP-Z7 / TC-Z8 → OP-Z8 | 같은 존의 위아래 패널을 같은 시간에 읽음. |
| TC-Z9 별도 패널 | TC-Z7·TC-Z8와 다른 느린 반복 흐름. 두 존 평균에 TC-Z9를 포함하지 않음. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| TC-Z7·TC-Z8 | LOT별 온도 수준이 여러 층으로 나뉨 | 강종을 섞은 수준 차이를 같은 강종 안에서 다시 확인해야 함. |
| **240114 · SPCEN / OP-Z8** | 약 **300분** 부근에서 끝으로 향하는 급락. 240058도 종료부 출력 하강 | 극값 하나만 선택하면 빠질 수 있는 동반 종료 변화. |
| TC-Z9 | TC-Z7·TC-Z8와 달리 느린 반복 오르내림 | 수준 차이와 주기 차이는 다른 항목. |

**결론:** 이 그림은 **TC-Z7·TC-Z8의 LOT 수준 차이와 OP의 종료 변화**를 보여줍니다. 높은 층 하나만으로 이상을 판정하지 않습니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240121 ↔ 240037·240133 · SPCE | TC-Z7·TC-Z8 각 컬럼의 수준·시간 지속성 확인 | 그림 10-B-온도·10-E; 목표값·속도·두께 |
| 240091 ↔ 240044·240061·240069 · SPHC | 단일 온도 신호군의 높은 수준이 지속되는지 확인 | 그림 10-E-참고 |
| 240038 ↔ 240114 · SPCEN | TC-Z7·TC-Z8 수준과 OP-Z7·OP-Z8 종료 상태 대조 | n=2 참고; 동일 운전상태 구간 |
| TC-Z9 / 긴 LOT | 반복 주기와 유효 관측 길이 확인 | 그림 13 |


In [ ]:
# 이 셀에서 해당 공정의 matplotlib 그래프를 직접 생성합니다.
figure_number = 8
group = '냉각대'
cols = all_groups[group]
height = 3.8*len(cols) + 2.45
fig, axes = plt.subplots(len(cols), 1, figsize=(17.5, height), squeeze=False, sharex=True)
fig.subplots_adjust(left=.07, right=.985, top=1-2.0/height, bottom=.45/height, hspace=.52)
for ax, col in zip(axes.flat, cols):
    scale = 1e10 if col == "CP-Z4M" else 1
    points = point_map[col]
    highlighted_lots = {int(point.LOT_NO) for point in points}
    for (lot, seg), d in analysis_ts.groupby(["LOT_NO", "SEG"]):
        grade = lot_map.loc[lot, "grade"]
        ax.plot(d.elapsed_min, d[col]*scale, color=colors[grade],
                alpha=.95 if lot in highlighted_lots else .70,
                lw=1.3 if lot in highlighted_lots else 1.0)
    ax.set_title(f"{col} · {signal_names[col]}", loc="left", pad=35, fontsize=12,
                 color="black", fontweight="bold")
    ax.set_ylabel(UNIT[col], fontsize=10)
    ax.set_xlabel("LOT 시작 후 경과 시간 [분]", fontsize=10)
    ax.tick_params(axis="x", labelbottom=True)
    ax.grid(alpha=.13)
    ax.spines[["top", "right"]].set_visible(False)
    # 긴 설명은 패널 위에, 그래프 안에는 원형 표식과 짧은 LOT 번호를 둡니다.
    ax.text(1, 1.15, f"30초 차분 SD 중앙값: {short_std[col].median()*scale:.3f} {displayed_unit(col)}",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color="#425466", clip_on=False)
    notes = []
    for point in points:
        ax.scatter([point.elapsed_min], [point[col]*scale], s=53, facecolors="none", edgecolors=RED, lw=1.5, zorder=6)
        x_min, x_max = ax.get_xlim()
        y_min, y_max = ax.get_ylim()
        near_right = point.elapsed_min > x_min + .82*(x_max-x_min)
        dx = -12 if near_right else 12
        dy = -15 if point[col]*scale > (y_min+y_max)/2 else 13
        ax.annotate(str(int(point.LOT_NO)), (point.elapsed_min, point[col]*scale),
                    xytext=(dx, dy), textcoords="offset points", color=RED, fontsize=9,
                    fontweight="bold", ha="right" if near_right else "left",
                    va="top" if dy < 0 else "bottom", zorder=7,
                    bbox=dict(facecolor="white", edgecolor="none", alpha=.82, pad=1.2),
                    arrowprops=dict(arrowstyle="-", color=RED, lw=.7))
        notes.append(f"○ LOT {int(point.LOT_NO)} · {point.elapsed_min:.1f}분 · {point[col]*scale:.3f} {displayed_unit(col)} ({point['구분']})")
    if notes:
        note_lines = ["   |   ".join(notes[i:i+2]) for i in range(0, len(notes), 2)]
        ax.text(0, 1.025, "\n".join(note_lines), transform=ax.transAxes, ha="left", va="bottom",
                fontsize=9.5, color=RED, clip_on=False)
    signal_plot_audit.append({"신호": col, "열": 1, "설명위치": "패널 밖", "표식수": len(points)})
fig.text(.07, 1-.12/height, f"그림 {figure_number}. {group} — {group_findings[group]}",
         ha="left", va="top", fontsize=15, fontweight="bold")
handles = [Line2D([], [], color=colors[grade], lw=2, label=grade) for grade in present_grades]
handles.append(Line2D([], [], marker="o", markerfacecolor="none", color=RED, ls="", label="LOT 확인 지점"))
fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(.065, 1-.48/height),
           ncol=len(handles), fontsize=9, frameon=False)
fig.text(.07, 1-.89/height, "같은 색 = 같은 강종 · TC 바로 아래에 같은 존 OP · 붉은 원형 표식과 LOT 번호 = 확인 지점",
         fontsize=10, color="#425466", ha="left", va="top")
save_and_show_figures()


#### 그림 8 · TC-OUT1·TC-OUT2 — 그림 읽기와결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 각 패널·선 | TC-OUT1과 TC-OUT2를 따로 봄. 서로 다른 LOT의 위상이 달라 겹친 선은 빽빽하게 보일 수 있음. |
| 한 LOT를 따라 읽기 | 반복성·온도 수준·진폭을 구분하고 선이 끝나는 시각은 관측 종료로 읽음. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| TC-OUT1·TC-OUT2 | 여러 LOT에서 반복적인 오르내림 | 반복 자체와 평소 반복 패턴의 변화를 구분할 필요. |
| LOT별 냉각 수준 | 같은 높이에 완전히 겹치지 않는 선들 | 강종·운전조건에 따른 수준 차이가 섞일 수 있음. |

**결론:** 이 그림으로 **반복 형태**를 확인했습니다. 주기 수치는 뒤의 자기상관·스펙트럼 분석에서 확인합니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240024 ↔ 240108 · SPFH590 | TC-OUT1·TC-OUT2 각 컬럼의 공통 수준 확인 | 그림 10-E-참고; 냉각 목표·라인속도 |
| TC-OUT1·TC-OUT2 / 동일 조건 LOT | 주기·진폭·위상 차이 확인 | 그림 13; 냉각 제어·가동 기록 |


In [ ]:
sig_summary = pd.DataFrame({"신호": SIG, "원본 결측행": [int(temp[c].isna().sum()) for c in SIG],
                           "LOT별 30초 차분 표준편차 중앙값": [short_std[c].median() for c in SIG],
                           "그림에 표시한 LOT": [", ".join(dict.fromkeys(str(int(p.LOT_NO)) for p in point_map[c])) or "없음" for c in SIG]})
display(sig_summary)

# 관측 후반 동반 하강의 온도·출력 비교 근거입니다.
display(terminal_falls_table.round(3))


#### 그림 4~8 · 컬럼별 검토 결과 — 그림 읽기와결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 존별 그림 뒤 집계표 | 원본 결측행·30초 차분 std·표시 LOT를 확인. 서로 단위가 다른 std는 직접 비교하지 않음. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| **TC-Z3·OP-Z3** | 각 신호군 내 차분 std 최대 **14/14 LOT** | 상태와 제어를 같은 시간축에서 상세 비교할 핵심 짝. |
| **240058·240114** | 가열 TC 네 개 모두 하강, OP-Z1~Z8도 기준보다 끝 5분 중앙값 10%p 이상 하락 | 후반 하강은 함께 움직이는 운전 전환 후보. |
| **240069 / CP-Z4** | 574.5분의 대표 피크 약 **0.894** | 수준과 변동성·일시적 피크를 나눠 확인. |
| TC-Z7·TC-Z8 / TC-OUT1·TC-OUT2 / TC-Z9 | LOT 수준 차이와 반복 형태가 각각 보임 | 강종 내 비교와 주기 분석의 다른 목적으로 유지. |

**결론:** 22개 컬럼을 같은 의미의 이상 점수로 합치지 않습니다. **수집 품질·공통 전환·개별 변동·반복 패턴**에 따라 비교 목적을 구분합니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 전체 LOT / TC-Z3·OP-Z3 | LOT별 차이를 공통 승온·끝 하강과 분리 | 바로 다음 그림 9 |
| 전체 LOT / 온도·OP-Z3·CP-Z4 | 서로 다른 신호군 후보를 찾고 동일 강종으로 좁히기 | 그림 10-A → 10-B → 시간 지속성 |
| CP-Z4M | 모니터 값의 정의와 온도와의 관계 확인 | 태그 정의·계측 사양; 현재 자료만으로 별도 고장 지표 채택하지 않음 |

**후반 하강의 계산 근거**

| LOT | 강종 | 하강 구간 시작 [분] | 마지막 관측 [분] | TC-Z3 기준 [℃] | TC-Z3 마지막 [℃] | TC-Z3 차이 [℃] | 20℃ 이상 하강 존 수 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 240058 | DP590 | 177.5 | 216.5 | 858.9 | 814.9 | -44.0 | 4 |
| 240114 | SPCEN | 280.0 | 319.0 | 857.3 | 814.2 | -43.1 | 4 |

[↑ 목차](#toc)


<a id="lots"></a>
## 5. 14 LOT를 비교하고 상세 사례 고르기

**분석 질문:** 변동이 큰 LOT는 초반 승온, 한 존 이탈, 제어출력 변화 중 무엇을 포함하는가?

**계산 방법:** 원본 TC-Z3 표준편차로 상세 사례를 선택하고 동일 강종 후보의 반복 LOT를 비교 대상으로 둡니다. 평균·표준편차·30초 차분 표준편차를 구분합니다.

**그림 읽는 법:** 빨강은 TC-Z3, 검정은 주변 존 평균입니다. 패널 제목에서 LOT 길이와 TC-Z3 표준편차를 확인하고 시작 상승·끝 하강으로 표시된 LOT를 비교합니다. 표의 표준편차는 추세와 흔들림 모두를 포함합니다.

In [ ]:
feature=lots[['LOT_NO','grade','start','minutes','rows']].copy().set_index('LOT_NO')
raw_stats=temp.groupby('LOT_NO')['TC-Z3'].agg(mean='mean',median='median',std='std')
feature=feature.join(raw_stats).join(short_std[['TC-Z3','OP-Z3']].rename(columns={'TC-Z3':'TC_diff_std','OP-Z3':'OP_diff_std'}))
feature['relative_std']=analysis_ts.groupby('LOT_NO').Z3_relative.std()
feature['OP_zero_rows']=temp.groupby('LOT_NO')['OP-Z3'].agg(lambda s:int(s.eq(0).sum()))
feature['OP_missing_rows']=temp.groupby('LOT_NO')['OP-Z3'].agg(lambda s:int(s.isna().sum()))
CASE_LOT=int(feature['std'].idxmax())
CASE_GRADE=feature.loc[CASE_LOT,'grade']
peers=feature.index[feature.grade.eq(CASE_GRADE)].tolist()
fig,axes=plt.subplots(4,4,figsize=(17,13),squeeze=False)
for ax,r in zip(axes.flat,lots.itertuples()):
    for seg,d in analysis_ts[analysis_ts.LOT_NO.eq(r.LOT_NO)].groupby('SEG'):
        ax.plot(d.elapsed_min,d['TC-Z3'],color='#c0392b',lw=.9)
        ref=d[['TC-Z4','TC-Z5','TC-Z6']].mean(axis=1).where(d[['TC-Z4','TC-Z5','TC-Z6']].notna().all(axis=1))
        ax.plot(d.elapsed_min,ref,color='black',lw=.8,alpha=.65)
    ax.set_ylim(805,885); ax.set_ylabel('TC-Z3 온도 [℃]'); ax.set_xlabel('경과 [분]')
    note=''
    if r.LOT_NO in [240061,240117]: note='시작 상승'
    elif r.LOT_NO in [240058,240114]: note='끝 하강'
    if note:
        ax.text(.04,.12,note,transform=ax.transAxes,color='#c0392b',fontweight='bold',bbox={'facecolor':'#fcf3cf','edgecolor':'none'})
    ax.set_title(f'{r.LOT_NO} · {r.grade} · {r.minutes:.0f}분\nTC-Z3 표준편차 {feature.loc[r.LOT_NO,"std"]:.2f}',fontsize=9)
for ax in list(axes.flat)[len(lots):]: ax.set_visible(False)
fig.legend(handles=[plt.Line2D([0],[0],color='#c0392b',label='TC-Z3'),plt.Line2D([0],[0],color='black',label='TC-Z4~Z6 평균')],loc='lower center',ncol=2)
fig.suptitle(f'그림 9. 시작 상승·끝 하강도 전체 변동에 포함 — 최대 변동 LOT {CASE_LOT}',fontsize=14,fontweight='bold')
fig.tight_layout(rect=(0,.04,1,.96)); save_and_show_figures()
display(feature.reset_index().round(3))

#### 그림 9 · 14 LOT와 사례 선택 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| LOT별 패널 | 각 LOT의 TC-Z3와 TC-Z4~Z6 평균을 같은 시간축에서 비교. 패널마다 관측 길이가 다름. |
| 표의 std·차분 std | 전체 std는 느린 승온·하강도 포함. 30초 차분 std는 인접 유효 칸의 짧은 움직임. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| **240061 · SPHC** | 원본 TC-Z3 평균 **852.455℃**, std **11.303℃**(최대), 관측 **225.4분** | 승온·결측·출력 변화가 함께 있어 주 상세 사례로 채택. |
| 240061·240117 / 240058·240114 | 앞의 두 LOT는 시작 상승, 뒤의 두 LOT는 끝 하강 | 전체 std가 크다는 이유만으로 개별 센서 이상을 붙이면 운전 전환이 섞임. |
| **SPHC 비교군** | **240044·240061·240069·240091**, n=4 | 같은 강종 안에서 결측·승온·TC-Z3 상대 이탈을 비교할 수 있음. |

**결론:** 240061은 **여러 현상을 같은 시간축에서 분리하는 사례**입니다. 전체 표준편차 최대를 고장 판정으로 사용하지 않습니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 전체 14 LOT | 온도·OP-Z3·CP-Z4 중 2개 이상 신호군 후보를 찾고 같은 강종으로 재비교 | 바로 다음 그림 10-A·B |
| 240061 ↔ 240044·240069·240091 | 결측·승온을 분리하고 각 TC·OP의 유효 구간 확인 | 그림 10-D·11·12·15 |
| SPHC 네 LOT | 결측·저출력·안정 구간 변동·정비 맥락별 채택 이유 정리 | 마지막 LOT 선정 보고 |

**SPHC 비교의 원자료 집계**

| LOT | 강종 후보 | 지속 [분] | TC-Z3 평균 | TC-Z3 표준편차 | TC 30초 차분 SD | OP 30초 차분 SD | OP=0 원본행 | OP 결측행 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 240044 | SPHC | 74.633 | 859.886 | 2.860 | 0.638 | 7.998 | 0 | 75 |
| 240061 | SPHC | 225.383 | 852.455 | 11.303 | 0.640 | 7.921 | 595 | 179 |
| 240069 | SPHC | 812.567 | 858.926 | 2.492 | 0.635 | 8.665 | 1235 | 0 |
| 240091 | SPHC | 641.200 | 860.059 | 1.827 | 0.426 | 6.088 | 413 | 0 |

[↑ 목차](#toc)


<a id="trend"></a>
## 6. 강종별 LOT 비교 — 전체 후보에서 확인 항목까지

**분석 흐름:** 전체 후보 → 같은 강종 비교 → LOT 차이 확인 → 시간 지속성 → 계획수리 위치 → 전체 LOT 확인표

**용어는 이렇게 읽습니다.**

| 표시 | 의미 | 읽는 방법 |
|---|---|---|
| 중앙값 | LOT의 대표 수준 | 평균과 달리 큰 피크의 영향을 덜 받음 |
| 변동폭(P90-P10) | 낮은 쪽 10%와 높은 쪽 10% 경계의 차이 | 값이 클수록 대부분의 관측값이 넓게 분포. 예: 856~862℃이면 변동폭 6℃ |
| CP-Z4 std | CP-Z4 표준편차 | 클수록 CP가 많이 흔들림. 짧은 피크만으로도 커질 수 있음 |
| CP-Z4 std(5분) | 최근 5분의 CP-Z4 표준편차 | 흔들림이 커진 시점과 지속 여부를 확인 |
| 차이 | 해당 LOT - 자신을 제외한 동일 강종 LOT 중앙값 | +는 더 높음, -는 더 낮음 |
| 배수 | 해당 LOT의 CP-Z4 std ÷ 비교 CP-Z4 std | 1배는 같음, 2배는 두 배. 고장 확률이 아님 |
| n | 해당 강종의 LOT 수 | n≥3 기술 비교, n=2 참고 비교, n=1 비교 불가 |
| %p | 출력 비율의 차이 | 60%에서 80%로 바뀌면 +20%p |

**같은 구간끼리 비교합니다.** 주 비교는 LOT 중간 **20~80%**입니다. 전체 구간에서만 나타나는 차이는 별도로 표시하여 초반 승온·종료 하강을 놓치지 않습니다. 전체 또는 중간에서 온도·OP·CP 중 **2개 이상 신호군**이 탐색 기준에 걸리면 넓은 후보로 둡니다.

**전체 LOT를 끝까지 확인합니다.** 넓은 후보는 11/14 LOT이지만, 한 신호군만 다른 240091과 결측이 있는 240044도 별도 확인합니다. 최종 표는 특정 사례만 선정하지 않고 **14개 LOT 모두**의 후보 이유, 같은 강종 차이, 시간 지속성, 수리 위치, 다음 확인을 정리합니다.

**그림 순서:** 10-A 후보 이유 → 10-B-온도·OP·CP 실제 차이 → 10-B-시간-1·2 지속성 → 10-C~F 대표 상세 사례 → 10-G-1 점검 목적·수리 위치 → 10-G-2 실제 정비 전후 관측 → 10-H 전체 확인표.

**계산 범위:** LOT별 30초 평균의 중앙값·변동폭·CP-Z4 std를 사용합니다. 동일 강종 비교는 자기 자신을 제외하며 모든 LOT를 동일 가중으로 집계합니다. 같은 강종을 맞춰도 목표온도·속도·두께 등의 운전조건 차이는 남을 수 있습니다. ±60일은 수리 전후 LOT 탐색 범위입니다.

**온도 지표의 원본 컬럼**

| 그림의 표시 | 사용한 컬럼 | 집계 방법 |
|---|---|---|
| TC-Z3~TC-Z6 평균 | `TC-Z3`, `TC-Z4`, `TC-Z5`, `TC-Z6` | 각 시각의 네 컬럼 평균 |
| TC-Z7·TC-Z8 평균 | `TC-Z7`, `TC-Z8` | 각 시각의 두 컬럼 평균. TC-Z9는 포함하지 않음 |
| TC-OUT1·TC-OUT2 평균 | `TC-OUT1`, `TC-OUT2` | 각 시각의 두 컬럼 평균 |
| OP-Z3 변동폭 | `OP-Z3` | 해당 출력의 P90-P10 |
| CP-Z4 std | `CP-Z4` | 해당 컬럼의 표준편차 |

`평균 변동폭`은 컬럼별 변동폭의 평균이 아니라 **시각별 평균온도 시계열의 P90-P10**입니다. 그림 10-B-온도 뒤에서 주요 LOT를 개별 TC 컬럼으로 다시 확인합니다.


### ① 전체 후보 — 값의 차이와 확인 우선순위를 구분

**비교 상대는 전체 14 LOT입니다.** 각 LOT의 중간 20~80%에서 온도·OP·CP 지표를 계산합니다. 수준은 낮거나 높은 쪽 20% 순위, 변동폭·CP-Z4 std는 큰 쪽 20% 순위에 해당하면 참고 대상으로 표시합니다. 중간 또는 전체 구간에서 두 신호군 이상이면 넓은 탐색 후보이며 여러 온도 지표는 온도군 하나로 셉니다.

| 색 | 확인 의미 | 이 그림에서 적용하는 기준 |
|---|---|---|
| **빨강** | **우선·중요 후속 확인** | OP-Z3 결측 우선 확인과 두 신호군 이상 탐색 후보의 후속 확인 |
| **초록** | **참고 확인** | 단일 신호군 참고 항목 |
| **흰색 `-`** | **기준 비해당** | 온도·OP·CP 칸에는 기준 미만 내용을 표시하지 않음 |
| **회색 `-`** | **자료 없음** | 해당 지표의 유효값 또는 비교 자료가 없음 |

**읽는 순서:** LOT·강종 → 신호군별 내용 → 마지막 확인 우선순위. 일반 값 칸은 `LOT 값 / 14 LOT 기준값 / 차이`를 적습니다. OP-Z3 결측 칸은 결측 행 수·시각·최초 유효값 시각을 표시합니다. 빨강 안에서 **① 결측 우선 확인 → ② 후보 후속 확인**으로 순서를 읽습니다. 색은 확인 필요성을 표시합니다.

**중간 구간:** 관측 시작을 0%, 끝을 100%로 놓고 시작·끝 각각 20%를 제외합니다. 200분 LOT에서는 40~160분입니다. 값의 분포를 나타내는 `P90-P10`과는 다르며 안정 운전 구간을 보장하지 않습니다.

**표시 방식:** 시간순 7 LOT씩 두 그림이며 두 그림 모두 전체 14 LOT의 같은 지표 중앙값을 기준으로 사용합니다. 결측은 신호군 수에 추가하지 않고 240044·240061의 품질 점검을 별도로 유지합니다.


In [ ]:
# 1단계: 계획수리·날짜를 사용하지 않고 전체 LOT 후보를 탐색합니다.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
S6_FEATURES=['가열·균열존','과시효대','냉각대','OP-Z3','CP-Z4','CP_sd','가열_P90-P10','OP_P90-P10']
S6_FAMILIES={'온도':['가열·균열존','과시효대','냉각대','가열_P90-P10'],'OP':['OP-Z3','OP_P90-P10'],'CP':['CP-Z4','CP_sd']}
S6_VARIABILITY={'CP_sd','가열_P90-P10','OP_P90-P10'}
S6_LABELS=['TC-Z3~TC-Z6 평균 중앙값\n[℃]',
 'TC-Z7·TC-Z8 평균 중앙값\n[℃]',
 'TC-OUT1·TC-OUT2 평균 중앙값\n[℃]',
 'OP-Z3 중앙값\n[%]',
 'CP-Z4 중앙값\n[무차원]',
 'CP-Z4 std\n[무차원]',
 'TC-Z3~TC-Z6 평균 변동폭\n[℃]',
 'OP-Z3 변동폭\n[%p]']
S6_NAMES=dict(zip(S6_FEATURES,[s.replace('\n',' ') for s in S6_LABELS]))
S6_TAIL=.20
s6_lots=lot_map.sort_values('start').copy()
s6_lots['grade_n']=s6_lots.grade.map(s6_lots.grade.value_counts())
s6_lots['support']=s6_lots.grade_n.map(lambda n:'기술 비교 (n≥3)' if n>=3 else '참고 비교 (n=2)' if n==2 else '비교 불가 (n=1)')
s6_analysis=analysis_ts.copy()
s6_analysis['progress_pct']=100*s6_analysis.elapsed_min/s6_analysis.LOT_NO.map(s6_lots.minutes)
# resample 경계 칸의 시각이 LOT 시작보다 앞서면 제외합니다.
s6_analysis=s6_analysis[s6_analysis.progress_pct.between(0,100)].copy()

def s6_metrics(frame):
    grouped=frame.groupby('LOT_NO')
    result=grouped[S6_FEATURES[:5]].median()
    result['CP_sd']=grouped['CP-Z4'].std()
    for feature,channel in [('가열_P90-P10','가열·균열존'),('OP_P90-P10','OP-Z3')]:
        result[feature]=grouped[channel].quantile(.9)-grouped[channel].quantile(.1)
    return result.reindex(s6_lots.index)[S6_FEATURES]

s6_windows={'전체':s6_analysis,'중간 20~80%':s6_analysis[s6_analysis.progress_pct.between(20,80)],'첫 10분':s6_analysis[s6_analysis.elapsed_min.lt(10)]}
s6_metrics_by_window={name:s6_metrics(frame) for name,frame in s6_windows.items()}
s6_full=s6_metrics_by_window['전체']; s6_middle=s6_metrics_by_window['중간 20~80%']
def s6_tail_flags(values,cut=S6_TAIL):
    rank=values.rank(pct=True,method='average')
    flags=pd.DataFrame({c:rank[c].ge(1-cut) if c in S6_VARIABILITY else rank[c].le(cut)|rank[c].ge(1-cut) for c in S6_FEATURES})
    families=pd.DataFrame({g:flags[cols].any(axis=1) for g,cols in S6_FAMILIES.items()})
    return rank,flags,families
s6_rank,s6_feature_flags,s6_flags=s6_tail_flags(s6_full)
_,_,s6_mid_flags=s6_tail_flags(s6_middle)
s6_screen=s6_lots[['grade','grade_n','support']].copy()
s6_screen['전체 신호군 수']=s6_flags.sum(axis=1); s6_screen['중간 신호군 수']=s6_mid_flags.sum(axis=1)
s6_screen['전체 후보']=s6_screen['전체 신호군 수'].ge(2); s6_screen['중간 후보']=s6_screen['중간 신호군 수'].ge(2)
s6_screen['탐색 후보']=s6_screen['전체 후보']|s6_screen['중간 후보']
s6_screen['전체 신호군']=[', '.join(s6_flags.columns[s6_flags.loc[lot]]) or '—' for lot in s6_screen.index]
s6_screen['중간 신호군']=[', '.join(s6_mid_flags.columns[s6_mid_flags.loc[lot]]) or '—' for lot in s6_screen.index]
s6_screen['OP-Z3 원본 결측행']=temp.groupby('LOT_NO')['OP-Z3'].apply(lambda s:int(s.isna().sum())).reindex(s6_screen.index)
# 값의 순위 후보와 별도로 OP-Z3 결측 LOT도 확인 대상으로 유지합니다.
s6_missing_rows = temp.loc[temp['OP-Z3'].isna()].groupby('LOT_NO').agg(
    first_sec=('sec','min'), last_sec=('sec','max'))
s6_screen['OP-Z3 첫 결측 [초]'] = s6_missing_rows.first_sec.reindex(s6_screen.index)
s6_screen['OP-Z3 마지막 결측 [초]'] = s6_missing_rows.last_sec.reindex(s6_screen.index)
s6_screen['결측 확인 대상'] = s6_screen['OP-Z3 원본 결측행'].gt(0)
s6_screen['값·품질 확인 대상'] = s6_screen['탐색 후보'] | s6_screen['결측 확인 대상']

def s6_value(value,c,delta=False):
    if pd.isna(value): return '—'
    digits=6 if c=='CP-Z4' else 4 if c=='CP_sd' else 2
    return format(value,('+' if delta else '')+f'.{digits}f')

S6_SIMPLE={'가열·균열존': 'TC-Z3~TC-Z6 평균',
 '과시효대': 'TC-Z7·TC-Z8 평균',
 '냉각대': 'TC-OUT1·TC-OUT2 평균',
 'OP-Z3': 'OP-Z3 출력',
 'CP-Z4': 'CP-Z4 수준',
 'CP_sd': 'CP-Z4 std',
 '가열_P90-P10': 'TC-Z3~TC-Z6 평균 변동폭',
 'OP_P90-P10': 'OP-Z3 변동폭'}
S6_UNITS={c:('℃' if c in ['가열·균열존','과시효대','냉각대','가열_P90-P10'] else '%p' if c=='OP_P90-P10' else '%' if c=='OP-Z3' else '') for c in S6_FEATURES}

def s6_reason(lot, group, window='중간 20~80%'):
    values = s6_metrics_by_window[window]
    ranks, flags, _ = s6_tail_flags(values)
    cols = [c for c in S6_FAMILIES[group] if flags.loc[lot,c]]
    if not cols:
        return '순위 표시 없음'
    selected = max(cols, key=lambda c:(c in S6_VARIABILITY, abs(ranks.loc[lot,c]-.5)))
    value = values.loc[lot,selected]
    reference = values[selected].median()  # 강종을 섞은 14 LOT의 같은 지표 중앙값
    direction = '큼' if selected in S6_VARIABILITY else '높음' if ranks.loc[lot,selected]>=.8 else '낮음'
    metric = 'CP-Z4 중앙값' if selected=='CP-Z4' else S6_SIMPLE[selected]
    unit = S6_UNITS[selected]
    delta_unit = '%p' if selected=='OP-Z3' else unit
    return (f'{metric} {direction}\n'
            f'LOT {s6_value(value,selected)}{unit} | 기준 {s6_value(reference,selected)}{unit}\n'
            f'차이 {s6_value(value-reference,selected,delta=True)}{delta_unit}')

# 검토 우선순위 색: 고장 확률이나 온도·출력의 높고 낮음을 나타내지 않습니다.
S6_REVIEW_STYLE = {
    '우선 확인': {'fg':'#B4232F','bg':'#FDEDEC','edge':'#E8B9B6'},
    '참고 확인': {'fg':'#16734A','bg':'#E9F5ED','edge':'#A6D4B7'},
    '기준 비해당': {'fg':'#66717D','bg':'#FFFFFF','edge':'#E4E7EB'},
}
# 표시용 범례와 별도로 비교 자료가 없을 때 사용할 기본 스타일입니다.
S6_MISSING_STYLE={'fg':'#7C8792','bg':'#F2F4F6','edge':'#D9DEE3'}

s6_a_reference=s6_middle.median()
s6_first_op_valid_sec=temp.loc[temp['OP-Z3'].notna()].groupby('LOT_NO').sec.min()
s6_a_family_state=pd.DataFrame(index=s6_lots.index,columns=list(S6_FAMILIES),dtype=object)
s6_a_lot_state=pd.Series(index=s6_lots.index,dtype=object)
for lot in s6_lots.index:
    for group,cols in S6_FAMILIES.items():
        has_data=s6_middle.loc[lot,cols].notna().any()
        state=('우선 확인' if group=='OP' and s6_screen.loc[lot,'결측 확인 대상'] else
               '자료 없음' if not has_data else
               ('우선 확인' if s6_screen.loc[lot,'탐색 후보'] else '참고 확인') if s6_mid_flags.loc[lot,group] else '기준 비해당')
        s6_a_family_state.loc[lot,group]=state
    s6_a_lot_state.loc[lot]=('우선 확인' if (s6_screen.loc[lot,'결측 확인 대상'] or s6_screen.loc[lot,'탐색 후보']) else
        '참고 확인' if max(s6_screen.loc[lot,'중간 신호군 수'],s6_screen.loc[lot,'전체 신호군 수'])>0 else
        '기준 비해당' if s6_full.loc[lot].notna().any() else '자료 없음')

a_ids=s6_lots.index.tolist();a_pages=[a_ids[i:i+7] for i in range(0,len(a_ids),7)]
a_figures=[];a_row_step=1.45
for a_page_number,a_page_ids in enumerate(a_pages,start=1):
    a_fig,a_axes=plt.subplots(1,4,figsize=(17.5,10.6),sharey=True,
        gridspec_kw={'width_ratios':[1.25,1.25,1.25,1.12]})
    a_figures.append(a_fig);a_y=np.arange(len(a_page_ids))*a_row_step
    for a_j,a_group in enumerate(['온도','OP','CP']):
        a_ax=a_axes[a_j]
        for a_i,a_lot in enumerate(a_page_ids):
            a_center=a_y[a_i];a_state=s6_a_family_state.loc[a_lot,a_group]
            a_style=S6_REVIEW_STYLE.get(a_state,S6_MISSING_STYLE).copy()
            if a_state=='기준 비해당': a_style.update(bg='white',fg='#9AA2AA',edge='#ECEFF2')
            a_ax.add_patch(plt.Rectangle((.01,a_center-.56),.98,1.12,
                fc=a_style['bg'],ec=a_style['edge'],lw=.7))
            if a_state in ['자료 없음','기준 비해당']:
                a_ax.text(.5,a_center,'-',
                    ha='center',va='center',fontsize=12,color=a_style['fg'])
                continue
            a_ax.add_patch(plt.Rectangle((.01,a_center-.56),.016,1.12,
                fc=a_style['fg'],ec='none'))
            if a_group=='OP' and s6_screen.loc[a_lot,'결측 확인 대상']:
                a_metric='OP-Z3 초반 결측'
                a_n=int(s6_screen.loc[a_lot,'OP-Z3 원본 결측행'])
                a_first=s6_screen.loc[a_lot,'OP-Z3 첫 결측 [초]']
                a_last=s6_screen.loc[a_lot,'OP-Z3 마지막 결측 [초]']
                a_values=f'{a_n}행 · {a_first:.0f}~{a_last:.0f}초'
                a_difference=f'{s6_first_op_valid_sec.get(a_lot,np.nan):.0f}초부터 유효 기록'
            else:
                a_metric,a_values,a_difference=s6_reason(a_lot,a_group).split('\n')
            a_ax.text(.065,a_center-.29,a_metric,ha='left',va='center',fontsize=13,
                fontweight='bold',color='#182b3a')
            a_ax.text(.065,a_center,a_values,ha='left',va='center',fontsize=12.2,
                color=a_style['fg'] if a_state=='우선 확인' else '#33475b')
            a_ax.text(.065,a_center+.29,a_difference,ha='left',va='center',fontsize=12.3,
                fontweight='bold',color=a_style['fg'])
        a_ax.set_title({'온도':'온도','OP':'제어출력(OP-Z3)','CP':'분위기(CP-Z4)'}[a_group],
            loc='left',fontweight='bold',pad=22,fontsize=14)
        a_ax.set_xlim(0,1);a_ax.set_xticks([]);a_ax.set_ylim(a_y[-1]+.73,-.73);a_ax.grid(False)
        a_ax.spines[['top','right','bottom','left']].set_visible(False)
    a_ax=a_axes[3]
    for a_i,a_lot in enumerate(a_page_ids):
        a_center=a_y[a_i];a_state=s6_a_lot_state.loc[a_lot];a_style=S6_REVIEW_STYLE.get(a_state,S6_MISSING_STYLE)
        a_middle_count=int(s6_screen.loc[a_lot,'중간 신호군 수'])
        a_full_count=int(s6_screen.loc[a_lot,'전체 신호군 수'])
        a_counts=f'중간 {a_middle_count}개 · 전체 {a_full_count}개'
        if s6_screen.loc[a_lot,'결측 확인 대상']:
            a_status='① 우선 확인 · 결측';a_next='아래 결측 비교에서 원인 점검'
        elif s6_screen.loc[a_lot,'탐색 후보'] or a_state=='참고 확인':
            a_status=('② 후속 확인 · 중간 후보' if s6_screen.loc[a_lot,'중간 후보'] else
                '② 후속 확인 · 전체 후보' if s6_screen.loc[a_lot,'전체 후보'] else '③ 참고 · 단일 신호군')
            a_next='같은 강종 비교' if s6_screen.loc[a_lot,'중간 후보'] else '시작·끝 변화 확인' if s6_screen.loc[a_lot,'전체 후보'] else '같은 강종 차이도 확인'
            if s6_lots.loc[a_lot,'grade_n']==1:a_next='n=1: 같은 강종 추가 LOT 필요'
        else:
            a_status='후보 기준 비해당' if a_state=='기준 비해당' else '평가할 자료 없음'
            a_next=''
        a_ax.add_patch(plt.Rectangle((.01,a_center-.56),.98,1.12,
            fc=a_style['bg'],ec=a_style['edge'],lw=.7))
        a_ax.text(.055,a_center-.29,a_status,ha='left',va='center',fontsize=12.8,
            fontweight='bold',color=a_style['fg'])
        a_ax.text(.055,a_center,a_counts,ha='left',va='center',fontsize=12.2,color='#33475b')
        if a_next:a_ax.text(.055,a_center+.29,a_next,ha='left',va='center',fontsize=11.4,color=a_style['fg'])
    a_ax.set_title('확인 우선순위',loc='left',fontweight='bold',pad=22,fontsize=14)
    a_ax.set_xlim(0,1);a_ax.set_xticks([]);a_ax.grid(False)
    a_ax.spines[['top','right','bottom','left']].set_visible(False)
    a_axes[0].set_yticks(a_y,[f"{lot} · {s6_lots.loc[lot,'grade']} (n={s6_lots.loc[lot,'grade_n']})"
        for lot in a_page_ids],fontsize=12)
    a_axes[0].tick_params(axis='y',length=0,pad=14)
    for tick,lot in zip(a_axes[0].get_yticklabels(),a_page_ids):tick.set_color(colors[s6_lots.loc[lot,'grade']])
    for ax in a_axes[1:]:ax.tick_params(axis='y',left=False,labelleft=False)
    a_fig.suptitle(f'그림 10-A-{a_page_number}. 전체 LOT 후보 탐색 — {a_page_number}/{len(a_pages)}',
        x=.03,y=.974,ha='left',fontweight='bold',fontsize=18)
    a_fig.text(.03,.922,'기준: 전체 14 LOT 중앙값 · 중간: LOT 진행률 20~80% · 값의 순위 후보와 결측 확인을 구분',
        fontsize=12,color='#425466')
    a_fig.legend(handles=[Patch(fc=style['bg'],ec=style['fg'],label=state)
        for state,style in S6_REVIEW_STYLE.items()],loc='upper left',bbox_to_anchor=(.025,.89),
        ncol=4,frameon=False,fontsize=11)
    a_fig.text(.03,.053,'빨강: 우선·중요 후속 확인 · 초록: 단일군 참고 · 신호군의 흰색 -: 기준 비해당 · 회색 -: 유효값 없음',
        fontsize=11.3,color='#425466')
    a_fig.subplots_adjust(left=.205,right=.989,top=.78,bottom=.11,wspace=.11)
    save_and_show_figures()
s6_sensitivity=pd.DataFrame({f'상하위 {int(100*cut)}%':s6_tail_flags(s6_full,cut)[2].sum(axis=1).ge(2)|s6_tail_flags(s6_middle,cut)[2].sum(axis=1).ge(2) for cut in [.15,.20,.25]})


#### 그림 10-A · 전체 후보 선정 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 온도·OP·CP 칸 | 표시 컬럼의 값과 전체 14 LOT 기준을 비교. 흰색 ‘-’는 해당 순위 기준 미만. |
| 오른쪽 확인 우선순위 | ① 초반 결측 점검, ② 다중신호 후속 비교, ③ 단일 신호군 참고. 빨강은 확인할 항목이지 고장 판정이 아님. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| **전체 후보 수** | 전체 또는 중간에서 2개 이상 신호군 **11 LOT** | 강종 효과가 섞인 넓은 탐색 결과. |
| **240044·240061 / OP-Z3** | 초반 결측 **75 / 179행**; 240044는 다중신호 후보 외 추가 점검 | 다중신호 후보와 수집 품질 확인을 합하면 **12 LOT**. |
| **240091 / TC-Z7·TC-Z8** | 온도 한 신호군만 다름 | 단일군 참고 대상도 유지. |
| 240001 · SGCC | 현재 다중신호 후보 기준 미만; n=1 | 정상 기준으로 고정할 근거는 부족. |

**결론:** 다음 단계는 **자기 자신을 제외한 동일 강종 LOT 비교**입니다. 평균온도 차이는 개별 TC로 나누고, 시작·끝과 중간 20~80%를 구분합니다.

**③ 후속 작업 — 무엇을 누구와 비교할 것인가?**

| 확인 대상 · 강종 | 비교 대상 | 비교할 컬럼·항목 | 그림에서 알 수 있는 점 | 다음에 확인해야 할 점 |
|---|---|---|---|---|
| **240044 · SPHC** | **240061·240069·240091**의 수집 시작 구간 | **OP-Z3 결측** ↔ 같은 시각의 **TC-Z3·OP-Z4·OP-Z5·OP-Z6** 기록 | OP-Z3 초반 **75행(0~81초)** 결측 | 다른 채널은 기록됐는가? 240061과 공통된 채널 초기화·수집 조건이 있는가? 태그·통신 로그 대조 |
| **240061 · SPHC** | **240044·240069·240091** | **OP-Z3 결측·변동폭**, **TC-Z3·TC-Z4·TC-Z5·TC-Z6**의 수준·변동폭 | OP-Z3 **179행(0~183초)** 결측, TC-Z3~TC-Z6 평균 변동폭 큼 | 결측과 승온을 먼저 분리. 승온 이후에도 TC·OP 차이가 남는지 확인 |
| **240069 · SPHC** | **240044·240061·240091** | **CP-Z4 중앙값·std**, **TC-Z3·TC-Z4·TC-Z5·TC-Z6** | CP-Z4 std 큼, TC-Z3~TC-Z6 평균 낮음 | CP-Z4 차이가 일시적 피크인가, 지속 변동인가? 평균온도의 차이가 어느 TC에서 나타나는가? |
| **240121 · SPCE** | **240037·240133** | **TC-Z7·TC-Z8 각각의 중앙값**, **OP-Z3 중앙값** | TC-Z7·TC-Z8 평균과 OP-Z3 수준 높음 | 같은 SPCE에서도 두 TC와 OP 차이가 남는가? 중간 구간에 지속되는가? 목표온도·속도·두께 대조 |
| **240091 · SPHC** | **240044·240061·240069** | **TC-Z7·TC-Z8 각각의 중앙값** | 온도 한 신호군만 높은 참고 항목 | 단일군이어도 수준 차이가 지속되는가? 두 존의 목표온도·운전조건 대조 |
| **240024 ↔ 240108 · SPFH590** | **서로 비교 · n=2 참고** | **TC-Z7·TC-Z8**, **TC-Z3~TC-Z6 평균**, **CP-Z4 중앙값·std** | 240024는 TC-Z7·TC-Z8 평균 낮음·CP-Z4 중앙값 낮음. 240108은 TC-Z3~TC-Z6 평균 높음·CP-Z4 std 큼 | 두 LOT에 공통되는 온도 수준은 강종 조건인지 확인. 작은 CP 수준 차이와 std 차이를 구분 |
| **240038 ↔ 240114 · SPCEN** | **서로 비교 · n=2 참고** | **TC-Z3~TC-Z6 평균 변동폭**, **TC-Z7·TC-Z8**, **OP-Z3 변동폭** | 240038은 TC·OP 변동폭 큼. 240114는 TC-Z7·TC-Z8 평균 높고 전체 구간에서 추가 후보 | 시작·종료 전환을 제외해도 차이가 남는가? 두 LOT의 목표값·가동상태 대조 |
| **240037 · SPCE** | **240121·240133** | **TC-OUT1·TC-OUT2**, **CP-Z4 중앙값·std** | 전체 구간에서는 후보지만 중간 구간의 표시 신호군은 0개 | 시작·끝의 CP 피크나 온도 수준 차이 때문에 선정됐는가? 중간 비교에서도 차이가 남는지 확인 |
| **240133 · SPCE** | **240037·240121** | **OP-Z3 중앙값**, **CP-Z4 중앙값·std** | OP-Z3와 CP-Z4 중앙값 높음 | 같은 강종에서도 실제 차이가 큰가? CP의 작은 순위 차이를 과장하지 않고 확인 |
| **240058 · DP590** | 동일 강종 LOT **없음** → 본 LOT의 중간·종료 구간, 추가 DP590 LOT | **TC-Z3·TC-Z4·TC-Z5·TC-Z6**, **OP-Z3**, **CP-Z4 std** | TC-Z3~TC-Z6 평균 변동폭·CP-Z4 std 큼 | 종료부에서 TC·OP·CP가 함께 하강하는지 확인. 현재 표본으로 강종 조건과 LOT 특이성을 분리할 수 없음 |
| **240117 · SPCD** | 동일 강종 LOT **없음** → 본 LOT의 시작·중간 구간, 추가 SPCD LOT | **OP-Z3 변동폭**, **TC-Z3·TC-Z4·TC-Z5·TC-Z6** | OP-Z3 변동폭 큼, 전체 구간에서 추가 후보 | 초반 승온과 OP-Z3 변화가 겹치는지 확인. SPCD 내부 비교를 위한 추가 LOT 필요 |

<a id="s6-missing-pair"></a>
**먼저 볼 원본 비교:** 아래 두 그림에서 240044·240061의 수집 시작 구간을 봅니다. 회색은 OP-Z3 결측 구간, 붉은 점은 OP-Z3 최초 유효값입니다. TC-Z3와 OP-Z4·OP-Z5·OP-Z6의 동시 기록을 함께 확인합니다.

>240038·240114·240037·240133·240117은 현재 요약 비교 중심이며, 개별 확대 시계열이 모두 있는 것은 아닙니다. 목표값·속도·두께·통신 로그 대조도 아직 수행하지 않은 추가 확인 사항입니다.

**어느 시간을 확대할지는 [구간별 선정 이유·근거 점검표](#interval-review)에 정리했습니다.** 240061의 승온 후 TC-Z3↔OP-Z3, 240069의 CP 피크, 정비 전후 창 길이도 실제 관측으로 추가 확인합니다.


In [ ]:
# OP-Z3 결측 두 LOT를 같은 기준으로 비교합니다. 원자료를 보간하거나 0으로 채우지 않습니다.
s6_quality_lots = s6_screen.index[s6_screen['결측 확인 대상']].tolist()
s6_quality_pair_rows = []
s6_quality_raw = {}
for quality_lot in s6_quality_lots:
    quality_all = temp[temp.LOT_NO.eq(quality_lot)].sort_values('MEAS_DT').copy()
    quality_absent = quality_all[quality_all['OP-Z3'].isna()]
    quality_valid = quality_all[quality_all['OP-Z3'].notna()]
    quality_first = quality_valid.iloc[0]
    quality_after = quality_all[quality_all.sec.ge(quality_first.sec)]
    s6_quality_pair_rows.append({
        'LOT':int(quality_lot), '강종':s6_lots.loc[quality_lot,'grade'],
        'OP-Z3 결측 [행]':len(quality_absent),
        '첫 결측 [초]':quality_absent.sec.min(), '마지막 결측 [초]':quality_absent.sec.max(),
        '최초 유효값 [초]':quality_first.sec, '최초 OP-Z3 [%]':quality_first['OP-Z3'],
        '결측 동안 TC-Z3 관측 [행]':int(quality_absent['TC-Z3'].notna().sum()),
        '결측 동안 OP-Z4 관측 [행]':int(quality_absent['OP-Z4'].notna().sum()),
        '결측 동안 OP-Z5 관측 [행]':int(quality_absent['OP-Z5'].notna().sum()),
        '결측 동안 OP-Z6 관측 [행]':int(quality_absent['OP-Z6'].notna().sum()),
        '유효값 시작 후 추가 결측 [행]':int(quality_after['OP-Z3'].isna().sum()),
    })
    s6_quality_raw[quality_lot] = quality_all[quality_all.elapsed_min.lt(8)].copy()
s6_quality_pair = pd.DataFrame(s6_quality_pair_rows).set_index('LOT')
s6_raw_progress = 100*temp.elapsed_min/temp.LOT_NO.map(s6_lots.minutes)
s6_quality_middle = temp[s6_raw_progress.between(20,80)].groupby('LOT_NO')['OP-Z3'].apply(
    lambda s:int(s.isna().sum())).reindex(s6_lots.index,fill_value=0)
s6_quality_pair['중간 20~80% 결측 [행]'] = s6_quality_middle.reindex(s6_quality_pair.index)

# LOT마다 넓은 그림 한 장: 위는 온도·출력, 아래는 같은 시각의 채널별 기록 여부입니다.
for quality_lot in s6_quality_lots:
    quality_d = s6_quality_raw[quality_lot]
    quality_r = s6_quality_pair.loc[quality_lot]
    quality_fig, quality_axes = plt.subplots(2,1,figsize=(15.5,7.5),sharex=True,
        gridspec_kw={'height_ratios':[3.3,1.4]})
    quality_ax, quality_availability = quality_axes
    quality_right = quality_ax.twinx()
    for quality_i,(_,quality_segment) in enumerate(quality_d.groupby('SEG')):
        quality_ax.plot(quality_segment.elapsed_min,quality_segment['TC-Z3'],
            color='#40566c',lw=1.6,label='TC-Z3 온도 [℃]' if quality_i==0 else None)
        quality_right.plot(quality_segment.elapsed_min,quality_segment['OP-Z3'],
            color='#2874a6',lw=1.4,label='OP-Z3 출력 [%]' if quality_i==0 else None)
    for first_sec,last_sec,missing_n in missing_runs(quality_lot,'OP-Z3'):
        quality_ax.axvspan(first_sec/60,(last_sec+1)/60,fc='#dce1e6',alpha=.8,lw=0,zorder=0)
    quality_first_min = quality_r['최초 유효값 [초]']/60
    quality_right.scatter([quality_first_min],[quality_r['최초 OP-Z3 [%]']],
        color='#b4232f',s=50,zorder=5)
    for quality_axis in [quality_ax,quality_availability]:
        quality_axis.axvline(quality_first_min,color='#b4232f',ls='--',lw=1,alpha=.8)
    quality_ax.set_ylabel('TC-Z3 온도 [℃]',color='#40566c',fontsize=12)
    quality_right.set_ylabel('OP-Z3 출력 [%]',color='#2874a6',fontsize=12)
    quality_right.set_ylim(0,105)
    quality_ax.set_xlim(0,8); quality_ax.grid(alpha=.16)
    quality_right.grid(False)
    quality_ax.spines['top'].set_visible(False)
    quality_right.spines['top'].set_visible(False)
    quality_channels=['TC-Z3','OP-Z3','OP-Z4','OP-Z5','OP-Z6']
    for channel_y,quality_channel in enumerate(quality_channels):
        channel_missing=quality_d[quality_channel].isna()
        quality_availability.scatter(quality_d.loc[~channel_missing,'elapsed_min'],
            np.full((~channel_missing).sum(),channel_y),marker='|',s=100,
            linewidths=1.5,color='#81929f')
        quality_availability.scatter(quality_d.loc[channel_missing,'elapsed_min'],
            np.full(channel_missing.sum(),channel_y),marker='|',s=100,
            linewidths=1.7,color='#b4232f')
    quality_availability.set_yticks(range(len(quality_channels)),quality_channels,fontsize=11)
    for channel_label in quality_availability.get_yticklabels():
        if channel_label.get_text()=='OP-Z3':
            channel_label.set_color('#b4232f'); channel_label.set_fontweight('bold')
    quality_availability.set_ylim(len(quality_channels)-.6,-.6)
    quality_availability.set_xlabel('LOT 시작 후 경과 [분]',fontsize=12)
    quality_availability.set_title('같은 시각의 기록 여부 · 회색: 관측 / 빨강: 결측',
        loc='left',fontsize=12,fontweight='bold',pad=10)
    quality_availability.spines[['top','right','left']].set_visible(False)
    quality_availability.tick_params(axis='y',length=0)
    quality_fig.suptitle(f'그림 10-A-품질-{quality_lot}. OP-Z3 초반 결측 원인 점검 · '+
        f"LOT {quality_lot} · {quality_r['강종']}",
        x=.08,y=.98,ha='left',fontweight='bold',fontsize=16)
    quality_fig.text(.08,.925,
        f"OP-Z3 결측 {quality_r['OP-Z3 결측 [행]']:.0f}행 · "
        f"{quality_r['첫 결측 [초]']:.0f}~{quality_r['마지막 결측 [초]']:.0f}초 · "
        f"최초 유효값: {quality_r['최초 유효값 [초]']:.0f}초 / {quality_r['최초 OP-Z3 [%]']:.2f}%",
        fontsize=12,color='#b4232f',fontweight='bold')
    quality_fig.legend(handles=[
        Line2D([],[],color='#40566c',lw=1.6,label='TC-Z3 · 왼쪽 축 [℃]'),
        Line2D([],[],color='#2874a6',lw=1.4,label='OP-Z3 · 오른쪽 축 [%]'),
        Patch(fc='#dce1e6',label='OP-Z3 결측 구간'),
        Line2D([],[],color='#b4232f',marker='o',ls='--',label='OP-Z3 최초 유효값 시각')],
        loc='upper left',bbox_to_anchor=(.074,.88),ncol=4,frameon=False,fontsize=10.5)
    quality_fig.text(.08,.025,'결측 동안에도 TC-Z3와 OP-Z4·Z5·Z6는 모두 기록됩니다. 두 축은 단위가 다르며 결측 OP 값을 0%로 해석하지 않습니다.',
        fontsize=10.5,color='#425466')
    quality_fig.subplots_adjust(left=.08,right=.92,top=.78,bottom=.12,hspace=.46)
    save_and_show_figures()

display(s6_quality_pair.round(4))


#### OP-Z3 결측 확대 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 각 LOT의 위패널 | 초반 8분에서 TC-Z3와 OP-Z3 기록 시작 시각을 확인. 회색은 OP-Z3 결측, 붉은 표시는 최초 유효값. |
| 아래 기록 유무 | 같은 시각 TC-Z3·OP-Z4·OP-Z5·OP-Z6는 있었는지 확인. ‘값 없음’과 출력 0%를 구분. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 확인 항목 | 240044 · SPHC | 240061 · SPHC | 의미 |
|---|---|---|---|
| 결측 행·시각 | **75행 · 0~81초** | **179행 · 0~183초** | 두 LOT 모두 수집 시작부터 OP-Z3 결측 |
| 결측 동안 TC-Z3 기록 | **75/75행** | **179/179행** | 온도 계측은 같은 시각에 계속 관측됨 |
| 결측 동안 OP-Z4·Z5·Z6 기록 | 각각 **75/75행** | 각각 **179/179행** | 다른 제어출력 채널도 기록되어 전체 수집 중단 패턴과 다름 |
| OP-Z3 최초 유효값 | **82초 · 98.83%** | **184초 · 100.00%** | 몇 분 뒤 OP-Z3 수집이 시작됨. 비어 있던 값을 0%로 해석할 수 없음 |
| 유효값 시작 후 추가 결측 | **0행** | **0행** | 해당 LOT의 결측은 시작 구간에 집중 |
| 중간 20~80% 결측 | **0행** | **0행** | 초반 결측을 분리하고 유효 중간 구간의 온도·출력 차이는 계속 비교 |

**결론:** 두 LOT 모두 **OP-Z3만 초반에 길게 비었다가 수집이 시작**됩니다. 큰 승온이 없는 240044에서도 반복되고, SPHC 240069·240091에는 이 결측이 없으므로 재가동이나 강종 자체를 원인으로 확정하지 않습니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240044 ↔ 240061 | 82초·184초 전후 OP-Z3 유효화 과정과 다른 OP 채널 대조 | 태그 초기화·PLC 연결·수집기·저장 전 메시지 로그 |
| 240044·240061 ↔ 240069·240091 | 결측 없는 SPHC와 LOT 시작·채널 초기 조건 비교 | 강종변경·수집 트리거·태그 품질 상태 |
| 240061 ↔ 나머지 SPHC | 결측은 제외하고 유효 중간 구간의 TC·OP 차이 계속 확인 | 다음 10-B·10-D; C(결측 점검 우선) 유지 |

**원인 확인 상태:** CSV와 그림으로 채널·시점을 좁혔고, 실제 초기화·통신·저장 원인은 아직 로그 대조가 필요합니다.


### ② 동일 강종 비교 — 순위 대신 실제 차이 확인

**읽는 순서:** 같은 강종 LOT끼리 묶인 행 → 기준선 → 점 옆의 실제 차이. 온도·OP는 **0**이 비교 중앙값과 같음을 뜻합니다. CP-Z4 std는 **1배**가 비교값과 같음을 뜻합니다. 붉은 테두리는 n≥3의 차이 기준, 초록 테두리는 n=2의 참고 차이이며 흰색 바탕의 회색 테두리 점은 차이 기준 미만입니다. n=1은 점 그래프에서 제외하고 마지막 전체 확인표에 남깁니다.


In [ ]:
# 2단계: 동일 강종 자기 제외 중앙값. 모든 LOT·모든 창에 같은 계산을 적용합니다.
s6_peer_ids={lot:s6_lots.index[s6_lots.grade.eq(r.grade)&s6_lots.index.to_series().ne(lot)].tolist() for lot,r in s6_lots.iterrows()}
s6_peer_ref={}; s6_delta={}
for window,values in s6_metrics_by_window.items():
    ref=pd.DataFrame(np.nan,index=values.index,columns=S6_FEATURES)
    for lot,peers in s6_peer_ids.items():
        if peers: ref.loc[lot]=values.loc[peers].median()
    s6_peer_ref[window]=ref; s6_delta[window]=values-ref
# 수준: 전체 robust scale과 LOT 내 SD 중앙값 중 큰 값. 작은 CP 중앙값 차이 과장 방지.
s6_scale=1.4826*(s6_full-s6_full.median()).abs().median()
s6_scale=s6_scale.where(s6_scale.gt(0),s6_full.std()).replace(0,np.nan)
s6_level_floor={c:max(float(s6_scale[c]),float(s6_analysis.groupby('LOT_NO')[c].std().median())) for c in S6_FEATURES[:5]}
def s6_difference_flags(window):
    values,ref,delta=s6_metrics_by_window[window],s6_peer_ref[window],s6_delta[window]
    flags=pd.DataFrame(False,index=values.index,columns=S6_FEATURES)
    for c in S6_FEATURES:
        if c in S6_VARIABILITY:
            ratio=values[c].div(ref[c].replace(0,np.nan))
            flags[c]=(ratio.ge(2)|ratio.le(.5))&delta[c].abs().ge(s6_scale[c])
        else:
            flags[c]=delta[c].abs().ge(s6_level_floor[c])
            for lot,peers in s6_peer_ids.items():
                if len(peers)>=2:
                    pv=values.loc[peers,c].dropna(); value=values.loc[lot,c]
                    distance=max(pv.min()-value,value-pv.max(),0) if len(pv) else np.nan
                    flags.loc[lot,c]=bool(flags.loc[lot,c] and distance>=s6_level_floor[c])
    return flags&ref.notna()
s6_diff_full=s6_difference_flags('전체'); s6_diff_middle=s6_difference_flags('중간 20~80%')
s6_peer_families=pd.DataFrame({g:s6_diff_middle[cols].any(axis=1) for g,cols in S6_FAMILIES.items()})
s6_delta_scale=s6_delta['중간 20~80%'].abs().max().replace(0,np.nan)
s6_comparison=pd.DataFrame([{'LOT':lot,'강종':r.grade,'강종 LOT수':r.grade_n,'비교 LOT':', '.join(map(str,s6_peer_ids[lot])) or '없음','지표':S6_NAMES[c],
    '전체 값':s6_full.loc[lot,c],'전체 Δ':s6_delta['전체'].loc[lot,c],'첫 10분 Δ':s6_delta['첫 10분'].loc[lot,c],
    '중간 값':s6_middle.loc[lot,c],'중간 비교 중앙값':s6_peer_ref['중간 20~80%'].loc[lot,c],'중간 Δ':s6_delta['중간 20~80%'].loc[lot,c],
    '중간 차이 확인':bool(s6_diff_middle.loc[lot,c]),'비교 수준':r.support} for lot,r in s6_lots.iterrows() for c in S6_FEATURES])
# 실제 시간의 5분 이동 SD: LOT·SEG 경계를 넘지 않습니다.
s6_analysis['CP_roll_sd']=np.nan
for (_,seg),d in s6_analysis.groupby(['LOT_NO','SEG']):
    s6_analysis.loc[d.index,'CP_roll_sd']=d.set_index('MEAS_DT')['CP-Z4'].rolling('5min',min_periods=5).std().to_numpy()

# 변동폭의 시간 지속성: 동일한 최근 5분 안에서 P90-P10을 계산합니다.
for out_col,in_col in [('가열_roll_width','가열·균열존'),('OP_roll_width','OP-Z3')]:
    s6_analysis[out_col]=np.nan
    for (_,seg),d in s6_analysis.groupby(['LOT_NO','SEG']):
        rolling=d.set_index('MEAS_DT')[in_col].rolling('5min',min_periods=5)
        s6_analysis.loc[d.index,out_col]=(rolling.quantile(.9)-rolling.quantile(.1)).to_numpy()
S6_TRACE_VARIABILITY={'CP_roll_sd','가열_roll_width','OP_roll_width'}
S6_TRACE_NAMES={'가열·균열존': 'TC-Z3~TC-Z6 평균',
 '과시효대': 'TC-Z7·TC-Z8 평균',
 '냉각대': 'TC-OUT1·TC-OUT2 평균',
 'OP-Z3': 'OP-Z3 출력',
 'CP-Z4': 'CP-Z4 수준',
 'CP_roll_sd': 'CP-Z4 std(5분)',
 '가열_roll_width': 'TC-Z3~TC-Z6 평균 변동폭(5분)',
 'OP_roll_width': 'OP-Z3 변동폭(5분)'}
# 진행률 5% 칸마다 LOT별 중앙값 → 비교 LOT 중앙값. 시간 보간·외삽 없이 동일 가중 비교.
s6_analysis['progress_bin']=np.minimum((s6_analysis.progress_pct/5).astype(int),19)
s6_trace_cols=['가열·균열존','과시효대','냉각대','OP-Z3','CP-Z4','CP_roll_sd','가열_roll_width','OP_roll_width']
s6_bin_values={}; s6_bin_ref={}; s6_bin_delta={}
for c in s6_trace_cols:
    grouped=s6_analysis.groupby(['LOT_NO','progress_bin'])[c]
    matrix=grouped.median().where(grouped.count().ge(3)).unstack('progress_bin').reindex(index=s6_lots.index,columns=range(20))
    ref=matrix.copy()*np.nan
    for lot,peers in s6_peer_ids.items():
        if peers:
            count=matrix.loc[peers].notna().sum()
            ref.loc[lot]=matrix.loc[peers].median().where(count.ge(2 if len(peers)>=2 else 1))
    s6_bin_values[c]=matrix; s6_bin_ref[c]=ref; s6_bin_delta[c]=matrix-ref
s6_persistence_rows=[]
for lot,r in s6_lots.iterrows():
    for c in s6_trace_cols:
        delta=s6_bin_delta[c].loc[lot]; target=s6_bin_values[c].loc[lot]; ref=s6_bin_ref[c].loc[lot]
        usable=delta.notna()&pd.Series([20<=5*b+2.5<=80 for b in range(20)],index=range(20))
        if c in S6_TRACE_VARIABILITY:
            ratio=target.div(ref.replace(0,np.nan)); high=ratio.ge(2); low=ratio.le(.5)
        else: high=delta.ge(s6_level_floor[c]); low=delta.le(-s6_level_floor[c])
        high_n,low_n=int((high&usable).sum()),int((low&usable).sum()); count=max(high_n,low_n); total=int(usable.sum())
        pct=100*count/total if total else np.nan
        s6_persistence_rows.append({'LOT':lot,'강종':r.grade,'채널':c,'비교 수준':r.support,'유효 중간 칸':total,'같은 방향 차이 칸':count,
            '방향':('높음' if high_n>=low_n else '낮음') if count else '—','중간 지속 비율 [%]':pct,'지속 후보':bool(total>=8 and pct>=70),'중간 Δ 중앙값':delta[usable].median()})
s6_persistence=pd.DataFrame(s6_persistence_rows)


# 3단계 핵심 요약: 비교 전후에 남는 신호군을 한 행에서 대조합니다.
# 계획수리 날짜와 수리 인근 여부는 이 계산에 사용하지 않습니다.
from matplotlib.colors import ListedColormap,BoundaryNorm
from matplotlib.patches import Rectangle

# 동일 존의 온도-출력 비교: 전체 후보 선별 기준은 유지하고 상세 그림만 원본 컬럼으로 확인합니다.
S6_ZONE_COLUMNS = ['TC-Z7', 'OP-Z7', 'TC-Z8', 'OP-Z8']
s6_zone_middle = s6_windows['중간 20~80%'].groupby('LOT_NO')[S6_ZONE_COLUMNS].median().reindex(s6_lots.index)
s6_zone_reference = s6_zone_middle.copy() * np.nan
s6_zone_bin_values = {}
s6_zone_bin_reference = {}
for zone_col in S6_ZONE_COLUMNS:
    grouped = s6_analysis.groupby(['LOT_NO', 'progress_bin'])[zone_col]
    values = grouped.median().where(grouped.count().ge(3)).unstack('progress_bin').reindex(index=s6_lots.index, columns=range(20))
    references = values.copy() * np.nan
    for zone_lot, zone_peers in s6_peer_ids.items():
        if zone_peers:
            s6_zone_reference.loc[zone_lot, zone_col] = s6_zone_middle.loc[zone_peers, zone_col].median()
            references.loc[zone_lot] = values.loc[zone_peers].median().where(
                values.loc[zone_peers].notna().sum().ge(2 if len(zone_peers) >= 2 else 1))
    s6_zone_bin_values[zone_col] = values
    s6_zone_bin_reference[zone_col] = references
s6_zone_case_stats = pd.DataFrame([
    {'LOT': zone_lot, '강종': s6_lots.loc[zone_lot, 'grade'], '컬럼': zone_col,
     '중간 중앙값': s6_zone_middle.loc[zone_lot, zone_col],
     '비교 중앙값': s6_zone_reference.loc[zone_lot, zone_col],
     '차이': s6_zone_middle.loc[zone_lot, zone_col] - s6_zone_reference.loc[zone_lot, zone_col],
     '단위': '℃' if zone_col.startswith('TC') else '%p'}
    for zone_lot in [240121, 240091] for zone_col in S6_ZONE_COLUMNS
])


In [ ]:

# 이 셀에서 실제 차이를 직접 그립니다.
b_cols=['가열·균열존', '과시효대', '냉각대', '가열_P90-P10']
b_title='그림 10-B-온도. 같은 강종과 비교한 온도 차이'
b_ids=s6_lots[s6_lots.grade_n.ge(2)].sort_values(['grade','start']).index.tolist()
b_height=4.2*len(b_cols)+2.3
b_fig,b_axes=plt.subplots(len(b_cols),1,figsize=(16,b_height),squeeze=False)
b_axes=b_axes[:,0]
b_fig.subplots_adjust(left=.20,right=.98,top=1-1.9/b_height,bottom=.95/b_height,hspace=.78)
b_fig.suptitle(b_title,x=.20,y=1-.12/b_height,ha='left',va='top',fontweight='bold',fontsize=15)
b_fig.legend(handles=[
    Line2D([0],[0],marker='o',color='none',markerfacecolor=S6_REVIEW_STYLE['우선 확인']['bg'],markeredgecolor=S6_REVIEW_STYLE['우선 확인']['fg'],markeredgewidth=2,label='n≥3 · 차이 기준 해당'),
    Line2D([0],[0],marker='o',color='none',markerfacecolor=S6_REVIEW_STYLE['참고 확인']['bg'],markeredgecolor=S6_REVIEW_STYLE['참고 확인']['fg'],markeredgewidth=2,label='n=2 · 참고 차이'),
    Line2D([0],[0],marker='o',color='none',markerfacecolor=S6_REVIEW_STYLE['기준 비해당']['bg'],markeredgecolor=S6_REVIEW_STYLE['기준 비해당']['fg'],label='차이 기준 미만')
],loc='upper left',bbox_to_anchor=(.20,1-.50/b_height),ncol=3,frameon=False,fontsize=9)
b_fig.text(.20,1-1.07/b_height,'같은 구간 비교: 중간 20~80% · 자신을 제외한 동일 강종 LOT 중앙값이 기준 · n=1은 아래 전체 확인표에서 보류',
           fontsize=9,color='#425466')
for b_ax,b_col in zip(b_axes,b_cols):
    b_ratio=b_col=='CP_sd'
    b_values=s6_middle[b_col].div(s6_peer_ref['중간 20~80%'][b_col]) if b_ratio else s6_delta['중간 20~80%'][b_col]
    b_baseline=1 if b_ratio else 0
    b_finite=b_values.loc[b_ids].dropna()
    b_lo=min(float(b_finite.min()),b_baseline)
    b_hi=max(float(b_finite.max()),b_baseline)
    b_span=max(b_hi-b_lo,.000001 if b_col=='CP-Z4' else .1)
    b_ax.set_xlim(b_lo-.22*b_span,b_hi+.22*b_span)
    if b_ratio:
        b_ax.set_xlim(0,b_hi+.22*b_span)
    for b_i,b_lot in enumerate(b_ids):
        b_value=b_values.loc[b_lot]
        b_flag=bool(s6_diff_middle.loc[b_lot,b_col])
        b_n=int(s6_lots.loc[b_lot,'grade_n'])
        b_state='우선 확인' if b_flag and b_n>=3 else '참고 확인' if b_flag else '기준 비해당'
        b_style=S6_REVIEW_STYLE[b_state];b_edge=b_style['fg']
        b_ax.plot([b_baseline,b_value],[b_i,b_i],color='#cfd5dd',lw=1.2,zorder=1)
        b_ax.scatter([b_value],[b_i],s=52,color=b_style['bg'],edgecolors=b_edge,lw=2 if b_flag else .8,zorder=3)
        b_delta_unit='%p' if b_col=='OP-Z3' else S6_UNITS[b_col]
        b_label=f'{b_value:.1f}배' if b_ratio else f'{b_value:+.6f}' if b_col=='CP-Z4' else f'{b_value:+.2f}{b_delta_unit}'
        # 수치는 점 바로 옆에 표시하며 기준선에서 멀어지는 쪽으로 배치합니다.
        b_label_left=bool(b_value<b_baseline and not b_ratio)
        b_ax.annotate(b_label,xy=(b_value,b_i),
                      xytext=(-10,0) if b_label_left else (10,0),textcoords='offset points',
                      ha='right' if b_label_left else 'left',va='center',fontsize=10,
                      color=b_edge,zorder=4,
                      bbox={'facecolor':'white','edgecolor':'none','pad':.12})
    b_ax.axvline(b_baseline,color='#263238',ls='--',lw=1.1)
    b_ax.set_yticks(range(len(b_ids)),[
        f"{lot} · {s6_lots.loc[lot,'grade']}" + (''if s6_lots.loc[lot,'grade_n']==2 else '')
        for lot in b_ids],fontsize=9)
    for tick,lot in zip(b_ax.get_yticklabels(),b_ids):
        tick.set_color(colors[s6_lots.loc[lot,'grade']])
    b_ax.set_ylim(len(b_ids)-.5,-.5)
    b_name=S6_SIMPLE[b_col]
    b_is_width=b_col in ['가열_P90-P10','OP_P90-P10']
    if b_is_width:
        b_name+=' (P90-P10)'
    b_ax.set_title(b_name+' · 동일 강종 비교',loc='left',fontweight='bold',fontsize=12,
                   pad=56 if b_is_width else 34)
    if b_is_width:
        b_ax.text(0,1.112,'P90-P10 = 90백분위값 - 10백분위값 · 가운데 80% 관측값의 분포 폭',
                  transform=b_ax.transAxes,color='#6B7280',fontsize=9,va='bottom')
    b_ax.text(0,1.035,'1배 = 비교 CP-Z4 std와 같음' if b_ratio else '0 = 비교 중앙값과 같음 · +는 높음 · -는 낮음',
               transform=b_ax.transAxes,color='#6B7280',fontsize=9,va='bottom')
    b_ax.set_xlabel('CP-Z4 std 배수' if b_ratio else '같은 강종과의 차이 ['+('%p' if b_col=='OP-Z3' else S6_UNITS[b_col] or '무차원')+']')
    if b_col=='CP-Z4':
        from matplotlib.ticker import FuncFormatter
        b_ax.xaxis.set_major_formatter(FuncFormatter(lambda x,_:f'{x:.6f}'))
    b_ax.grid(axis='x',alpha=.16); b_ax.grid(axis='y',visible=False)
    b_ax.spines[['top','right','left']].set_visible(False)
save_and_show_figures()

# 평균 지표의 차이가 어느 TC 컬럼에서 나타나는지 따로 확인합니다.
S6_TC_COLUMNS=['TC-Z3','TC-Z4','TC-Z5','TC-Z6','TC-Z7','TC-Z8','TC-OUT1','TC-OUT2']
s6_tc_grouped=s6_windows['중간 20~80%'].groupby('LOT_NO')[S6_TC_COLUMNS]
s6_tc_level=s6_tc_grouped.median().reindex(s6_lots.index)
s6_tc_width=(s6_tc_grouped.quantile(.9)-s6_tc_grouped.quantile(.1)).reindex(s6_lots.index)
s6_tc_detail_rows=[]
for s6_tc_lot,s6_tc_lot_info in s6_lots.iterrows():
    s6_tc_peers=s6_peer_ids[s6_tc_lot]
    for s6_tc_column in S6_TC_COLUMNS:
        for s6_tc_metric,s6_tc_matrix in [('중앙값',s6_tc_level),('변동폭(P90-P10)',s6_tc_width)]:
            s6_tc_value=s6_tc_matrix.loc[s6_tc_lot,s6_tc_column]
            s6_tc_reference=s6_tc_matrix.loc[s6_tc_peers,s6_tc_column].median() if s6_tc_peers else np.nan
            s6_tc_detail_rows.append({'LOT':int(s6_tc_lot),'강종':s6_tc_lot_info.grade,'n':int(s6_tc_lot_info.grade_n),
                '컬럼':s6_tc_column,'지표':s6_tc_metric,'LOT [℃]':s6_tc_value,
                '동일 강종 기준 [℃]':s6_tc_reference,'차이 [℃]':s6_tc_value-s6_tc_reference})
s6_tc_details=pd.DataFrame(s6_tc_detail_rows)
s6_tc_case_table=s6_tc_details.loc[
    (s6_tc_details.LOT.isin([240091,240121])&s6_tc_details['컬럼'].isin(['TC-Z7','TC-Z8'])&s6_tc_details['지표'].eq('중앙값'))|
    (s6_tc_details.LOT.eq(240061)&s6_tc_details['컬럼'].isin(['TC-Z3','TC-Z4','TC-Z5','TC-Z6'])&s6_tc_details['지표'].eq('변동폭(P90-P10)'))|
    (s6_tc_details.LOT.eq(240069)&s6_tc_details['컬럼'].isin(['TC-Z3','TC-Z4','TC-Z5','TC-Z6'])&s6_tc_details['지표'].eq('중앙값'))].copy()
display(s6_tc_case_table.round(3))


#### 그림 10-B-온도 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 행·동그라미 옆 수치 | 같은 강종의 자기 제외 중앙값 대비 차이(℃). 0은 같은 수준, +는 높음, -는 낮음. |
| 패널 | TC-Z3~TC-Z6 평균, TC-Z7·TC-Z8 평균, TC-OUT1·TC-OUT2 평균, TC-Z3~TC-Z6 평균 변동폭을 따로 읽음. |
| 색·변동폭 | 빨강은 n≥3 차이 기준, 초록은 n=2 참고 차이. P90-P10은 90백분위값 - 10백분위값. 모두 중간 20~80% 비교. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| **240121 · SPCE / TC-Z7·TC-Z8** | 평균 **+3.08℃**; 개별 TC-Z7 **+3.144℃**, TC-Z8 **+3.008℃** | 두 존 모두 같은 SPCE보다 높은 수준. |
| **240091 · SPHC / TC-Z7·TC-Z8** | 평균 **+3.14℃**; 개별 TC-Z7 **+3.210℃**, TC-Z8 **+3.059℃** | 온도 한 신호군만 달라도 후속 확인할 근거가 남음. |
| **240061 · SPHC / TC-Z3·Z4·Z5·Z6** | 평균 변동폭 **9.78℃ / 비교 0.96℃ → +8.82℃**; 개별 네 TC 모두 변동폭 증가 | TC-Z3만의 문제가 아니라 중간 구간에 걸친 공통 승온의 영향도 포함. |
| **240069 · SPHC / TC-Z3·Z4·Z5·Z6** | 중앙값 차이: TC-Z3 **-1.088℃**, Z4·Z5·Z6 **+0.052~+0.092℃** | 평균온도가 낮은 이유는 주로 TC-Z3의 수준 차이. |
| 240038 ↔ 240114 · SPCEN | TC-Z7·TC-Z8 평균 **±3.23℃**, n=2 | 참고 차이이며 두 LOT 중 어느 쪽이 특이한지는 미확정. |
| 240024 ↔ 240108 · SPFH590 | TC-Z7·TC-Z8 평균 **326.15 / 326.19℃** | 두 LOT의 공통 저온 수준은 강종 조건 가능성. |

**결론:** 평균온도의 차이를 **개별 TC의 차이**로 확인했습니다. 같은 강종을 맞춰도 목표값·속도·두께 차이는 남습니다.

**개별 TC 컬럼의 계산 근거**

| LOT · 강종 | 컬럼 | 지표 | LOT [℃] | 동일 강종 기준 [℃] | 차이 [℃] |
|---|---|---|---:|---:|---:|
| 240061 · SPHC | **TC-Z3** | 변동폭(P90-P10) | 12.079 | 3.379 | +8.700 |
| 240061 · SPHC | **TC-Z4** | 변동폭(P90-P10) | 9.856 | 0.861 | +8.996 |
| 240061 · SPHC | **TC-Z5** | 변동폭(P90-P10) | 9.785 | 0.563 | +9.222 |
| 240061 · SPHC | **TC-Z6** | 변동폭(P90-P10) | 10.044 | 0.488 | +9.556 |
| 240069 · SPHC | **TC-Z3** | 중앙값 | 859.218 | 860.306 | -1.088 |
| 240069 · SPHC | **TC-Z4** | 중앙값 | 859.652 | 859.600 | +0.052 |
| 240069 · SPHC | **TC-Z5** | 중앙값 | 859.696 | 859.643 | +0.053 |
| 240069 · SPHC | **TC-Z6** | 중앙값 | 859.738 | 859.647 | +0.092 |
| 240091 · SPHC | **TC-Z7** | 중앙값 | 332.975 | 329.765 | +3.210 |
| 240091 · SPHC | **TC-Z8** | 중앙값 | 333.298 | 330.239 | +3.059 |
| 240121 · SPCE | **TC-Z7** | 중앙값 | 333.309 | 330.165 | +3.144 |
| 240121 · SPCE | **TC-Z8** | 중앙값 | 333.517 | 330.509 | +3.008 |

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240121 ↔ 240037·240133; 240091 ↔ 240044·240061·240069 | TC-Z7·TC-Z8의 높은 수준이 시간 내내 이어지는지 확인 | 다음 진행률 지속성 지도·상세 시계열, 존별 목표온도·속도·두께 |
| 240061 ↔ 240044·240069·240091 | 승온이 끝난 뒤에도 각 TC 변동폭이 큰지 재비교 | 재가동 시각·목표값; 결측 없는 동일 운전상태 구간 |
| 240069 ↔ 240044·240061·240091 | TC-Z3 수준 차이와 OP-Z3 변화 시점을 대조 | TC-Z3 원자료·설정값·제어 이력 |


In [ ]:

# 이 셀에서 실제 차이를 직접 그립니다.
b_cols=['OP-Z3', 'OP_P90-P10']
b_title='그림 10-B-OP. OP-Z3 수준과 변동폭 비교'
b_ids=s6_lots[s6_lots.grade_n.ge(2)].sort_values(['grade','start']).index.tolist()
b_height=4.2*len(b_cols)+2.3
b_fig,b_axes=plt.subplots(len(b_cols),1,figsize=(16,b_height),squeeze=False)
b_axes=b_axes[:,0]
b_fig.subplots_adjust(left=.20,right=.98,top=1-1.9/b_height,bottom=.95/b_height,hspace=.78)
b_fig.suptitle(b_title,x=.20,y=1-.12/b_height,ha='left',va='top',fontweight='bold',fontsize=15)
b_fig.legend(handles=[
    Line2D([0],[0],marker='o',color='none',markerfacecolor=S6_REVIEW_STYLE['우선 확인']['bg'],markeredgecolor=S6_REVIEW_STYLE['우선 확인']['fg'],markeredgewidth=2,label='n≥3 · 차이 기준 해당'),
    Line2D([0],[0],marker='o',color='none',markerfacecolor=S6_REVIEW_STYLE['참고 확인']['bg'],markeredgecolor=S6_REVIEW_STYLE['참고 확인']['fg'],markeredgewidth=2,label='n=2 · 참고 차이'),
    Line2D([0],[0],marker='o',color='none',markerfacecolor=S6_REVIEW_STYLE['기준 비해당']['bg'],markeredgecolor=S6_REVIEW_STYLE['기준 비해당']['fg'],label='차이 기준 미만')
],loc='upper left',bbox_to_anchor=(.20,1-.50/b_height),ncol=3,frameon=False,fontsize=9)
b_fig.text(.20,1-1.07/b_height,'같은 구간 비교: 중간 20~80% · 자신을 제외한 동일 강종 LOT 중앙값이 기준 · n=1은 아래 전체 확인표에서 보류',
           fontsize=9,color='#425466')
for b_ax,b_col in zip(b_axes,b_cols):
    b_ratio=b_col=='CP_sd'
    b_values=s6_middle[b_col].div(s6_peer_ref['중간 20~80%'][b_col]) if b_ratio else s6_delta['중간 20~80%'][b_col]
    b_baseline=1 if b_ratio else 0
    b_finite=b_values.loc[b_ids].dropna()
    b_lo=min(float(b_finite.min()),b_baseline)
    b_hi=max(float(b_finite.max()),b_baseline)
    b_span=max(b_hi-b_lo,.000001 if b_col=='CP-Z4' else .1)
    b_ax.set_xlim(b_lo-.22*b_span,b_hi+.22*b_span)
    if b_ratio:
        b_ax.set_xlim(0,b_hi+.22*b_span)
    for b_i,b_lot in enumerate(b_ids):
        b_value=b_values.loc[b_lot]
        b_flag=bool(s6_diff_middle.loc[b_lot,b_col])
        b_n=int(s6_lots.loc[b_lot,'grade_n'])
        b_state='우선 확인' if b_flag and b_n>=3 else '참고 확인' if b_flag else '기준 비해당'
        b_style=S6_REVIEW_STYLE[b_state];b_edge=b_style['fg']
        b_ax.plot([b_baseline,b_value],[b_i,b_i],color='#cfd5dd',lw=1.2,zorder=1)
        b_ax.scatter([b_value],[b_i],s=52,color=b_style['bg'],edgecolors=b_edge,lw=2 if b_flag else .8,zorder=3)
        b_delta_unit='%p' if b_col=='OP-Z3' else S6_UNITS[b_col]
        b_label=f'{b_value:.1f}배' if b_ratio else f'{b_value:+.6f}' if b_col=='CP-Z4' else f'{b_value:+.2f}{b_delta_unit}'
        # 수치는 점 바로 옆에 표시하며 기준선에서 멀어지는 쪽으로 배치합니다.
        b_label_left=bool(b_value<b_baseline and not b_ratio)
        b_ax.annotate(b_label,xy=(b_value,b_i),
                      xytext=(-10,0) if b_label_left else (10,0),textcoords='offset points',
                      ha='right' if b_label_left else 'left',va='center',fontsize=10,
                      color=b_edge,zorder=4,
                      bbox={'facecolor':'white','edgecolor':'none','pad':.12})
    b_ax.axvline(b_baseline,color='#263238',ls='--',lw=1.1)
    b_ax.set_yticks(range(len(b_ids)),[
        f"{lot} · {s6_lots.loc[lot,'grade']}" + ('' if s6_lots.loc[lot,'grade_n']==2 else '')
        for lot in b_ids],fontsize=9)
    for tick,lot in zip(b_ax.get_yticklabels(),b_ids):
        tick.set_color(colors[s6_lots.loc[lot,'grade']])
    b_ax.set_ylim(len(b_ids)-.5,-.5)
    b_name=S6_SIMPLE[b_col]
    b_is_width=b_col in ['가열_P90-P10','OP_P90-P10']
    if b_is_width:
        b_name+=' (P90-P10)'
    b_ax.set_title(b_name+' · 동일 강종 비교',loc='left',fontweight='bold',fontsize=12,
                   pad=56 if b_is_width else 34)
    if b_is_width:
        b_ax.text(0,1.112,'P90-P10 = 90백분위값 - 10백분위값 · 가운데 80% 관측값의 분포 폭',
                  transform=b_ax.transAxes,color='#6B7280',fontsize=9,va='bottom')
    b_ax.text(0,1.035,'1배 = 비교 CP-Z4 std와 같음' if b_ratio else '0 = 비교 중앙값과 같음 · +는 높음 · -는 낮음',
               transform=b_ax.transAxes,color='#6B7280',fontsize=9,va='bottom')
    b_ax.set_xlabel('CP-Z4 std 배수' if b_ratio else '같은 강종과의 차이 ['+('%p' if b_col=='OP-Z3' else S6_UNITS[b_col] or '무차원')+']')
    if b_col=='CP-Z4':
        from matplotlib.ticker import FuncFormatter
        b_ax.xaxis.set_major_formatter(FuncFormatter(lambda x,_:f'{x:.6f}'))
    b_ax.grid(axis='x',alpha=.16); b_ax.grid(axis='y',visible=False)
    b_ax.spines[['top','right','left']].set_visible(False)
save_and_show_figures()


#### 그림 10-B-OP — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 윗패널 / 아랫패널 | 윗패널은 OP-Z3 중앙값의 차이, 아랫패널은 OP-Z3 변동폭(P90-P10)의 차이. |
| 동그라미 옆 수치 | 단위는 %p. 0은 자기 제외 동일 강종 LOT와 같음. +는 더 높거나 더 넓음, -는 더 낮거나 더 좁음. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| 240038 ↔ 240114 · SPCEN | 변동폭 차이 **±32.10%p**; 초록색 n=2 참고 | 두 LOT의 출력 움직임 폭이 다름. 출력 수준이 높다는 뜻과 구분. |
| **240061 · SPHC** | 중간 OP-Z3 중앙값 **-0.39%p**, 변동폭 **+18.56%p**; 두 항목 모두 설정한 차이 기준 미만 | 전체 비교에서 후보였지만 같은 강종에서는 OP 차이만으로 상세 후보를 확정하지 않음. |
| 240121·240133 · SPCE | 중앙값 차이 **+0.24 / +0.77%p**, 기준 미만 | 전체 LOT에서 높아 보인 출력이 같은 SPCE 안에서는 큰 차이로 남지 않음. |
| **240044·240061 · SPHC** | OP-Z3 초반 결측 **75 / 179행** | 점 그래프와 별개로 수집 품질 점검은 계속 필요. |

**결론:** **출력 수준·변동폭·결측은 별개의 확인 항목**입니다. 기준 미만인 점도 정상 판정은 아닙니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240038 ↔ 240114 | 큰 변동폭 차이가 시작·종료 전환에 집중되는지 확인 | 진행률별 OP-Z3 변동폭(5분), 두 LOT의 출력 목표·가동상태 |
| 240044 ↔ 240061; 대조 240069·240091 | OP-Z3 최초 유효값 시각과 다른 채널의 수집 상태 대조 | 앞의 결측 확대 그림, PLC·태그·수집 로그 |
| 240061 ↔ 나머지 SPHC | 승온 이후 OP-Z3 저출력(<1%)과 TC-Z3 변화가 함께 나타나는지 확인 | 실제 시간축의 TC-Z3·OP-Z3 및 제어 기록 |


In [ ]:

# 이 셀에서 실제 차이를 직접 그립니다.
b_cols=['CP-Z4', 'CP_sd']
b_title='그림 10-B-CP. CP-Z4 수준과 std 증가 비교'
b_ids=s6_lots[s6_lots.grade_n.ge(2)].sort_values(['grade','start']).index.tolist()
b_height=4.2*len(b_cols)+2.3
b_fig,b_axes=plt.subplots(len(b_cols),1,figsize=(16,b_height),squeeze=False)
b_axes=b_axes[:,0]
b_fig.subplots_adjust(left=.20,right=.98,top=1-1.9/b_height,bottom=.95/b_height,hspace=.78)
b_fig.suptitle(b_title,x=.20,y=1-.12/b_height,ha='left',va='top',fontweight='bold',fontsize=15)
b_fig.legend(handles=[
    Line2D([0],[0],marker='o',color='none',markerfacecolor=S6_REVIEW_STYLE['우선 확인']['bg'],markeredgecolor=S6_REVIEW_STYLE['우선 확인']['fg'],markeredgewidth=2,label='n≥3 · 차이 기준 해당'),
    Line2D([0],[0],marker='o',color='none',markerfacecolor=S6_REVIEW_STYLE['참고 확인']['bg'],markeredgecolor=S6_REVIEW_STYLE['참고 확인']['fg'],markeredgewidth=2,label='n=2 · 참고 차이'),
    Line2D([0],[0],marker='o',color='none',markerfacecolor=S6_REVIEW_STYLE['기준 비해당']['bg'],markeredgecolor=S6_REVIEW_STYLE['기준 비해당']['fg'],label='차이 기준 미만')
],loc='upper left',bbox_to_anchor=(.20,1-.50/b_height),ncol=3,frameon=False,fontsize=9)
b_fig.text(.20,1-1.07/b_height,'같은 구간 비교: 중간 20~80% · 자신을 제외한 동일 강종 LOT 중앙값이 기준 · n=1은 아래 전체 확인표에서 보류',
           fontsize=9,color='#425466')
for b_ax,b_col in zip(b_axes,b_cols):
    b_ratio=b_col=='CP_sd'
    b_values=s6_middle[b_col].div(s6_peer_ref['중간 20~80%'][b_col]) if b_ratio else s6_delta['중간 20~80%'][b_col]
    b_baseline=1 if b_ratio else 0
    b_finite=b_values.loc[b_ids].dropna()
    b_lo=min(float(b_finite.min()),b_baseline)
    b_hi=max(float(b_finite.max()),b_baseline)
    b_span=max(b_hi-b_lo,.000001 if b_col=='CP-Z4' else .1)
    b_ax.set_xlim(b_lo-.22*b_span,b_hi+.22*b_span)
    if b_ratio:
        b_ax.set_xlim(0,b_hi+.22*b_span)
    for b_i,b_lot in enumerate(b_ids):
        b_value=b_values.loc[b_lot]
        b_flag=bool(s6_diff_middle.loc[b_lot,b_col])
        b_n=int(s6_lots.loc[b_lot,'grade_n'])
        b_state='우선 확인' if b_flag and b_n>=3 else '참고 확인' if b_flag else '기준 비해당'
        b_style=S6_REVIEW_STYLE[b_state];b_edge=b_style['fg']
        b_ax.plot([b_baseline,b_value],[b_i,b_i],color='#cfd5dd',lw=1.2,zorder=1)
        b_ax.scatter([b_value],[b_i],s=52,color=b_style['bg'],edgecolors=b_edge,lw=2 if b_flag else .8,zorder=3)
        b_delta_unit='%p' if b_col=='OP-Z3' else S6_UNITS[b_col]
        b_label=f'{b_value:.1f}배' if b_ratio else f'{b_value:+.6f}' if b_col=='CP-Z4' else f'{b_value:+.2f}{b_delta_unit}'
        # 수치는 점 바로 옆에 표시하며 기준선에서 멀어지는 쪽으로 배치합니다.
        b_label_left=bool(b_value<b_baseline and not b_ratio)
        b_ax.annotate(b_label,xy=(b_value,b_i),
                      xytext=(-10,0) if b_label_left else (10,0),textcoords='offset points',
                      ha='right' if b_label_left else 'left',va='center',fontsize=10,
                      color=b_edge,zorder=4,
                      bbox={'facecolor':'white','edgecolor':'none','pad':.12})
    b_ax.axvline(b_baseline,color='#263238',ls='--',lw=1.1)
    b_ax.set_yticks(range(len(b_ids)),[
        f"{lot} · {s6_lots.loc[lot,'grade']}" + ('' if s6_lots.loc[lot,'grade_n']==2 else '')
        for lot in b_ids],fontsize=9)
    for tick,lot in zip(b_ax.get_yticklabels(),b_ids):
        tick.set_color(colors[s6_lots.loc[lot,'grade']])
    b_ax.set_ylim(len(b_ids)-.5,-.5)
    b_name=S6_SIMPLE[b_col]
    b_is_width=b_col in ['가열_P90-P10','OP_P90-P10']
    if b_is_width:
        b_name+=' (P90-P10)'
    b_ax.set_title(b_name+' · 동일 강종 비교',loc='left',fontweight='bold',fontsize=12,
                   pad=56 if b_is_width else 34)
    if b_is_width:
        b_ax.text(0,1.112,'P90-P10 = 90백분위값 - 10백분위값 · 가운데 80% 관측값의 분포 폭',
                  transform=b_ax.transAxes,color='#6B7280',fontsize=9,va='bottom')
    b_ax.text(0,1.035,'1배 = 비교 CP-Z4 std와 같음' if b_ratio else '0 = 비교 중앙값과 같음 · +는 높음 · -는 낮음',
               transform=b_ax.transAxes,color='#6B7280',fontsize=9,va='bottom')
    b_ax.set_xlabel('CP-Z4 std 배수' if b_ratio else '같은 강종과의 차이 ['+('%p' if b_col=='OP-Z3' else S6_UNITS[b_col] or '무차원')+']')
    if b_col=='CP-Z4':
        from matplotlib.ticker import FuncFormatter
        b_ax.xaxis.set_major_formatter(FuncFormatter(lambda x,_:f'{x:.6f}'))
    b_ax.grid(axis='x',alpha=.16); b_ax.grid(axis='y',visible=False)
    b_ax.spines[['top','right','left']].set_visible(False)
save_and_show_figures()


#### 그림 10-B-CP — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 윗패널 | CP-Z4 중앙값의 실제 차이. 작은 차이도 순위에는 걸릴 수 있으므로 수치 확인. |
| 아랫패널 | CP-Z4 std(표준편차)의 배수. 1배는 동일 강종 비교값과 같음. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| **240069 · SPHC / CP-Z4 std** | 중간 **0.0363 / 비교 0.0027 → 13.5배** | 같은 강종에서도 CP 변동성이 큼. |
| 240069 · SPHC / CP-Z4 중앙값 | 자기 제외 차이 **+0.000096**, 수준 차이 기준 미만 | CP의 지속적인 수준 상승과 std 증가를 구분해야 함. |
| 나머지 LOT의 CP | 동일 강종 차이 기준에 해당하는 CP 항목은 240069의 std | 전체 LOT의 순위에서 높거나 낮다고 표시된 CP가 모두 상세 후보로 남지는 않음. |

**결론:** 240069에서 남는 핵심은 **CP-Z4의 변동성**입니다. std만으로 지속 변동인지 짧은 피크인지 알 수 없습니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240069 ↔ 240044·240061·240091 | CP-Z4 std(5분)이 높은 시점·지속 길이를 확인 | 다음 시간 지도와 그림 10-C |
| 240069 / CP-Z4 피크 시각 | 피크 원자료와 분위기 설정 변경·분석계 상태 대조 | CP-Z4 원자료, 분위기 제어·교정·통신 로그 |


**이 단계의 판단:** 전체 LOT 순위에 걸렸다는 이유와 동일 강종에서 실제 차이가 남는다는 이유를 구분했습니다. 다음 단계에서는 그 차이가 시작·중간·종료 중 어디에서 이어지는지 확인합니다. 단일 온도군의 차이가 남는 240091도 전체 확인표에 포함합니다.

### ③ LOT 차이 확인 — 한 신호군의 차이와 다중신호 후보를 구분

전체 후보 선정과 동일 강종 차이는 별도 항목으로 유지합니다. 2개 이상 신호군의 넓은 후보가 아니어도 같은 강종 차이 또는 결측 근거가 있으면 참고 항목으로 남깁니다. 최종 10-H 확인표에서 **전체 14개 LOT**의 처리 이유를 확인합니다.

In [ ]:
s6_summary_ids=s6_lots.index.tolist()
s6_summary_before=pd.DataFrame(False,index=s6_summary_ids,columns=list(S6_FAMILIES))
s6_summary_window={}
for lot in s6_summary_ids:
    use_middle=s6_screen.loc[lot,'중간 신호군 수']>=s6_screen.loc[lot,'전체 신호군 수']
    s6_summary_before.loc[lot]=(s6_mid_flags if use_middle else s6_flags).loc[lot]
    s6_summary_window[lot]='중간' if use_middle else '전체'
s6_summary_after=s6_peer_families.loc[s6_summary_ids].astype(float)
s6_summary_after.loc[s6_lots.loc[s6_summary_ids,'grade_n'].eq(1)]=np.nan


**남겨야 할 확인 대상:** 240069·240121 외에 240091의 TC-Z7·TC-Z8 평균온도, 240044·240061의 출력 결측, 240038·240114의 동일 강종 참고 차이, 240058·240114의 종료 하강, 240117의 초반 승온을 함께 봅니다. 240024·240108은 공통된 낮은 TC-Z7·TC-Z8 평균온도를 참고 비교하고, 240037·240133은 전체 후보와 강종 내 차이 기준이 달라 보류한 이유를 남깁니다.

In [ ]:
# 4단계 핵심 요약: 진행률별 자기 제외 차이가 어디에서 유지되는지 보여줍니다.
# 0은 '기준 미만', NaN은 '비교 자료 없음'으로 구분합니다.


### ④ 시간 지속성 — 같은 강종과 남은 차이가 계속 이어지는가?

**이 단계에서 볼 질문은 하나입니다. 같은 강종과 비교해 다른 값이 중간 구간 전반에 반복되는가, 일부 시점에만 나타나는가?** 앞의 10-B는 차이의 크기, 아래 그림은 **차이가 나타나는 위치와 반복 정도**를 확인합니다. 전체 후보를 새로 뽑거나 고장 원인을 판정하는 그림은 아닙니다. **각 칸에는 실제 측정값 숫자 대신 같은 강종과의 비교 결과를 표시합니다. 실제 차이의 크기는 앞의 10-B-온도·OP·CP에서 읽습니다.**

**먼저 볼 사례:** 240121·240091은 지속적인 온도 수준 차이, 240069는 CP의 국소 변동, 240061은 승온과 변동폭을 분리합니다. 핵심 사례와 n=2·비교군 참고를 두 그림으로 나눴습니다.

| 표시 | 무엇을 뜻하는가? |
|---|---|
| **한 행** | 한 LOT의 한 컬럼/평균 지표. TC 평균은 표시된 컬럼의 시각별 평균 |
| **한 칸 = 진행률 5%** | LOT 길이의 5% 구간 중앙값. 같은 강종의 다른 LOT에서 같은 진행률 구간과 비교. 실제 분 길이는 LOT마다 다름 |
| **std·변동폭** | std는 표준편차, 변동폭은 P90-P10. 여기서는 두 지표로 단기 흔들림을 비교 |
| **최근 5분** | CP std·온도/OP 변동폭을 계산하는 이동 창. **가로 한 칸이 5분이라는 뜻은 아님** |
| **높음·낮음** | 온도·출력·CP 수준이 같은 강종 비교 기준보다 높거나 낮은 차이 기준에 해당 |
| **큼·작음** | 최근 5분 std/변동폭이 비교값의 2배 이상 또는 0.5배 이하. 유효 구간의 대표값끼리 비교 |
| **흰색 '-' / 회색 '-'** | 차이 기준 미만 / 비교 자료 없음. 흰색을 정상 판정으로 읽지 않음 |
| **오른쪽** | 중간 20~80%의 높은/낮은 구간 수를 따로 집계하고 다음 확인을 제시 |

**색은 기존 점검 우선순위를 유지합니다.** 빨강은 n≥3이며 중간 대표값 차이도 남은 항목, 초록은 n=2 또는 시간 구간의 참고 차이입니다. 색 자체가 '높음·낮음'을 뜻하지 않으므로 칸의 글자를 읽습니다. 유효 중간 칸 ≥8, 같은 방향 차이 ≥70%를 지속 후보의 탐색 기준으로 쓰며, 많은 구간의 차이가 곧 고장이라는 뜻은 아닙니다.


In [ ]:
s6_time_rows=[(240069,'CP_roll_sd'),(240069,'CP-Z4'),(240061,'가열_roll_width'),(240061,'가열·균열존'),(240061,'OP-Z3'),
              (240121,'과시효대'),(240091,'과시효대'),(240024,'과시효대'),(240108,'과시효대'),
              (240024,'가열_roll_width'),(240108,'가열_roll_width'),
              (240038,'과시효대'),(240114,'과시효대'),(240038,'OP_roll_width'),(240114,'OP_roll_width'),(240133,'과시효대')]
s6_time_state=[]; s6_time_labels=[]; s6_time_results=[]
for lot,c in s6_time_rows:
    delta=s6_bin_delta[c].loc[lot]
    if c in S6_TRACE_VARIABILITY:
        ratio=s6_bin_values[c].loc[lot].div(s6_bin_ref[c].loc[lot].replace(0,np.nan))
        high=ratio.ge(2); low=ratio.le(.5); valid=delta.notna()&ratio.notna()
    else:
        high=delta.ge(s6_level_floor[c]); low=delta.le(-s6_level_floor[c]); valid=delta.notna()
    state=pd.Series(0.,index=range(20)); state[high]=1.; state[low]=-1.; state=state.where(valid)
    s6_time_state.append(state.to_numpy())
    r=s6_lots.loc[lot]; support=' · 참고' if r.grade_n==2 else ' · 단일군 참고' if not s6_screen.loc[lot,'탐색 후보'] else ''
    channel=S6_TRACE_NAMES[c]
    s6_time_labels.append(f'{lot} · {r.grade}{support}\n{channel}')
    p=s6_persistence[s6_persistence.LOT.eq(lot)&s6_persistence['채널'].eq(c)].iloc[0]
    s6_time_results.append(p)
s6_time_state=np.array(s6_time_state)
# 방향은 +/-로, 확인 우선순위는 색으로 각각 표시합니다.
s6_time_review=np.zeros_like(s6_time_state)
s6_time_priority=[]
for i,(lot,c) in enumerate(s6_time_rows):
    s6_time_feature={'CP_roll_sd':'CP_sd','가열_roll_width':'가열_P90-P10','OP_roll_width':'OP_P90-P10'}.get(c,c)
    s6_bin_priority=bool(s6_lots.loc[lot,'grade_n']>=3 and s6_diff_middle.loc[lot,s6_time_feature])
    s6_time_priority.append(s6_bin_priority)
    s6_bin_valid=np.isfinite(s6_time_state[i]); s6_bin_different=s6_bin_valid&(s6_time_state[i]!=0)
    s6_time_review[i,s6_bin_valid]=1
    s6_time_review[i,s6_bin_different]=3 if s6_bin_priority else 2
s6_time_style_order=['자료 없음','기준 비해당','참고 확인','우선 확인']
s6_page_indices=list(range(7))
s6_core_time_indices=s6_page_indices.copy()
s6_page_title='그림 10-B-시간-1. 같은 강종과 남은 차이 — 중간 내내인가, 일부인가?'

# 진행률 구간의 값/변동 차이를 글자로 표시하고 다음 확인을 연결합니다.
s6_page_labels=[]
for s6_i in s6_page_indices:
    s6_lot,s6_channel=s6_time_rows[s6_i]
    s6_metric={
        'CP_roll_sd':'CP-Z4 std(최근 5분)','CP-Z4':'CP-Z4 수준',
        '가열_roll_width':'TC-Z3~TC-Z6 평균\n변동폭(최근 5분)',
        '가열·균열존':'TC-Z3~TC-Z6 평균온도','OP-Z3':'OP-Z3 출력 수준',
        '과시효대':'TC-Z7·TC-Z8 평균온도',
        'OP_roll_width':'OP-Z3 변동폭(최근 5분)'}[s6_channel]
    s6_page_labels.append(f'{s6_lot} · {s6_lots.loc[s6_lot,"grade"]}\n{s6_metric}')
s6_page_fig,(s6_page_ax,s6_page_summary)=plt.subplots(
    1,2,figsize=(18,.85*len(s6_page_indices)+3.4),sharey=True,
    gridspec_kw={'width_ratios':[8.5,3.7]})
s6_page_ax.imshow(s6_time_review[s6_page_indices],aspect='auto',
    cmap=ListedColormap([S6_REVIEW_STYLE.get(s,S6_MISSING_STYLE)['bg'] for s in s6_time_style_order]),
    norm=BoundaryNorm([-.5,.5,1.5,2.5,3.5],4),
    extent=(0,100,len(s6_page_indices)-.5,-.5))
for s6_local,s6_i in enumerate(s6_page_indices):
    s6_lot,s6_channel=s6_time_rows[s6_i]
    s6_is_variability=s6_channel in S6_TRACE_VARIABILITY
    for s6_j,s6_direction in enumerate(s6_time_state[s6_i]):
        if not np.isfinite(s6_direction):
            s6_label='-';s6_fg='#87919A'
        elif s6_direction==0:
            s6_label='-';s6_fg='#BBC2C8'
        else:
            s6_label=('큼' if s6_direction>0 else '작음') if s6_is_variability else ('높음' if s6_direction>0 else '낮음')
            s6_fg=S6_REVIEW_STYLE.get(s6_time_style_order[int(s6_time_review[s6_i,s6_j])],S6_MISSING_STYLE)['fg']
        s6_page_ax.text(s6_j*5+2.5,s6_local,s6_label,ha='center',va='center',
            fontsize=9.2,color=s6_fg,fontweight='bold' if s6_direction!=0 and np.isfinite(s6_direction) else 'normal')
s6_page_ax.set_yticks(range(len(s6_page_indices)),s6_page_labels,fontsize=11.2,color='#263238')
s6_page_ax.tick_params(axis='y',length=0,pad=12)
s6_page_ax.set_xticks(np.arange(0,101,10))
s6_page_ax.set_xticks(np.arange(0,101,5),minor=True)
s6_page_ax.set_yticks(np.arange(-.5,len(s6_page_indices)),minor=True)
s6_page_ax.grid(which='minor',color='#F1F3F5',lw=1)
s6_page_ax.grid(which='major',visible=False)
s6_page_ax.tick_params(which='minor',length=0)
s6_page_ax.set_xlabel('LOT 진행률 [%] · 한 칸 = LOT 길이의 5%',fontsize=11,labelpad=12)
s6_page_ax.add_patch(Rectangle((20,-.5),60,len(s6_page_indices),fill=False,edgecolor='#616B74',lw=1.2))
s6_page_ax.set_title('② 차이는 언제 나타나는가?',loc='left',fontweight='bold',fontsize=13,pad=32)
for s6_x,s6_label in [(10,'시작 0~20%'),(50,'주 비교: 중간 20~80%'),(90,'종료 80~100%')]:
    s6_page_ax.text(s6_x,1.02,s6_label,transform=s6_page_ax.get_xaxis_transform(),
        ha='center',fontsize=10,color='#66717D')
s6_page_summary.set_title('③ 얼마나 반복되며, 다음에 무엇을 볼까?',loc='left',
    fontweight='bold',fontsize=12.5,pad=32)
s6_page_summary.set_xlim(0,1);s6_page_summary.set_xticks([])
s6_page_summary.tick_params(axis='y',which='both',left=False,right=False,labelleft=False)
s6_page_summary.spines[['top','bottom','left','right']].set_visible(False)
s6_page_summary.grid(visible=False)
for s6_local,s6_i in enumerate(s6_page_indices):
    s6_lot,s6_channel=s6_time_rows[s6_i];s6_p=s6_time_results[s6_i]
    s6_middle_state=s6_time_state[s6_i,4:16]
    s6_high=int((s6_middle_state==1).sum());s6_low=int((s6_middle_state==-1).sum())
    s6_n=int(np.isfinite(s6_middle_state).sum());s6_count=max(s6_high,s6_low)
    s6_variability=s6_channel in S6_TRACE_VARIABILITY
    s6_high_word,s6_low_word=('큼','작음') if s6_variability else ('높음','낮음')
    s6_direction=s6_high_word if s6_high>=s6_low else s6_low_word
    s6_pct=100*s6_count/s6_n if s6_n else np.nan
    if not s6_n:
        s6_state='자료 없음';s6_count_line='비교할 중간 구간 없음';s6_pattern='-';s6_next='동일 강종 관측 확보'
    else:
        s6_state='기준 비해당' if not s6_count else '우선 확인' if s6_time_priority[s6_i] else '참고 확인'
        s6_count_line=f'{s6_high_word} {s6_high}/{s6_n}칸 · {s6_low_word} {s6_low}/{s6_n}칸'
        s6_pattern=(f'{s6_direction} {s6_pct:.0f}% · '+
            ('중간 전반' if s6_count==s6_n else '여러 중간 구간' if s6_p['지속 후보'] else '일부 구간')) if s6_count else '중간 구간의 차이 기준 미만'
        if s6_channel=='CP_roll_sd':s6_next='CP 피크 전후 확대'
        elif s6_channel=='CP-Z4':s6_next='수준과 피크·std를 구분'
        elif s6_lot==240061 and s6_channel=='가열_roll_width':s6_next='승온 후 변동 재확인'
        elif s6_lot==240061 and s6_channel=='가열·균열존':s6_next='승온 구간을 분리'
        elif s6_lot==240061 and s6_channel=='OP-Z3':s6_next='짧은 저출력은 별도 확인'
        elif s6_lot in [240121,240091]:s6_next='각 TC의 목표값 대조'
        elif s6_lot==240133:s6_next='240037과 직접 비교'
        elif s6_lots.loc[s6_lot,'grade_n']==2:
            s6_next='같은 강종 조건 대조 (n=2)' if s6_count else '두 LOT의 공통 수준 확인'
        else:s6_next='같은 강종 원자료 확인'
    s6_style=S6_REVIEW_STYLE.get(s6_state,S6_MISSING_STYLE)
    s6_page_summary.add_patch(Rectangle((.01,s6_local-.43),.98,.86,
        facecolor=s6_style['bg'],edgecolor=s6_style['edge'],lw=.7))
    s6_page_summary.text(.045,s6_local-.24,s6_count_line,va='center',fontsize=10.5,color=s6_style['fg'])
    s6_page_summary.text(.045,s6_local,s6_pattern,va='center',fontsize=11,
        fontweight='bold',color=s6_style['fg'])
    s6_page_summary.text(.045,s6_local+.24,f'다음: {s6_next}',va='center',fontsize=10.5,color='#425466')
s6_page_fig.suptitle(s6_page_title,x=.025,y=.983,ha='left',fontweight='bold',fontsize=16)
s6_page_fig.text(.025,.937,'목적: 같은 강종과의 차이가 중간 내내 이어지는지, 일부 시점에만 나타나는지 구분',
    fontsize=11,color='#425466')
s6_page_fig.text(.025,.899,
    '비교: 자신을 제외한 동일 강종 LOT의 같은 진행률 구간 중앙값 · 한 칸은 5%이며, 최근 5분은 변동 지표의 계산 창',
    fontsize=10.6,color='#66717D')
s6_page_fig.legend(handles=[Patch(fc=S6_REVIEW_STYLE.get(s,S6_MISSING_STYLE)['bg'],
    ec=S6_REVIEW_STYLE.get(s,S6_MISSING_STYLE)['fg'],label=s) for s in ['우선 확인','참고 확인','기준 비해당','자료 없음']],
    loc='upper left',bbox_to_anchor=(.218,.861),ncol=4,frameon=False,fontsize=10)
s6_page_fig.text(.025,.765,'① LOT · 비교 컬럼',fontsize=12.5,fontweight='bold',color='black')
s6_page_fig.text(.025,.058,
    '높음/낮음 = 값의 수준 차이 · 큼/작음 = std 또는 변동폭의 차이 · 색은 점검 우선순위 · 흰색 -는 기준 미만이며 정상 판정이 아님',
    fontsize=10,color='#66717D')
s6_page_fig.text(.025,.028,
    '오른쪽은 중간 20~80%의 12칸 집계 · 높은/낮은 방향을 따로 셈 · 한 칸의 중앙값 비교로 짧은 피크·저출력이 보이지 않을 수 있음',
    fontsize=10,color='#66717D')
s6_page_fig.subplots_adjust(left=.225,right=.985,top=.73,bottom=.15,wspace=.065)
save_and_show_figures()

p69=s6_persistence[s6_persistence.LOT.eq(240069)&s6_persistence['채널'].eq('CP_roll_sd')].iloc[0]
p121=s6_persistence[s6_persistence.LOT.eq(240121)&s6_persistence['채널'].eq('과시효대')].iloc[0]


#### 그림 10-B-시간-1 — 계속 다른 LOT와 일부 구간만 다른 LOT를 구분

**① 읽는 순서:** 왼쪽에서 **LOT·컬럼**, 가운데에서 **높음/낮음 또는 큼/작음의 위치**, 오른쪽에서 **중간 구간 수·다음 확인**을 읽습니다. 시작·끝 20%는 운전 전환의 영향을 보기 위한 참고 구간입니다.

**② 확인한 결과 — 이 그림으로 무엇을 알게 되었는가?**

| 대상·컬럼 | 중간 구간의 관측 | 알게 된 점 · 다음 확인 |
|---|---|---|
| **240121 · SPCE / TC-Z7·TC-Z8 평균** | **높음 12/12칸(100%)**, 대표값 차이 **+3.08℃** | 일부 피크보다 **중간 전반의 수준 차이**. TC-Z7·TC-Z8 각각과 해당 존 OP-Z7·OP-Z8, 목표온도를 대조 |
| **240091 · SPHC / TC-Z7·TC-Z8 평균** | **높음 12/12칸(100%)**, 대표값 차이 **+3.14℃** | 단일 온도군이어도 **반복되는 차이**를 확인할 이유가 있음. 같은 SPHC의 목표값·운전조건과 대조 |
| **240069 · SPHC / CP-Z4 std(최근 5분)** | 큰 방향 **4/12칸(33%)** | **국소적인 CP 변동**. LOT 전체에 큰 변동이 지속된다는 뜻은 아님. **574.5분 피크 전후** 확대 |
| **240069 / CP-Z4 수준** | 높은/낮은 수준 기준 초과 **0/12칸** | 큰 std와 지속적인 CP 수준 상승은 다른 현상. **0칸이어도 짧은 피크는 존재** |
| **240061 · SPHC / TC-Z3~TC-Z6 평균 변동폭(최근 5분)** | 큰 방향 **6/12칸(50%)** | 일부 중간 구간만 큼. **승온을 제외한 뒤 변동을 재확인** |
| **240061 / TC-Z3~TC-Z6 평균온도·OP-Z3** | 온도 낮은 방향 **6/12칸**, 출력 낮은 방향 **1/12칸** | 중간 20~80%에도 승온이 남음. OP 구간 중앙값만으로 짧은 0%를 모두 찾을 수 없어 저출력 시각은 별도 확인 |

**결론:** **240121·240091은 중간 구간 전반의 온도 수준**, **240069는 CP 피크 주변**, **240061은 승온 이후 TC-Z3↔OP-Z3**를 서로 다른 목적으로 확대합니다. 한 행의 색을 이상 점수로 합치지 않습니다. 12/12칸은 구간 중앙값의 반복이며 모든 1초 값에서 차이가 났다는 의미는 아닙니다.

**③ 후속 작업**

| 확인 대상·비교군 | 자세히 볼 구간·항목 | 이어지는 분석·필요 자료 |
|---|---|---|
| **240121 ↔ 240037·240133**, **240091 ↔ 240044·240061·240069** | 각각 중간 **130.2~521.0분 / 128.2~513.0분**, TC-Z7·OP-Z7·TC-Z8·OP-Z8 | 뒤의 같은 존 상세 그림; 목표값·속도·두께 |
| **240069 ↔ 나머지 SPHC** | CP 피크 **574.5분**, 확대 **564.5~584.5분**. 피크를 제외해도 큰 std가 남는가? | 그림 10-C와 [그림 19-B의 피크 제거 민감도](#interval-review); 분석계·분위기 로그 |
| **240061 ↔ 나머지 SPHC** | **0~73.5분 승온**과 이후를 구분. **128~134.5분 TC-Z3↔OP-Z3** | 그림 10-D·19-A; 목표값·가동상태·수집 로그 |


In [ ]:
s6_page_indices=list(range(7,len(s6_time_rows)))
s6_reference_time_indices=s6_page_indices.copy()
s6_page_title='그림 10-B-시간-2. 참고 비교 — 반복되는 차이도 비교군부터 확인'

# 진행률 구간의 값/변동 차이를 글자로 표시하고 다음 확인을 연결합니다.
s6_page_labels=[]
for s6_i in s6_page_indices:
    s6_lot,s6_channel=s6_time_rows[s6_i]
    s6_metric={
        'CP_roll_sd':'CP-Z4 std(최근 5분)','CP-Z4':'CP-Z4 수준',
        '가열_roll_width':'TC-Z3~TC-Z6 평균\n변동폭(최근 5분)',
        '가열·균열존':'TC-Z3~TC-Z6 평균온도','OP-Z3':'OP-Z3 출력 수준',
        '과시효대':'TC-Z7·TC-Z8 평균온도',
        'OP_roll_width':'OP-Z3 변동폭(최근 5분)'}[s6_channel]
    s6_page_labels.append(f'{s6_lot} · {s6_lots.loc[s6_lot,"grade"]}\n{s6_metric}')
s6_page_fig,(s6_page_ax,s6_page_summary)=plt.subplots(
    1,2,figsize=(18,.85*len(s6_page_indices)+3.4),sharey=True,
    gridspec_kw={'width_ratios':[8.5,3.7]})
s6_page_ax.imshow(s6_time_review[s6_page_indices],aspect='auto',
    cmap=ListedColormap([S6_REVIEW_STYLE.get(s,S6_MISSING_STYLE)['bg'] for s in s6_time_style_order]),
    norm=BoundaryNorm([-.5,.5,1.5,2.5,3.5],4),
    extent=(0,100,len(s6_page_indices)-.5,-.5))
for s6_local,s6_i in enumerate(s6_page_indices):
    s6_lot,s6_channel=s6_time_rows[s6_i]
    s6_is_variability=s6_channel in S6_TRACE_VARIABILITY
    for s6_j,s6_direction in enumerate(s6_time_state[s6_i]):
        if not np.isfinite(s6_direction):
            s6_label='-';s6_fg='#87919A'
        elif s6_direction==0:
            s6_label='-';s6_fg='#BBC2C8'
        else:
            s6_label=('큼' if s6_direction>0 else '작음') if s6_is_variability else ('높음' if s6_direction>0 else '낮음')
            s6_fg=S6_REVIEW_STYLE.get(s6_time_style_order[int(s6_time_review[s6_i,s6_j])],S6_MISSING_STYLE)['fg']
        s6_page_ax.text(s6_j*5+2.5,s6_local,s6_label,ha='center',va='center',
            fontsize=9.2,color=s6_fg,fontweight='bold' if s6_direction!=0 and np.isfinite(s6_direction) else 'normal')
s6_page_ax.set_yticks(range(len(s6_page_indices)),s6_page_labels,fontsize=11.2,color='#263238')
s6_page_ax.tick_params(axis='y',length=0,pad=12)
s6_page_ax.set_xticks(np.arange(0,101,10))
s6_page_ax.set_xticks(np.arange(0,101,5),minor=True)
s6_page_ax.set_yticks(np.arange(-.5,len(s6_page_indices)),minor=True)
s6_page_ax.grid(which='minor',color='#F1F3F5',lw=1)
s6_page_ax.grid(which='major',visible=False)
s6_page_ax.tick_params(which='minor',length=0)
s6_page_ax.set_xlabel('LOT 진행률 [%] · 한 칸 = LOT 길이의 5%',fontsize=11,labelpad=12)
s6_page_ax.add_patch(Rectangle((20,-.5),60,len(s6_page_indices),fill=False,edgecolor='#616B74',lw=1.2))
s6_page_ax.set_title('② 차이는 언제 나타나는가?',loc='left',fontweight='bold',fontsize=13,pad=32)
for s6_x,s6_label in [(10,'시작 0~20%'),(50,'주 비교: 중간 20~80%'),(90,'종료 80~100%')]:
    s6_page_ax.text(s6_x,1.02,s6_label,transform=s6_page_ax.get_xaxis_transform(),
        ha='center',fontsize=10,color='#66717D')
s6_page_summary.set_title('③ 얼마나 반복되며, 다음에 무엇을 볼까?',loc='left',
    fontweight='bold',fontsize=12.5,pad=32)
s6_page_summary.set_xlim(0,1);s6_page_summary.set_xticks([])
s6_page_summary.tick_params(axis='y',which='both',left=False,right=False,labelleft=False)
s6_page_summary.spines[['top','bottom','left','right']].set_visible(False)
s6_page_summary.grid(visible=False)
for s6_local,s6_i in enumerate(s6_page_indices):
    s6_lot,s6_channel=s6_time_rows[s6_i];s6_p=s6_time_results[s6_i]
    s6_middle_state=s6_time_state[s6_i,4:16]
    s6_high=int((s6_middle_state==1).sum());s6_low=int((s6_middle_state==-1).sum())
    s6_n=int(np.isfinite(s6_middle_state).sum());s6_count=max(s6_high,s6_low)
    s6_variability=s6_channel in S6_TRACE_VARIABILITY
    s6_high_word,s6_low_word=('큼','작음') if s6_variability else ('높음','낮음')
    s6_direction=s6_high_word if s6_high>=s6_low else s6_low_word
    s6_pct=100*s6_count/s6_n if s6_n else np.nan
    if not s6_n:
        s6_state='자료 없음';s6_count_line='비교할 중간 구간 없음';s6_pattern='-';s6_next='동일 강종 관측 확보'
    else:
        s6_state='기준 비해당' if not s6_count else '우선 확인' if s6_time_priority[s6_i] else '참고 확인'
        s6_count_line=f'{s6_high_word} {s6_high}/{s6_n}칸 · {s6_low_word} {s6_low}/{s6_n}칸'
        s6_pattern=(f'{s6_direction} {s6_pct:.0f}% · '+
            ('중간 전반' if s6_count==s6_n else '여러 중간 구간' if s6_p['지속 후보'] else '일부 구간')) if s6_count else '중간 구간의 차이 기준 미만'
        if s6_channel=='CP_roll_sd':s6_next='CP 피크 전후 확대'
        elif s6_channel=='CP-Z4':s6_next='수준과 피크·std를 구분'
        elif s6_lot==240061 and s6_channel=='가열_roll_width':s6_next='승온 후 변동 재확인'
        elif s6_lot==240061 and s6_channel=='가열·균열존':s6_next='승온 구간을 분리'
        elif s6_lot==240061 and s6_channel=='OP-Z3':s6_next='짧은 저출력은 별도 확인'
        elif s6_lot in [240121,240091]:s6_next='각 TC의 목표값 대조'
        elif s6_lot==240133:s6_next='240037과 직접 비교'
        elif s6_lots.loc[s6_lot,'grade_n']==2:
            s6_next='같은 강종 조건 대조 (n=2)' if s6_count else '두 LOT의 공통 수준 확인'
        else:s6_next='같은 강종 원자료 확인'
    s6_style=S6_REVIEW_STYLE.get(s6_state,S6_MISSING_STYLE)
    s6_page_summary.add_patch(Rectangle((.01,s6_local-.43),.98,.86,
        facecolor=s6_style['bg'],edgecolor=s6_style['edge'],lw=.7))
    s6_page_summary.text(.045,s6_local-.24,s6_count_line,va='center',fontsize=10.5,color=s6_style['fg'])
    s6_page_summary.text(.045,s6_local,s6_pattern,va='center',fontsize=11,
        fontweight='bold',color=s6_style['fg'])
    s6_page_summary.text(.045,s6_local+.24,f'다음: {s6_next}',va='center',fontsize=10.5,color='#425466')
s6_page_fig.suptitle(s6_page_title,x=.025,y=.983,ha='left',fontweight='bold',fontsize=16)
s6_page_fig.text(.025,.937,'목적: 같은 강종과의 차이가 중간 내내 이어지는지, 일부 시점에만 나타나는지 구분',
    fontsize=11,color='#425466')
s6_page_fig.text(.025,.899,
    '비교: 자신을 제외한 동일 강종 LOT의 같은 진행률 구간 중앙값 · 한 칸은 5%이며, 최근 5분은 변동 지표의 계산 창',
    fontsize=10.6,color='#66717D')
s6_page_fig.legend(handles=[Patch(fc=S6_REVIEW_STYLE.get(s,S6_MISSING_STYLE)['bg'],
    ec=S6_REVIEW_STYLE.get(s,S6_MISSING_STYLE)['fg'],label=s) for s in ['우선 확인','참고 확인','기준 비해당','자료 없음']],
    loc='upper left',bbox_to_anchor=(.218,.861),ncol=4,frameon=False,fontsize=10)
s6_page_fig.text(.025,.765,'① LOT · 비교 컬럼',fontsize=12.5,fontweight='bold',color='black')
s6_page_fig.text(.025,.058,
    '높음/낮음 = 값의 수준 차이 · 큼/작음 = std 또는 변동폭의 차이 · 색은 점검 우선순위 · 흰색 -는 기준 미만이며 정상 판정이 아님',
    fontsize=10,color='#66717D')
s6_page_fig.text(.025,.028,
    '오른쪽은 중간 20~80%의 12칸 집계 · 높은/낮은 방향을 따로 셈 · 한 칸의 중앙값 비교로 짧은 피크·저출력이 보이지 않을 수 있음',
    fontsize=10,color='#66717D')
s6_page_fig.subplots_adjust(left=.225,right=.985,top=.73,bottom=.15,wspace=.065)
save_and_show_figures()

assert sorted(s6_core_time_indices+s6_reference_time_indices)==list(range(len(s6_time_rows)))


#### 그림 10-B-시간-2 — 참고 차이는 비교군과 조건부터 확인

**① 읽는 순서:** n=2 LOT는 상대 LOT 하나와 비교한 참고입니다. 구간 차이가 반복돼도 어느 LOT가 특이한지 판정하기 어렵습니다. 240133은 n=3이지만 높은 240121이 비교 기준을 올리는 영향도 함께 확인합니다.

**② 확인한 결과**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| **240024 ↔ 240108 · SPFH590 / TC-Z7·TC-Z8 평균** | 두 LOT 모두 **0/12칸**, 중간 중앙값 약 **326.15 / 326.19℃** | 전체 LOT보다 낮아도 같은 SPFH590 두 LOT끼리는 비슷함. 공통 운전조건 참고 |
| 두 SPFH590 / TC-Z3~TC-Z6 평균 변동폭 | 큰/작은 방향 각각 **2/12칸(17%)** | 변동 차이는 일부 구간에 한정. n=2이므로 조건을 맞춘 참고 비교 |
| **240038 ↔ 240114 · SPCEN / TC-Z7·TC-Z8 평균** | 각각 낮음/높음 **12/12칸(100%)** | 두 LOT의 지속적인 수준 차이는 있음. 목표값·규격·속도를 확인해야 어느 쪽을 상세 점검할지 정할 수 있음 |
| 두 SPCEN / OP-Z3 변동폭 | 각각 큰/작은 방향 **2/12칸(17%)** | 큰 대표 변동폭 차이와 시간 내내 큰 변동은 다름 |
| **240133 · SPCE / TC-Z7·TC-Z8 평균** | 낮은 방향 **10/12칸(83%)**, 중간 대표값 차이 기준 미만 | 비교 중앙값에 높은 240121이 포함됨. **240037과 직접 비교**하여 기준 구성의 영향을 확인 |

**결론:** 참고 그림의 반복 차이는 **비교군·운전조건을 확인할 근거**입니다. n=2에서 특정 LOT를 고장 후보로 확정하거나, 240133의 낮은 방향만 보고 저온 이상을 붙이지 않습니다.

**③ 후속 작업**

| 대상·비교군 | 다음 확인 | 필요한 자료 |
|---|---|---|
| 240024 ↔ 240108 · SPFH590 | 각 TC의 공통 수준, 냉각 진폭·주기와 목표값 | n=2 참고 상세 그림; 추가 동일 조건 LOT |
| 240038 ↔ 240114 · SPCEN | TC-Z7·TC-Z8 목표값, OP-Z3 큰/작은 구간. 240114 종료 하강은 분리 | 목표온도·소재·속도·정지 기록 |
| 240133 ↔ 240037 · SPCE | 높은 240121을 뺀 직접 비교에서 공통 수준이 유지되는가? | SPCE 원자료·목표값; 비교군 구성 점검 |


### 상세 그림에서 비교 컬럼이 다른 이유

**전체 후보 단계에서는 모든 LOT에 같은 8개 지표를 적용했습니다.** 아래 상세 그림은 그중 **같은 강종 비교에서 남은 차이**와 **그 차이를 해석할 주변 조건**을 확대합니다. LOT마다 임의의 센서를 골라 점수를 바꾼 것이 아닙니다.

| 사례·비교군 | 선택한 컬럼과 이유 | 컬럼 사이의 연관성 | 이 비교로 구분할 것 |
|---|---|---|---|
| **240061 ↔ 240044·240069·240091 · SPHC** | **TC-Z3·TC-Z4·TC-Z5·TC-Z6**, **OP-Z3**: 공통 승온·변동폭과 OP-Z3 초반 결측이 남음 | TC 네 개는 공통 온도 상승의 대조군. **TC-Z3↔OP-Z3은 같은 존**의 상태·출력 짝 | 여러 존이 함께 오르는 승온과 TC-Z3만의 이탈; 실제 출력 변화와 수집 결측 |
| **240069 ↔ 나머지 SPHC** | **CP-Z4·CP-Z4 std**, TC-Z3~Z6 평균·OP-Z3: CP std가 **13.5배** | CP는 분위기 관련 계측, TC는 온도, OP는 제어출력. 서로 다른 신호군에서 같은 시각 변화가 있었는지 확인 | CP 피크만의 변화인지, 온도·출력까지 동반하는 공통 변화인지 |
| **240121 ↔ 240037·240133 · SPCE**; **240091 ↔ 나머지 SPHC** | **TC-Z7·TC-Z8**: 각각의 높은 수준과 평균의 **12/12칸 지속 차이**. 출력은 **OP-Z7·OP-Z8** | **TC-Z7↔OP-Z7, TC-Z8↔OP-Z8은 같은 존**. 이전 OP-Z3는 다른 존의 출력 조건을 보는 참고값일 뿐 직접 제어 짝은 아님 | 지속 온도 차이가 두 존 공통인지, 해당 존 출력도 달랐는지. 목표값·운전조건 차이와 점검 후보를 구분 |
| **240024 ↔ 240108 · SPFH590** | **TC-Z7·TC-Z8**, **TC-OUT1·TC-OUT2**: 두 LOT의 낮은 수준이 반복되는지 확인 | 서로 다른 공정의 온도 수준이 동일 강종에서 반복되는지 보는 조건 비교. 어느 한 존이 다른 존의 원인이라는 뜻은 아님 | 같은 강종의 공통 조건과 한 LOT만 다른 값을 구분. n=2이므로 참고 |
| **240058 · DP590** | **TC-Z3·TC-Z4·TC-Z5·TC-Z6**, **OP-Z3**, **CP-Z4**: 종료부에 함께 하강 | 공통 운전 전환이 여러 신호군에 나타나는지 확인 | 종료 상태 변화와 개별 센서 이상. n=1이라 강종 효과·LOT 특이성은 분리 불가 |

**평균을 먼저 본 이유:** 인접 TC의 공통 수준을 간단히 찾기 위해서입니다. 평균은 한 컬럼의 이탈을 감출 수 있으므로, 주요 후보는 앞의 **개별 TC 표**와 아래 **TC→같은 존 OP** 그림으로 다시 확인합니다. 온도는 ℃, 출력은 %이므로 선의 높이 자체를 직접 비교하지 않습니다.

**TC-OUT 주의:** 냉각 신호는 주기적으로 진동합니다. 두 LOT의 진행률 50%는 실제 경과 시간과 진동 위상이 같다는 뜻이 아닙니다. 진행률별 순간 차이는 위상 때문에 생길 수 있어, 냉각은 **수준·진폭·주기**를 따로 읽습니다. 이 자료에는 소재의 존 간 이동시간이 없어 다른 존의 같은 시각 변화를 동일 소재의 전후 변화로 확정하지 않습니다.


#### 대표 후보의 실제 시계열 확인

아래 상세 그래프는 확인 항목을 설명하는 대표 사례입니다. **[실제 확대 구간과 근거 점검](#interval-review)**에서 분·시각·비교군을 지정하고 국소 변화와 창 길이의 영향도 확인합니다. 전체 목록은 마지막 10-H의 **14개 LOT 확인표**에 정리합니다. 각 그래프의 검정 점선은 자기 제외 비교 중앙값이고 옅은 파란 구간은 중간 20~80%입니다.

In [ ]:
# 그림 10-C: 이 셀의 matplotlib 코드로 240069와 동일 강종 LOT를 직접 그립니다.
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

lot69_no = 240069
lot69_grade = s6_lots.loc[lot69_no, 'grade']
lot69_peers = s6_peer_ids[lot69_no]  # SPHC: 240044, 240061, 240091
lot69_data = s6_analysis[s6_analysis.LOT_NO.eq(lot69_no)]
lot69_channels = [
    ('가열·균열존', 'TC-Z3~TC-Z6 평균온도', '온도 [℃]'),
    ('OP-Z3', 'OP-Z3 · 3존 제어출력', '제어출력 [%]'),
    ('CP-Z4', 'CP-Z4 · 카본포텐셜', 'CP-Z4 [무차원]'),
    ('CP_roll_sd', 'CP-Z4 std(5분)', 'CP-Z4 std [무차원]'),
]
lot69_styles = ['-', '--', ':']
lot69_fig, lot69_axes = plt.subplots(4, 1, figsize=(16, 15.9))
lot69_fig.subplots_adjust(left=.09, right=.98, top=.843, bottom=.05, hspace=.60)
lot69_fig.suptitle(
    '그림 10-C. SPHC 240069 — CP-Z4 피크와 지속적인 LOT 차이 구분',
    x=.09, y=.992, ha='left', va='top', fontsize=15, fontweight='bold'
)

# 비교 LOT는 회색, 확인 LOT는 강종 색, 자기 제외 중앙값은 검정 점선입니다.
lot69_handles = [Line2D([0], [0], color=colors[lot69_grade], lw=2, label='240069 · 확인 LOT')]
lot69_handles += [
    Line2D([0], [0], color='#9aa3ad', ls=lot69_styles[j], label=str(peer))
    for j, peer in enumerate(lot69_peers)
]
lot69_handles += [
    Line2D([0], [0], color='#263238', ls='--', label='자기 제외 · 5% 칸 중앙값'),
    Patch(facecolor='#edf4fa', label='중간 20~80%'),
]
lot69_fig.legend(handles=lot69_handles, loc='upper left',
                 bbox_to_anchor=(.09, .964), ncol=4, frameon=False, fontsize=9)
lot69_fig.text(.09, .922, 'SPHC n=4 · 같은 강종 비교 · LOT별 실제 수집 길이를 0~100%로 정렬',
              fontsize=10, color='#425466')

for lot69_ax, (lot69_col, lot69_title, lot69_unit) in zip(lot69_axes, lot69_channels):
    lot69_ax.axvspan(20, 80, color='#edf4fa', zorder=0)

    # 연속구간마다 따로 그려 실제 계측 공백을 보호합니다.
    for lot69_j, lot69_peer in enumerate(lot69_peers):
        lot69_peer_data = s6_analysis[s6_analysis.LOT_NO.eq(lot69_peer)]
        for _, lot69_segment in lot69_peer_data.groupby('SEG'):
            lot69_ax.plot(lot69_segment.progress_pct, lot69_segment[lot69_col],
                          color='#9aa3ad', ls=lot69_styles[lot69_j], lw=.85, alpha=.65)
    for _, lot69_segment in lot69_data.groupby('SEG'):
        lot69_ax.plot(lot69_segment.progress_pct, lot69_segment[lot69_col],
                      color=colors[lot69_grade], lw=1.5, alpha=.95, zorder=3)
    lot69_ax.plot(np.arange(20) * 5 + 2.5, s6_bin_ref[lot69_col].loc[lot69_no],
                  color='#263238', ls='--', lw=1.2, zorder=4)

    # 소제목과 확인 내용을 선 바깥의 서로 다른 줄에 배치합니다.
    lot69_ax.set_title(lot69_title, loc='left', color='black',
                       fontweight='bold', fontsize=12, pad=38)
    lot69_p = s6_persistence[
        s6_persistence.LOT.eq(lot69_no) & s6_persistence['채널'].eq(lot69_col)
    ].iloc[0]
    lot69_note = (
        f"LOT 240069 · 중간 유효 {int(lot69_p['유효 중간 칸'])}칸 중 "
        f"기준을 넘는 같은 방향 차이 {int(lot69_p['같은 방향 차이 칸'])}칸"
    )
    if lot69_col == 'CP-Z4':
        lot69_cp_delta = s6_delta['중간 20~80%'].loc[lot69_no, 'CP-Z4']
        lot69_note += f" · CP-Z4 중앙값 Δ {lot69_cp_delta:+.6f} · 최대 피크 {lot69_data['CP-Z4'].max():.4f}"
    elif lot69_col == 'CP_roll_sd':
        lot69_sd_ratio = (
            s6_middle.loc[lot69_no, 'CP_sd']
            / s6_peer_ref['중간 20~80%'].loc[lot69_no, 'CP_sd']
        )
        lot69_note += f" · 중간 CP-Z4 std {lot69_sd_ratio:.1f}배 · 이동 SD는 국소 변화 확인"
    else:
        lot69_note += f" · 중간 중앙값 Δ {s6_delta['중간 20~80%'].loc[lot69_no, lot69_col]:+.2f}"
    lot69_ax.text(0, 1.055, lot69_note, transform=lot69_ax.transAxes,
                  ha='left', va='bottom', fontsize=9, clip_on=False,
                  color='#c0392b' if lot69_col.startswith('CP') else '#425466')

    # CP-Z4 피크의 실제 관측 지점에 붉은 원과 LOT 번호를 붙입니다.
    if lot69_col in ['CP-Z4', 'CP_roll_sd']:
        lot69_valid = lot69_data[lot69_data[lot69_col].notna()]
        lot69_peak = lot69_valid.loc[lot69_valid[lot69_col].idxmax()]
        lot69_x, lot69_y = lot69_peak.progress_pct, lot69_peak[lot69_col]
        lot69_ax.scatter([lot69_x], [lot69_y], s=40, facecolors='none',
                         edgecolors='#c0392b', lw=1.4, zorder=5)
        lot69_ax.annotate('240069', (lot69_x, lot69_y), xytext=(-12, 14),
                          textcoords='offset points', ha='right',
                          fontsize=9, fontweight='bold', color='#c0392b',
                          arrowprops={'arrowstyle': '-', 'color': '#c0392b', 'lw': .8},
                          bbox={'fc': 'white', 'ec': 'none', 'alpha': .8})

    lot69_ax.set_ylabel(lot69_unit)
    lot69_ax.set_xlabel('LOT 진행률 [%]')
    lot69_ax.set_xlim(0, 100)
    lot69_ax.margins(y=.13)
    lot69_ax.grid(alpha=.18)
    lot69_ax.spines[['top', 'right']].set_visible(False)

lot69_fig.text(.09, .009,
               '지속성 탐색: 중간 12칸 중 8칸 이상 유효, 같은 방향 차이가 70% 이상인 경우 · 붉은 원은 실제 피크 위치',
               fontsize=9, color='#425466')
save_and_show_figures()


#### 그림 10-C · SPHC 240069 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 선·배경 | 보라색은 240069, 회색은 240044·240061·240091, 검정 점선은 자기 제외 진행률 칸 중앙값. 파랑 배경은 중간 20~80%. |
| CP 두 패널 | CP-Z4 값과 최근 5분 std를 따로 봄. 붉은 원은 피크 위치. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| CP-Z4 피크 | 최대 30초 평균 **0.8944**, 시작 후 **574.5분 / 진행률 70.7%** | 눈에 띄는 CP 피크가 존재. |
| 중간 CP-Z4 std | **0.0363 / 비교 0.0027 = 13.5배**; 높은 이동 std는 **4/12칸(33%)** | 같은 강종 차이는 남지만 변동이 시간 내내 지속되지는 않음. |
| CP-Z4 수준·온도·출력 | CP 중앙값 차이 **+0.000096**. 평균온도·OP-Z3의 같은 방향 기준 초과는 중간 각각 **0/12칸** | CP·온도·OP의 지속적인 동시 변화가 확인된 사례는 아님. |

**결론:** **A. CP-Z4 변동성 상세분석 후보**입니다. 짧은 피크와 지속적인 수준 상승을 구분했습니다. **[그림 19-B의 추가 점검](#interval-review)**에서는 피크 ±10분을 제외하면 std 비율이 **13.48배에서 1.98배로 감소**합니다. 큰 중간 std의 상당 부분은 피크 주변에 의존합니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240069 ↔ 240044·240061·240091 | 564.5~584.5분의 30초 확대·피크 제거 민감도는 그림 19-B에서 확인. 1초 CP와 동시 이벤트를 대조 | 1초 원자료·분위기 제어·분석계 교정 기록 |
| 240069 / TC-Z3·OP-Z3 | CP 피크와 온도·출력 변화의 실제 시각이 겹치는지 확인 | 동시각 시계열; 이후 정비 맥락은 그림 10-G |
| 240069 / BM 기록 | LOT 종료 2.3시간 뒤 BM·ELE(절연 저하)와 관련 계측 점검 | 정비 상세·장치 대응 관계; 시간 인접만으로 원인 확정 불가 |


In [ ]:
# 그림 10-C의 수치 근거: 그래프와 해석을 읽은 뒤 비교합니다.
display(s6_comparison[s6_comparison.LOT.eq(240069)].round(6))
display(s6_persistence[s6_persistence.LOT.eq(240069)].round(4))


In [ ]:
# 계산된 LOT 시계열과 동일 강종 비교선을 이 셀에서 직접 그립니다.
case_lot = 240061
case_columns = ['가열·균열존', 'OP-Z3']
case_title = '그림 10-D. SPHC 240061 — 초반 승온·결측을 분리한 동일 강종 비교'
case_peers = s6_peer_ids[case_lot]
case_grade = s6_lots.loc[case_lot, 'grade']
case_n = len(case_columns)
case_height = 3.2 * case_n + 3.1
case_fig, case_axes = plt.subplots(case_n, 1, figsize=(16, case_height), squeeze=False)
case_axes = case_axes[:, 0]
case_fig.subplots_adjust(left=0.09, right=0.98, top=1 - 2.5 / case_height, bottom=0.75 / case_height, hspace=0.6)
case_fig.suptitle(case_title, x=0.09, y=1 - 0.12 / case_height, ha='left', va='top', fontsize=15, fontweight='bold')
case_styles = ['-', '--', ':', '-.']
case_handles = [Line2D([0], [0], color=colors[case_grade], lw=2, label=f'{case_lot} · 확인 LOT')]
case_handles += [Line2D([0], [0], color='#9aa3ad', lw=1.2, ls=case_styles[case_i % 4], label=str(case_peer)) for case_i, case_peer in enumerate(case_peers)]
if case_peers:
    case_handles += [Line2D([0], [0], color='#263238', lw=1.5, ls='--', label='자기 제외 · 5% 칸 중앙값')]
case_handles += [Patch(facecolor='#edf4fa', label='중간 20~80%')]
if 'OP-Z3' in case_columns and s6_screen.loc[case_lot, 'OP-Z3 원본 결측행'] > 0:
    case_handles += [Patch(facecolor='#cfd4da', label='OP-Z3 원본 결측')]
case_fig.legend(handles=case_handles, loc='upper left', bbox_to_anchor=(0.09, 1 - 0.54 / case_height), ncol=4, frameon=False, fontsize=9)
case_fig.text(0.09, 1 - 1.25 / case_height, f"{case_grade} n={s6_lots.loc[case_lot, 'grade_n']} · {s6_lots.loc[case_lot, 'support']} · 공백은 연결하지 않음 · 진행률은 실제 수집 길이 기준", fontsize=10, color='#425466')
case_names = {'가열·균열존': 'TC-Z3~TC-Z6 평균온도', '과시효대': 'TC-Z7·TC-Z8 평균온도', '냉각대': 'TC-OUT1·TC-OUT2 평균온도', 'OP-Z3': 'OP-Z3 · 3존 제어출력', 'CP-Z4': 'CP-Z4 · 카본포텐셜', 'CP_roll_sd': 'CP-Z4 std(5분)'}
for case_ax, case_c in zip(case_axes, case_columns):
    case_ax.axvspan(20, 80, color='#edf4fa', zorder=0)
    for case_i, case_peer in enumerate(case_peers):
        for case__, case_seg in s6_analysis[s6_analysis.LOT_NO.eq(case_peer)].groupby('SEG'):
            case_ax.plot(case_seg.progress_pct, case_seg[case_c], color='#9aa3ad', lw=0.85, alpha=0.65, ls=case_styles[case_i % 4])
    case_d = s6_analysis[s6_analysis.LOT_NO.eq(case_lot)]
    for case__, case_seg in case_d.groupby('SEG'):
        case_ax.plot(case_seg.progress_pct, case_seg[case_c], color=colors[case_grade], lw=1.5, alpha=0.95, zorder=3)
    if case_peers:
        case_ax.plot(np.arange(20) * 5 + 2.5, s6_bin_ref[case_c].loc[case_lot], color='#263238', ls='--', lw=1.2, zorder=4)
    if case_c == 'OP-Z3':
        for case_start, case_end, case_count in missing_runs(case_lot, 'OP-Z3'):
            case_ax.axvspan(100 * case_start / (60 * s6_lots.loc[case_lot, 'minutes']), 100 * (case_end + 1) / (60 * s6_lots.loc[case_lot, 'minutes']), color='#cfd4da', zorder=2)
    case_ax.set_title(case_names[case_c], loc='left', color='black', fontweight='bold', fontsize=12, pad=38)
    case_p = s6_persistence[s6_persistence.LOT.eq(case_lot) & s6_persistence['채널'].eq(case_c)].iloc[0]
    if case_peers:
        case_pct = f"{case_p['중간 지속 비율 [%]']:.0f}%" if pd.notna(case_p['중간 지속 비율 [%]']) else '—'
        case_text = f"LOT {case_lot} · 중간 유효 {int(case_p['유효 중간 칸'])}칸 중 같은 방향 차이 {int(case_p['같은 방향 차이 칸'])}칸 ({case_pct})"
        if case_c in S6_FEATURES:
            case_text += f" · 중간 요약 Δ {s6_value(s6_delta['중간 20~80%'].loc[case_lot, case_c], case_c, True)}"
        if case_c == 'CP_roll_sd':
            case_text += ' · 2배/1/2 기준 · 피크와 지속 상승 구분'
        case_cp_peak_note = case_c == 'CP-Z4' and s6_diff_middle.loc[case_lot, 'CP_sd']
        if case_cp_peak_note:
            case_text += f' · 최대 피크 {case_d[case_c].max():.4f}'
        if case_c == 'OP-Z3' and s6_screen.loc[case_lot, 'OP-Z3 원본 결측행']:
            case_text += f" · 원본 결측 {s6_screen.loc[case_lot, 'OP-Z3 원본 결측행']}행"
        case_ax.text(0, 1.055, case_text, transform=case_ax.transAxes, ha='left', va='bottom', color='#c0392b' if case_cp_peak_note or case_p['같은 방향 차이 칸'] or (case_c == 'OP-Z3' and s6_screen.loc[case_lot, 'OP-Z3 원본 결측행']) else '#425466', fontsize=9, clip_on=False)
        case_delta = s6_bin_delta[case_c].loc[case_lot]
        if case_c == 'CP_roll_sd':
            case_ratio = s6_bin_values[case_c].loc[case_lot].div(s6_bin_ref[case_c].loc[case_lot].replace(0, np.nan))
            case_eligible = case_delta.notna() & (case_ratio.ge(2) | case_ratio.le(0.5))
        else:
            case_eligible = case_delta.abs().ge(s6_level_floor[case_c])
        case_point = None
        if case_c in ['CP-Z4', 'CP_roll_sd'] and s6_diff_middle.loc[case_lot, 'CP_sd']:
            case_valid = case_d[case_d[case_c].notna()]
            if len(case_valid):
                case_point = case_valid.loc[case_valid[case_c].idxmax()]
        elif case_eligible.any():
            case_middle_eligible = case_eligible & pd.Series([20 <= 5 * case_b + 2.5 <= 80 for case_b in range(20)], index=range(20))
            case_selected = case_middle_eligible if case_middle_eligible.any() else case_eligible
            case_b = case_delta[case_selected].abs().idxmax()
            case_valid = case_d[case_d.progress_bin.eq(case_b) & case_d[case_c].notna()]
            if len(case_valid):
                case_point = case_valid.loc[(case_valid.progress_pct - (5 * case_b + 2.5)).abs().idxmin()]
        if case_point is not None:
            case_x, case_y = (case_point.progress_pct, case_point[case_c])
            case_ax.scatter([case_x], [case_y], s=40, facecolors='none', edgecolors='#c0392b', lw=1.4, zorder=5)
            case_point_label = f'{case_lot}' + (' · 초반' if case_x < 20 else ' · 종료부' if case_x > 80 else '')
            case_ax.annotate(case_point_label, (case_x, case_y), xytext=(-12 if case_x > 70 else 12, 14), textcoords='offset points', ha='right' if case_x > 70 else 'left', fontsize=9, fontweight='bold', color='#c0392b', arrowprops={'arrowstyle': '-', 'color': '#c0392b', 'lw': 0.8}, bbox={'fc': 'white', 'ec': 'none', 'alpha': 0.75})
    else:
        case_ax.text(0, 1.055, f'LOT {case_lot} · 동일 강종 비교 LOT 없음 → 운전 전환 확인, 강종/LOT 차이 분리 불가', transform=case_ax.transAxes, color='#c0392b', fontsize=9, clip_on=False)
    case_ax.set_ylabel('CP-Z4 [무차원]' if case_c == 'CP-Z4' else 'CP-Z4 std [무차원]' if case_c == 'CP_roll_sd' else '제어출력 [%]' if case_c.startswith('OP') else '온도 [℃]')
    case_ax.set_xlabel('LOT 진행률 [%] · 시간 보간 없음')
    case_ax.set_xlim(0, 100)
    case_ax.margins(y=0.13)
    case_ax.grid(alpha=0.18)
    case_ax.spines[['top', 'right']].set_visible(False)
case_fig.text(0.09, 0.14 / case_height, '지속 후보: 중간 12칸 중 8칸 이상 비교 가능하고 같은 방향 차이가 70% 이상. 붉은 원은 실제 확인 지점이며 지속 구간 표시가 아닙니다.', fontsize=9, color='#425466')
save_and_show_figures()
display(s6_comparison[s6_comparison.LOT.eq(240061)].round(6))
display(s6_persistence[s6_persistence.LOT.eq(240061)].round(4))


#### 그림 10-D · SPHC 240061 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 비교선 | 240061과 240044·240069·240091. 위는 TC-Z3~TC-Z6 평균, 아래는 OP-Z3. |
| 초반 회색·중간 파랑 | 회색은 OP-Z3 원본 결측. 중간 20~80%에도 긴 승온 일부가 포함됨. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| OP-Z3 결측 | **179행 / 0~183초**, 최초 유효 **184초** | 수집 품질 점검을 먼저 수행할 대상. |
| TC-Z3~TC-Z6 평균 변동폭 | 전체 **25.08℃ → 중간 9.78℃**; 중간 자기 제외 차이 **+8.82℃** | 끝 20%·처음 20%를 제외해도 승온 영향이 모두 제거되지는 않음. |
| 변동 지속성 | 최근 5분 평균온도 변동폭이 큰 중간 구간 **6/12칸(50%)** | 전체 중간 구간의 지속 차이로 확정하지 않음. |

**결론:** **C. 결측 점검 우선 + 승온을 분리할 상세 사례**로 유지합니다. TC-Z3 단독 이탈과 공통 승온은 나눠 봅니다. **[그림 19-A의 추가 확대](#interval-review)**에서는 승온 후 **128~134.5분 OP-Z3<1%**와 **131분 TC-Z3 상대 상승**이 겹칩니다. 이를 고장으로 확정하려면 목표값과 제어 모드를 확인해야 합니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240061 ↔ 240044; 대조 240069·240091 | 초반 OP-Z3 결측 발생·해소 조건을 대조 | [앞의 결측 확대](#s6-missing-pair), 태그·PLC·수집 로그 |
| 240061 ↔ 240044·240069·240091 | 승온이 끝난 뒤 TC-Z3·Z4·Z5·Z6와 OP-Z3를 다시 비교 | 실제 재가동 시각·목표온도; 뒤의 그림 11·15 |
| 240061 / 안정 비교 구간 | TC-Z3 차분 std와 OP-Z3 저출력 발생 시점을 대조 | 마지막 LOT 선정 보고의 동일 상태 기준 |


In [ ]:
# 같은 존의 TC와 OP를 이 셀에서 직접 그립니다.
zone_case_lot = 240121
zone_case_grade = s6_lots.loc[zone_case_lot, 'grade']
zone_case_peers = s6_peer_ids[zone_case_lot]
zone_case_data = s6_analysis[s6_analysis.LOT_NO.eq(zone_case_lot)]
zone_case_fig, zone_case_axes = plt.subplots(4, 1, figsize=(16, 15.6), sharex=True)
zone_case_fig.subplots_adjust(left=.09, right=.98, top=.855, bottom=.05, hspace=.62)
zone_case_fig.suptitle(
    f'그림 10-E. {zone_case_grade} {zone_case_lot} — TC-Z7·TC-Z8과 같은 존 OP-Z7·OP-Z8 비교',
    x=.09, y=.992, ha='left', fontsize=15, fontweight='bold')
zone_styles = ['-', '--', ':']
zone_handles = [Line2D([], [], color=colors[zone_case_grade], lw=2, label=f'{zone_case_lot} · 확인 LOT')]
zone_handles += [Line2D([], [], color='#9AA3AD', ls=zone_styles[j], label=str(peer))
                 for j, peer in enumerate(zone_case_peers)]
zone_handles += [Line2D([], [], color='#263238', ls='--', label='자기 제외 · 5% 칸 중앙값'),
                 Patch(facecolor='#EDF4FA', label='중간 20~80%')]
zone_case_fig.legend(handles=zone_handles, loc='upper left', bbox_to_anchor=(.09,.965),
                     ncol=4, frameon=False, fontsize=9)
zone_case_fig.text(.09,.922,
    'TC는 해당 존의 온도, OP는 같은 존의 출력 · 온도 수준 차이와 출력 조건을 함께 확인 · 인과 효과로 해석하지 않음',
    fontsize=9.5, color='#6B7280')
for zone_ax, zone_col in zip(zone_case_axes, S6_ZONE_COLUMNS):
    zone_ax.axvspan(20, 80, color='#EDF4FA', zorder=0)
    for j, peer in enumerate(zone_case_peers):
        for _, segment in s6_analysis[s6_analysis.LOT_NO.eq(peer)].groupby('SEG'):
            zone_ax.plot(segment.progress_pct, segment[zone_col], color='#9AA3AD',
                         ls=zone_styles[j], lw=.85, alpha=.65)
    for _, segment in zone_case_data.groupby('SEG'):
        zone_ax.plot(segment.progress_pct, segment[zone_col], color=colors[zone_case_grade],
                     lw=1.4, zorder=3)
    zone_ax.plot(np.arange(20)*5+2.5, s6_zone_bin_reference[zone_col].loc[zone_case_lot],
                 color='#263238', ls='--', lw=1.1, zorder=4)
    is_temperature = zone_col.startswith('TC')
    zone_unit = '℃' if is_temperature else '%'
    zone_diff_unit = '℃' if is_temperature else '%p'
    zone_delta = s6_zone_middle.loc[zone_case_lot, zone_col] - s6_zone_reference.loc[zone_case_lot, zone_col]
    zone_ax.set_title(f'{zone_col} · {zone_col[-1]}존 '+('온도' if is_temperature else '제어출력'),
                      loc='left', fontsize=12, fontweight='bold', color='black', pad=38)
    zone_ax.text(0,1.055,
        f'LOT {zone_case_lot} · 중간 중앙값 {s6_zone_middle.loc[zone_case_lot,zone_col]:.3f}{zone_unit}'
        f' · 동일 강종 자기 제외 기준 {s6_zone_reference.loc[zone_case_lot,zone_col]:.3f}{zone_unit}'
        f' · 차이 {zone_delta:+.3f}{zone_diff_unit}',
        transform=zone_ax.transAxes, ha='left', va='bottom', fontsize=9, clip_on=False,
        color='#B4232F' if is_temperature else '#6B7280')
    if is_temperature:
        available=zone_case_data[zone_case_data.progress_pct.between(20,80)&zone_case_data[zone_col].notna()]
        point=available.loc[(available.progress_pct-50).abs().idxmin()]
        zone_ax.scatter([point.progress_pct],[point[zone_col]],s=42,facecolors='none',edgecolors='#B4232F',zorder=5)
        zone_ax.annotate(str(zone_case_lot),(point.progress_pct,point[zone_col]),xytext=(12,12),
                         textcoords='offset points',color='#B4232F',fontsize=9,
                         bbox={'fc':'white','ec':'none','alpha':.8})
    zone_ax.set_ylabel('온도 [℃]' if is_temperature else '제어출력 [%]')
    zone_ax.set_xlabel('LOT 진행률 [%] · 시간 보간 없음')
    zone_ax.tick_params(axis='x',labelbottom=True)
    zone_ax.set_xlim(0,100); zone_ax.margins(y=.16); zone_ax.grid(alpha=.15)
    zone_ax.spines[['top','right']].set_visible(False)
zone_case_fig.text(.09,.01,'출력 차이는 제어·운전조건의 참고값입니다. 목표값·속도·두께 없이 온도 차이의 원인을 확정하지 않습니다.',
                   fontsize=9,color='#6B7280')
save_and_show_figures()
display(s6_zone_case_stats[s6_zone_case_stats.LOT.eq(zone_case_lot)].round(4))


#### 그림 10-E · SPCE 240121 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 비교선 | 240121과 같은 SPCE 240037·240133. 검정 점선은 두 비교 LOT의 진행률별 중앙값. |
| 중간 20~80% | TC-Z7→OP-Z7, TC-Z8→OP-Z8 순으로 각 TC 수준과 같은 존 출력 조건을 확인. |

**컬럼 선택 이유:** 두 TC의 지속적인 높은 수준을 확인한 뒤, 직접 같은 존의 출력인 OP-Z7·OP-Z8을 대조합니다. OP-Z3는 다른 존의 참고 조건이며 두 TC의 직접 제어출력이 아닙니다.

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| TC-Z7·TC-Z8 평균 중앙값 | **333.42℃ / 비교 330.34℃ → +3.08℃** | 같은 SPCE 안에서도 높은 온도 수준. |
| 시간 지속성 | 중간 **12/12칸(100%)**에서 높은 방향 차이 | 일시적 피크보다 지속적인 조건 차이. |
| 개별 TC·다른 신호군 | TC-Z7 **+3.144℃**, TC-Z8 **+3.008℃**. OP·CP에는 동일한 차이 근거가 남지 않음 | 두 TC가 함께 높지만 다중신호 지속 이상으로 확대하지 않음. |
| **240121 / OP-Z7·OP-Z8** | 같은 강종 중간 중앙값 차이 **OP-Z7 -1.687%p**, **OP-Z8 -0.105%p** | 두 TC가 높다고 해당 존 출력도 더 높게 유지된 것은 아님. 목표값·열부하·제어조건 대조 필요 |

**결론:** **A. TC-Z7·TC-Z8의 지속 수준 차이 상세분석 후보**입니다. 설비 이상 여부는 운전조건 대조 후 판단합니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240121 ↔ 240037·240133 | TC-Z7↔OP-Z7, TC-Z8↔OP-Z8의 목표값·속도·두께와 제어조건을 대조 | 운전 설정·소재 규격 이력 |
| 240121 / 11월 계획수리 | 수리 후 13.3일이라는 위치와 동일 강종 반대편 관측 유무 확인 | 그림 10-G; 수리 전 동일 SPCE 비교 LOT 확보 |


In [ ]:
# 같은 존의 TC와 OP를 이 셀에서 직접 그립니다.
zone_case_lot = 240091
zone_case_grade = s6_lots.loc[zone_case_lot, 'grade']
zone_case_peers = s6_peer_ids[zone_case_lot]
zone_case_data = s6_analysis[s6_analysis.LOT_NO.eq(zone_case_lot)]
zone_case_fig, zone_case_axes = plt.subplots(4, 1, figsize=(16, 15.6), sharex=True)
zone_case_fig.subplots_adjust(left=.09, right=.98, top=.855, bottom=.05, hspace=.62)
zone_case_fig.suptitle(
    f'그림 10-E-참고. {zone_case_grade} {zone_case_lot} — TC-Z7·TC-Z8과 같은 존 OP-Z7·OP-Z8 비교',
    x=.09, y=.992, ha='left', fontsize=15, fontweight='bold')
zone_styles = ['-', '--', ':']
zone_handles = [Line2D([], [], color=colors[zone_case_grade], lw=2, label=f'{zone_case_lot} · 확인 LOT')]
zone_handles += [Line2D([], [], color='#9AA3AD', ls=zone_styles[j], label=str(peer))
                 for j, peer in enumerate(zone_case_peers)]
zone_handles += [Line2D([], [], color='#263238', ls='--', label='자기 제외 · 5% 칸 중앙값'),
                 Patch(facecolor='#EDF4FA', label='중간 20~80%')]
zone_case_fig.legend(handles=zone_handles, loc='upper left', bbox_to_anchor=(.09,.965),
                     ncol=4, frameon=False, fontsize=9)
zone_case_fig.text(.09,.922,
    'TC는 해당 존의 온도, OP는 같은 존의 출력 · 온도 수준 차이와 출력 조건을 함께 확인 · 인과 효과로 해석하지 않음',
    fontsize=9.5, color='#6B7280')
for zone_ax, zone_col in zip(zone_case_axes, S6_ZONE_COLUMNS):
    zone_ax.axvspan(20, 80, color='#EDF4FA', zorder=0)
    for j, peer in enumerate(zone_case_peers):
        for _, segment in s6_analysis[s6_analysis.LOT_NO.eq(peer)].groupby('SEG'):
            zone_ax.plot(segment.progress_pct, segment[zone_col], color='#9AA3AD',
                         ls=zone_styles[j], lw=.85, alpha=.65)
    for _, segment in zone_case_data.groupby('SEG'):
        zone_ax.plot(segment.progress_pct, segment[zone_col], color=colors[zone_case_grade],
                     lw=1.4, zorder=3)
    zone_ax.plot(np.arange(20)*5+2.5, s6_zone_bin_reference[zone_col].loc[zone_case_lot],
                 color='#263238', ls='--', lw=1.1, zorder=4)
    is_temperature = zone_col.startswith('TC')
    zone_unit = '℃' if is_temperature else '%'
    zone_diff_unit = '℃' if is_temperature else '%p'
    zone_delta = s6_zone_middle.loc[zone_case_lot, zone_col] - s6_zone_reference.loc[zone_case_lot, zone_col]
    zone_ax.set_title(f'{zone_col} · {zone_col[-1]}존 '+('온도' if is_temperature else '제어출력'),
                      loc='left', fontsize=12, fontweight='bold', color='black', pad=38)
    zone_ax.text(0,1.055,
        f'LOT {zone_case_lot} · 중간 중앙값 {s6_zone_middle.loc[zone_case_lot,zone_col]:.3f}{zone_unit}'
        f' · 동일 강종 자기 제외 기준 {s6_zone_reference.loc[zone_case_lot,zone_col]:.3f}{zone_unit}'
        f' · 차이 {zone_delta:+.3f}{zone_diff_unit}',
        transform=zone_ax.transAxes, ha='left', va='bottom', fontsize=9, clip_on=False,
        color='#B4232F' if is_temperature else '#6B7280')
    if is_temperature:
        available=zone_case_data[zone_case_data.progress_pct.between(20,80)&zone_case_data[zone_col].notna()]
        point=available.loc[(available.progress_pct-50).abs().idxmin()]
        zone_ax.scatter([point.progress_pct],[point[zone_col]],s=42,facecolors='none',edgecolors='#B4232F',zorder=5)
        zone_ax.annotate(str(zone_case_lot),(point.progress_pct,point[zone_col]),xytext=(12,12),
                         textcoords='offset points',color='#B4232F',fontsize=9,
                         bbox={'fc':'white','ec':'none','alpha':.8})
    zone_ax.set_ylabel('온도 [℃]' if is_temperature else '제어출력 [%]')
    zone_ax.set_xlabel('LOT 진행률 [%] · 시간 보간 없음')
    zone_ax.tick_params(axis='x',labelbottom=True)
    zone_ax.set_xlim(0,100); zone_ax.margins(y=.16); zone_ax.grid(alpha=.15)
    zone_ax.spines[['top','right']].set_visible(False)
zone_case_fig.text(.09,.01,'출력 차이는 제어·운전조건의 참고값입니다. 목표값·속도·두께 없이 온도 차이의 원인을 확정하지 않습니다.',
                   fontsize=9,color='#6B7280')
save_and_show_figures()
display(s6_zone_case_stats[s6_zone_case_stats.LOT.eq(zone_case_lot)].round(4))


#### 그림 10-E-참고 · SPHC 240091 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 비교선 | 240091과 240044·240061·240069. TC-Z7→OP-Z7, TC-Z8→OP-Z8 순으로 두 TC 수준과 해당 존 출력 조건을 확인. |
| 후보의 의미 | 전체 다중신호 후보에는 없지만 단일 온도 신호군의 차이를 따로 유지. |

**컬럼 선택 이유:** 두 TC의 지속적인 높은 수준을 확인한 뒤, 직접 같은 존의 출력인 OP-Z7·OP-Z8을 대조합니다. OP-Z3는 다른 존의 참고 조건이며 두 TC의 직접 제어출력이 아닙니다.

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| TC-Z7·TC-Z8 평균 중앙값 | **333.14℃ / 비교 330.00℃ → +3.14℃** | 같은 SPHC에서도 온도 수준 차이. |
| 시간 지속성·개별 TC | 중간 **12/12칸(100%)**; TC-Z7 **+3.210℃**, TC-Z8 **+3.059℃** | 두 존의 높은 수준이 지속. |
| 비교 기준으로서 역할 | 안정 구간 TC-Z3 차분 std **0.426℃**, SPHC 중 최소 | TC-Z3의 상대 기준으로 쓸 수 있지만 모든 컬럼의 정상 기준은 아님. |
| **240091 / OP-Z7·OP-Z8** | 같은 강종 중간 중앙값 차이 **OP-Z7 -1.311%p**, **OP-Z8 -0.412%p** | 지속적으로 높은 두 TC와 더 높은 출력이 함께 나타난 것은 아님. 온도 설정·운전조건을 먼저 확인 |

**결론:** **TC-Z3 비교 역할과 TC-Z7·TC-Z8 점검 역할을 구분**합니다. 단일 신호군이어도 지속 차이를 후속 확인합니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240091 ↔ 240044·240061·240069 | TC-Z7↔OP-Z7, TC-Z8↔OP-Z8의 목표값·속도·두께를 대조 | 운전조건 기록 |
| 240091 / 이후 BM | 종료 52.2시간 뒤 BM 기록을 확인하고 정상 기준 고정 여부 검토 | 정비 상세와 추가 동일 조건 SPHC LOT |


In [ ]:
# 계산된 LOT 시계열과 동일 강종 비교선을 이 셀에서 직접 그립니다.
case_lot = 240024
case_columns = ['과시효대', '냉각대']
case_title = '그림 10-F-참고. SPFH590 240024 ↔ 240108 — 낮은 온도 수준이 반복되는가?'
case_peers = s6_peer_ids[case_lot]
case_grade = s6_lots.loc[case_lot, 'grade']
case_n = len(case_columns)
case_height = 3.2 * case_n + 3.1
case_fig, case_axes = plt.subplots(case_n, 1, figsize=(16, case_height), squeeze=False)
case_axes = case_axes[:, 0]
case_fig.subplots_adjust(left=0.09, right=0.98, top=1 - 2.5 / case_height, bottom=0.75 / case_height, hspace=0.6)
case_fig.suptitle(case_title, x=0.09, y=1 - 0.12 / case_height, ha='left', va='top', fontsize=15, fontweight='bold')
case_styles = ['-', '--', ':', '-.']
case_handles = [Line2D([0], [0], color=colors[case_grade], lw=2, label=f'{case_lot} · 확인 LOT')]
case_handles += [Line2D([0], [0], color='#9aa3ad', lw=1.2, ls=case_styles[case_i % 4], label=str(case_peer)) for case_i, case_peer in enumerate(case_peers)]
if case_peers:
    case_handles += [Line2D([0], [0], color='#263238', lw=1.5, ls='--', label='자기 제외 · 5% 칸 중앙값')]
case_handles += [Patch(facecolor='#edf4fa', label='중간 20~80%')]
if 'OP-Z3' in case_columns and s6_screen.loc[case_lot, 'OP-Z3 원본 결측행'] > 0:
    case_handles += [Patch(facecolor='#cfd4da', label='OP-Z3 원본 결측')]
case_fig.legend(handles=case_handles, loc='upper left', bbox_to_anchor=(0.09, 1 - 0.54 / case_height), ncol=4, frameon=False, fontsize=9)
case_fig.text(0.09, 1 - 1.25 / case_height, f"{case_grade} n={s6_lots.loc[case_lot, 'grade_n']} · {s6_lots.loc[case_lot, 'support']} · 공백은 연결하지 않음 · 진행률은 실제 수집 길이 기준", fontsize=10, color='#425466')
case_names = {'가열·균열존': 'TC-Z3~TC-Z6 평균온도', '과시효대': 'TC-Z7·TC-Z8 평균온도', '냉각대': 'TC-OUT1·TC-OUT2 평균온도', 'OP-Z3': 'OP-Z3 · 3존 제어출력', 'CP-Z4': 'CP-Z4 · 카본포텐셜', 'CP_roll_sd': 'CP-Z4 std(5분)'}
for case_ax, case_c in zip(case_axes, case_columns):
    case_ax.axvspan(20, 80, color='#edf4fa', zorder=0)
    for case_i, case_peer in enumerate(case_peers):
        for case__, case_seg in s6_analysis[s6_analysis.LOT_NO.eq(case_peer)].groupby('SEG'):
            case_ax.plot(case_seg.progress_pct, case_seg[case_c], color='#9aa3ad', lw=0.85, alpha=0.65, ls=case_styles[case_i % 4])
    case_d = s6_analysis[s6_analysis.LOT_NO.eq(case_lot)]
    for case__, case_seg in case_d.groupby('SEG'):
        case_ax.plot(case_seg.progress_pct, case_seg[case_c], color=colors[case_grade], lw=1.5, alpha=0.95, zorder=3)
    if case_peers:
        case_ax.plot(np.arange(20) * 5 + 2.5, s6_bin_ref[case_c].loc[case_lot], color='#263238', ls='--', lw=1.2, zorder=4)
    if case_c == 'OP-Z3':
        for case_start, case_end, case_count in missing_runs(case_lot, 'OP-Z3'):
            case_ax.axvspan(100 * case_start / (60 * s6_lots.loc[case_lot, 'minutes']), 100 * (case_end + 1) / (60 * s6_lots.loc[case_lot, 'minutes']), color='#cfd4da', zorder=2)
    case_ax.set_title(case_names[case_c], loc='left', color='black', fontweight='bold', fontsize=12, pad=38)
    case_p = s6_persistence[s6_persistence.LOT.eq(case_lot) & s6_persistence['채널'].eq(case_c)].iloc[0]
    if case_peers:
        case_pct = f"{case_p['중간 지속 비율 [%]']:.0f}%" if pd.notna(case_p['중간 지속 비율 [%]']) else '—'
        case_text = f"LOT {case_lot} · 중간 유효 {int(case_p['유효 중간 칸'])}칸 중 같은 방향 차이 {int(case_p['같은 방향 차이 칸'])}칸 ({case_pct})"
        if case_c in S6_FEATURES:
            case_text += f" · 중간 요약 Δ {s6_value(s6_delta['중간 20~80%'].loc[case_lot, case_c], case_c, True)}"
        if case_c == 'CP_roll_sd':
            case_text += ' · 2배/1/2 기준 · 피크와 지속 상승 구분'
        case_cp_peak_note = case_c == 'CP-Z4' and s6_diff_middle.loc[case_lot, 'CP_sd']
        if case_cp_peak_note:
            case_text += f' · 최대 피크 {case_d[case_c].max():.4f}'
        if case_c == 'OP-Z3' and s6_screen.loc[case_lot, 'OP-Z3 원본 결측행']:
            case_text += f" · 원본 결측 {s6_screen.loc[case_lot, 'OP-Z3 원본 결측행']}행"
        case_ax.text(0, 1.055, case_text, transform=case_ax.transAxes, ha='left', va='bottom', color='#c0392b' if case_cp_peak_note or case_p['같은 방향 차이 칸'] or (case_c == 'OP-Z3' and s6_screen.loc[case_lot, 'OP-Z3 원본 결측행']) else '#425466', fontsize=9, clip_on=False)
        case_delta = s6_bin_delta[case_c].loc[case_lot]
        if case_c == 'CP_roll_sd':
            case_ratio = s6_bin_values[case_c].loc[case_lot].div(s6_bin_ref[case_c].loc[case_lot].replace(0, np.nan))
            case_eligible = case_delta.notna() & (case_ratio.ge(2) | case_ratio.le(0.5))
        else:
            case_eligible = case_delta.abs().ge(s6_level_floor[case_c])
        case_point = None
        if case_c in ['CP-Z4', 'CP_roll_sd'] and s6_diff_middle.loc[case_lot, 'CP_sd']:
            case_valid = case_d[case_d[case_c].notna()]
            if len(case_valid):
                case_point = case_valid.loc[case_valid[case_c].idxmax()]
        elif case_eligible.any():
            case_middle_eligible = case_eligible & pd.Series([20 <= 5 * case_b + 2.5 <= 80 for case_b in range(20)], index=range(20))
            case_selected = case_middle_eligible if case_middle_eligible.any() else case_eligible
            case_b = case_delta[case_selected].abs().idxmax()
            case_valid = case_d[case_d.progress_bin.eq(case_b) & case_d[case_c].notna()]
            if len(case_valid):
                case_point = case_valid.loc[(case_valid.progress_pct - (5 * case_b + 2.5)).abs().idxmin()]
        if case_point is not None:
            case_x, case_y = (case_point.progress_pct, case_point[case_c])
            case_ax.scatter([case_x], [case_y], s=40, facecolors='none', edgecolors='#c0392b', lw=1.4, zorder=5)
            case_point_label = f'{case_lot}' + (' · 초반' if case_x < 20 else ' · 종료부' if case_x > 80 else '')
            case_ax.annotate(case_point_label, (case_x, case_y), xytext=(-12 if case_x > 70 else 12, 14), textcoords='offset points', ha='right' if case_x > 70 else 'left', fontsize=9, fontweight='bold', color='#c0392b', arrowprops={'arrowstyle': '-', 'color': '#c0392b', 'lw': 0.8}, bbox={'fc': 'white', 'ec': 'none', 'alpha': 0.75})
    else:
        case_ax.text(0, 1.055, f'LOT {case_lot} · 동일 강종 비교 LOT 없음 → 운전 전환 확인, 강종/LOT 차이 분리 불가', transform=case_ax.transAxes, color='#c0392b', fontsize=9, clip_on=False)
    case_ax.set_ylabel('CP-Z4 [무차원]' if case_c == 'CP-Z4' else 'CP-Z4 std [무차원]' if case_c == 'CP_roll_sd' else '제어출력 [%]' if case_c.startswith('OP') else '온도 [℃]')
    case_ax.set_xlabel('LOT 진행률 [%] · 시간 보간 없음')
    case_ax.set_xlim(0, 100)
    case_ax.margins(y=0.13)
    case_ax.grid(alpha=0.18)
    case_ax.spines[['top', 'right']].set_visible(False)
case_fig.text(0.09, 0.14 / case_height, '지속 후보: 중간 12칸 중 8칸 이상 비교 가능하고 같은 방향 차이가 70% 이상. 붉은 원은 실제 확인 지점이며 지속 구간 표시가 아닙니다.', fontsize=9, color='#425466')
save_and_show_figures()
display(s6_comparison[s6_comparison.LOT.isin([240024,240108])].round(6))


#### 그림 10-E-참고 · SPFH590 240024 ↔ 240108 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 두 LOT | 240024와 240108을 서로 비교. 동일 강종 n=2여서 참고 비교. |
| 두 온도 패널 | TC-Z7·TC-Z8 평균과 TC-OUT1·TC-OUT2 평균의 공통 수준을 확인. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| TC-Z7·TC-Z8 평균 중앙값 | 중간 **326.15 / 326.19℃**, 차이 약 **0.04℃** | 전체 LOT보다 낮은 수준이 두 SPFH590에서 반복. |
| TC-OUT1·TC-OUT2 평균 중앙값 | 중간 **277.44 / 276.89℃** | 냉각 온도도 비슷한 수준. |
| TC-Z3~TC-Z6 평균 변동폭 | 동일 강종 차이 **±0.66℃**, 시간 차이 **2/12칸(17%)** | 일부 차이는 남지만 n=2 참고 항목. |

**결론:** **B. 같은 강종의 공통 운전조건 가능성**이 있습니다. 두 LOT만으로 강종의 정상 범위를 정하지 않습니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240024 ↔ 240108 | 각 TC-Z7·TC-Z8·TC-OUT1·TC-OUT2와 목표값을 대조 | 소재 규격·속도·목표온도 |
| 추가 SPFH590 LOT | 공통 낮은 수준의 반복성과 변동폭 범위를 확인 | 동일 운전조건 반복 표본 확보 |

**선택 이유와 한계:** TC-Z7·TC-Z8과 TC-OUT1·TC-OUT2는 두 SPFH590 LOT에서 공통된 낮은 수준이 반복되는지 보는 조건 비교입니다. 냉각의 진행률별 차이는 진동 위상이 달라 생길 수 있어 순간 차이를 이상으로 세지 않고 주기·진폭도 따로 봅니다.


In [ ]:
# 계산된 LOT 시계열과 동일 강종 비교선을 이 셀에서 직접 그립니다.
case_lot = 240058
case_columns = ['가열·균열존', 'OP-Z3', 'CP-Z4']
case_title = '그림 10-F-보류. DP590 240058 — 강종/LOT 차이를 분리할 비교군이 없음'
case_peers = s6_peer_ids[case_lot]
case_grade = s6_lots.loc[case_lot, 'grade']
case_n = len(case_columns)
case_height = 3.2 * case_n + 3.1
case_fig, case_axes = plt.subplots(case_n, 1, figsize=(16, case_height), squeeze=False)
case_axes = case_axes[:, 0]
case_fig.subplots_adjust(left=0.09, right=0.98, top=1 - 2.5 / case_height, bottom=0.75 / case_height, hspace=0.6)
case_fig.suptitle(case_title, x=0.09, y=1 - 0.12 / case_height, ha='left', va='top', fontsize=15, fontweight='bold')
case_styles = ['-', '--', ':', '-.']
case_handles = [Line2D([0], [0], color=colors[case_grade], lw=2, label=f'{case_lot} · 확인 LOT')]
case_handles += [Line2D([0], [0], color='#9aa3ad', lw=1.2, ls=case_styles[case_i % 4], label=str(case_peer)) for case_i, case_peer in enumerate(case_peers)]
if case_peers:
    case_handles += [Line2D([0], [0], color='#263238', lw=1.5, ls='--', label='자기 제외 · 5% 칸 중앙값')]
case_handles += [Patch(facecolor='#edf4fa', label='중간 20~80%')]
if 'OP-Z3' in case_columns and s6_screen.loc[case_lot, 'OP-Z3 원본 결측행'] > 0:
    case_handles += [Patch(facecolor='#cfd4da', label='OP-Z3 원본 결측')]
case_fig.legend(handles=case_handles, loc='upper left', bbox_to_anchor=(0.09, 1 - 0.54 / case_height), ncol=4, frameon=False, fontsize=9)
case_fig.text(0.09, 1 - 1.25 / case_height, f"{case_grade} n={s6_lots.loc[case_lot, 'grade_n']} · {s6_lots.loc[case_lot, 'support']} · 공백은 연결하지 않음 · 진행률은 실제 수집 길이 기준", fontsize=10, color='#425466')
case_names = {'가열·균열존': 'TC-Z3~TC-Z6 평균온도', '과시효대': 'TC-Z7·TC-Z8 평균온도', '냉각대': 'TC-OUT1·TC-OUT2 평균온도', 'OP-Z3': 'OP-Z3 · 3존 제어출력', 'CP-Z4': 'CP-Z4 · 카본포텐셜', 'CP_roll_sd': 'CP-Z4 std(5분)'}
for case_ax, case_c in zip(case_axes, case_columns):
    case_ax.axvspan(20, 80, color='#edf4fa', zorder=0)
    for case_i, case_peer in enumerate(case_peers):
        for case__, case_seg in s6_analysis[s6_analysis.LOT_NO.eq(case_peer)].groupby('SEG'):
            case_ax.plot(case_seg.progress_pct, case_seg[case_c], color='#9aa3ad', lw=0.85, alpha=0.65, ls=case_styles[case_i % 4])
    case_d = s6_analysis[s6_analysis.LOT_NO.eq(case_lot)]
    for case__, case_seg in case_d.groupby('SEG'):
        case_ax.plot(case_seg.progress_pct, case_seg[case_c], color=colors[case_grade], lw=1.5, alpha=0.95, zorder=3)
    if case_peers:
        case_ax.plot(np.arange(20) * 5 + 2.5, s6_bin_ref[case_c].loc[case_lot], color='#263238', ls='--', lw=1.2, zorder=4)
    if case_c == 'OP-Z3':
        for case_start, case_end, case_count in missing_runs(case_lot, 'OP-Z3'):
            case_ax.axvspan(100 * case_start / (60 * s6_lots.loc[case_lot, 'minutes']), 100 * (case_end + 1) / (60 * s6_lots.loc[case_lot, 'minutes']), color='#cfd4da', zorder=2)
    case_ax.set_title(case_names[case_c], loc='left', color='black', fontweight='bold', fontsize=12, pad=38)
    case_p = s6_persistence[s6_persistence.LOT.eq(case_lot) & s6_persistence['채널'].eq(case_c)].iloc[0]
    if case_peers:
        case_pct = f"{case_p['중간 지속 비율 [%]']:.0f}%" if pd.notna(case_p['중간 지속 비율 [%]']) else '—'
        case_text = f"LOT {case_lot} · 중간 유효 {int(case_p['유효 중간 칸'])}칸 중 같은 방향 차이 {int(case_p['같은 방향 차이 칸'])}칸 ({case_pct})"
        if case_c in S6_FEATURES:
            case_text += f" · 중간 요약 Δ {s6_value(s6_delta['중간 20~80%'].loc[case_lot, case_c], case_c, True)}"
        if case_c == 'CP_roll_sd':
            case_text += ' · 2배/1/2 기준 · 피크와 지속 상승 구분'
        case_cp_peak_note = case_c == 'CP-Z4' and s6_diff_middle.loc[case_lot, 'CP_sd']
        if case_cp_peak_note:
            case_text += f' · 최대 피크 {case_d[case_c].max():.4f}'
        if case_c == 'OP-Z3' and s6_screen.loc[case_lot, 'OP-Z3 원본 결측행']:
            case_text += f" · 원본 결측 {s6_screen.loc[case_lot, 'OP-Z3 원본 결측행']}행"
        case_ax.text(0, 1.055, case_text, transform=case_ax.transAxes, ha='left', va='bottom', color='#c0392b' if case_cp_peak_note or case_p['같은 방향 차이 칸'] or (case_c == 'OP-Z3' and s6_screen.loc[case_lot, 'OP-Z3 원본 결측행']) else '#425466', fontsize=9, clip_on=False)
        case_delta = s6_bin_delta[case_c].loc[case_lot]
        if case_c == 'CP_roll_sd':
            case_ratio = s6_bin_values[case_c].loc[case_lot].div(s6_bin_ref[case_c].loc[case_lot].replace(0, np.nan))
            case_eligible = case_delta.notna() & (case_ratio.ge(2) | case_ratio.le(0.5))
        else:
            case_eligible = case_delta.abs().ge(s6_level_floor[case_c])
        case_point = None
        if case_c in ['CP-Z4', 'CP_roll_sd'] and s6_diff_middle.loc[case_lot, 'CP_sd']:
            case_valid = case_d[case_d[case_c].notna()]
            if len(case_valid):
                case_point = case_valid.loc[case_valid[case_c].idxmax()]
        elif case_eligible.any():
            case_middle_eligible = case_eligible & pd.Series([20 <= 5 * case_b + 2.5 <= 80 for case_b in range(20)], index=range(20))
            case_selected = case_middle_eligible if case_middle_eligible.any() else case_eligible
            case_b = case_delta[case_selected].abs().idxmax()
            case_valid = case_d[case_d.progress_bin.eq(case_b) & case_d[case_c].notna()]
            if len(case_valid):
                case_point = case_valid.loc[(case_valid.progress_pct - (5 * case_b + 2.5)).abs().idxmin()]
        if case_point is not None:
            case_x, case_y = (case_point.progress_pct, case_point[case_c])
            case_ax.scatter([case_x], [case_y], s=40, facecolors='none', edgecolors='#c0392b', lw=1.4, zorder=5)
            case_point_label = f'{case_lot}' + (' · 초반' if case_x < 20 else ' · 종료부' if case_x > 80 else '')
            case_ax.annotate(case_point_label, (case_x, case_y), xytext=(-12 if case_x > 70 else 12, 14), textcoords='offset points', ha='right' if case_x > 70 else 'left', fontsize=9, fontweight='bold', color='#c0392b', arrowprops={'arrowstyle': '-', 'color': '#c0392b', 'lw': 0.8}, bbox={'fc': 'white', 'ec': 'none', 'alpha': 0.75})
    else:
        case_ax.text(0, 1.055, f'LOT {case_lot} · 동일 강종 비교 LOT 없음 → 운전 전환 확인, 강종/LOT 차이 분리 불가', transform=case_ax.transAxes, color='#c0392b', fontsize=9, clip_on=False)
    case_ax.set_ylabel('CP-Z4 [무차원]' if case_c == 'CP-Z4' else 'CP-Z4 std [무차원]' if case_c == 'CP_roll_sd' else '제어출력 [%]' if case_c.startswith('OP') else '온도 [℃]')
    case_ax.set_xlabel('LOT 진행률 [%] · 시간 보간 없음')
    case_ax.set_xlim(0, 100)
    case_ax.margins(y=0.13)
    case_ax.grid(alpha=0.18)
    case_ax.spines[['top', 'right']].set_visible(False)
case_fig.text(0.09, 0.14 / case_height, '지속 후보: 중간 12칸 중 8칸 이상 비교 가능하고 같은 방향 차이가 70% 이상. 붉은 원은 실제 확인 지점이며 지속 구간 표시가 아닙니다.', fontsize=9, color='#425466')
save_and_show_figures()
display(s6_comparison[s6_comparison.LOT.eq(240058)].round(6))


#### 그림 10-F · DP590 240058 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 시간·선 | 본 LOT의 TC-Z3~TC-Z6 평균, OP-Z3, CP-Z4를 진행률 순서로 봄. 동일 DP590 비교선은 없음. |
| 후반 | 여러 TC·OP·CP가 같은 끝 구간에서 함께 내려가는지 확인. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| TC-Z3·TC-Z4·TC-Z5·TC-Z6 | 후반 동반 하강 **177.5~216.5분**; TC-Z3 약 **858.9 → 814.9℃** | TC-Z3 하나의 단독 급락보다 공통 운전 전환 형태. |
| OP-Z3·CP-Z4 | 후반에 함께 하강. OP-Z1~Z8도 끝 5분 중앙값이 기준보다 각각 10%p 이상 감소 | 여러 신호가 동반하는 종료 상태 확인이 필요. |
| DP590 표본 | **n=1**, 동일 강종 비교 LOT 없음 | 강종 특성과 이 LOT의 특이성을 분리할 수 없음. |

**결론:** **D. 강종 내 비교 보류 + 종료 전환 확인 대상**입니다. 큰 하강은 후속 점검하지만 고장으로 확정하지 않습니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240058 ↔ 240114 | 두 종료부의 TC·OP 하강 형태와 순서를 비교 | 가동·정지계획·출력 목표; 강종이 달라 형태 참고만 가능 |
| 240058 / 추가 DP590 LOT | 동일 조건에서 중간 운전 수준과 종료 형태를 비교 | 추가 DP590 표본·목표값·소재 규격 |
| 240058 / 계획수리 | 수리 전날이라는 위치와 실제 정지 시각 대조 | 그림 10-G·설비 가동 로그 |


### ⑤ 계획수리 연계 — 후보 확인 후에 실제 날짜를 붙이기

**보고 싶은 것:** 선정한 LOT가 수리 전후 어디에 있고, 반대편에 같은 강종 LOT가 있는가? ±60일은 비교 LOT 탐색 범위입니다.


In [ ]:
# 5단계: LOT 차이·시간 지속성 확인 후에만 계획수리 위치를 붙입니다.
s6_review=s6_screen[['grade','grade_n','support','탐색 후보','전체 신호군 수','중간 신호군 수','OP-Z3 원본 결측행']].copy()
s6_review['동일 강종 중간 차이 신호군 수']=s6_peer_families.sum(axis=1)
s6_review['동일 강종 중간 차이 지표']=[', '.join(S6_NAMES[c] for c in S6_FEATURES if s6_diff_middle.loc[lot,c]) or '—' for lot in s6_review.index]
s6_review['반복 수준 지표']=''
for lot,r in s6_lots.iterrows():
    ids=s6_lots.index[s6_lots.grade.eq(r.grade)]; repeated=[]
    if len(ids)>=2:
        for c in S6_FEATURES[:5]:
            shifts=s6_middle.loc[ids,c]-s6_middle[c].median()
            if shifts.ge(s6_level_floor[c]).all() or shifts.le(-s6_level_floor[c]).all(): repeated.append(S6_NAMES[c])
    s6_review.loc[lot,'반복 수준 지표']=', '.join(repeated) or '—'
def s6_classify(lot):
    r=s6_review.loc[lot]
    if r.grade_n==1: return 'D. 비교 불가 / 보류'
    if r['OP-Z3 원본 결측행']>0: return 'C. 결측 영향 먼저 확인'
    if r.grade_n==2: return 'B. 동일 강종 반복 · 조건 차이 가능' if r['반복 수준 지표']!='—' else '보류: n=2 참고 비교'
    if s6_diff_middle.loc[lot].any():
        return 'A. 강종 내 차이 · 상세분석 후보' if r['탐색 후보'] else '단일군 참고: 같은 강종 내 차이'
    if r['반복 수준 지표']!='—': return 'B. 동일 강종 반복 · 조건 차이 가능'
    return '보류: 강종 내 지속 차이 근거 부족'
s6_review['분류']=[s6_classify(lot) for lot in s6_review.index]
s6_review['지속 차이 채널']=[', '.join(s6_persistence.loc[s6_persistence.LOT.eq(lot)&s6_persistence['지속 후보'],'채널']) or '—' for lot in s6_review.index]
s6_review['해석 제한']=['강종/LOT 차이 분리 불가' if r.grade_n==1 else 'n=2 참고 비교; 강종 내 판정 불가' if r.grade_n==2 else '중간 차이가 한 신호군; 다중군 이상 확정 아님' if s6_review.loc[lot,'동일 강종 중간 차이 신호군 수']==1 else 'n≥3 기술 비교; 고장/인과 판정 아님' for lot,r in s6_lots.iterrows()]
REMOTE_DAYS=60
s6_repair_rows=[]
for plan in plans.itertuples():
    for lot,r in s6_lots.iterrows():
        if r.end<plan.STRT_DT:
            side='수리 전'; gap=(plan.STRT_DT-r.end).total_seconds()/86400
            opposite=s6_lots[s6_lots.grade.eq(r.grade)&s6_lots.start.ge(plan.end_exclusive)&s6_lots.start.le(plan.end_exclusive+pd.Timedelta(days=REMOTE_DAYS))]
        elif r.start>=plan.end_exclusive:
            side='수리 후'; gap=(r.start-plan.end_exclusive).total_seconds()/86400
            opposite=s6_lots[s6_lots.grade.eq(r.grade)&s6_lots.end.lt(plan.STRT_DT)&s6_lots.end.ge(plan.STRT_DT-pd.Timedelta(days=REMOTE_DAYS))]
        else: side='수리와 관측 중첩'; gap=0.; opposite=s6_lots.iloc[:0]
        if gap<=REMOTE_DAYS:
            s6_repair_rows.append({'계획수리':plan.PLAN_ID,'수리 시작':plan.STRT_DT,'종료 다음날':plan.end_exclusive,'LOT':lot,'강종':r.grade,'분류':s6_review.loc[lot,'분류'],'위치':side,'경계 거리 [일]':gap,'반대편 동일 강종 LOT':', '.join(map(str,opposite.index)) or '없음','반대편 표본수':len(opposite),'해석 범위':'같은 강종 원격 비교 가능; 효과 확정 불가' if len(opposite) else '수리 인근에서 관측됨; 같은 강종 전후 비교 불가'})
s6_repair_link=pd.DataFrame(s6_repair_rows)
s6_pair_rows=[]
for plan in plans.itertuples():
    pre=s6_lots[s6_lots.end.lt(plan.STRT_DT)&s6_lots.end.ge(plan.STRT_DT-pd.Timedelta(days=REMOTE_DAYS))]
    post=s6_lots[s6_lots.start.ge(plan.end_exclusive)&s6_lots.start.le(plan.end_exclusive+pd.Timedelta(days=REMOTE_DAYS))]
    for grade in sorted(set(pre.grade)&set(post.grade)):
        before_ids=pre.index[pre.grade.eq(grade)].tolist(); after_ids=post.index[post.grade.eq(grade)].tolist()
        if not s6_review.loc[before_ids+after_ids,'탐색 후보'].any(): continue
        for c in S6_FEATURES:
            before=s6_middle.loc[before_ids,c].median(); after=s6_middle.loc[after_ids,c].median()
            s6_pair_rows.append({'계획수리':plan.PLAN_ID,'강종':grade,'수리 전 LOT':', '.join(map(str,before_ids)),'수리 후 LOT':', '.join(map(str,after_ids)),'전 표본수':len(before_ids),'후 표본수':len(after_ids),'지표':S6_NAMES[c],'전 중간 중앙값':before,'후 중간 중앙값':after,'후-전':after-before})
s6_repair_pairs=pd.DataFrame(s6_pair_rows)
# 세로축은 LOT 분류가 아니라 점검 목적입니다. LOT·강종과 실제 관측일을 점검 항목에 연결합니다.
s6_g_red='#B4232F';s6_g_green='#18794E';s6_g_gray='#7B8794';s6_g_plan='#D6EAF8'
s6_g_cp69=s6_analysis.loc[s6_analysis[s6_analysis.LOT_NO.eq(240069)]['CP-Z4'].idxmax()]
s6_g_warm_ends={}
for s6_g_lot in [240061,240117]:
    s6_g_d=s6_analysis[s6_analysis.LOT_NO.eq(s6_g_lot)].sort_values('MEAS_DT')
    s6_g_cross=s6_g_d[s6_g_d['가열·균열존'].ge(855)]
    s6_g_warm_ends[s6_g_lot]=float(s6_g_cross.elapsed_min.iloc[0]) if len(s6_g_cross) else np.nan
s6_g_fall_text=[]
for s6_g_lot in [240058,240114]:
    s6_g_fall=terminal_falls_table[terminal_falls_table.LOT.eq(s6_g_lot)].iloc[0]
    s6_g_fall_text.append(f"{s6_g_lot}: {s6_g_fall['하강 구간 시작 [분]']:.1f}~{s6_g_fall['마지막 관측 [분]']:.1f}분")
s6_g_check_rows=[
    {'목적':'OP-Z3 초반 결측','대상':[(240044,'중요'),(240061,'중요')],
     '제목':'0~81초 / 0~183초 · OP-Z3만 결측',
     '점검':'같은 SPHC 두 LOT의 시작 구간을 비교\nTC-Z3·OP-Z4·Z5·Z6 기록 + 수집 로그'},
    {'목적':'TC-Z3~TC-Z6 종료 하강','대상':[(240058,'중요'),(240114,'중요')],
     '제목':'CR1 계획수리 전 마지막 관측',
     '점검':' / '.join(s6_g_fall_text)+'\nTC-Z3·Z4·Z5·Z6 ↔ OP-Z3~Z6 · 정지 시각 대조'},
    {'목적':'TC-Z3~TC-Z6 초반 승온','대상':[(240061,'중요'),(240117,'중요')],
     '제목':'CR1 계획수리 후 첫 관측',
     '점검':f"240061: 0~{s6_g_warm_ends[240061]:.1f}분 / 240117: 0~{s6_g_warm_ends[240117]:.1f}분\n승온 종료 후 TC-Z3 ↔ OP-Z3 차이가 남는지 확인"},
    {'목적':'CP-Z4 피크·변동','대상':[(240069,'중요'),(240037,'참고')],
     '제목':f'240069: {s6_g_cp69.elapsed_min-10:.1f}~{s6_g_cp69.elapsed_min+10:.1f}분',
     '점검':'CP-Z4 피크 ±10분 제외 전/후 std\n240069 ↔ SPHC 240044·240061·240091'},
    {'목적':'TC-Z7·Z8 지속 차이','대상':[(240091,'중요'),(240121,'중요'),(240133,'참고')],
     '제목':'중간 20~80% · TC-Z7과 TC-Z8 각각',
     '점검':'240091 ↔ 나머지 SPHC / 240121 ↔ SPCE 240037·240133\n같은 존 OP-Z7·OP-Z8 + 목표값·속도·두께'},
    {'목적':'실제 정비 전후 관측','대상':[(240117,'중요'),(240133,'중요')],
     '제목':'감지 · 착수 · 완료를 구분',
     '점검':'240117: TBM 01859·CBM 01854 전후 관측 있음\n240133: CBM 02105 감지 관측만 · 완료 후 없음'},
    {'목적':'SPFH590 조건 참고','대상':[(240024,'참고'),(240108,'참고')],
     '제목':'240024 ↔ 240108 · n=2 참고 비교',
     '점검':'중간 TC-Z7·TC-Z8 / TC-OUT1·TC-OUT2\n두 LOT의 공통 수준과 운전조건 확인'},
    {'목적':'SPCEN 조건 참고','대상':[(240038,'참고'),(240114,'참고')],
     '제목':'240038 ↔ 240114 · n=2 참고 비교',
     '점검':'중간 TC-Z3~Z6·OP-Z3 변동폭 / TC-Z7·TC-Z8\n240114 종료 하강을 제외해도 차이가 남는가?'},
    {'목적':'동일 강종 표본 확보','대상':[(240001,'보류')],
     '제목':'SGCC 240001 · 현재 후보 기준 미만',
     '점검':'같은 SGCC LOT 없음 · 정상 기준으로 지정하지 않음\nDP590 240058·SPCD 240117도 n=1 한계 유지'},
]
s6_g_all_lots={lot for row in s6_g_check_rows for lot,_ in row['대상']}
assert s6_g_all_lots==set(s6_lots.index)

# 위쪽은 CR1 계획수리와 CAL01의 실제 작업, 아래쪽은 점검할 LOT·강종입니다.
fig=plt.figure(figsize=(22,14.2))
s6_g_grid=fig.add_gridspec(2,2,height_ratios=[1.7,7.8],width_ratios=[1.95,1],
    left=.15,right=.985,top=.83,bottom=.075,hspace=.23,wspace=.075)
s6_g_work_ax=fig.add_subplot(s6_g_grid[0,0])
s6_g_context_ax=fig.add_subplot(s6_g_grid[0,1]);s6_g_context_ax.axis('off')
s6_g_lot_ax=fig.add_subplot(s6_g_grid[1,0],sharex=s6_g_work_ax)
s6_g_check_ax=fig.add_subplot(s6_g_grid[1,1]);s6_g_check_ax.axis('off')
s6_g_work_y={'TBM':1,'CBM':2,'BM':3}
s6_g_type_marker={'TBM':'s','CBM':'D','BM':'X'}
s6_g_selected_wo={'WO-2024-00717','WO-2024-01094','WO-2024-01859','WO-2024-01854','WO-2024-02105'}
s6_g_plan_rows=[]
for s6_g_p in plans.itertuples():
    for s6_g_ax in [s6_g_work_ax,s6_g_lot_ax]:
        s6_g_ax.axvspan(s6_g_p.STRT_DT,s6_g_p.end_exclusive,color=s6_g_plan,alpha=.7,zorder=0)
    s6_g_work_ax.hlines(0,s6_g_p.STRT_DT,s6_g_p.end_exclusive,color='#83B8D8',lw=7,zorder=2)
    s6_g_work_ax.annotate(f'{s6_g_p.STRT_DT:%m/%d}~{s6_g_p.END_DT:%m/%d}',
        (s6_g_p.STRT_DT,0),xytext=(0,10),textcoords='offset points',ha='center',va='bottom',fontsize=9)
    s6_g_jobs=cal[cal.STRT_DT.ge(s6_g_p.STRT_DT)&cal.STRT_DT.lt(s6_g_p.end_exclusive)]
    s6_g_plan_rows.append({'계획수리':s6_g_p.PLAN_ID,'기간':f'{s6_g_p.STRT_DT:%m/%d}~{s6_g_p.END_DT:%m/%d}',
        '계획 범위':s6_g_p.SCOPE,'CAL01 작업':len(s6_g_jobs),
        '유형':', '.join(f'{t} {int(k)}건' for t,k in s6_g_jobs.MNT_TYPE.value_counts().items())})
for s6_g_w in cal.itertuples():
    s6_g_y=s6_g_work_y[s6_g_w.MNT_TYPE]
    s6_g_color=s6_g_red if s6_g_w.WO_NO in s6_g_selected_wo else '#939DA6'
    s6_g_work_ax.hlines(s6_g_y,s6_g_w.STRT_DT,s6_g_w.END_DT,color=s6_g_color,lw=2,zorder=2)
    s6_g_work_ax.scatter(s6_g_w.STRT_DT,s6_g_y,marker=s6_g_type_marker[s6_g_w.MNT_TYPE],
        s=28 if s6_g_w.WO_NO in s6_g_selected_wo else 16,color=s6_g_color,zorder=3)
s6_g_work_ax.set_yticks([0,1,2,3],['CR1 계획수리','TBM · 일정 기반','CBM · 상태 기반','BM · 사후정비'],fontsize=10)
s6_g_work_ax.set_ylim(3.5,-.75);s6_g_work_ax.tick_params(axis='x',labelbottom=False)
s6_g_work_ax.set_title('CR1 계획수리 / CAL01 실제 착수~완료',loc='left',fontweight='bold',pad=16)
s6_g_context_ax.text(0,.98,'계획 범위와 실제 작업을 구분',va='top',fontweight='bold',fontsize=12)
s6_g_context_ax.text(0,.77,'06/10~06/14 · '+str(plans.iloc[0].SCOPE)+'\n11/05~11/10 · '+str(plans.iloc[1].SCOPE),
    va='top',fontsize=10.5,linespacing=1.8,color='#425466')
s6_g_context_ax.text(0,.36,'정비 트랙의 빨간 표식: 아래 확대할 5개 작업\n연간 축에서는 짧은 작업이 점처럼 보입니다.',
    va='top',fontsize=10,color='#66717D',linespacing=1.8)
for s6_g_i,s6_g_row in enumerate(s6_g_check_rows):
    s6_g_lot_ax.axhline(s6_g_i+.5,color='#EDF0F3',lw=.9,zorder=0)
    for s6_g_lot,s6_g_status in s6_g_row['대상']:
        s6_g_r=s6_lots.loc[s6_g_lot]
        s6_g_color={'중요':s6_g_red,'참고':s6_g_green,'보류':s6_g_gray}[s6_g_status]
        s6_g_lot_ax.hlines(s6_g_i,s6_g_r.start,s6_g_r.end,color=s6_g_color,lw=4,zorder=2)
        s6_g_lot_ax.scatter(s6_g_r.start,s6_g_i,s=62,facecolor='white',edgecolor=s6_g_color,lw=1.8,zorder=3)
        s6_g_label_at_edge=s6_g_r.start<pd.Timestamp('2024-01-16')
        s6_g_lot_ax.annotate(f'{s6_g_lot} · {s6_g_r.grade}',(s6_g_r.start,s6_g_i),
            xytext=(8 if s6_g_label_at_edge else 0,12),textcoords='offset points',
            ha='left' if s6_g_label_at_edge else 'center',va='bottom',fontsize=10,color=s6_g_color,fontweight='bold')
        s6_g_lot_ax.annotate(f'{s6_g_r.start:%m/%d}',(s6_g_r.start,s6_g_i),
            xytext=(0,-12),textcoords='offset points',ha='center',va='top',fontsize=9,color='#66717D')
    s6_g_y=1-(s6_g_i+.12)/len(s6_g_check_rows)
    s6_g_check_ax.text(0,s6_g_y,s6_g_row['제목'],transform=s6_g_check_ax.transAxes,
        ha='left',va='top',fontsize=10.5,fontweight='bold',color='black')
    s6_g_check_ax.text(0,s6_g_y-.030,s6_g_row['점검'],transform=s6_g_check_ax.transAxes,
        ha='left',va='top',fontsize=9.8,color='#425466',linespacing=1.6)
s6_g_lot_ax.set_yticks(range(len(s6_g_check_rows)),[r['목적'] for r in s6_g_check_rows],fontsize=11,fontweight='bold')
s6_g_lot_ax.set_ylim(len(s6_g_check_rows)-.5,-.5)
s6_g_lot_ax.set_title('점검 목적별로 LOT·강종을 찾기',loc='left',fontweight='bold',pad=20)
s6_g_check_ax.set_title('확인할 구간 · 컬럼 · 비교 대상',loc='left',fontweight='bold',pad=20)
s6_g_lot_ax.set_xlim(pd.Timestamp('2024-01-01'),pd.Timestamp('2025-01-05'))
s6_g_lot_ax.xaxis.set_major_locator(mdates.MonthLocator())
s6_g_lot_ax.xaxis.set_major_formatter(mdates.DateFormatter('%m월'))
s6_g_lot_ax.set_xlabel('실제 관측일 [2024년] · ○ LOT 시작 / 짧은 가로선 = 실제 관측 시작~끝',labelpad=12)
for s6_g_ax in [s6_g_work_ax,s6_g_lot_ax]:
    s6_g_ax.grid(axis='x',alpha=.14);s6_g_ax.spines[['top','right']].set_visible(False)
s6_g_overview_axes=[s6_g_work_ax,s6_g_lot_ax]
s6_g_overview_notes=[s6_g_context_ax,s6_g_check_ax]
fig.suptitle('그림 10-G-1. 무엇을 점검할 것인가? — LOT·강종과 계획수리·정비 시점을 함께 보기',
    x=.15,y=.982,ha='left',fontweight='bold',fontsize=16)
fig.text(.15,.943,'세로축 = 점검 항목 · 같은 LOT가 서로 다른 점검 행에 반복될 수 있음 · 후보를 찾은 후 날짜와 정비 이력을 연결',
    fontsize=10.5,color='#425466')
fig.legend(handles=[
    Line2D([0],[0],marker='o',mfc='white',mec=s6_g_red,color='none',label='빨강: 우선·중요 후속 점검',ms=7),
    Line2D([0],[0],marker='o',mfc='white',mec=s6_g_green,color='none',label='초록: 참고 비교',ms=7),
    Line2D([0],[0],marker='o',mfc='white',mec=s6_g_gray,color='none',label='회색: 추가 표본 확보',ms=7),
    Patch(fc=s6_g_plan,ec='none',label='파랑 배경: CR1 계획수리')],
    loc='upper left',bbox_to_anchor=(.146,.918),ncol=4,frameon=False,fontsize=10)
fig.text(.15,.018,'색은 점검 우선순위이며 고장 판정이 아님 · 855℃는 승온 구간 구분용 참고값 · ±60일은 동일 강종 전후 LOT 탐색 범위',
    fontsize=9.5,color='#66717D')
save_and_show_figures()

# 연간 축에서 겹치는 짧은 정비는 감지 기준 시간축으로 확대합니다.
# LOT 트랙의 회색은 센서 파일에 관측이 없다는 뜻이며, 채널 NaN과 구분합니다.
s6_g_event_specs=[
    (240044,['WO-2024-00717'],'WO-2024-00717','OP-Z3 결측과 BM 직전 관측을 구분'),
    (240069,['WO-2024-01094'],'WO-2024-01094','CP-Z4 피크와 BM의 시간 간격 확인'),
    (240117,['WO-2024-01859','WO-2024-01854'],'WO-2024-01859','TBM·CBM 실제 작업 전후를 같은 LOT에서 확인'),
    (240133,['WO-2024-02105'],'WO-2024-02105','CBM 감지 관측과 교체 후 관측 부재를 구분')]
s6_g_work_rows=[];s6_g_actual_rows=[]
for s6_g_lot,s6_g_orders,s6_g_anchor_no,_ in s6_g_event_specs:
    s6_g_r=s6_lots.loc[s6_g_lot]
    s6_g_raw=temp[temp.LOT_NO.eq(s6_g_lot)]
    for s6_g_order in s6_g_orders:
        s6_g_w=cal[cal.WO_NO.eq(s6_g_order)].iloc[0]
        s6_g_parts=cal_parts[cal_parts.WO_NO.eq(s6_g_order)]
        s6_g_parts_text='; '.join(f'{r.PART_NM} · QTY {r.QTY} · {r.CHG_RSN}' for r in s6_g_parts.itertuples())
        s6_g_before_ok=bool(s6_g_r.start<=s6_g_w.STRT_DT-pd.Timedelta(minutes=10) and s6_g_r.end>=s6_g_w.STRT_DT)
        s6_g_after_ok=bool(s6_g_r.start<=s6_g_w.END_DT and s6_g_r.end>=s6_g_w.END_DT+pd.Timedelta(minutes=10))
        s6_g_work_rows.append({'LOT':s6_g_lot,'강종':s6_g_r.grade,'작업':s6_g_order,'유형':s6_g_w.MNT_TYPE,
            '감지':s6_g_w.DETC_DT,'착수':s6_g_w.STRT_DT,'완료':s6_g_w.END_DT,
            '기록 내용':str(s6_g_w.REMARK).strip() if pd.notna(s6_g_w.REMARK) else '미기록',
            '교체 여부':s6_g_w.PART_CHG,'부품교체 기록':s6_g_parts_text or '교체 내역 없음',
            '착수 직전 10분 관측':s6_g_before_ok,'완료 후 10분 관측':s6_g_after_ok,
            '실제 전후 비교':s6_g_before_ok and s6_g_after_ok,
            'LOT 종료→감지 [시간]':(s6_g_w.DETC_DT-s6_g_r.end).total_seconds()/3600})
        if s6_g_before_ok and s6_g_after_ok:
            s6_g_pre=s6_g_raw[s6_g_raw.MEAS_DT.ge(s6_g_w.STRT_DT-pd.Timedelta(minutes=10))&s6_g_raw.MEAS_DT.lt(s6_g_w.STRT_DT)]
            s6_g_post=s6_g_raw[s6_g_raw.MEAS_DT.ge(s6_g_w.END_DT)&s6_g_raw.MEAS_DT.lt(s6_g_w.END_DT+pd.Timedelta(minutes=10))]
            for s6_g_col in ['TC-Z3','OP-Z3','CP-Z4']:
                s6_g_actual_rows.append({'작업':s6_g_order,'LOT':s6_g_lot,'컬럼':s6_g_col,
                    '전 평균':s6_g_pre[s6_g_col].mean(),'후 평균':s6_g_post[s6_g_col].mean(),
                    '전 std':s6_g_pre[s6_g_col].std(),'후 std':s6_g_post[s6_g_col].std(),
                    '전 유효행':int(s6_g_pre[s6_g_col].count()),'후 유효행':int(s6_g_post[s6_g_col].count())})
s6_g_work_links=pd.DataFrame(s6_g_work_rows)
s6_g_actual_pairs=pd.DataFrame(s6_g_actual_rows)
s6_g_plan_context=pd.DataFrame(s6_g_plan_rows)

fig=plt.figure(figsize=(21,13))
s6_g_zoom_grid=fig.add_gridspec(8,2,height_ratios=[.7,1.4]*4,width_ratios=[1.85,1],
    left=.15,right=.985,top=.83,bottom=.08,hspace=.22,wspace=.075)
s6_g_event_axes=[];s6_g_event_headers=[];s6_g_event_notes=[]
for s6_g_i,(s6_g_lot,s6_g_orders,s6_g_anchor_no,s6_g_question) in enumerate(s6_g_event_specs):
    s6_g_header=fig.add_subplot(s6_g_zoom_grid[2*s6_g_i,0]);s6_g_header.axis('off');s6_g_event_headers.append(s6_g_header)
    ax=fig.add_subplot(s6_g_zoom_grid[2*s6_g_i+1,0]);s6_g_event_axes.append(ax)
    s6_g_side=fig.add_subplot(s6_g_zoom_grid[2*s6_g_i:2*s6_g_i+2,1]);s6_g_side.axis('off');s6_g_event_notes.append(s6_g_side)
    s6_g_r=s6_lots.loc[s6_g_lot];s6_g_anchor=cal[cal.WO_NO.eq(s6_g_anchor_no)].iloc[0].DETC_DT
    s6_g_header.text(0,.75,f'{s6_g_lot} · {s6_g_r.grade} — {s6_g_question}',
        va='center',fontsize=11,fontweight='bold',color='black')
    s6_g_anchor_note=(f'0 = TBM 01859 감지 {s6_g_anchor:%m/%d %H:%M} · CBM 감지: 전날 15:00, 확대 범위 밖'
        if s6_g_lot==240117 else f'0 = {s6_g_anchor_no[-5:]} 감지 {s6_g_anchor:%m/%d %H:%M} · 단위: 시간 · 패널별 범위 다름')
    s6_g_header.text(0,.13,s6_g_anchor_note,fontsize=9.3,va='center',color='#66717D')
    ax.axhspan(-.24,.24,color='#F0F2F5',zorder=0)
    s6_g_sensor_start=(s6_g_r.start-s6_g_anchor).total_seconds()/3600
    s6_g_sensor_end=(s6_g_r.end-s6_g_anchor).total_seconds()/3600
    ax.hlines(0,s6_g_sensor_start,s6_g_sensor_end,color=s6_g_red,lw=7,zorder=2)
    ax.scatter([s6_g_sensor_start,s6_g_sensor_end],[0,0],s=28,color=s6_g_red,zorder=3)
    s6_g_tracks=['센서 LOT'];s6_g_notes=[]
    for s6_g_j,s6_g_order in enumerate(s6_g_orders,start=1):
        s6_g_w=cal[cal.WO_NO.eq(s6_g_order)].iloc[0]
        s6_g_a=s6_g_work_links[s6_g_work_links['작업'].eq(s6_g_order)].iloc[0]
        s6_g_detect=(s6_g_w.DETC_DT-s6_g_anchor).total_seconds()/3600
        s6_g_start=(s6_g_w.STRT_DT-s6_g_anchor).total_seconds()/3600
        s6_g_end=(s6_g_w.END_DT-s6_g_anchor).total_seconds()/3600
        ax.hlines(s6_g_j,s6_g_start,s6_g_end,color='#263238',lw=5,zorder=2)
        ax.scatter(s6_g_detect,s6_g_j,marker='D',s=34,facecolors='white',edgecolors=s6_g_red,zorder=3)
        ax.scatter(s6_g_start,s6_g_j,marker='o',s=38,facecolors='white',edgecolors='#263238',zorder=3)
        ax.scatter(s6_g_end,s6_g_j,marker='s',s=30,color='#263238',zorder=3)
        s6_g_tracks.append(f'{s6_g_w.MNT_TYPE} {s6_g_order[-5:]}')
        if s6_g_a['실제 전후 비교']:
            ax.hlines(0,s6_g_start-1/6,s6_g_start,color='#83B8D8',lw=11,zorder=1)
            ax.hlines(0,s6_g_end,s6_g_end+1/6,color='#83B8D8',lw=11,zorder=1)
        s6_g_notes.append(f'{s6_g_w.MNT_TYPE} {s6_g_order[-5:]} · {s6_g_w.STRT_DT:%m/%d %H:%M}~{s6_g_w.END_DT:%H:%M}')
    ax.axvline(0,color=s6_g_gray,ls=':',lw=.8)
    ax.set_yticks(range(len(s6_g_tracks)),s6_g_tracks,fontsize=9)
    ax.set_ylim(len(s6_g_tracks)-.55,-.65)
    s6_g_zoom_limits={240044:(-7,10),240069:(-17,13),240117:(-1.2,8.2),240133:(-2.5,9.5)}
    s6_g_zoom_ticks={240044:[-6,-3,0,3,6,9],240069:[-15,-10,-5,0,5,10],
        240117:[0,2,4,6,8],240133:[-2,0,2,4,6,8]}
    ax.set_xlim(*s6_g_zoom_limits[s6_g_lot]);ax.set_xticks(s6_g_zoom_ticks[s6_g_lot])
    ax.tick_params(axis='x',labelsize=9)
    ax.grid(axis='x',alpha=.13);ax.spines[['top','right','left']].set_visible(False)
    if s6_g_i==len(s6_g_event_specs)-1:
        ax.set_xlabel('기준 감지 시각으로부터 경과 [시간] · 패널별 범위 다름',fontsize=9,labelpad=7)
    # 조치와 비교 가능성은 곡선·막대 바깥에 표시합니다.
    if s6_g_lot==240044:
        s6_g_gap=s6_g_work_links[s6_g_work_links.LOT.eq(240044)]['LOT 종료→감지 [시간]'].iloc[0]
        s6_g_head=f'LOT 종료 {s6_g_gap:.1f}시간 뒤 BM 감지'
        s6_g_action='동작 불량 · 교체 N · 상세 조치 미기록'
        s6_g_next='OP-Z3 결측 원인 ↔ SPHC 240061·240069·240091\n실제 작업 직전·완료 후 센서 없음 → 효과 미확인'
    elif s6_g_lot==240069:
        s6_g_a=s6_g_work_links[s6_g_work_links.LOT.eq(240069)].iloc[0]
        s6_g_head=f"LOT 종료 {s6_g_a['LOT 종료→감지 [시간]']:.1f}시간 뒤 BM 감지"
        s6_g_action='절연 저하 / '+s6_g_a['부품교체 기록']
        s6_g_next='CP 피크 구간 ↔ 나머지 SPHC · 부품/태그 대응 확인\n완료 후 센서 없음 → 교체 후 변화 미확인'
    elif s6_g_lot==240117:
        s6_g_head='두 작업 모두 착수 전·완료 후 10분 관측 있음'
        s6_g_action='TBM: 계획 점검·교체 N / CBM: 마모량 초과·교체 N'
        s6_g_next='TBM은 승온과 중첩 · CBM 후 10분의 세 신호 std 감소\n20분 창에서는 증가 → 지속 개선은 미확인\n작업 중첩 + SPCD n=1 · 상세 비교 그림 17-A·B / 19-C'
    else:
        s6_g_a=s6_g_work_links[s6_g_work_links.LOT.eq(240133)].iloc[0]
        s6_g_head='감지는 LOT 안 · 착수·완료는 LOT 밖'
        s6_g_action=s6_g_a['부품교체 기록']
        s6_g_next='CP-Z4 감지 전후 ↔ SPCE 240037·240121\n교체 완료 후 센서 없음 → CP 변화 미확인'
    s6_g_side.text(0,.94,s6_g_head,va='top',fontsize=10.3,fontweight='bold',color=s6_g_red)
    s6_g_side.text(0,.72,' / '.join(s6_g_notes),va='top',fontsize=9.2,color='#66717D')
    s6_g_side.text(0,.50,s6_g_action,va='top',fontsize=9.6,color='#425466')
    s6_g_side.text(0,.27,s6_g_next,va='top',fontsize=9.5,color='#425466',linespacing=1.6)
fig.suptitle('그림 10-G-2. 정비 전후를 실제로 비교할 수 있는 LOT는? — 감지·착수·완료 확대',
    x=.15,y=.98,ha='left',fontweight='bold',fontsize=16)
fig.text(.15,.943,'LOT·강종 → 감지와 작업 시간 → 기록된 조치 → 전후 관측 유무 · 각 패널의 0은 기준 감지 시각이며 눈금 범위는 다름',fontsize=10.5,color='#425466')
fig.legend(handles=[
    Line2D([0],[0],color=s6_g_red,lw=5,label='빨간 막대: 실제 센서 관측'),
    Line2D([0],[0],color='#263238',lw=4,label='검정 막대: 착수~완료'),
    Line2D([0],[0],ls='None',marker='D',mfc='white',mec=s6_g_red,label='◇ 감지'),
    Line2D([0],[0],ls='None',marker='o',mfc='white',mec='#263238',label='○ 착수'),
    Line2D([0],[0],ls='None',marker='s',color='#263238',label='■ 완료'),
    Patch(fc='#83B8D8',ec='none',label='파랑: 전후 10분 비교 창')],
    loc='upper left',bbox_to_anchor=(.145,.918),ncol=3,frameon=False,fontsize=10)
fig.text(.15,.018,'LOT 트랙의 회색: 관측 자료 없음 · OP-Z3 채널 결측과 구분 · 240117의 CBM 감지는 전날이며 확대 범위 밖',
    fontsize=9.5,color='#66717D')
save_and_show_figures()

# 아래 마크다운 결과표와 후속 검증에 사용합니다.
s6_known_pairs=s6_repair_pairs[['계획수리','강종','수리 전 LOT','수리 후 LOT']].drop_duplicates()
s6_pair_text='; '.join(f'{r["강종"]}: {r["수리 전 LOT"]} → {r["수리 후 LOT"]}' for _,r in s6_known_pairs.iterrows()) or '없음'
assert s6_g_work_links['실제 전후 비교'].sum()==2
assert s6_g_work_links.loc[s6_g_work_links['실제 전후 비교'],'LOT'].eq(240117).all()
assert s6_g_actual_pairs[['전 유효행','후 유효행']].gt(0).all().all()


<a id="inspection-maintenance"></a>
#### 그림 10-G-1·2 — 무엇을 점검하며, 정비 전후를 어디에서 볼 수 있는가?

**① 읽는 순서:** 10-G-1의 **세로축은 점검 목적**입니다. 해당 행의 **LOT·강종 → 실제 관측일과 CR1 계획수리 → 오른쪽 구간·컬럼·비교 대상**을 읽습니다. 빨강은 우선·중요 후속 점검, 초록은 참고 비교입니다. 같은 LOT가 여러 목적에 반복되어도 여러 고장으로 세지 않습니다.

**10-G-2는 짧은 정비의 확대 그림입니다.** 빨간 막대는 실제 센서 관측, 검정 막대는 착수~완료입니다. **◇ 감지 / ○ 착수 / ■ 완료**를 구분합니다. 센서 트랙의 회색은 **관측 자료 없음**이며 OP-Z3만의 채널 결측과는 다릅니다. 파랑 구간은 착수 직전·완료 후 각 10분의 비교 창입니다. **각 패널의 시간 눈금 범위는 다릅니다.** 240117의 0은 TBM 감지이며, CBM 감지는 전날 15:00으로 확대 범위 밖입니다.

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 점검 목적 | LOT·강종 | 자세히 볼 구간·컬럼 | 비교 대상·알게 된 점 |
|---|---|---|---|
| **결측 원인** | **240044·240061 / SPHC** | OP-Z3 **0~81초 / 0~183초** ↔ 동시각 TC-Z3·OP-Z4·Z5·Z6 | 두 LOT의 수집 시작 조건 대조. 다른 채널은 기록되어 있어 OP-Z3 초기 수집 문제를 먼저 확인 |
| **종료 하강** | **240058 / DP590**, **240114 / SPCEN** | 종료부 TC-Z3·Z4·Z5·Z6와 OP-Z3~Z6 동반 하강 | 각각 6월·11월 CR1 수리 전 마지막 관측. 서로는 형태 참고만 비교하고 정지 로그 대조 |
| **초반 승온** | **240061 / SPHC**, **240117 / SPCD** | 시작~승온 종료, 이후 TC-Z3↔OP-Z3 | CR1 수리 후 첫 관측. 240061은 같은 SPHC 240044·240069·240091과 승온 이후 비교; SPCD는 n=1 |
| **CP 피크·변동** | **240069 / SPHC** | CP-Z4 **564.5~584.5분**, 피크 제외 전/후 std | 나머지 SPHC와 비교. LOT 종료 **2.3시간 뒤 BM 감지**는 시간 위치이며 CP 피크의 원인 근거는 아님 |
| **지속 수준 차이** | **240091 / SPHC**, **240121 / SPCE** | 중간 **20~80%**, TC-Z7↔OP-Z7·TC-Z8↔OP-Z8 | 각각 자기 제외 동일 강종 LOT와 비교. 두 TC의 차이가 **12/12칸** 지속; 목표값·속도·두께 대조 필요 |
| **실제 작업 전후** | **240117 / SPCD** | TBM 01859·CBM 01854의 착수 직전→완료 후 TC-Z3·OP-Z3·CP-Z4 | 두 작업 모두 같은 LOT에 전후 관측 있음. 작업 중첩·승온 때문에 개별 정비 효과 분리에는 한계 |
| **교체 후 관측 확보** | **240133 / SPCE** | CBM 02105 감지 주변 / 가스 분석계 셀 교체 후 CP-Z4 | 감지는 LOT 안이지만 착수·완료는 밖. 교체 후 센서가 없어 효과 미확인 |
| **강종 조건 참고** | **240024↔240108 / SPFH590**, **240038↔240114 / SPCEN** | 중간 온도 수준·변동폭과 같은 존 OP | n=2 참고 비교. 반복되는 강종 조건과 개별 LOT 차이를 추가 표본으로 분리 |

**CR1 계획수리와 CAL01 정비는 기록 단위가 다릅니다.** 6월 계획은 **06/10~06/14, 연속소둔로 노내 정비**, 11월 계획은 **11/05~11/10, 냉연 라인 전 설비 정기수리**입니다. 개별 작업은 `EQP_TAG=P1-CR1-CAL01`의 TBM·CBM·BM 이력으로 따로 연결합니다.

| 실제 작업·유형 | 연결 LOT·강종 | 기록된 내용·조치 | 전후 관측으로 알 수 있는 범위 |
|---|---|---|---|
| **00717 / BM** | **240044 / SPHC** | 동작 불량 · 부품교체 N · 상세 조치 미기록 | LOT 종료 **3.7시간 뒤 감지**. 실제 작업 직전·완료 후 센서 없음 |
| **01094 / BM** | **240069 / SPHC** | 절연 저하 · **O링 세트, QTY 3, 예방 교체** | LOT 종료 **2.3시간 뒤 감지**. 교체 후 센서 없음; 부품과 CP 태그 관계 확인 필요 |
| **01859 / TBM** | **240117 / SPCD** | 11/13 **07:40~09:35**, 계획 점검 수행 · 부품교체 N | 전후 관측 있음. 전은 승온 중; 후에 TC-Z3 수준은 오르고 OP·CP std도 증가 |
| **01854 / CBM** | **240117 / SPCD** | 11/13 **08:54~12:44**, 마모량 초과 · 부품교체 N · 상세 조치 미기록 | 완료 후 **10분**의 TC-Z3·OP-Z3·CP-Z4 std 감소. **20분**으로 늘리면 증가하여 지속 개선 근거 부족 |
| **02105 / CBM** | **240133 / SPCE** | **가스 분석계 셀, QTY 3, 예방 교체** | 감지 주변 관측만 있음. 교체 완료 후 CP 변화는 확인 불가 |

**같은 강종의 계획수리 전후 비교:** 6월 **SPHC 240044 → 240061·240069**는 양쪽 표본이 있습니다. 다만 240044는 수리 37.2일 전, 240069는 22.7일 후의 원격 관측입니다. **±60일은 비교 LOT 탐색 범위**이며 실제 수리 직전·완료 직후 관측이나 동일 운전조건을 뜻하지 않습니다.

**③ 다음 후속 점검**

| 우선 대상 | 해야 할 비교·확인 | 이어지는 분석·필요 자료 |
|---|---|---|
| **240044·240061 / SPHC** | OP-Z3 시작 결측의 공통 수집 조건, 240044 BM의 실제 고장 부위·조치 | 그림 11의 OP-Z3 결측 확대·17-C / 태그 품질·통신·수집 로그·정비 보고서 |
| **240061·240058·240114·240117** | 승온·종료 전환과 실제 재가동·정지 시각, 이후 남는 TC↔OP 차이 | 그림 15·19-A / 목표값·라인속도·가동상태 |
| **240069 / SPHC** | CP 피크 제외 민감도, BM 교체 부품과 CP 계측의 대응, 완료 후 값 | 그림 19-B·17-C / 분위기·교정·교체 시각·완료 후 센서 |
| **240091 / SPHC·240121 / SPCE** | 같은 강종 중간 구간에서 TC-Z7·Z8 목표값·OP-Z7·Z8 비교 | 그림 10-E / 속도·두께·목표온도 |
| **240117 / SPCD** | 겹친 TBM·CBM의 작업 상세와 5·10·20분 전후 변화, 비정비 대조 구간 | 그림 17-A·B·19-C / 동일 상태 SPCD LOT 추가 확보 |
| **240133 / SPCE** | 분석계 셀 교체·교정 후 CP-Z4 관측 확보 | 그림 17-C / 장착·교정·태그 정의·교체 후 센서 |

**현재 알 수 있는 것:** 점검 목적에 맞는 **LOT·강종·구간·비교군**, 정비와의 시간 위치, 실제 전후 비교 가능 여부입니다. **원인과 정비 효과**는 운전조건·정비 상세·완료 후 센서 대조가 필요합니다.


### 최종 확인 — A·B·C·D와 단일군 참고

| 구분 | 의미 |
|---|---|
| A | n≥3에서 중간 구간의 같은 강종 차이가 남는 상세 확인 후보 |
| B | 동일 강종에서 반복되는 공통 수준. n=2는 참고 비교 |
| C | 결측 원인을 먼저 확인하되 결측 없는 구간의 차이도 함께 검토 |
| D | 같은 강종이 하나뿐이어서 강종 조건과 LOT 차이를 분리할 수 없음 |
| 단일군 참고 | 다중신호 후보는 아니지만 한 신호군의 같은 강종 차이가 남음 |
| 보류 | n=2 참고 차이 또는 차이 기준 미만. 보류 이유를 기록 |

아래는 일부 대표 사례의 목록이 아니라 **전체 14개 LOT의 확인표**입니다. D 또는 보류라는 표시는 고장이나 정상 판정이 아닙니다.


In [ ]:

# 전체 14 LOT를 하나씩 검토합니다. 두 대표 사례로 최종 목록을 제한하지 않습니다.
from textwrap import fill

s6_next_actions={
    240001:'추가 SGCC LOT 확보 · 현재 다중신호 기준 미만',
    240024:'240108과 목표온도·소재규격 대조 · 공통 저온 수준 확인',
    240037:'전체 구간 CP-Z4 피크와 중간 차이 소멸 여부 확인',
    240038:'240114와 목표온도·출력 변동폭 대조 · n=2 참고',
    240044:'OP-Z3 초반 결측 75행의 센서·통신 로그 확인',
    240058:'177.5~216.5분 다중 존 하강의 종료·가동상태 확인',
    240061:'OP-Z3 결측 179행과 긴 승온을 분리하여 확인',
    240069:'CP-Z4 피크·제어 기록 + TC-Z3 중앙값 대조',
    240091:'TC-Z7·TC-Z8 목표온도·속도·두께 확인 · 단일 온도군 참고',
    240108:'240024와 공통 TC-Z7·TC-Z8 평균 수준·TC-Z3~TC-Z6 평균 변동폭 참고 비교',
    240114:'280~319분 다중 존 하강 · 240038과 운전조건 대조',
    240117:'초반 승온 상태와 추가 SPCD LOT 확보',
    240121:'TC-Z7·TC-Z8 목표온도·속도·두께 확인 · 지속 수준 차이',
    240133:'240037과도 대조 · 높은 240121이 비교 기준에 주는 영향 확인',
}
s6_focus={
    240024:'가열_roll_width',240037:'CP_roll_sd',240038:'OP_roll_width',
    240044:'OP-Z3',240058:'가열·균열존',240061:'가열_roll_width',
    240069:'CP_roll_sd',240091:'과시효대',240108:'가열_roll_width',
    240114:'OP_roll_width',240117:'가열·균열존',240121:'과시효대',240133:'과시효대'
}
s6_audit_rows=[]
for lot,r in s6_lots.iterrows():
    peer_ids=s6_peer_ids[lot]
    differences=[]
    for c in S6_FEATURES:
        if not s6_diff_middle.loc[lot,c]:
            continue
        v=s6_middle.loc[lot,c]; ref=s6_peer_ref['중간 20~80%'].loc[lot,c]
        differences.append(f'{S6_SIMPLE[c]} {v/ref:.1f}배' if c=='CP_sd' else
                           f'{S6_SIMPLE[c]} {v-ref:+.2f}{S6_UNITS[c]}')
    same='비교 LOT 없음' if not peer_ids else '; '.join(differences) or '수준·변동폭 차이 기준 미만'
    if r.grade_n==2:
        same+=' · 참고'
    temporal='같은 강종 비교 불가' if not peer_ids else '다중신호 기준 미만'
    if peer_ids and lot in s6_focus:
        channel=s6_focus[lot]
        p=s6_persistence[s6_persistence.LOT.eq(lot)&s6_persistence['채널'].eq(channel)].iloc[0]
        temporal=f"{S6_TRACE_NAMES[channel]}: {int(p['같은 방향 차이 칸'])}/{int(p['유효 중간 칸'])}칸 ({p['중간 지속 비율 [%]']:.0f}%)"
        if channel in S6_TRACE_VARIABILITY:
            temporal+=' · 변동 지속성'
        if r.grade_n==2:
            temporal+=' · 참고'
    evidence='중간: '+s6_screen.loc[lot,'중간 신호군']
    if s6_screen.loc[lot,'전체 후보'] and not s6_screen.loc[lot,'중간 후보']:
        evidence+='; 전체에서 추가 후보'
    if s6_screen.loc[lot,'OP-Z3 원본 결측행']:
        evidence+=f"; OP-Z3 결측 {int(s6_screen.loc[lot,'OP-Z3 원본 결측행'])}행"
    if lot in [240058,240114]:
        tail=terminal_falls_table[terminal_falls_table.LOT.eq(lot)].iloc[0]
        evidence+=f"; 종료 TC-Z3~TC-Z6 평균 {tail['가열 평균 차이 [℃]']:.1f}℃"
    repairs=s6_repair_link[s6_repair_link.LOT.eq(lot)]
    if len(repairs):
        link=repairs.sort_values('경계 거리 [일]').iloc[0]
        repair=f"{link['수리 시작']:%m/%d} {link['위치']} {link['경계 거리 [일]']:.1f}일"
        repair+=' · 반대편 '+link['반대편 동일 강종 LOT']
    else:
        repair='±60일 범위의 수리 없음'
    s6_audit_rows.append({'LOT':lot,'강종':r.grade,'강종 LOT수':int(r.grade_n),
        '검토 구분':s6_review.loc[lot,'분류'],'후보·전환·결측 근거':evidence,
        '같은 강종 차이':same,'시간 지속성':temporal,
        '수리 위치':repair,'다음 확인':s6_next_actions[lot]})
s6_audit=pd.DataFrame(s6_audit_rows).set_index('LOT')
s6_transition_ids=set(terminal_falls_table.LOT.astype(int))
for lot,d in s6_analysis.groupby('LOT_NO'):
    heating=d.dropna(subset=['가열·균열존'])
    if len(heating) and heating['가열·균열존'].iloc[0]<855 and heating['가열·균열존'].ge(855).any():
        s6_transition_ids.add(int(lot))
s6_h_states={}
for lot,r in s6_audit.iterrows():
    grade_n=s6_lots.loc[lot,'grade_n'];missing=s6_screen.loc[lot,'결측 확인 대상']
    flagged=s6_diff_middle.loc[lot].any()
    p=s6_persistence[s6_persistence.LOT.eq(lot)&s6_persistence['채널'].eq(s6_focus.get(lot,''))]
    persistent=bool(len(p) and p['지속 후보'].iloc[0])
    overall=('우선 확인' if missing or lot in s6_transition_ids or (flagged and grade_n>=3) else
        '참고 확인' if s6_screen.loc[lot,'탐색 후보'] or s6_flags.loc[lot].any() or s6_mid_flags.loc[lot].any() else '기준 비해당')
    same=('자료 없음' if grade_n==1 else '우선 확인' if flagged and grade_n>=3 else
        '참고 확인' if flagged or s6_review.loc[lot,'반복 수준 지표']!='—' else '기준 비해당')
    temporal=('자료 없음' if grade_n==1 or p.empty or p['유효 중간 칸'].iloc[0]==0 else
        '우선 확인' if persistent and flagged and grade_n>=3 else
        '참고 확인' if p['같은 방향 차이 칸'].iloc[0]>0 else '기준 비해당')
    action='참고 확인' if overall=='기준 비해당' and grade_n==1 else overall
    s6_h_states[lot]=[overall,overall,same,temporal,action]
s6_priority_ids=[lot for lot,states in s6_h_states.items() if states[0]=='우선 확인']


# 정확한 컬럼명과 다음 확인을 7 LOT씩 넓게 표시합니다.
h_columns=[('LOT · 강종',2.1,19),('후보·전환·결측 근거',3.3,27),
           ('같은 강종 차이',4.4,36),('시간 지속성',3.8,30),('다음 확인',4.7,38)]
h_all_ids=s6_audit.index.tolist()
for h_page_number,h_first in enumerate(range(0,len(h_all_ids),7),start=1):
    h_ids=h_all_ids[h_first:h_first+7]
    h_fig,h_ax=plt.subplots(figsize=(19,12.2))
    h_ax.set_xlim(0,sum(w for _,w,_ in h_columns));h_ax.set_ylim(len(h_ids)+.35,-1.2);h_ax.axis('off')
    for j,(title,width,wrap) in enumerate(h_columns):
        x=sum(w for _,w,_ in h_columns[:j]);h_ax.text(x+.08,-.62,title,fontweight='bold',fontsize=11,va='center')
    for i,lot in enumerate(h_ids):
        r=s6_audit.loc[lot]
        texts=[f"{lot} · {r['강종']}\nn={r['강종 LOT수']} · "+r['검토 구분'].split('·')[0],
               r['후보·전환·결측 근거'],r['같은 강종 차이'],r['시간 지속성'],r['다음 확인']]
        for j,((_,width,wrap),value) in enumerate(zip(h_columns,texts)):
            x=sum(w for _,w,_ in h_columns[:j]);h_style=S6_REVIEW_STYLE.get(s6_h_states[lot][j],S6_MISSING_STYLE)
            h_ax.add_patch(plt.Rectangle((x,i),width,1,fc=h_style['bg'],ec='white',lw=1.5))
            content='\n'.join(fill(part,width=wrap,break_long_words=True,break_on_hyphens=False) for part in value.split('\n'))
            h_ax.text(x+.08,i+.50,content,ha='left',va='center',fontsize=10,color=h_style['fg'],linespacing=1.35)
    h_fig.suptitle(f'그림 10-H-{h_page_number}. 전체 LOT 확인표 — {h_page_number}/2',x=.025,y=.98,ha='left',fontweight='bold',fontsize=16)
    h_fig.text(.025,.937,f'빨강: 우선·중요 후속 확인 {len(s6_priority_ids)} LOT · 초록: 참고 · 흰색: 기준 미만 · 회색: 비교 자료 없음',fontsize=10,color='#425466')
    h_fig.text(.025,.04,'시간 지속성: 중간 20~80%의 같은 방향 차이 비율 · n=2 참고 · 평균 변동폭은 시각별 평균온도에서 계산',fontsize=9,color='#425466')
    h_fig.text(.025,.019,'계획수리 위치는 뒤 마크다운 표에서 확인합니다. 각 TC 컬럼과 평균 지표를 구분하여 읽습니다.',fontsize=9,color='#425466')
    h_fig.subplots_adjust(left=.025,right=.99,top=.89,bottom=.07)
    save_and_show_figures()


#### 그림 10-H · 전체 LOT 최종 확인 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 한 행·각 칸 | LOT별 결측, 같은 강종 차이, 시간 지속성, 수리 위치를 순서대로 대조. |
| 확인 색 | 빨강은 우선·중요 후속 확인, 초록은 참고 확인, 흰색은 기준 미만, 회색은 비교 자료 없음. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| **240044·240061 · SPHC** | OP-Z3 결측 **75 / 179행** | **C. 수집 품질 먼저 확인**; 240061은 승온·변동도 함께 분석. |
| **240069 · SPHC** | CP-Z4 std **13.5배**, 시간 차이 **4/12칸** | **A. CP 상세분석**; 지속 상승보다 일부 구간 피크를 확인. |
| **240121 · SPCE / 240091 · SPHC** | TC-Z7·TC-Z8 평균 각각 **+3.08 / +3.14℃**, 둘 다 **12/12칸** | 지속 수준 차이. 240091은 단일군 참고이지만 후속 확인 필요. |
| 240024·240108 / 240038·240114 | n=2 공통 조건·차이 참고 | 강종 조건과 개별 LOT 차이를 추가 자료로 구분. |
| 240058·240117 | n=1; 종료 하강 / 초반 승온 | 전환은 점검하면서 **D. 강종 내 판정 보류** 유지. |

| 구분하려던 현상 | 해당 LOT | 다음에 볼 것 |
|---|---|---|
| **지속적인 온도 수준 차이** | 240121·240091 | TC-Z7·TC-Z8 각각의 목표온도·운전조건 |
| **일부 구간의 큰 변동** | 240069 | CP-Z4 피크 전후 확대 |
| **승온이 섞인 온도·변동폭 차이** | 240061 | 승온을 제외한 TC-Z3↔OP-Z3 |

**결론:** 후보는 한 종류가 아닙니다. **결측 → 운전 전환 → 같은 강종 차이 → 지속성 → 정비 맥락** 순서로 확인해야 다음 점검 목적이 분명해집니다.

**③ 후속 작업 — 무엇을 누구와 비교할 것인가?**

| LOT | 강종 | 검토 구분 | 같은 강종 차이 | 시간 지속성 | 수리 위치 | 다음 확인 |
|---|---|---|---|---|---|---|
| 240001 | SGCC | D. 비교 불가 / 보류 | 비교 LOT 없음 | 같은 강종 비교 불가 | ±60일 범위의 수리 없음 | 추가 SGCC LOT 확보 · 현재 다중신호 기준 미만 |
| 240024 | SPFH590 | B. 동일 강종 반복 · 조건 차이 가능 | TC-Z3~TC-Z6 평균 변동폭 -0.66℃ · 참고 | TC-Z3~TC-Z6 평균 변동폭(5분): 2/12칸 (17%) · 변동 지속성 · 참고 | ±60일 범위의 수리 없음 | 240108과 목표온도·소재규격 대조 · 공통 저온 수준 확인 |
| 240037 | SPCE | 보류: 강종 내 지속 차이 근거 부족 | 수준·변동폭 차이 기준 미만 | CP-Z4 std(5분): 3/12칸 (25%) · 변동 지속성 | 06/10 수리 전 56.6일 · 반대편 없음 | 전체 구간 CP-Z4 피크와 중간 차이 소멸 여부 확인 |
| 240038 | SPCEN | 보류: n=2 참고 비교 | TC-Z7·TC-Z8 평균온도 -3.23℃; TC-Z3~TC-Z6 평균 변동폭 +1.85℃; OP-Z3 변동폭 +32.10%p · 참고 | OP-Z3 변동폭(5분): 2/12칸 (17%) · 변동 지속성 · 참고 | 06/10 수리 전 54.4일 · 반대편 없음 | 240114와 목표온도·출력 변동폭 대조 · n=2 참고 |
| 240044 | SPHC | C. 결측 영향 먼저 확인 | 수준·변동폭 차이 기준 미만 | OP-Z3 출력: 2/12칸 (17%) | 06/10 수리 전 37.2일 · 반대편 240061, 240069 | OP-Z3 초반 결측 75행의 센서·통신 로그 확인 |
| 240058 | DP590 | D. 비교 불가 / 보류 | 비교 LOT 없음 | 같은 강종 비교 불가 | 06/10 수리 전 0.6일 · 반대편 없음 | 177.5~216.5분 다중 존 하강의 종료·가동상태 확인 |
| 240061 | SPHC | C. 결측 영향 먼저 확인 | TC-Z3~TC-Z6 평균 변동폭 +8.82℃ | TC-Z3~TC-Z6 평균 변동폭(5분): 6/12칸 (50%) · 변동 지속성 | 06/10 수리 후 0.0일 · 반대편 240044 | OP-Z3 결측 179행과 긴 승온을 분리하여 확인 |
| 240069 | SPHC | A. 강종 내 차이 · 상세분석 후보 | CP-Z4 std 13.5배 | CP-Z4 std(5분): 4/12칸 (33%) · 변동 지속성 | 06/10 수리 후 22.7일 · 반대편 240044 | CP-Z4 피크·제어 기록 + TC-Z3 중앙값 대조 |
| 240091 | SPHC | 단일군 참고: 같은 강종 내 차이 | TC-Z7·TC-Z8 평균온도 +3.14℃ | TC-Z7·TC-Z8 평균온도: 12/12칸 (100%) | 11/05 수리 전 56.6일 · 반대편 없음 | TC-Z7·TC-Z8 목표온도·속도·두께 확인 · 단일 온도군 참고 |
| 240108 | SPFH590 | B. 동일 강종 반복 · 조건 차이 가능 | TC-Z3~TC-Z6 평균 변동폭 +0.66℃ · 참고 | TC-Z3~TC-Z6 평균 변동폭(5분): 2/12칸 (17%) · 변동 지속성 · 참고 | 11/05 수리 전 17.5일 · 반대편 없음 | 240024와 공통 TC-Z7·TC-Z8 평균 수준·TC-Z3~TC-Z6 평균 변동폭 참고 비교 |
| 240114 | SPCEN | 보류: n=2 참고 비교 | TC-Z7·TC-Z8 평균온도 +3.23℃; TC-Z3~TC-Z6 평균 변동폭 -1.85℃; OP-Z3 변동폭 -32.10%p · 참고 | OP-Z3 변동폭(5분): 2/12칸 (17%) · 변동 지속성 · 참고 | 11/05 수리 전 0.1일 · 반대편 없음 | 280~319분 다중 존 하강 · 240038과 운전조건 대조 |
| 240117 | SPCD | D. 비교 불가 / 보류 | 비교 LOT 없음 | 같은 강종 비교 불가 | 11/05 수리 후 2.3일 · 반대편 없음 | 초반 승온 상태와 추가 SPCD LOT 확보 |
| 240121 | SPCE | A. 강종 내 차이 · 상세분석 후보 | TC-Z7·TC-Z8 평균온도 +3.08℃ | TC-Z7·TC-Z8 평균온도: 12/12칸 (100%) | 11/05 수리 후 13.3일 · 반대편 없음 | TC-Z7·TC-Z8 목표온도·속도·두께 확인 · 지속 수준 차이 |
| 240133 | SPCE | 보류: 강종 내 지속 차이 근거 부족 | 수준·변동폭 차이 기준 미만 | TC-Z7·TC-Z8 평균온도: 10/12칸 (83%) | 11/05 수리 후 47.6일 · 반대편 없음 | 240037과도 대조 · 높은 240121이 비교 기준에 주는 영향 확인 |

**원인 확인에 남은 자료:** 목표온도·속도·두께·가동상태·수집 로그. 반대편 동일 강종이 없으면 수리 인근 위치까지만 설명합니다. ±60일은 탐색 범위입니다.


In [ ]:
# 해석 문장도 현재 계산된 수치와 시간 지속성에서 생성합니다.
def s6_delta_sentence(lot,c):
    value=s6_middle.loc[lot,c]; ref=s6_peer_ref['중간 20~80%'].loc[lot,c]
    return f'{s6_value(value,c)} 대 {s6_value(ref,c)} (Δ {s6_value(value-ref,c,True)})'
def s6_persistence_sentence(lot,c):
    p=s6_persistence[s6_persistence.LOT.eq(lot)&s6_persistence['채널'].eq(c)].iloc[0]
    return f'중간 유효 {int(p["유효 중간 칸"])}칸 중 {int(p["같은 방향 차이 칸"])}칸에서 기준을 넘는 같은 방향 차이'
candidate_ids=s6_screen.index[s6_screen['탐색 후보']].tolist()
cp69=s6_middle.loc[240069,'CP_sd']; cp69ref=s6_peer_ref['중간 20~80%'].loc[240069,'CP_sd']
cp_peak=s6_analysis.loc[s6_analysis[s6_analysis.LOT_NO.eq(240069)]['CP-Z4'].idxmax()]
spfh=s6_middle.loc[[240024,240108],['과시효대','냉각대']]
pair_description='; '.join(f'{r["계획수리"]} {r["강종"]}: {r["수리 전 LOT"]} → {r["수리 후 LOT"]}' for _,r in s6_repair_pairs[['계획수리','강종','수리 전 LOT','수리 후 LOT']].drop_duplicates().iterrows()) or '없음'


### 분석 결과의 활용

전체 후보와 최종 확인 대상은 동일한 목록이 아닙니다. 같은 강종에서도 남는 차이, 한 신호군의 지속 차이, 결측, 종료 전환, 공통 운전조건, 비교 표본 부족을 각각 구분했습니다. **전체 14개 LOT의 다음 확인과 계획수리 위치는 위 10-H 확인표**를 기준으로 읽습니다.

계획수리 전후 위치는 해당 LOT의 발생 시점을 설명하는 보조 정보입니다. 운전조건과 연속 관측이 부족하므로 수리의 인과 효과는 확정하지 않습니다.

[↑ 목차](#toc)


<a id="case"></a>
## 7. SPHC 240061 — 공정 흐름과 OP-Z3 결측을 한 그림에서 확인

**분석 질문:** 낮은 온도는 여러 존의 공통 승온인가? TC-Z3·OP-Z3 변화 시각에 다른 공정·CP도 변하는가? OP-Z3가 없는 시작 구간에도 다른 채널은 기록되는가?

**위에서 아래로 읽습니다:** **세정 → 건조 TC·OP → 가열 TC·OP → OP-Z3 결측 확대 → CP → 과시효 TC·OP → TC-Z9 → 냉각**. 각 공정의 TC 다음에 같은 존의 OP를 놓았습니다. **가열 출력 패널 바로 아래에서 OP-Z3의 시작 0~5분을 확대**하고, **한 확대 그래프의 왼쪽 축은 TC-Z3~Z6 온도, 오른쪽 축은 OP-Z3~Z6 출력**입니다. TC는 실선, OP는 점선으로 같은 초 단위에서 대조합니다.

**범위·단위:** 제공된 **22개 신호**를 모두 포함합니다. 전체 흐름은 LOT 시작 후 **분** 단위의 유효 30초 평균, 결측 확대 한 패널은 **초** 단위 원본입니다. 온도는 ℃, 출력은 %, CP-Z4는 무차원이며 CP-Z4M만 ×10^-10으로 표시합니다. 전체 흐름과 확대의 시간 단위가 다른 것을 먼저 확인합니다.

**색·선:** **노랑 = 첫 10분**, **초록 = 마지막 10분**, **옅은 빨강 = 가열 TC·OP 확인 128~135분**, **회색 빗금 = OP-Z3가 기록되지 않은 시각**. 같은 존의 TC·OP는 같은 선색이며 붉은 TC-Z3·OP-Z3를 우선 대조합니다. 검정은 표시 TC의 동시각 평균이고 CLN-IN은 단일 컬럼 값입니다.

**계산 기준:** 30초 칸에 원본 유효 관측 15개 이상이 있는 경우만 표시합니다. 처음·마지막 10분 평균은 원본에서 계산합니다. 붉은 구간은 OP-Z3 저출력(<1%)의 연속 구간 중 TC-Z3 최대값이 가장 큰 구간입니다. 결측을 보간하거나 0%로 채우지 않습니다.


In [ ]:
# TC·OP를 같은 존의 위/아래 패널로 직접 생성합니다. 외부 이미지나 파일을 사용하지 않습니다.
case=analysis_ts[analysis_ts.LOT_NO.eq(CASE_LOT)].copy()
case_raw=temp[temp.LOT_NO.eq(CASE_LOT)].copy()
first=case_raw[case_raw.MEAS_DT.lt(case_raw.MEAS_DT.min()+pd.Timedelta(minutes=10))]
last=case_raw[case_raw.MEAS_DT.gt(case_raw.MEAS_DT.max()-pd.Timedelta(minutes=10))]
case_total_minutes=float(lot_map.loc[CASE_LOT,'minutes'])
case_first_color='#FCF3CF';case_last_color='#D5F5E3';case_roi_color='#FBE9E9'
case_pair_colors={'Z1':'#2874A6','Z2':'#7E8790','Z3':'#B4232F','Z4':'#2874A6',
    'Z5':'#25836B','Z6':'#8C9199','Z7':'#2874A6','Z8':'#7E8790'}
case_missing_n=int(case_raw['OP-Z3'].isna().sum());case_zero_n=int(case_raw['OP-Z3'].eq(0).sum())
case_missing_intervals=[]
for _,case_seg in case_raw.groupby('SEG'):
    case_seg=case_seg.sort_values('sec')
    edges=np.diff(np.r_[False,case_seg['OP-Z3'].isna().to_numpy(),False].astype(int))
    for a,b in zip(np.flatnonzero(edges==1),np.flatnonzero(edges==-1)-1):
        case_missing_intervals.append((float(case_seg.sec.iloc[a]),float(case_seg.sec.iloc[b]),int(b-a+1)))
# 전체 유효 30초 저출력 칸을 센서 공백/LOT 경계를 넘지 않는 연속 구간으로 묶습니다.
case_low_rows=[]
for seg,d in case.groupby('SEG'):
    d=d.sort_values('MEAS_DT').copy()
    low=d['OP-Z3'].lt(1)
    blocks=(low.ne(low.shift())|d.MEAS_DT.diff().ne(pd.Timedelta(seconds=30))).cumsum()
    for _,q in d[low].groupby(blocks[low]):
        case_low_rows.append({'SEG':int(seg),'시작 [분]':float(q.elapsed_min.min()),
            '끝 [분]':min(case_total_minutes,float(q.elapsed_min.max())+.5),
            '유효 30초 칸':len(q),'TC-Z3 최대':float(q['TC-Z3'].max())})
case_low_windows=pd.DataFrame(case_low_rows)
case_focus=case_low_windows.sort_values('TC-Z3 최대',ascending=False).iloc[0]
case_focus_a=float(case_focus['시작 [분]']);case_focus_b=float(case_focus['끝 [분]'])
case_focus_points=case[case.elapsed_min.ge(case_focus_a)&case.elapsed_min.lt(case_focus_b)]
case_focus_peak=case_focus_points.loc[case_focus_points['TC-Z3'].idxmax()]
case_tc_op_rows=[]
for z in range(1,9):
    tc=f'TC-Z{z}';op=f'OP-Z{z}'
    case_tc_op_rows.append({'TC':tc,'OP':op,'TC 첫10분':first[tc].mean(),'TC 끝10분':last[tc].mean(),
        'OP 첫10분':first[op].mean(),'OP 끝10분':last[op].mean(),
        'OP 변동폭(P90-P10)':case[op].quantile(.9)-case[op].quantile(.1)})
case_tc_op_summary=pd.DataFrame(case_tc_op_rows).set_index('TC')
# 이전 후속 셀에서 사용하는 원본 처음/마지막 10분 표를 유지합니다.
first_last=[]
for group,cols in TC_GROUPS.items():
    f=first[cols].mean(axis=1).where(first[cols].notna().all(axis=1))
    l=last[cols].mean(axis=1).where(last[cols].notna().all(axis=1))
    first_last.append({'공정':group,'첫10분 유효행':f.count(),'끝10분 유효행':l.count(),
        '첫10분 평균':f.mean(),'끝10분 평균':l.mean(),'차이 (끝-처음)':l.mean()-f.mean()})
first_last=pd.DataFrame(first_last)

# 공정 순서의 한 그림. 가열 출력 바로 아래에서 TC·OP의 초반 결측을 한 축 쌍으로 확대합니다.
case_zoom=case_raw[case_raw.sec.between(0,300)].copy()
case_zoom_missing=case_zoom[case_zoom["OP-Z3"].isna()]
case_zoom_first_valid=case_zoom[case_zoom["OP-Z3"].notna()].iloc[0]
case_zoom_tc_cols=["TC-Z3","TC-Z4","TC-Z5","TC-Z6"]
case_zoom_op_cols=["OP-Z4","OP-Z5","OP-Z6"]
case_zoom_tc_n=int(case_zoom_missing[case_zoom_tc_cols].notna().all(axis=1).sum())
case_zoom_op_n=int(case_zoom_missing[case_zoom_op_cols].notna().all(axis=1).sum())
case_flow_colors={**case_pair_colors,"Z9":"#7E8790"}
case_flow_specs=[
    ("세정조 입측 · CLN-IN",["CLN-IN"],"full"),
    ("건조존 온도 · TC-Z1·TC-Z2",["TC-Z1","TC-Z2"],"full"),
    ("건조존 출력 · OP-Z1·OP-Z2",["OP-Z1","OP-Z2"],"full"),
    ("가열·균열존 온도 · TC-Z3·TC-Z4·TC-Z5·TC-Z6",case_zoom_tc_cols,"full"),
    ("가열·균열존 출력 · OP-Z3·OP-Z4·OP-Z5·OP-Z6",["OP-Z3",*case_zoom_op_cols],"full"),
    ("바로 확대: 시작 0~5분 · 왼쪽 TC 온도 / 오른쪽 OP 출력",
        [*case_zoom_tc_cols,"OP-Z3",*case_zoom_op_cols],"missing_pair"),
    ("가열 존 분위기 · CP-Z4",["CP-Z4"],"full"),
    ("분위기 모니터 · CP-Z4M",["CP-Z4M"],"full"),
    ("과시효대 온도 · TC-Z7·TC-Z8",["TC-Z7","TC-Z8"],"full"),
    ("과시효대 출력 · OP-Z7·OP-Z8",["OP-Z7","OP-Z8"],"full"),
    ("슬러지 제거부 온도 · TC-Z9",["TC-Z9"],"full"),
    ("냉각대 온도 · TC-OUT1·TC-OUT2",["TC-OUT1","TC-OUT2"],"full"),
]
case_flow_columns={c for _,cols,kind in case_flow_specs if kind=="full" for c in cols}
assert case_flow_columns==set(SIG), "제공된 22개 신호가 공정 흐름 그림에 모두 포함되어야 합니다."
# 결측 확대 한 패널에는 범례 두 줄과 오른쪽 축을 위한 여백을 확보합니다.
case_flow_ratios=[value for _,_,kind in case_flow_specs
    for value in ([1.08,2.10,.42] if kind=="missing_pair" else [.68,1.75,.42])]
case_flow_height=3.2+sum(case_flow_ratios)
case_flow_fig=plt.figure(figsize=(18,case_flow_height))
case_flow_grid=case_flow_fig.add_gridspec(
    3*len(case_flow_specs),1,height_ratios=case_flow_ratios,
    left=.075,right=.94,top=1-2.60/case_flow_height,bottom=.65/case_flow_height,hspace=.10)
case_flow_axes=[];case_flow_headers=[];case_flow_audit=[]

def case_flow_color(column):
    if column=="CLN-IN": return "black"
    if column=="CP-Z4": return "#7D3C98"
    if column=="CP-Z4M": return "#89929B"
    if column.startswith("TC-OUT"): return "#2874A6" if column=="TC-OUT1" else "#89929B"
    return case_flow_colors[column.split("-")[1]]

for j,(title,cols,kind) in enumerate(case_flow_specs):
    head=case_flow_fig.add_subplot(case_flow_grid[3*j]);head.set_axis_off()
    ax=case_flow_fig.add_subplot(case_flow_grid[3*j+1])
    case_flow_headers.append(head);case_flow_axes.append(ax)
    if kind=="missing_pair":
        case_missing_tc_ax=ax
        case_missing_op_ax=ax.twinx()
        case_missing_op_ax.patch.set_visible(False)
        tc_handles=[];op_handles=[]
        for c in case_zoom_tc_cols:
            for _,d in case_zoom.groupby("SEG"):
                ax.plot(d.sec,d[c],color=case_flow_color(c),ls="-",
                    lw=1.5 if c=="TC-Z3" else 1,alpha=1 if c=="TC-Z3" else .86)
            tc_handles.append(Line2D([],[],color=case_flow_color(c),ls="-",lw=1.3,label=c))
        for _,d in case_zoom.groupby("SEG"):
            mean=d[case_zoom_tc_cols].mean(axis=1).where(d[case_zoom_tc_cols].notna().all(axis=1))
            ax.plot(d.sec,mean,color="black",lw=1.2)
        tc_handles.append(Line2D([],[],color="black",lw=1.2,label="TC 평균 · 왼쪽 축"))
        for c in ["OP-Z3",*case_zoom_op_cols]:
            for _,d in case_zoom.groupby("SEG"):
                case_missing_op_ax.plot(d.sec,d[c],color=case_flow_color(c),ls="--",
                    lw=1.6 if c=="OP-Z3" else 1.1,alpha=1 if c=="OP-Z3" else .85)
            op_handles.append(Line2D([],[],color=case_flow_color(c),ls="--",lw=1.3,label=c))
        for a,b,n in case_missing_intervals:
            if a<=300:
                ax.axvspan(max(0,a),min(300,b+1),fc="#D8DEE4",ec="#9AA4AE",
                    hatch="///",lw=0,alpha=.65,zorder=0)
        tc_handles.append(Patch(fc="#D8DEE4",ec="#9AA4AE",hatch="///",label="회색 빗금: OP-Z3 결측"))
        case_missing_op_ax.scatter([case_zoom_first_valid.sec],[case_zoom_first_valid["OP-Z3"]],
            s=45,facecolors="white",edgecolors="#B4232F",lw=1.5,zorder=5)
        op_handles.append(Line2D([],[],ls="",marker="o",mfc="white",mec="#B4232F",
            label="첫 유효값 · 오른쪽 축"))
        ax.set_xlim(0,300);ax.set_xticks([0,60,120,180,240,300])
        ax.set_xlabel("LOT 시작 후 경과 [초] · 같은 시각 원본값",fontsize=9.5,labelpad=3)
        ax.set_ylabel("TC 온도 [℃] · 왼쪽",fontsize=10);ax.margins(y=.16)
        case_missing_op_ax.set_ylabel("OP 출력 [%] · 오른쪽",fontsize=10,labelpad=8)
        case_missing_op_ax.set_ylim(-4,105);case_missing_op_ax.set_yticks([0,25,50,75,100])
        case_missing_op_ax.tick_params(axis="y",labelsize=9,pad=4)
        ax.grid(alpha=.13);case_missing_op_ax.grid(False)
        ax.spines[["top","right"]].set_visible(False)
        case_missing_op_ax.spines[["top","left"]].set_visible(False)
        stat=(f"OP-Z3 결측 {case_missing_n}행(0~183초) · 첫 유효 "
              f"{case_zoom_first_valid.sec:.0f}초 / {case_zoom_first_valid['OP-Z3']:.1f}%"
              f" · 같은 {len(case_zoom_missing)}행에서 TC 4개·다른 OP 3개 모두 기록")
        head.text(0,.90,f"{j+1:02d}. {title}",fontsize=12,fontweight="bold",va="center")
        head.text(0,.64,stat,fontsize=9.7,color="#B4232F",va="center")
        tc_legend=head.legend(handles=tc_handles,loc="lower left",bbox_to_anchor=(0,.32),
            ncol=len(tc_handles),frameon=False,fontsize=9.1,borderaxespad=0,borderpad=0,
            handlelength=2.2,columnspacing=1.6)
        head.add_artist(tc_legend)
        head.legend(handles=op_handles,loc="lower left",bbox_to_anchor=(0,.01),
            ncol=len(op_handles),frameon=False,fontsize=9.1,borderaxespad=0,borderpad=0,
            handlelength=2.2,columnspacing=1.6)
        ax.tick_params(axis="x",pad=3,labelsize=9)
        case_flow_audit.append({"패널":j+1,"컬럼":tuple(cols),"구간":kind,"x축":"sec",
            "왼쪽 축":"TC [℃]","오른쪽 축":"OP [%]"})
        continue

    scale=1e10 if cols==["CP-Z4M"] else 1
    handles=[]
    for c in cols:
        color=case_flow_color(c)
        for _,d in case.groupby("SEG"):
            ax.plot(d.elapsed_min,d[c]*scale,color=color,
                lw=1.5 if c in ["TC-Z3","OP-Z3"] else 1.1 if c=="CP-Z4" else .95,
                alpha=1 if c in ["TC-Z3","OP-Z3","CP-Z4"] else .86)
        handles.append(Line2D([],[],color=color,
            lw=1.5 if c in ["TC-Z3","OP-Z3"] else 1,label=c))
    is_temperature=cols[0].startswith("TC") or cols==["CLN-IN"]
    if is_temperature and len(cols)>1:
        for _,d in case.groupby("SEG"):
            mean=d[cols].mean(axis=1).where(d[cols].notna().all(axis=1))
            ax.plot(d.elapsed_min,mean,color="black",lw=1.1)
        handles.append(Line2D([],[],color="black",lw=1.1,label="검정: 표시 TC의 동시각 평균"))
    ax.axvspan(0,10,color=case_first_color,alpha=.7,zorder=0)
    ax.axvspan(case_total_minutes-10,case_total_minutes,color=case_last_color,alpha=.65,zorder=0)
    ax.axvspan(case_focus_a,case_focus_b,color=case_roi_color,lw=0,zorder=0)
    ax.set_xlim(0,case_total_minutes)
    ax.set_xlabel("LOT 시작 후 경과 [분] · 유효 30초 평균",fontsize=9.5,labelpad=3)
    fm=first[cols].mean(axis=1).where(first[cols].notna().all(axis=1)).mean()*scale
    lm=last[cols].mean(axis=1).where(last[cols].notna().all(axis=1)).mean()*scale
    unit="℃" if is_temperature else "%" if cols[0].startswith("OP") else ""
    precision=6 if cols==["CP-Z4"] else 3 if cols==["CP-Z4M"] else 2
    stat=(f"첫 10분 {fm:.{precision}f}{unit} → 끝 10분 {lm:.{precision}f}{unit}"
          f" · 차이 {lm-fm:+.{precision}f}{unit}")
    if cols==case_zoom_tc_cols:
        stat+=(f" · 붉은 구간 TC-Z3 최대 {case_focus_peak['TC-Z3']:.2f}℃"
               f" ({case_focus_peak.elapsed_min:.1f}분)")
        ax.scatter([case_focus_peak.elapsed_min],[case_focus_peak["TC-Z3"]],
            facecolors="white",edgecolors="#B4232F",lw=1.4,s=40,zorder=5)
    if cols==["OP-Z3",*case_zoom_op_cols]:
        for a,b,n in case_missing_intervals:
            ax.axvspan(a/60,(b+1)/60,fc="#D8DEE4",ec="#9AA4AE",hatch="///",lw=0,zorder=1)
        handles.append(Patch(fc="#D8DEE4",ec="#9AA4AE",hatch="///",label="OP-Z3 원본 결측"))
        stat=(f"OP-Z3 결측 {case_missing_n}행 → 바로 아래 TC·OP 함께 확대"
              f" · 원본 0값 {case_zero_n}행 · 30초 평균 저출력(<1%) {case['OP-Z3'].lt(1).sum()}칸")
    if is_temperature:
        ax.set_ylabel("온도 [℃]",fontsize=10);ax.margins(y=.14)
    elif cols[0].startswith("OP"):
        ax.set_ylabel("제어출력 [%]",fontsize=10)
        ax.set_ylim(-4,105);ax.set_yticks([0,25,50,75,100])
    else:
        ax.set_ylabel("CP-Z4M [×10^-10]" if cols==["CP-Z4M"] else "CP-Z4 [무차원]",fontsize=10)
        ax.margins(y=.14)
    head.text(0,.86,f"{j+1:02d}. {title}",fontsize=12,fontweight="bold",va="center")
    head.text(0,.52,stat,fontsize=9.7,color="#425466",va="center")
    head.legend(handles=handles,loc="lower left",bbox_to_anchor=(0,.005),ncol=len(handles),
        frameon=False,fontsize=9.1,borderaxespad=0,borderpad=0,handlelength=2.2,columnspacing=1.6)
    ax.tick_params(axis="x",pad=3,labelsize=9)
    ax.grid(alpha=.13);ax.spines[["top","right"]].set_visible(False)
    case_flow_audit.append({"패널":j+1,"컬럼":tuple(cols),"구간":kind,"x축":"elapsed_min",
        "왼쪽 축":"기존 단위","오른쪽 축":""})
case_flow_audit=pd.DataFrame(case_flow_audit)
case_flow_fig.suptitle(
    f"그림 11. SPHC {CASE_LOT} — 공정 흐름의 TC·OP·CP와 OP-Z3 결측 확대",
    x=.075,y=1-.12/case_flow_height,ha="left",fontsize=16,fontweight="bold")
case_flow_fig.text(.075,1-.60/case_flow_height,
    "확인 목적: 공통 승온과 Z3의 변화를 구분하고, OP-Z3가 비는 시각을 바로 아래에서 확인",
    fontsize=11,color="#425466")
case_flow_fig.text(.075,1-.98/case_flow_height,
    "세정 → 건조 TC·OP → 가열 TC·OP → 결측 확대: TC 왼쪽·OP 오른쪽(초) → CP → 과시효 TC·OP → TC-Z9 → 냉각",
    fontsize=10.2,color="#425466")
case_flow_fig.legend(handles=[
    Patch(fc=case_first_color,label="노랑: 첫 10분"),
    Patch(fc=case_roi_color,label=f"옅은 빨강: 가열 TC·OP 확인 {case_focus_a:.0f}~{case_focus_b:.0f}분"),
    Patch(fc=case_last_color,label="초록: 마지막 10분"),
    Patch(fc="#D8DEE4",ec="#9AA4AE",hatch="///",label="회색 빗금: OP-Z3 원본 결측")],
    loc="upper left",bbox_to_anchor=(.07,1-1.34/case_flow_height),ncol=4,frameon=False,fontsize=10)
case_flow_fig.text(.075,1-1.91/case_flow_height,
    "같은 존 TC·OP는 같은 선색 · 결측 확대는 TC 실선/OP 점선 · 확대만 초 단위 · 검정은 표시 TC 평균(CLN-IN은 단일값)",
    fontsize=10,color="#66717D")
case_flow_fig.text(.075,.20/case_flow_height,
    "결측 보간·0 채움 없음 · 확대의 두 세로축은 단위가 다르므로 같은 시각의 기록·변화 방향을 대조",
    fontsize=10,color="#66717D")
save_and_show_figures()


#### 그림 11 — 흐름에 따라 확인한 결과와 다음 비교

| 위에서 아래로 볼 곳 | 이번에 확인한 결과 | 무엇을 알게 되었는가? |
|---|---|---|
| **건조 TC-Z1·Z2 ↔ OP-Z1·Z2** | TC 평균 첫/끝 **96.62 → 99.05℃**. 두 존 모두 온도 상승·평균 출력 감소 | 온도와 출력이 반드시 같은 방향으로 움직이지 않음. 같은 존의 TC·OP를 대조 |
| **가열 TC-Z3~Z6 ↔ OP-Z3~Z6** | 초반~**73.5분** 공통 승온. 가열 평균 **821.95 → 859.13℃** | TC-Z3 하나만의 저온으로 해석하지 않음 |
| **OP-Z3 바로 아래 TC·OP 통합 결측 확대** | **179행 / 0~183초** 결측. **184초 / 100%**부터 실제 기록. 같은 179행에서 **TC 네 컬럼·다른 OP 세 컬럼 모두 유효** | 전체 수집 공백보다 OP-Z3 채널 점검을 우선. 원본 0값 **595행**과 구분 |
| **붉은 128~135분의 가열 TC·OP** | OP-Z3 30초 평균 **1% 미만**, TC-Z3는 **131.5분 / 870.60℃**까지 상승 | 같은 존의 온도·출력 관계를 자세히 확인할 구간. 목표값·제어 모드·출력 제한 대조 |
| **같은 붉은 구간의 CP-Z4·다른 공정 온도** | CP-Z4 피크가 보임. CLN-IN은 완만한 변화, 냉각 온도는 반복 흐름을 이어감 | 여러 공정의 동시 급변 근거는 부족. **Z3의 온도·출력 변화와 CP-Z4 피크**가 후속 확인 대상 |
| **과시효 TC-Z7·Z8 ↔ OP-Z7·Z8** | TC 평균 **316.25 → 329.78℃**. 두 OP의 첫/끝 평균은 감소 | 다른 공정에도 공통 온도 상승이 있음. 해당 존 출력과 직접 비교 |
| **TC-Z9·CP-Z4M·냉각 TC-OUT1·OUT2** | 전체 흐름과 컬럼별 변화를 함께 확인. 냉각 온도는 반복적인 오르내림 | CP-Z4M과 CP-Z4의 척도를 구분. 반복 저점은 주변 주기·변동폭과 대조하여 해석 |

**확대 읽기:** 왼쪽 **TC 온도[℃] 실선**, 오른쪽 **OP 출력[%] 점선**입니다. 회색 빗금은 **OP-Z3만 기록되지 않은 시각**이며, 그 안에서 다른 TC·OP의 선이 이어지는지 확인합니다. 두 세로축의 선 높이를 직접 비교하지 않습니다.

**첫/끝 평균은 수준 비교**, **128~135분은 특정 변화 구간 비교**, **0~183초는 수집 품질 비교**입니다. 배경색이 정상·고장을 판정하지 않습니다. 동반 관측만으로 원인 관계를 확정하지 않습니다.

| 후속 확인 대상 | 누구·무엇과 비교하는가? | 필요한 자료·이어지는 그림 |
|---|---|---|
| **OP-Z3 초반 결측** | **240061 ↔ 240044(75행 / 0~81초)**, 대조 **240069·240091**의 같은 채널·수집 시작 | 태그 초기화·통신·저장 로그. 이 그림의 OP-Z3 바로 아래 확대 |
| **240061의 128~135분** | 같은 시각 **TC-Z3↔OP-Z3·CP-Z4**, 다른 가열 TC·OP. 같은 SPHC **240044·240069·240091**의 유사 운전 상태 | 목표온도·제어 모드·가동상태·가스 공급·분석계 교정 로그. **19-A 확대**, **15 상대 온도 변화**, **14 시차** |
| **건조·과시효 TC·OP** | 같은 SPHC의 **TC-Z1↔OP-Z1·TC-Z2↔OP-Z2·TC-Z7↔OP-Z7·TC-Z8↔OP-Z8** | 목표값·라인속도·두께. 다른 존 OP를 직접 제어출력으로 연결하지 않음 |
| **냉각 반복·TC-Z9의 느린 변화** | 충분히 긴 연속 구간의 같은 SPHC와 반복 간격·수준·변동폭 비교 | **13은 240069의 사례**이므로 LOT를 구분. 240061은 승온 제외 후 최소 3주기 관측부터 확인 |

CLN-IN·TC-OUT1·OUT2·TC-Z9에 대응하는 OP는 제공 자료에 없습니다. 다른 공정의 물리적 전달을 비교하려면 라인속도·재료 이동시간도 확인해야 합니다.


<a id="period"></a>
## 8. 반복 온도 하강 — 무엇을 점검 대상으로 볼 것인가?

**왜 보는가?** 앞선 그림에서 TC-OUT1·TC-OUT2는 반복적인 오르내림, TC-Z9는 느린 변화가 보였습니다. 온도가 내려간 횟수만으로 LOT를 선정하기 전에 **반복 간격과 온도 변동폭이 달라졌는지** 확인할 기준을 마련합니다.

**이 단계에서 확인할 것:** SPHC 240069의 온도 하강이 반복되는 형태인지, 반복 간격을 얼마나 신뢰할 수 있는지 봅니다. 현재 그림은 **240069와 다른 SPHC의 주기 차이를 판정한 결과가 아닙니다.** 다음 비교 대상은 **240091**이며 운전상태·구간 길이를 맞춰야 합니다.

**그림 읽는 순서:** **그림 13의 실제 온도 → 오른쪽 ‘이번 결과·다음 점검’ → 그림 13-근거의 반복 간격 교차 확인**입니다. 수치 계산보다 무엇을 알게 되었고 무엇이 남았는지를 먼저 읽습니다.

**계산 범위:** 신호별 가장 긴 연속 유효 1분 평균을 사용하고 보간하지 않습니다. 선형 추세를 제거한 뒤 10~240분 중 관측 길이에 최소 3주기가 들어갈 간격을 탐색합니다. **탐색 범위의 끝값은 주기 판단 보류**로 두며 주기 요약에서 제외합니다. 변동폭 **P90-P10 = 90분위값 - 10분위값**입니다.


In [ ]:
def longest_valid(frame,col):
    blocks=[]
    for (lot,seg),d in frame.groupby(['LOT_NO','SEG']):
        d=d.sort_values('MEAS_DT').copy()
        ok=d[col].notna()
        run=(ok.ne(ok.shift()) | d.MEAS_DT.diff().ne(pd.Timedelta(minutes=1))).cumsum()
        for _,block in d[ok].groupby(run[ok]): blocks.append(block)
    return max(blocks,key=len) if blocks else pd.DataFrame()
# 최소 3주기 조건으로 제한한 탐색 범위의 끝값은 주기 확인 결과로 집계하지 않습니다.
period_rows=[]
for lot,d in analysis_ts1.groupby('LOT_NO'):
    for col in ['TC-OUT1','TC-OUT2','TC-Z9']:
        block=longest_valid(d,col)
        if len(block)<60: continue
        y=detrend(block[col].to_numpy())
        freq,power=periodogram(y,fs=1,window='hann')
        periods=np.divide(1,freq,out=np.full_like(freq,np.inf),where=freq>0)
        eligible=(periods>=10)&(periods<=min(240,len(block)/3))
        if not eligible.any(): continue
        eligible_ids=np.flatnonzero(eligible)
        idx=eligible_ids[np.argmax(power[eligible])]
        local_peak=(0<idx<len(power)-1 and power[idx]>power[idx-1] and power[idx]>power[idx+1])
        boundary=bool(idx in (eligible_ids[0],eligible_ids[-1]))
        state=('탐색 경계-주기 판단 보류' if boundary else
               '국소 피크 후보' if local_peak else '뚜렷한 피크 없음-판정 유보')
        period_rows.append({'LOT':int(lot),'강종':lot_map.loc[lot,'grade'],'신호':col,
            '구간 시작 [분]':float(block.elapsed_min.iloc[0]),'구간 끝 [분]':float(block.elapsed_min.iloc[-1]),
            '유효길이 [분]':len(block),'탐색최대주기 [분]':min(240,len(block)/3),
            '스펙트럼최대주기 [분]':periods[idx],'포함주기 수':len(block)/periods[idx],
            '온도 변동폭(P90-P10) [℃]':float(block[col].quantile(.9)-block[col].quantile(.1)),
            '탐색 경계':boundary,'상태':state})
period_table=pd.DataFrame(period_rows)
PERIOD_LOT=int(lots.set_index('LOT_NO').minutes.idxmax())
period_case=analysis_ts1[analysis_ts1.LOT_NO.eq(PERIOD_LOT)]
period_selected=period_table[period_table.LOT.eq(PERIOD_LOT)].set_index('신호')
selected_periods=period_selected['스펙트럼최대주기 [분]']
period_summary=(period_table[period_table.상태.eq('국소 피크 후보')]
    .groupby('신호').agg(피크확인LOT수=('LOT','nunique'),
        주기중앙값분=('스펙트럼최대주기 [분]','median'),
        주기최소분=('스펙트럼최대주기 [분]','min'),주기최대분=('스펙트럼최대주기 [분]','max')))

s13_period_cols=['TC-OUT1','TC-OUT2','TC-Z9']
s13_period_labels={'TC-OUT1':'TC-OUT1 · 냉각대 1 온도',
    'TC-OUT2':'TC-OUT2 · 냉각대 2 온도','TC-Z9':'TC-Z9 · 슬러지 제거부 온도'}
s13_period_blocks={c:longest_valid(period_case,c) for c in s13_period_cols}
s13_period_blue='#2874A6';s13_period_red='#B4232F';s13_period_gray='#66717D'
s13_period_grade=lot_map.loc[PERIOD_LOT,'grade']

# 주요 그림은 실제 변화와 해석·다음 점검을 함께 보여줍니다.
fig=plt.figure(figsize=(20,11.5))
s13_grid=fig.add_gridspec(6,2,height_ratios=[.6,1.8]*3,width_ratios=[2.2,1],
    left=.07,right=.98,top=.82,bottom=.095,hspace=.26,wspace=.09)
s13_main_axes=[];s13_main_headers=[];s13_main_notes=[]
for s13_row,s13_col in enumerate(s13_period_cols):
    s13_block=s13_period_blocks[s13_col];s13_result=period_selected.loc[s13_col]
    s13_header=fig.add_subplot(s13_grid[2*s13_row,0]);s13_header.axis('off');s13_main_headers.append(s13_header)
    ax=fig.add_subplot(s13_grid[2*s13_row+1,0]);s13_main_axes.append(ax)
    s13_note=fig.add_subplot(s13_grid[2*s13_row:2*s13_row+2,1]);s13_note.axis('off');s13_main_notes.append(s13_note)
    s13_header.text(0,.78,s13_period_labels[s13_col]+' — 하강 뒤 다시 상승하는가?',
        fontsize=11.8,fontweight='bold',va='center',color='black')
    s13_header.text(0,.08,
        f"사용 구간: LOT 시작 후 {s13_result['구간 시작 [분]']:.1f}~{s13_result['구간 끝 [분]']:.1f}분 · 연속 1분 평균 {len(s13_block)}칸",
        fontsize=9.6,color=s13_period_gray,va='center')
    ax.plot(s13_block.elapsed_min,s13_block[s13_col],color=s13_period_blue,lw=1.3)
    ax.set_ylabel('온도 [℃]');ax.grid(alpha=.14);ax.spines[['top','right']].set_visible(False)
    ax.margins(x=.015,y=.12)
    ax.set_xlabel('LOT 시작 후 경과 [분]',fontsize=10,labelpad=7)
    s13_width=s13_result['온도 변동폭(P90-P10) [℃]']
    if s13_result['상태']=='국소 피크 후보':
        s13_note.text(0,.96,f"이번 결과: 약 {s13_result['스펙트럼최대주기 [분]']:.1f}분 간격의 반복 후보",
            fontsize=11,fontweight='bold',va='top',color='black')
        s13_note.text(0,.74,f'온도 변동폭(P90-P10): {s13_width:.2f}℃\n같은 형태의 오르내림이 여러 번 나타남',
            fontsize=10,color='#425466',va='top',linespacing=1.6)
        s13_note.text(0,.45,'점검할 것은 반복 간격·온도 변동폭의 변화',
            fontsize=10.5,fontweight='bold',va='top',color=s13_period_red)
        s13_note.text(0,.24,'다음: SPHC 240069 ↔ 240091\n같은 운전상태·같은 길이 구간으로 대조\n현재 그림만으로 240069의 정상/이상은 판정하지 않음',
            fontsize=9.6,color='#425466',va='top',linespacing=1.5)
    else:
        s13_note.text(0,.96,'이번 결과: 느린 변화 · 정확한 주기 판단 보류',
            fontsize=11,fontweight='bold',va='top',color=s13_period_red)
        s13_note.text(0,.74,
            f"최대 {s13_result['스펙트럼최대주기 [분]']:.1f}분 = 탐색 상한\n관측 길이 ÷ 3 조건의 경계에 걸린 값",
            fontsize=10,color='#425466',va='top',linespacing=1.6)
        s13_note.text(0,.45,'172.3분을 확정된 운전 주기로 사용하지 않음',
            fontsize=10.5,fontweight='bold',va='top',color=s13_period_red)
        s13_note.text(0,.24,'다음: 더 긴 연속 관측 + SPHC 240091 대조\n슬러지 제거 운전 시각과 온도 흐름을 확인\n느린 변화와 반복 주기의 근거를 구분',
            fontsize=9.6,color='#425466',va='top',linespacing=1.5)
fig.suptitle(f'그림 13. {s13_period_grade} {PERIOD_LOT} — 온도 하강이 반복되는가, 무엇을 추가 점검할 것인가?',
    x=.07,y=.98,ha='left',fontweight='bold',fontsize=16)
fig.text(.07,.936,
    f"확인 결과: TC-OUT1 약 {selected_periods['TC-OUT1']:.1f}분 · TC-OUT2 약 {selected_periods['TC-OUT2']:.1f}분 반복 후보 / TC-Z9 주기 판단 보류",
    fontsize=11,color='#425466')
fig.text(.07,.899,'왜 보는가: 반복 저점의 횟수만으로 후보를 선정하지 않고, 같은 강종에서 반복 간격·온도 변동폭이 달라지는지 점검',
    fontsize=10.3,color=s13_period_gray)
fig.text(.07,.032,'변동폭(P90-P10) = 90분위값 - 10분위값 · 파란 선: 해당 컬럼의 1분 평균 · 공백 보간 없음 · 반복 확인은 정상 판정이나 정비 효과 확인이 아님',
    fontsize=9.4,color=s13_period_gray)
save_and_show_figures()

# 계산 근거는 별도로 배치합니다. 질문과 용어 설명을 각 패널 위에 표시합니다.
fig=plt.figure(figsize=(18,13))
s13_grid=fig.add_gridspec(6,2,height_ratios=[.86,1.85]*3,
    left=.075,right=.98,top=.84,bottom=.085,hspace=.33,wspace=.25)
s13_evidence_axes=[];s13_evidence_headers=[];s13_ac_first_peak={}
for s13_row,s13_col in enumerate(s13_period_cols):
    s13_block=s13_period_blocks[s13_col];s13_result=period_selected.loc[s13_col]
    y=detrend(s13_block[s13_col].to_numpy())
    max_lag=min(240,len(y)//3)
    ac=np.correlate(y,y,mode='full')[len(y)-1:len(y)+max_lag]/np.dot(y,y)
    s13_ac_peaks=np.flatnonzero((ac[1:-1]>ac[:-2])&(ac[1:-1]>ac[2:])&(ac[1:-1]>0))+1
    s13_ac_peaks=s13_ac_peaks[s13_ac_peaks>=10]
    s13_ac_first_peak[s13_col]=float(s13_ac_peaks[0]) if len(s13_ac_peaks) else np.nan
    for s13_j in [0,1]:
        s13_header=fig.add_subplot(s13_grid[2*s13_row,s13_j]);s13_header.axis('off');s13_evidence_headers.append(s13_header)
        ax=fig.add_subplot(s13_grid[2*s13_row+1,s13_j]);s13_evidence_axes.append(ax)
        if s13_j==0:
            s13_header.text(0,.82,s13_col+' — 몇 분 뒤 비슷한 움직임이 나타나는가?',
                fontsize=11.4,fontweight='bold',va='center')
            s13_header.text(0,.43,'자기상관: 시간을 옮겼을 때 원래 흐름과 닮은 정도',
                fontsize=9.4,va='center',color=s13_period_gray)
            s13_ac_note=(f'첫 양의 봉우리: 약 {s13_ac_first_peak[s13_col]:.0f}분' if len(s13_ac_peaks)
                else f'관측 시차 0~{max_lag}분 안에서 다시 높아지는 봉우리 없음')
            s13_header.text(0,.05,s13_ac_note,fontsize=9.7,va='center',color='#425466')
            ax.plot(np.arange(len(ac)),ac,color=s13_period_blue,lw=1.2)
            ax.axhline(0,color='#8A949E',lw=.7)
            if len(s13_ac_peaks):
                s13_first=int(s13_ac_peaks[0])
                ax.scatter([s13_first],[ac[s13_first]],s=35,facecolors='white',edgecolors=s13_period_blue,lw=1.5,zorder=3)
            ax.set_xlabel('시간을 옮긴 간격 [분]');ax.set_ylabel('자기상관 [-1~1]');ax.set_ylim(-1.08,1.1)
        else:
            freq,power=periodogram(y,fs=1,window='hann')
            periods=np.divide(1,freq,out=np.full_like(freq,np.inf),where=freq>0)
            eligible=(periods>=10)&(periods<=min(240,len(y)/3))
            xx=periods[eligible];yy=power[eligible];order=np.argsort(xx)
            s13_header.text(0,.82,s13_col+' — 어떤 반복 간격이 두드러지는가?',
                fontsize=11.4,fontweight='bold',va='center')
            s13_header.text(0,.43,'스펙트럼: 반복 간격별 신호 강도 · 패널별 최대=1',
                fontsize=9.4,va='center',color=s13_period_gray)
            s13_limit=float(s13_result['탐색최대주기 [분]']);s13_peak=float(s13_result['스펙트럼최대주기 [분]'])
            s13_edge=bool(s13_result['탐색 경계'])
            s13_peak_color=s13_period_red if s13_edge else '#263238'
            s13_header.text(0,.05,
                f"{'빨간' if s13_edge else '검정'} 점선: {s13_peak:.1f}분 · "+('탐색 상한 → 판단 보류' if s13_edge else '범위 안의 국소 피크 후보'),
                fontsize=9.7,va='center',color=s13_peak_color)
            ax.plot(xx[order],yy[order]/yy.max(),color=s13_period_blue,lw=1.2)
            ax.axvline(s13_peak,color=s13_peak_color,ls='--',lw=1.1)
            ax.scatter([s13_peak],[1],s=35,facecolors='white',edgecolors=s13_peak_color,lw=1.5,zorder=3)
            ax.set_xlabel('반복 간격 [분]');ax.set_ylabel('상대 강도 [패널 최대=1]');ax.set_ylim(-.04,1.12)
            ax.set_xlim(10,s13_limit*1.07)
        ax.grid(alpha=.12);ax.spines[['top','right']].set_visible(False)
fig.suptitle(f'그림 13-근거. {s13_period_grade} {PERIOD_LOT} — 반복 간격을 두 방법으로 대조',
    x=.075,y=.98,ha='left',fontweight='bold',fontsize=16)
fig.text(.075,.936,'왼쪽: 비슷한 흐름이 다시 나타나는 간격 / 오른쪽: 반복 신호가 큰 간격 · 0분 자기상관=1은 자기 자신과의 비교',
    fontsize=10.5,color='#425466')
fig.text(.075,.900,'TC-Z9는 시차 범위 안의 재봉우리가 없고 스펙트럼 최대가 탐색 상한에 위치하므로 정확한 주기는 보류',
    fontsize=10.2,color=s13_period_red)
fig.text(.075,.025,'선형 추세 제거: 전체적인 기울기를 빼고 반복 형태 점검 · 자기상관과 상대 강도는 정상 범위·고장 확률이 아님 · 경계 피크는 주기 요약에서 제외',
    fontsize=9.3,color=s13_period_gray)
save_and_show_figures()

# 표에서도 탐색 경계와 내부 피크를 구분하여 해석의 과장을 막습니다.
period_selected_results=period_selected.reset_index()[[
    '신호','구간 시작 [분]','구간 끝 [분]','유효길이 [분]','스펙트럼최대주기 [분]',
    '온도 변동폭(P90-P10) [℃]','상태']]
display(period_selected_results.round(3))
display(period_summary.rename(columns={'피크확인LOT수':'경계 제외 주기 후보 LOT수'}).round(2))
display(period_table.round(2))
assert not period_table.loc[period_table.상태.eq('국소 피크 후보'),'탐색 경계'].any()
assert period_selected.loc['TC-Z9','탐색 경계']
assert np.isclose(selected_periods['TC-Z9'],period_selected.loc['TC-Z9','탐색최대주기 [분]'])


#### 그림 13 · 반복 온도 하강 — 이번에 알게 된 점과 다음 점검

**핵심 질문:** SPHC 240069의 반복적인 온도 하강을 어떻게 읽고, 같은 강종과 무엇을 비교해야 하는가?

**① 이번에 확인한 결과**

| 대상·컬럼 | 관측 근거 | 이번에 알게 된 점 |
|---|---|---|
| **SPHC 240069 / TC-OUT1** | 사용 구간 **105.0~621.0분**. 주기 후보 **43.08분**, 자기상관 재봉우리 약 **43분**. 온도 변동폭(P90-P10) **25.21℃** | 실제 온도 흐름과 두 계산 방법에 반복 근거가 있음. 다음 비교는 같은 SPHC 240091의 주기·변동폭 |
| **SPHC 240069 / TC-OUT2** | 사용 구간 **105.0~621.0분**. 주기 후보 **47.00분**, 자기상관 재봉우리 약 **48분**. 온도 변동폭(P90-P10) **16.87℃** | 실제 온도 흐름과 두 계산 방법에 반복 근거가 있음. 다음 비교는 같은 SPHC 240091의 주기·변동폭 |
| **SPHC 240069 / TC-Z9** | 같은 사용 구간의 최대값 **172.33분 = 탐색 상한 172.33분**. 자기상관 범위 안에서 재봉우리 없음 | **정확한 주기 판단 보류**. 느린 변화는 보이지만 최대값만으로 반복 간격을 확정할 수 없음 |

**냉각대의 반복 저점은 반복 형태의 일부로 구분해 기록합니다.** 반복이 존재한다는 사실이 정상 판정은 아닙니다. 240069를 상세 점검하려면 같은 SPHC의 **주기·온도 수준·변동폭이 달라지는지** 추가 비교해야 합니다.

**TC-Z9의 172.3분은 관측 길이 ÷ 3으로 정한 탐색 상한**입니다. 사용 구간에서 느린 오르내림은 보이지만 그 값을 확정된 운전 주기나 이상 주기로 사용하지 않습니다. 자기상관도 분석한 시차 안에서 다시 높아지는 양의 봉우리가 나타나지 않았습니다.

**② 근거 그림은 이렇게 읽습니다.**

| 볼 곳 | 묻는 질문 | 이번에 확인한 의미 |
|---|---|---|
| 그림 13의 실제 온도 | 내려간 뒤 비슷한 형태로 다시 올라오는가? | 두 냉각 온도에 반복적인 오르내림이 보임. TC-Z9의 변화는 더 느림 |
| 자기상관 | 몇 분을 옮기면 흐름이 다시 비슷해지는가? | 0분은 자기 자신이라 1. 이후 다시 높아지는 봉우리로 반복 간격을 대조 |
| 주기 스펙트럼 | 어느 반복 간격의 신호가 두드러지는가? | TC-OUT1·OUT2는 탐색 범위 안의 후보. TC-Z9는 상한에 걸려 보류 |
| 패널별 상대 강도=1 | 신호가 얼마나 강한가? | 각 패널에서 가장 큰 값을 1로 둔 상대값. 컬럼 간 절대 강도나 정상 여부를 비교하는 수치가 아님 |

**여러 LOT에서의 참고 근거 — 탐색 경계 제외**

| 컬럼 | 탐색 경계 제외 후보 LOT수 | 내부 피크의 주기 범위 | 해석 범위 |
|---|---|---|---|
| **TC-OUT1** | **11 LOT** | **40.36~45.57분** | 전체 강종을 섞은 참고 근거. 동일 강종 차이 또는 정상 범위 판정은 아님 |
| **TC-OUT2** | **11 LOT** | **43.40~49.33분** | 전체 강종을 섞은 참고 근거. 동일 강종 차이 또는 정상 범위 판정은 아님 |
| **TC-Z9** | **2 LOT** | **43.40~172.00분** | TC-Z9 후보는 240001(SGCC)·240058(DP590). 종료 전환 영향과 반복 형태를 추가 점검; 대표 주기로 합치지 않음 |

TC-OUT1·TC-OUT2의 **240038은 탐색 상한에 걸려 보류**했습니다. TC-Z9도 경계에 걸린 최대값을 요약에서 제외했습니다. 이 표의 후보 수는 고장 건수나 주기의 물리적 검증 횟수가 아닙니다.

**③ 다음 후속 작업 — 무엇을 누구와 비교할 것인가?**

| 대상·비교군 | 비교할 구간·컬럼 | 확인하려는 것 | 필요한 조건·자료 |
|---|---|---|---|
| **240069 ↔ 240091 / SPHC** | 동일 운전상태의 같은 길이 연속 구간, **TC-OUT1·TC-OUT2 각각** | 반복 간격·중앙값·P90-P10이 다른가? 한 LOT에서만 패턴이 달라지는가? | 냉각 목표값·라인속도·두께. 240091도 정상 정답으로 지정하지 않음 |
| **240061 / SPHC** | 초반 승온을 제외한 냉각 온도 구간 | 재가동 전환을 제외해도 충분한 반복 수가 확보되고 같은 패턴이 남는가? | 제외 후 최소 3주기 관측 가능 여부부터 확인 |
| **240044 / SPHC** | 약 75분의 짧은 관측 | 40~50분 반복을 3회 확인하기에는 정보가 부족 | 더 긴 관측 확보. 계산된 경계 최대값을 실제 주기로 사용하지 않음 |
| **240069·240091 / TC-Z9** | 더 긴 연속 구간과 슬러지 제거 운전 시각 | 느린 온도 변화가 운전 주기와 반복적으로 대응하는가? | 실제 슬러지 제거 기록·목표값. 현재 172.3분은 보류 |
| 정비 전후의 냉각 온도 | 같은 강종·같은 운전상태·충분한 길이의 전후 관측 | 정비 전후에 반복 간격·변동폭이 바뀌는가? | 실제 착수·완료와 센서 관측. 이 그림은 정비 효과를 계산한 결과가 아님 |

**현재 판단 범위:** 두 냉각 온도의 **반복 주기 후보**와 TC-Z9의 **주기 판단 한계**를 확인했습니다. 240069가 다른 SPHC보다 특이한지, 변화가 고장·정비·운전조건 때문인지는 위 후속 비교가 필요합니다.

[↑ 목차](#toc)


<a id="lag"></a>
## 9. 온도와 제어출력 — 수준·차분의 시차상관

**분석 질문:** 온도와 출력의 관계가 수준 추세만의 결과인가, 짧은 변화에도 보이는가?

**계산 방법:** 30초 평균에서 Z1~Z8의 OP(t)와 TC(t+lag)를 비교합니다. SEG별로 결측 칸에서 유효 구간을 끊고 시차마다 유효 쌍 30개 이상일 때만 LOT 상관을 구합니다. LOT별 상관 중앙값을 표시합니다.

**그림 읽는 법:** 양의 lag는 OP 관측이 먼저, 음의 lag는 TC 관측이 먼저입니다. 왼쪽 수준과 오른쪽 30초 차분을 비교합니다. 강한 상관과 원인·결과는 구분합니다.

In [ ]:
lag_steps=np.arange(-6,7)
lag_minutes=lag_steps*.5
lag_records=[]
for lot,d in analysis_ts.groupby('LOT_NO'):
    d=d.sort_values('MEAS_DT').copy()
    for z in range(1,9):
        tc,op=f'TC-Z{z}',f'OP-Z{z}'
        ok=d[[tc,op]].notna().all(axis=1)
        # 결측·SEG·30초 칸 불연속에서 시차 계산 구간을 끊습니다.
        run=(d.SEG.ne(d.SEG.shift())|ok.ne(ok.shift())|d.MEAS_DT.diff().ne(pd.Timedelta(seconds=30))).cumsum()
        valid=d.loc[ok,[tc,op]].copy(); valid['run']=run[ok]
        for mode in ['수준','30초 차분']:
            values=valid.copy()
            if mode=='30초 차분': values[[tc,op]]=values.groupby('run')[[tc,op]].diff()
            for k in lag_steps:
                future=values.groupby('run')[tc].shift(-int(k))
                pairs=pd.concat([values[op],future],axis=1).dropna()
                r=pairs.iloc[:,0].corr(pairs.iloc[:,1]) if len(pairs)>=30 and pairs.std().gt(0).all() else np.nan
                lag_records.append({'LOT':int(lot),'존':f'Z{z}','방식':mode,'lag [분]':k*.5,'r':r,'유효쌍':len(pairs)})
lag_table=pd.DataFrame(lag_records)
fig,axes=plt.subplots(1,2,figsize=(16,6))
for ax,mode in zip(axes,['수준','30초 차분']):
    matrix=lag_table[lag_table.방식.eq(mode)].pivot_table(index='존',columns='lag [분]',values='r',aggfunc='median').reindex([f'Z{z}' for z in range(1,9)])
    im=ax.imshow(matrix,vmin=-1,vmax=1,cmap='RdBu_r',aspect='auto')
    ax.set_xticks(range(len(lag_minutes)),lag_minutes); ax.set_yticks(range(8),matrix.index)
    ax.set(xlabel='lag [분]: +는 OP 먼저',ylabel='동일 존 TC–OP',title=mode+' · LOT별 상관 중앙값')
    for i in range(8):
        for j in range(len(lag_minutes)): ax.text(j,i,f'{matrix.iloc[i,j]:.2f}',ha='center',va='center',fontsize=7,color='white' if abs(matrix.iloc[i,j])>.65 else 'black')
fig.colorbar(im,ax=axes,label='Pearson r',fraction=.025)
lag0_z3=lag_table[lag_table.존.eq('Z3')&lag_table.방식.eq('수준')&lag_table['lag [분]'].eq(0)].r
lag0_z3_diff=lag_table[lag_table.존.eq('Z3')&lag_table.방식.eq('30초 차분')&lag_table['lag [분]'].eq(0)].r
fig.suptitle(f'그림 14. Z3 동시각 음의 상관: 수준 {lag0_z3.lt(0).sum()}/14 → 차분 {lag0_z3_diff.lt(0).sum()}/14 LOT',fontsize=14,fontweight='bold')
fig.text(.5,.025,'확인 목적: 느린 수준의 관계가 단기 변화에도 나타나는가?  |  +시차: OP 먼저 · -시차: TC 먼저 · 상관은 인과·실제 지연을 확정하지 않음',ha='center',fontsize=9)
fig.subplots_adjust(top=.9,bottom=.14,wspace=.18,right=.88); save_and_show_figures()
lag_summary=[]
for (zone,mode),d in lag_table.groupby(['존','방식']):
    m=d.groupby('lag [분]').r.median()
    lag_summary.append({'존':zone,'방식':mode,'동시각 r 중앙값':m.loc[0.0],'상관계수 절댓값 최대 시차 [분]':m.abs().idxmax(),'해당 r':m.loc[m.abs().idxmax()],'유효 LOT 수 (lag0)':d[d['lag [분]'].eq(0)].r.count()})
lag_summary=pd.DataFrame(lag_summary)
display(lag_summary.round(3))

#### 그림 14 · 온도-제어출력 시차상관 — 그림 읽기와 결과

**① 읽는 순서**

| 볼 곳 | 읽는 법 |
|---|---|
| 수준 / 30초 차분 | 수준은 느린 추세 포함, 차분은 짧은 변화를 비교. 같은 존의 TC-Zn·OP-Zn 짝을 읽음. |
| 시차·색 | 양의 lag는 OP가 먼저, 음의 lag는 TC가 먼저. 상관의 부호와 크기는 원인·결과 판정이 아님. |

**② 확인한 결과 — 무엇을 알게 되었는가?**

| 대상·컬럼 | 관측 근거 | 알게 된 점 |
|---|---|---|
| **TC-Z3 ↔ OP-Z3 / 수준** | 동시각 r 중앙값 **-0.745**; 음수 **12/14 LOT** | 대체로 반대로 움직이지만 LOT의 공통 추세가 결과에 섞임. |
| **TC-Z3 ↔ OP-Z3 / 30초 차분** | 동시각 r 중앙값 **-0.646**; **14/14 LOT** 음수 | 단기 변화에도 음의 관계가 반복. |
| 240058·240114 / 수준 | Z3 수준 상관이 양수인 두 LOT | 종료 하강이 두 신호에 함께 포함되어 수준 상관이 달라질 수 있음. |
| Z3 최대 절댓값 시차 | 수준·차분 모두 후보 **+0.5분**, r 약 **-0.80~-0.81** | OP 변화가 먼저인 정렬에서 관계가 강함. 실제 제어 지연으로 확정할 수는 없음. |

**결론:** **수준과 차분을 함께 봐야** 운전 전환과 단기 제어 관계를 구분할 수 있습니다. 상관을 고장 점수나 인과 효과로 사용하지 않습니다.

**③ 후속 작업**

| 확인 대상·비교군 | 다음에 확인할 것 | 필요한 자료·이어지는 분석 |
|---|---|---|
| 240061 ↔ 240044·240069·240091 | 승온·결측을 분리한 구간의 TC-Z3·OP-Z3 관계 비교 | 같은 강종·같은 운전상태 원자료 |
| TC-Z3·OP-Z3 / 수집기 | 기록 시각·집계창·실제 제어 주기 확인 | PLC·제어 로그·타임스탬프 사양 |
| TC-Z7·OP-Z7 등 높은 상관 짝 | 공유 제어·계측 및 자료 생성 관계 확인 | 태그 정의·센서 연결 구조 |

**존별 계산값**

| 존 | 방식 | 동시각 r 중앙값 | 상관계수 절댓값 최대 시차 [분] | 해당 r | 유효 LOT 수 (lag0) |
| --- | --- | --- | --- | --- | --- |
| Z1 | 30초 차분 | -0.684 | 0.000 | -0.684 | 14 |
| Z1 | 수준 | -0.834 | -0.500 | -0.839 | 14 |
| Z2 | 30초 차분 | -0.562 | 0.000 | -0.562 | 14 |
| Z2 | 수준 | -0.626 | -2.000 | -0.673 | 14 |
| Z3 | 30초 차분 | -0.646 | 0.500 | -0.813 | 14 |
| Z3 | 수준 | -0.745 | 0.500 | -0.801 | 14 |
| Z4 | 30초 차분 | -0.693 | 0.000 | -0.693 | 14 |
| Z4 | 수준 | -0.703 | 0.000 | -0.703 | 14 |
| Z5 | 30초 차분 | -0.674 | 0.000 | -0.674 | 14 |
| Z5 | 수준 | -0.716 | 0.000 | -0.716 | 14 |
| Z6 | 30초 차분 | -0.696 | 0.000 | -0.696 | 14 |
| Z6 | 수준 | -0.770 | 0.500 | -0.771 | 14 |
| Z7 | 30초 차분 | -0.997 | 0.000 | -0.997 | 14 |
| Z7 | 수준 | -1.000 | 0.000 | -1.000 | 14 |
| Z8 | 30초 차분 | -0.880 | 0.000 | -0.880 | 14 |
| Z8 | 수준 | -0.997 | 0.000 | -0.997 | 14 |

[↑ 목차](#toc)


<a id="change"></a>
## 10. 승온 이후 TC-Z3의 차이 — 어느 구간을 확대 점검할 것인가?

**보려는 것:** SPHC **240061**의 초반 낮은 온도가 여러 존의 **공통 승온**인지 확인하고, 승온 뒤 **TC-Z3만 다른 세 TC와 크게 벌어지는 구간**을 찾습니다. 같은 시각의 TC-Z4·TC-Z5·TC-Z6를 비교하므로 느린 공통 상승과 TC-Z3의 상대 변화를 나눠 볼 수 있습니다.

**읽는 순서:** **① 노란 승온 구간에서 네 TC가 함께 오르는가 → ② 승온 이후 평균의 변화속도가 커지는 시각은 어디인가 → ③ 그 시각에 TC-Z3와 다른 세 TC의 차이가 커지는가.** 이후 해당 시각의 **OP-Z3·목표온도·가동상태**를 대조합니다.

| 표시 | 무엇을 뜻하는가? | 무엇을 확인하는가? |
|---|---|---|
| **노란 영역 · 0~73.5분** | TC-Z3·Z4·Z5·Z6 평균이 855℃에 도달하기 전의 관측 구간. 855℃는앞선 분석과 승온을 나누기 위한 참고값 | 네 TC가 함께 오르는지 확인하고, 승온 이후의 차이는 별도로 점검 |
| **가운데 빨간 점선 · ±1.081℃/분** | 전체 14 LOT의 평균 온도 변화속도 절댓값 중 상위 1% 경계 | 선 밖의 시각에서 각 TC를 확인. 평균 급변에는 TC-Z3의 단독 변화도 포함될 수 있음 |
| **아래 빨간 점선 · -6.65 / +5.64℃** | 이 LOT의 평소 존 간 차이를 기준으로 만든 확대 점검선 | TC-Z3가 다른 세 TC보다 크게 높아지거나 낮아진 구간 |
| **아래 빨간 원 · 관측값 33개** | 위 점검선 밖에 있는 **30초 관측값 33개**, 연속 **4구간**으로 묶임 | 고장 건수로 세지 않고, 구간별 TC-Z3·다른 TC·OP-Z3 원자료를 확대 |

**LOT 내 상대차 점수(robust z):** `TC-Z3 - (TC-Z4·TC-Z5·TC-Z6 평균)`이 **이 LOT의 차이 중앙값**에서 얼마나 벗어났는지 나타냅니다. **다른 LOT나 같은 강종과 비교한 점수가 아닙니다.** 그래프는 점수 대신 실제 온도 차이 **℃**와 점검선을 표시합니다. 계산 근거는 그림 아래에 정리합니다.


In [ ]:
delta_t=analysis_ts.groupby(['LOT_NO','SEG']).MEAS_DT.diff().dt.total_seconds()/60
analysis_ts['heat_rate']=analysis_ts.groupby(['LOT_NO','SEG'])['가열·균열존'].diff()/delta_t
analysis_ts.loc[delta_t.ne(.5),'heat_rate']=np.nan
rate_cut=analysis_ts.heat_rate.abs().quantile(.99)
rel_median=analysis_ts.groupby('LOT_NO').Z3_relative.transform('median')
rel_mad=(analysis_ts.Z3_relative-rel_median).abs().groupby(analysis_ts.LOT_NO).transform('median')
analysis_ts['relative_z']=(analysis_ts.Z3_relative-rel_median)/(1.4826*rel_mad.replace(0,np.nan))
analysis_ts['rate_candidate']=analysis_ts.heat_rate.abs().gt(rate_cut)
analysis_ts['relative_candidate']=analysis_ts.relative_z.abs().gt(4)
analysis_ts['low_heat_reference']=analysis_ts['가열·균열존'].lt(855)&analysis_ts['가열·균열존'].notna()
case=analysis_ts[analysis_ts.LOT_NO.eq(CASE_LOT)].copy()

# 점수는 기존 LOT 내 중앙값/MAD 기준을 유지하고, 그래프에서는 ℃ 점검선으로 바꿔 읽습니다.
s15_median=float(case.Z3_relative.median())
s15_mad=float((case.Z3_relative-s15_median).abs().median())
s15_scale=1.4826*s15_mad
s15_lower=s15_median-4*s15_scale
s15_upper=s15_median+4*s15_scale
s15_grade=lot_map.loc[CASE_LOT,'grade']

# 30초 관측값을 실제 연속 구간으로 묶습니다. SEG·결측·시간 공백을 넘지 않습니다.
s15_episode_rows=[]
for s15_kind in ['low_heat_reference','rate_candidate','relative_candidate']:
    s15_mask=case[s15_kind]
    s15_run=(s15_mask.ne(s15_mask.shift())|case.SEG.ne(case.SEG.shift())
             |case.MEAS_DT.diff().dt.total_seconds().ne(30)).cumsum()
    for _,s15_group in case[s15_mask].groupby(s15_run[s15_mask]):
        s15_peak=s15_group.loc[s15_group.relative_z.abs().idxmax()]
        s15_episode_rows.append({
            '종류':s15_kind,'LOT':int(CASE_LOT),'강종':s15_grade,
            '시작 [분]':float(s15_group.elapsed_min.min()),
            '마지막 관측 [분]':float(s15_group.elapsed_min.max()),
            '시작 시각':s15_group.MEAS_DT.min(),'마지막 관측 시각':s15_group.MEAS_DT.max(),
            '관측값 [개]':len(s15_group),'관측값 환산 [분]':.5*len(s15_group),
            '가장 큰 차이 시각 [분]':float(s15_peak.elapsed_min),
            'TC-Z3-다른 TC 평균 [℃]':float(s15_peak.Z3_relative),
            'OP-Z3 [%]':float(s15_peak['OP-Z3']),
            '최대 |robust z|':float(abs(s15_peak.relative_z))})
s15_episodes=pd.DataFrame(s15_episode_rows)
s15_relative_episodes=s15_episodes[s15_episodes['종류'].eq('relative_candidate')].copy()
s15_relative_episodes['구간 [분]']=s15_relative_episodes.apply(
    lambda r:f"{r['시작 [분]']:.1f}~{r['마지막 관측 [분]']:.1f}",axis=1)
s15_low_episodes=s15_episodes[s15_episodes['종류'].eq('low_heat_reference')]
s15_first_above=case.loc[case['가열·균열존'].ge(855),'elapsed_min'].min()
s15_first_heat=float(case['가열·균열존'].iloc[0])
s15_above_heat=float(case.loc[case.elapsed_min.eq(s15_first_above),'가열·균열존'].iloc[0])
s15_priority=s15_relative_episodes.loc[s15_relative_episodes['최대 |robust z|'].idxmax()]
s15_rate_n=int(case.rate_candidate.sum())
s15_low_rate_n=int((case.rate_candidate&case.low_heat_reference).sum())
s15_relative_n=int(case.relative_candidate.sum())

s15_red='#B4232F';s15_gray='#66717D';s15_purple='#7D3C98';s15_yellow='#FCF3CF'
# 제목·범례·의미는 별도 축에 배치합니다. 곡선 위에는 글자를 쓰지 않습니다.
s15_fig=plt.figure(figsize=(20,13.5))
s15_grid=s15_fig.add_gridspec(6,2,height_ratios=[.94,2.42]*3,width_ratios=[3.1,1],
    left=.075,right=.975,top=.855,bottom=.105,hspace=.16,wspace=.12)
s15_axes=[];s15_headers=[];s15_notes=[]
for s15_i in range(3):
    s15_head=s15_fig.add_subplot(s15_grid[2*s15_i,0]);s15_head.set_axis_off()
    s15_ax=s15_fig.add_subplot(s15_grid[2*s15_i+1,0],sharex=s15_axes[0] if s15_axes else None)
    s15_note=s15_fig.add_subplot(s15_grid[2*s15_i:2*s15_i+2,1]);s15_note.set_axis_off()
    s15_headers.append(s15_head);s15_axes.append(s15_ax);s15_notes.append(s15_note)
    for _,s15_low in s15_low_episodes.iterrows():
        s15_ax.axvspan(s15_low['시작 [분]'],s15_low['마지막 관측 [분]']+.5,
                      color=s15_yellow,alpha=.7,zorder=0,lw=0)
    s15_ax.grid(alpha=.13);s15_ax.spines[['top','right']].set_visible(False)
    s15_ax.margins(y=.12)
    if s15_i<2:s15_ax.tick_params(axis='x',labelbottom=False)

for _,s15_d in case.groupby('SEG'):
    for s15_tc,s15_ls in zip(['TC-Z4','TC-Z5','TC-Z6'],['-','--',':']):
        s15_axes[0].plot(s15_d.elapsed_min,s15_d[s15_tc],color='#A5ADB5',ls=s15_ls,lw=.8,alpha=.75)
    s15_axes[0].plot(s15_d.elapsed_min,s15_d['TC-Z3'],color=s15_purple,lw=1.1)
    s15_axes[0].plot(s15_d.elapsed_min,s15_d['가열·균열존'],color='black',lw=1.5)
    s15_axes[1].plot(s15_d.elapsed_min,s15_d.heat_rate,color='#2874A6',lw=1.4)
    s15_axes[2].plot(s15_d.elapsed_min,s15_d.Z3_relative,color=s15_purple,lw=1.6)
s15_axes[0].axhline(855,color=s15_gray,ls='--',lw=1)
s15_axes[1].axhline(rate_cut,color=s15_red,ls='--',lw=1.3)
s15_axes[1].axhline(-rate_cut,color=s15_red,ls='--',lw=1.3)
s15_axes[1].axhline(0,color='#B9C0C7',lw=.65,zorder=0)
s15_axes[2].axhline(s15_median,color=s15_gray,ls=':',lw=1)
s15_axes[2].axhline(s15_lower,color=s15_red,ls='--',lw=1.2)
s15_axes[2].axhline(s15_upper,color=s15_red,ls='--',lw=1.2)
s15_hit=case[case.relative_candidate]
s15_axes[2].scatter(s15_hit.elapsed_min,s15_hit.Z3_relative,s=28,
                   facecolors='none',edgecolors=s15_red,lw=1.3,zorder=5)
s15_axes[0].set_ylabel('온도 [℃]',fontsize=11)
s15_axes[1].set_ylabel('평균 온도 변화속도 [℃/분]',fontsize=11)
s15_axes[2].set_ylabel('TC-Z3 - 다른 세 TC 평균 [℃]',fontsize=11)
s15_axes[2].set_xlabel(f'LOT {CASE_LOT} 수집 시작 후 경과 [분]',labelpad=10,fontsize=11)

s15_titles=[
    '① 노란 구간: TC-Z3·Z4·Z5·Z6가 함께 승온하는가?',
    '② 평균 변화속도: 어느 시각의 변화를 다시 볼 것인가?',
    '③ 승온 이후: TC-Z3와 다른 세 TC의 차이가 커지는가?']
s15_descriptions=[
    f'노란 영역: 평균 <855℃ · 이 LOT의 첫 855℃ 도달은 {s15_first_above:.1f}분',
    f'빨간 점선: 전체 14 LOT |변화속도|의 상위 1% 경계 ±{rate_cut:.3f}℃/분',
    '빨간 점선: LOT 내 상대차 점수 |robust z| >4에 해당하는 ℃ 경계 · 빨간 원: 선 밖 관측값']
s15_handles=[
    [Line2D([],[],color=s15_purple,lw=1.2,label='보라 실선: TC-Z3'),
     Line2D([],[],color='black',lw=1.6,label='검정 실선: 네 TC 평균'),
     Line2D([],[],color=s15_gray,ls='--',label='회색 점선: 855℃ 참고값'),
     Line2D([],[],color='#A5ADB5',ls='-',label='TC-Z4'),
     Line2D([],[],color='#A5ADB5',ls='--',label='TC-Z5'),
     Line2D([],[],color='#A5ADB5',ls=':',label='TC-Z6')],
    [Line2D([],[],color='#2874A6',lw=1.4,label='파랑 실선: 평균 온도 변화속도'),
     Line2D([],[],color=s15_red,ls='--',label=f'빨간 점선: ±{rate_cut:.3f}℃/분 확대 점검선')],
    [Line2D([],[],color=s15_purple,lw=1.6,label='보라 실선: TC-Z3 - TC-Z4·Z5·Z6 평균'),
     Line2D([],[],color=s15_gray,ls=':',label=f'회색 점선: 이 LOT 차이 중앙값 {s15_median:+.2f}℃'),
     Line2D([],[],color=s15_red,ls='--',label=f'빨간 점선: {s15_lower:+.2f} / {s15_upper:+.2f}℃ 점검선'),
     Line2D([],[],ls='',marker='o',mfc='none',mec=s15_red,label='빨간 원: 원자료 확대 대상')]]
for s15_i,s15_head in enumerate(s15_headers):
    s15_head.text(0,.90,s15_titles[s15_i],ha='left',va='center',fontsize=13,fontweight='bold',color='black')
    s15_head.text(0,.63,s15_descriptions[s15_i],ha='left',va='center',fontsize=10,color=s15_gray)
    s15_head.legend(handles=s15_handles[s15_i],loc='lower left',bbox_to_anchor=(0,0),
                    ncol=3 if s15_i==0 else 2,fontsize=9.7,frameon=False,
                    handlelength=2.4,columnspacing=1.8,borderaxespad=0,borderpad=0)

s15_note_lines=[
    [('노란 구간에서 볼 것','black',13,True),
     (f'0~{s15_first_above:.1f}분 · 공통 승온','black',12,True),
     ('네 TC가 함께 상승하는지 확인',s15_gray,11.5,False),
     (f'평균 {s15_first_heat:.1f} → {s15_above_heat:.1f}℃',s15_gray,11.5,False),
     ('855℃는 승온을 나눈 참고값',s15_gray,11,False),
     ('승온 이후는 아래 상대차로 확인',s15_red,11.5,True)],
    [('빨간 점선에서 볼 것','black',13,True),
     (f'변화속도 ±{rate_cut:.3f}℃/분',s15_red,12,True),
     ('전체 관측의 상위 1% 경계',s15_gray,11.5,False),
     (f'초과 {s15_rate_n}개 중 {s15_low_rate_n}개는 승온 중',s15_gray,11.5,False),
     (f'승온 후 {s15_rate_n-s15_low_rate_n}개: 각 TC 동반 여부 확인',s15_red,11.5,True),
     ('운전 목표값은 별도로 대조',s15_gray,11,False)],
    [('빨간 원에서 볼 것','black',13,True),
     ('+는 TC-Z3가 높음 / -는 낮음',s15_gray,11.5,False),
     (f'{s15_relative_n}개 관측값 → 연속 {len(s15_relative_episodes)}구간',s15_red,12,True),
     (f"우선 확대: {s15_priority['시작 [분]']:.1f}~{s15_priority['마지막 관측 [분]']:.1f}분",s15_red,12,True),
     (f"{s15_priority['가장 큰 차이 시각 [분]']:.1f}분: 차이 {s15_priority['TC-Z3-다른 TC 평균 [℃]']:+.2f}℃",s15_red,11.5,True),
     (f"같은 시각 OP-Z3 {s15_priority['OP-Z3 [%]']:.1f}% · 목표값 대조",s15_gray,11,False)]]
for s15_note,s15_lines in zip(s15_notes,s15_note_lines):
    for s15_y,(s15_text,s15_color,s15_size,s15_bold) in zip([.96,.79,.63,.47,.31,.15],s15_lines):
        s15_note.text(0,s15_y,s15_text,ha='left',va='top',fontsize=s15_size,
                      color=s15_color,fontweight='bold' if s15_bold else 'normal')

s15_fig.suptitle(f'그림 15. {s15_grade} {CASE_LOT} — 승온 이후 어느 TC-Z3 구간을 확대 점검할 것인가?',
                x=.075,y=.980,ha='left',fontsize=17,fontweight='bold')
s15_fig.text(.075,.942,
    f'확인 결과: 노란 승온 0~{s15_first_above:.1f}분을 분리 → TC-Z3 상대차 관측값 {s15_relative_n}개를 연속 {len(s15_relative_episodes)}구간으로 묶음',
    fontsize=12,color='#425466')
s15_fig.legend(handles=[Patch(fc=s15_yellow,ec='none',
    label=f'노란 영역: 평균 855℃ 미만 · 0~{s15_first_above:.1f}분 승온 확인')],
    loc='upper left',bbox_to_anchor=(.07,.918),frameon=False,fontsize=10.8)
s15_fig.text(.075,.051,
    'robust z: TC-Z3와 다른 세 TC의 차이가 이 LOT의 차이 중앙값에서 얼마나 벗어났는지 나타내는 점수 · 아래 패널은 실제 차이와 점검선을 ℃로 표시',
    fontsize=10,color=s15_gray)
s15_fig.text(.075,.025,
    f'{s15_relative_n}개는 30초 관측값의 수입니다. 고장 건수로 세지 않고 연속 구간의 온도·OP-Z3·목표값·가동상태를 대조합니다.',
    fontsize=10,color=s15_gray)
save_and_show_figures()

# 이후 셀에서 사용하는 전체 LOT 집계 변수는 그대로 유지합니다.
candidate_summary=analysis_ts.groupby('LOT_NO').agg(유효변화율칸=('heat_rate','count'),변화율후보칸=('rate_candidate','sum'),상대이탈후보칸=('relative_candidate','sum'),참고855미만칸=('low_heat_reference','sum'))
candidate_summary['855미만안의변화율후보']=analysis_ts[analysis_ts.low_heat_reference].groupby('LOT_NO').rate_candidate.sum().reindex(candidate_summary.index,fill_value=0)
candidate_summary=candidate_summary.join(lot_map[['grade']]).reset_index()

# 마지막 시각은 마지막 관측 칸의 시작입니다. 칸 수는 독립된 고장 건수가 아닙니다.
display(s15_relative_episodes[['구간 [분]','관측값 [개]','가장 큰 차이 시각 [분]',
                              'TC-Z3-다른 TC 평균 [℃]','OP-Z3 [%]']].round(3))


#### 그림 15 — 승온과 TC-Z3 상대차를 나눠 어떤 구간을 점검하는가?

**알게 된 것:** **0~73.5분의 공통 승온을 따로 보면, 승온 이후에도 TC-Z3가 다른 세 TC와 크게 벌어지는 4구간이 남습니다.** 따라서 이 LOT의 온도 변동을 초반 승온만으로 설명하기 어렵습니다. 가장 큰 상대 상승인 **129~135.5분**, 이어지는 상대 하강 **140.5~144분**의 원자료를 우선 봅니다.

| 볼 곳 | 실제 결과 | 무엇을 알게 되었는가? |
|---|---|---|
| **노란 영역 · 0~73.5분** | 네 TC 평균은 시작 **815.3℃**, 첫 855℃ 도달은 **73.5분**. 변화속도 기준 초과 **21개 중 15개**가 이 구간에 있음 | 초반 큰 변화에 공통 승온이 섞임. TC-Z3 단독 이탈과 나눠 분석해야 함 |
| **가운데 파란 선·빨간 점선** | 평균 온도의 변화속도 [℃/분], 점선은 전체 14 LOT의 절댓값 **99분위수 ±1.081℃/분** | 선 밖의 시각을 다시 볼 기준. 평균에는 TC-Z3 단독 변화도반영되므로 각 TC를 함께 확인 |
| **아래 보라 선·빨간 원** | TC-Z3에서 같은 시각 **TC-Z4·Z5·Z6 평균을 뺀 차이**. 점검선 밖의 **30초 관측값 33개 → 연속 4구간** | 공통 상승과 달리 TC-Z3가 상대적으로 높아지거나 낮아지는 구간을 지정 가능 |
| **131분 · 03:15:00** | TC-Z3 **870.536℃**, 다른 세 TC 평균보다 **+10.366℃**, 같은 시각 **OP-Z3 0%** | 승온 이후의 TC-Z3 상대 상승과 제어출력 감소가 겹침. 목표온도·제어 모드를 확인할 우선 지점 |

**33개의 의미:** 30초로 집계한 관측값 33개이며, 칸 수로 환산하면 **16.5분**입니다. 실제 연속 운전 상태의 확정 유지시간이나 고장 33건을 뜻하지 않습니다. 다음 표의 구간 끝은 **마지막 30초 관측 칸의 시작 시각**입니다.

| 확대할 구간 · SPHC 240061 | 관측값 수 | 가장 큰 상대차 · 같은 시각 OP-Z3 | 확인해야 할 것 |
|---|---:|---|---|
| **76.5~78분 · 02:20:30~02:22:00** | 4개 | **77.5분: +7.058℃ / OP-Z3 0%** | 승온 직후의 상대 상승과 출력 감소가 함께 있는가? |
| **81.5~84.5분 · 02:25:30~02:28:30** | 7개 | **83분: -7.915℃ / OP-Z3 94.283%** | 직전 상승 뒤 TC-Z3만 낮아지는가? 다른 TC와 OP-Z3의 회복 흐름 대조 |
| **129~135.5분 · 03:13:00~03:19:30** | 14개 | **131분: +10.366℃ / OP-Z3 0%** | 가장 큰 상대 상승. **TC-Z3 실제값·목표값·OP-Z3·TC-Z4~Z6**를 함께 확대 |
| **140.5~144분 · 03:24:30~03:28:00** | 8개 | **142분: -7.968℃ / OP-Z3 94.283%** | 앞선 상대 상승 이후의 하강·회복을 함께 점검. 별개의 고장으로 세지 않음 |

**왜 이 비교인가?** 여러 존이 함께 오르면 `TC-Z3 - 다른 세 TC 평균`의 공통 상승 성분은 줄어듭니다. 이 차이가 승온 후에도 크게 벌어지면 **TC-Z3와 연결된 온도·제어·계측 조건**을 더 자세히 볼 이유가 됩니다. 존별 목표값이 다를 수도 있으므로, 관측된 차이를 바로 센서 고장으로 연결하지 않습니다.

**robust z 계산 근거 — LOT 내부의 상대차 점수**

`d(t) = TC-Z3(t) - mean(TC-Z4, TC-Z5, TC-Z6)(t)`

`robust z(t) = [d(t) - median(d)] / [1.4826 × MAD(d)]`

- `median(d)`는 **240061 내부**의 차이 중앙값 **-0.505℃**입니다.
- `MAD`는 중앙값에서 떨어진 거리의 중앙값이며 **1.036℃**, 보정 척도 `1.4826 × MAD`는 **1.536℃**입니다.
- `|robust z| > 4`를 온도 차이로 바꾼 점검선은 **-6.651 / +5.641℃**입니다. 빨간 원은 이 선 밖의 관측값입니다.
- 이 기준은 현재 자료에서 사후 계산한 확대 점검 규칙입니다. **고장 라벨**은 현장 고장 발생 여부가 확인되어 시각·대상에 연결된 정답 기록을 뜻하는데, 이 점수만으로는 그런 기록이 만들어지지 않습니다.

**후속 점검 — 무엇을 누구와 비교하는가?**

| 우선순위 · 대상 | 비교 대상·컬럼 | 확인할 질문 · 이어지는 분석 |
|---|---|---|
| **① 240061 / 129~135.5분**, 이후 **140.5~144분** | 같은 시각 **TC-Z3 ↔ TC-Z4·Z5·Z6**, **TC-Z3 ↔ OP-Z3** | 상대 상승·하강이 출력 억제·회복과 어떻게 겹치는가? **그림 19-A**에서 상승 구간을 확대. 하강 구간의 제어 흐름도 별도로 대조 |
| **② 240061 / 승온 이후**, 참고 **76.5~84.5분** | 같은 SPHC **240044·240069·240091**의 승온 제외 구간 | 같은 운전상태에서도 상대차의 크기·빈도·연속 구간이 다른가? LOT 길이가 달라 관측값 수 대신 유효 관측 대비 비율·구간 지속시간도 비교 |
| **③ 원인 확인** | 존별 **목표온도·제어 모드·가동상태·속도·두께·계측 로그** | 목표값 차이, 제어 반응, 외란, 계측 문제 중 무엇으로 설명되는가? 수집 초반 OP-Z3 결측은 별도 로그로 확인 |

**판단 범위:** 이 그림은 **확대할 시각을 지정**했습니다. 온도가 오를 때 OP-Z3를 줄이는 것은 정상적인 제어 반응일 수도 있습니다. 같은 강종·같은 운전상태의 비교와 목표값 대조 후에 점검 원인을 좁힙니다.

[↑ 목차](#toc)


<a id="maintenance"></a>
## 11. 정비 연결 — 감지 주변 변화와 실제 작업 전후를 구분

**분석 질문:** 감지 시각 주변에는 어떤 변화가 관측되며, 실제 착수·완료 후 변화도 확인할 수 있는가?

**보는 순서:** **그림 16**에서 관측 범위를 확인 → **그림 17-감지-1·2**에서 작업별 감지 주변 변화 → **그림 17-A·B**에서 실제 작업 전후 → **그림 17-C**에서 작업 후 관측이 없는 사례 → **그림 19-C**에서 5·10·20분 창의 결과가 유지되는지 확인합니다.

**감지 주변 계산:** 같은 LOT의 원본값으로 평균·std를 계산합니다. 비교 창은 감지 전후에 같은 길이를 사용합니다. **01859는 감지 전 관측이 5분뿐이므로 전후 5분**, **02105는 전후 10분**입니다. 곡선은 유효 원본 15개 이상인 30초 평균이며 시간 공백을 연결하지 않습니다.

**각 감지 그림의 읽는 순서:** 위의 **센서 관측·감지·착수·완료 시간축** → 아래 **TC-Z3·OP-Z3·CP-Z4 곡선** → 곡선 위 헤더의 **전후 평균**, 오른쪽의 **std·해석**을 봅니다. 회색 비교창은 감지 전, 옅은 붉은 비교창은 감지 후이며 두 사례 모두 해당 작업의 착수 전입니다. 빗금은 해당 LOT 관측 범위 밖을 뜻합니다.

**2/44건의 의미:** CAL01 정비 44건 중 감지 시각에 센서가 있는 작업이 2건이라는 자료 범위입니다. 정비 후 개선을 확인한 건수와는 다릅니다. **CBM 01854는 감지가 관측 밖이어도 작업 전후는 관측 안**이므로 그림 17-A·B에서 따로 확인합니다.


### 먼저 정할 것 — 어느 LOT의 어떤 작업 전후를 비교하는가?

**첨부 사례의 LOT는 240133과 240117입니다.** LOT는 센서 관측 묶음, SPCE·SPCD·SPHC는 강종, TBM·CBM·BM은 **정비 작업의 유형**입니다. **같은 LOT에 여러 작업이 연결될 수 있습니다.** 아래의 주 비교는 각 LOT 내부의 시간 구간 비교이며, 서로 다른 강종의 240133과 240117을 정비 효과의 대조군으로 바로 쓰지 않습니다.

| 주 확인 대상 | 연결할 작업·맥락 | 먼저 비교할 것 | 같은 강종 비교·이어지는 위치 |
|---|---|---|---|
| **240117 · SPCD** | **01859 · TBM** / 계획 점검 | 본 LOT의 **착수 전 07:30~07:40 ↔ 완료 후 09:35~09:45** | 그림 **17-A·B·19-C**. SPCD n=1이므로 외부 동일 강종 비교 LOT 없음 |
| **240117 · SPCD** | **01854 · CBM** / 마모량 초과 | 같은 LOT의 다른 작업. **착수 전 08:44~08:54 ↔ 완료 후 12:44~12:54** | 그림 **17-A·B·19-C**. TBM과 08:54~09:35 중첩·운전상태를 함께 확인 |
| **240133 · SPCE** | **02105 · CBM** / 분석계 셀 교체 | **감지 전 15:44~15:54 ↔ 감지 후 15:54~16:04**. 실제 완료 후 관측 없음 | 감지 그림 **17-감지-2**, 공백 **17-C**. CP는 **240037을 먼저**, **240121도 보조** 대조 |
| **240044 · SPHC** | **00717 · BM** / 동작 불량 | OP-Z3 초반 결측과 BM 감지까지의 시간 간격. 실제 착수 전·완료 후 비교는 관측 부족 | **240061·240069·240091**의 같은 상태와 비교. 그림 **10-A·D·G·H, 17-C** |
| **240061 · SPHC** | **CR1 계획수리 후 첫 관측** | OP-Z3 결측·0~73.5분 승온을 분리한 뒤 TC-Z3↔OP-Z3 확대 | **240044·240069·240091**. 그림 **10-D·G·H, 11·12·15·19-A** |
| **240069 · SPHC** | **01094 · BM** / 절연 저하 | CP 피크·OP-Z3 저출력과 BM 시각의 위치. CP 피크가 감지와 같은 시각인지 별도 확인 | **240044·240061·240091**. 그림 **10-C·G·H, 17-C·19-B** |
| **240091 · SPHC** | 상대 비교 LOT / TC-Z7·Z8 지속 차이도 있음 | 다른 SPHC의 TC-Z3 변동을 대조하면서 본 LOT의 TC-Z7·Z8도 따로 확인 | **240044·240061·240069**. 그림 **10-E-추가·G·H**. 모든 컬럼의 정상 기준으로 지정하지 않음 |

**SPHC 네 LOT를 모두 사용합니다.** 다만 질문이 다릅니다. 240044·240069는 BM과의 시간 위치, 240061은 수리 후 승온·결측, 240091은 같은 강종의 상대 흐름을 확인합니다. 전체 14 LOT의 검토·다음 점검은 **그림 10-G·H**에 있습니다.

### 왜 TC-Z3·OP-Z3·CP-Z4인가?

| 비교 컬럼 | 선택 근거 | 이 비교로 구분할 것 |
|---|---|---|
| **TC-Z3 ↔ TC-Z4·TC-Z5·TC-Z6** | 앞선 **그림 6**에서 TC-Z3의 30초 변화량 std가 가열 TC 4개 중 **14/14 LOT 최대** | 여러 TC가 함께 승온·하강하는가, TC-Z3의 상대 변화인가? |
| **TC-Z3 ↔ OP-Z3** | 제공된 컬럼 정의상 같은 존의 온도·출력. OP-Z3의 30초 변화량 std도 출력 8개 중 **14/14 LOT 최대** | 온도 변화와 출력 억제·회복의 동시 흐름. 제어 반응의 원인은 목표값·모드와 대조 |
| **CP-Z4** | 온도·출력과 구분한 분위기 관련 신호. 수준·std·피크를 각각 확인 | 분위기 수준 변화와 흔들림, 일시적 피크를 분리. 분석계 셀과 CP-Z4의 직접 태그 대응은 현재 CSV에서 확인되지 않음 |

**존 선택은 변화와 제어를 비교할 대표 채널 선정입니다. 해당 정비가 TC-Z3의 부품을 수리했다는 뜻은 아닙니다.** 작업 대상 부위·태그 기록이 확인되면 연결된 채널을 우선하고, 다른 존에 변화가 있으면 그 존을 추가합니다.

**감지 시각에 센서 관측이 있으면 유용합니다.** 감지 전후 흐름을 확인할 수 있기 때문입니다. ‘감지 시각 관측 있음’은 분석 가능한 자료 범위를 나타내며, 고장 상태나 센서의 문제를 뜻하지 않습니다. 감지는 기록된 한 시각이므로 ‘감지 중’보다 ‘감지 시각’으로 표시합니다.

**관측 공백을 확인해서 연결할 것:** **센서 변화 → 작업 기록·조치 → 실제 완료 후 변화**가 모두 관측되는지 판단합니다. 앞·뒤가 있으면 작업 전후 비교를 수행하고, 뒤가 없으면 어떤 시점·채널을 추가 수집해야 하는지 지정합니다. 공백 자체로 설비 정지나 고장 원인을 판정하지 않습니다.


In [ ]:
cal['month']=cal.DETC_DT.dt.month
monthly=pd.crosstab(cal.month,cal.MNT_TYPE).reindex(range(1,13),fill_value=0).reindex(columns=['TBM','BM','CBM'],fill_value=0)
mnt_summary=cal.groupby('MNT_TYPE').agg(작업수=('WO_NO','size'),정지시간평균=('DOWN_MIN','mean'),정지시간중앙값=('DOWN_MIN','median'),정지시간합계=('DOWN_MIN','sum'))
fig,axes=plt.subplots(1,2,figsize=(15,5))
monthly.plot.bar(stacked=True,color=[mnt_colors[c] for c in monthly.columns],ax=axes[0],rot=0)
axes[0].set(title='CAL01 월별 정비 감지 작업 수',xlabel='2024년 월',ylabel='고유 작업 수')
for i,total in enumerate(monthly.sum(axis=1)): axes[0].text(i,total+.1,str(total),ha='center',fontsize=8)
for i,kind in enumerate(['TBM','BM','CBM']):
    d=cal[cal.MNT_TYPE.eq(kind)]
    jitter=np.linspace(-.15,.15,len(d))
    axes[1].scatter(i+jitter,d.DOWN_MIN,color=mnt_colors[kind],alpha=.65)
    axes[1].plot([i-.2,i+.2],[d.DOWN_MIN.median()]*2,color='black',lw=3)
    axes[1].text(i,d.DOWN_MIN.max()+15,f'n={len(d)}\n중앙값 {d.DOWN_MIN.median():.0f}분',ha='center',fontsize=9)
axes[1].margins(y=.22)
axes[1].set_xticks([0,1,2],['TBM','BM','CBM']); axes[1].set(title='정지시간 분포 · 검정 = 중앙값',ylabel='기록 DOWN_MIN [분]')
finish(fig,f'그림 16. CAL01 정비 {len(cal)}작업 — 유형별 작업 수와 정지시간을 구분',
       '비교 목적: 기록 빈도와 정비 부담 확인  |  왼쪽 = 월별 감지 작업 수 · 오른쪽 점 = 작업별 DOWN_MIN · 검정 = 중앙값 · 고장률·정비 효과 아님')
display(mnt_summary.round(2))

# 감지 시각에 센서가 있는 작업을 선정하고, 전후 동일 길이의 창을 계산합니다.
observed_works=cal[cal.DETC_DT_observed].copy()
s17_records=[];s17_window_rows=[]
for s17_work in observed_works.itertuples():
    s17_lot=int(lot_map.index[(lot_map.start<=s17_work.DETC_DT)&(lot_map.end>=s17_work.DETC_DT)][0])
    s17_raw=temp[temp.LOT_NO.eq(s17_lot)].copy()
    s17_pre_available=(s17_work.DETC_DT-s17_raw.MEAS_DT.min()).total_seconds()/60
    s17_post_available=(s17_raw.MEAS_DT.max()-s17_work.DETC_DT).total_seconds()/60
    s17_width=float(min(10,s17_pre_available,s17_post_available))
    s17_pre=s17_raw[s17_raw.MEAS_DT.ge(s17_work.DETC_DT-pd.Timedelta(minutes=s17_width))
                    &s17_raw.MEAS_DT.lt(s17_work.DETC_DT)]
    s17_post=s17_raw[s17_raw.MEAS_DT.ge(s17_work.DETC_DT)
                     &s17_raw.MEAS_DT.lt(s17_work.DETC_DT+pd.Timedelta(minutes=s17_width))]
    # 둘 다 해당 작업의 착수 전 관측인지 시각으로 확인합니다.
    s17_before_start=s17_work.DETC_DT+pd.Timedelta(minutes=s17_width)<=s17_work.STRT_DT
    s17_window_rows.append({
        '작업':s17_work.WO_NO,'LOT':s17_lot,'강종':lot_map.loc[s17_lot,'grade'],'유형':s17_work.MNT_TYPE,
        '비교창 [분]':s17_width,'감지':s17_work.DETC_DT,'착수':s17_work.STRT_DT,'완료':s17_work.END_DT,
        '센서 시작':s17_raw.MEAS_DT.min(),'센서 종료':s17_raw.MEAS_DT.max(),
        '두 비교창 모두 착수 전':bool(s17_before_start),
        # 세 TC가 모두 기록된 원본 행만 공통 온도 평균에 사용합니다.
        '다른 세 TC 평균 차이':s17_post[['TC-Z4','TC-Z5','TC-Z6']].mean(axis=1).where(
            s17_post[['TC-Z4','TC-Z5','TC-Z6']].notna().all(axis=1)).mean()
            -s17_pre[['TC-Z4','TC-Z5','TC-Z6']].mean(axis=1).where(
                s17_pre[['TC-Z4','TC-Z5','TC-Z6']].notna().all(axis=1)).mean()})
    for s17_col in ['TC-Z3','OP-Z3','CP-Z4']:
        s17_before_std=s17_pre[s17_col].std();s17_after_std=s17_post[s17_col].std()
        s17_records.append({
            '작업':s17_work.WO_NO,'유형':s17_work.MNT_TYPE,'LOT':s17_lot,'신호':s17_col,
            '전 관측창 [분]':s17_width,'후 관측창 [분]':s17_width,
            '전 유효행':int(s17_pre[s17_col].count()),'후 유효행':int(s17_post[s17_col].count()),
            '전평균':s17_pre[s17_col].mean(),'후평균':s17_post[s17_col].mean(),
            '차이 (후-전)':s17_post[s17_col].mean()-s17_pre[s17_col].mean(),
            '전 std':s17_before_std,'후 std':s17_after_std,
            'std 후/전':s17_after_std/s17_before_std if s17_before_std>0 else np.nan})
before_after=pd.DataFrame(s17_records)
s17_detect_windows=pd.DataFrame(s17_window_rows).set_index('작업')
observed_status=observed_works[['WO_NO','MNT_TYPE','DETC_DT','STRT_DT','END_DT',
    'DETC_DT_observed','STRT_DT_observed','END_DT_observed']].copy()


#### 그림 16 — 정비 기록과 센서 관측 범위

| 확인한 결과 | 무엇을 알게 되었는가? |
|---|---|
| **CAL01 44작업: TBM 22 · BM 12 · CBM 10** | 기록된 작업 유형별 빈도·DOWN_MIN 분포를 확인. 가동시간 대비 고장률이나 정비 효과를 계산한 것은 아님 |
| **감지 시각 센서 있음: 2/44건** | **01859 / SPCD 240117**, **02105 / SPCE 240133**에서 감지 주변 변화를 볼 수 있음 |
| 감지 관측과 작업 전후 관측은 별도 확인 | 01854는 감지 시각이 관측 밖이지만 실제 착수·완료 전후는 240117에서 확인 가능 |

**다음에 볼 것:** 아래에서 **작업별 감지 그림 → 결과·후속 점검**을 읽습니다. 이어지는 그림 17-A·B는 감지가 아니라 실제 착수 전·완료 후 비교입니다. DOWN_MIN은 기록된 정지시간으로, 착수~완료 경과시간과 구분합니다.


In [ ]:

# 이 셀에서 작업 하나의 시간축과 세 센서 곡선을 직접 생성합니다. 이미지 파일을 사용하지 않습니다.
s17_work_no='WO-2024-01859'
s17_work=cal[cal.WO_NO.eq(s17_work_no)].iloc[0]
s17_info=s17_detect_windows.loc[s17_work_no]
s17_lot=int(s17_info.LOT);s17_grade=s17_info['강종'];s17_width=float(s17_info['비교창 [분]'])
s17_metrics=before_after[before_after['작업'].eq(s17_work_no)].set_index('신호')
s17_raw=temp[temp.LOT_NO.eq(s17_lot)].copy()
s17_trace=analysis_ts[analysis_ts.LOT_NO.eq(s17_lot)].copy()
s17_trace['감지 기준 [분]']=(s17_trace.MEAS_DT-s17_work.DETC_DT).dt.total_seconds()/60
s17_trace=s17_trace[s17_trace['감지 기준 [분]'].between(-20,20)]
s17_first=(s17_info['센서 시작']-s17_work.DETC_DT).total_seconds()/60
s17_last=(s17_info['센서 종료']-s17_work.DETC_DT).total_seconds()/60
s17_red='#B4232F';s17_gray='#66717D';s17_pre_color='#F1F3F5';s17_post_color='#FDF0F0'
s17_series_color={'TC-Z3':'#263238','OP-Z3':'#2874A6','CP-Z4':'#7D3C98'}
s17_is_warmup=s17_work_no=='WO-2024-01859'
s17_heading=('감지 직후 온도 상승은 정비 전 승온인가?' if s17_is_warmup
             else '감지 주변 CP-Z4의 수준과 변동은 어떻게 달라졌는가?')
s17_fig=plt.figure(figsize=(20,14.5))
s17_grid=s17_fig.add_gridspec(8,2,height_ratios=[.96,1.04]+[.96,2.0]*3,
    width_ratios=[3.1,1],left=.075,right=.975,top=.865,bottom=.12,hspace=.23,wspace=.13)
s17_stage_header=s17_fig.add_subplot(s17_grid[0,:]);s17_stage_header.set_axis_off()
s17_stage=s17_fig.add_subplot(s17_grid[1,:])
s17_headers=[];s17_axes=[];s17_notes=[]

# 실제 시각에서 센서 관측과 착수~완료를 다른 행으로 표시합니다.
for _,s17_segment in s17_raw.groupby('SEG'):
    s17_stage.hlines(1,s17_segment.MEAS_DT.min(),s17_segment.MEAS_DT.max(),color='#7E8790',lw=8)
s17_stage.hlines(0,s17_work.STRT_DT,s17_work.END_DT,color='black',lw=7)
s17_stage.scatter([s17_work.STRT_DT],[0],facecolors='white',edgecolors='black',s=45,zorder=5)
s17_stage.scatter([s17_work.END_DT],[0],marker='s',color='black',s=40,zorder=5)
s17_stage.axvline(s17_work.DETC_DT,color='black',ls='--',lw=1.3)
s17_timeline_left=min(s17_info['센서 시작'],s17_work.DETC_DT)-pd.Timedelta(minutes=15)
s17_timeline_right=max(s17_info['센서 종료'],s17_work.END_DT)+pd.Timedelta(minutes=15)
s17_stage.set_xlim(s17_timeline_left,s17_timeline_right);s17_stage.set_ylim(-.45,1.45)
s17_stage.set_yticks([1,0],['센서 관측 LOT','실제 작업'])
s17_stage.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4,maxticks=7))
s17_stage.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
s17_stage.spines[['top','right','left']].set_visible(False);s17_stage.grid(axis='x',alpha=.14)
s17_stage_header.text(0,.87,'① 먼저 시각을 대조: 감지 후인가, 실제 작업 완료 후인가?',
    ha='left',va='center',fontsize=13,fontweight='bold')
s17_stage_header.text(0,.56,
    f"{s17_work.DETC_DT:%Y-%m-%d} · 감지 {s17_work.DETC_DT:%H:%M} / 착수 {s17_work.STRT_DT:%H:%M} / 완료 {s17_work.END_DT:%H:%M}"
    f" · 센서 관측 {s17_info['센서 시작']:%H:%M:%S}~{s17_info['센서 종료']:%H:%M:%S}",
    ha='left',va='center',fontsize=11,color=s17_gray)
s17_stage_header.legend(handles=[
    Line2D([],[],color='#7E8790',lw=5,label='회색 가로선: 실제 LOT 센서 관측'),
    Line2D([],[],color='black',lw=5,label='검정 가로선: 착수~완료 · ○ 착수 / ■ 완료'),
    Line2D([],[],color='black',ls='--',label='검정 세로 점선: 기록된 감지 시각')],
    loc='lower left',bbox_to_anchor=(0,0),ncol=3,frameon=False,fontsize=10,borderaxespad=0)

for s17_i,s17_col in enumerate(['TC-Z3','OP-Z3','CP-Z4']):
    s17_head=s17_fig.add_subplot(s17_grid[2+2*s17_i,0]);s17_head.set_axis_off()
    s17_ax=s17_fig.add_subplot(s17_grid[3+2*s17_i,0],sharex=s17_axes[0] if s17_axes else None)
    s17_note=s17_fig.add_subplot(s17_grid[2+2*s17_i:4+2*s17_i,1]);s17_note.set_axis_off()
    s17_headers.append(s17_head);s17_axes.append(s17_ax);s17_notes.append(s17_note)
    s17_value=s17_metrics.loc[s17_col]
    s17_precision=6 if s17_col=='CP-Z4' else 3
    s17_unit='℃' if s17_col.startswith('TC') else '%' if s17_col.startswith('OP') else ''
    s17_diff_unit='%p' if s17_col.startswith('OP') else s17_unit
    s17_ax.axvspan(-s17_width,0,color=s17_pre_color,zorder=0,lw=0)
    s17_ax.axvspan(0,s17_width,color=s17_post_color,zorder=0,lw=0)
    # 시작 전·종료 후를 센서 결측값과 혼동하지 않게 LOT 관측 밖이라고 표시합니다.
    for s17_a,s17_b in [(-20,min(20,s17_first)),(max(-20,s17_last),20)]:
        if s17_b>s17_a:
            s17_ax.axvspan(s17_a,s17_b,fc='#E7EBEF',ec='#B1BAC3',hatch='///',lw=0,zorder=0)
    for _,s17_segment in s17_trace.groupby('SEG'):
        if s17_col=='TC-Z3':
            for s17_peer,s17_ls in zip(['TC-Z4','TC-Z5','TC-Z6'],['-','--',':']):
                s17_ax.plot(s17_segment['감지 기준 [분]'],s17_segment[s17_peer],
                    color='#A4ADB6',ls=s17_ls,lw=.9,alpha=.8)
        s17_ax.plot(s17_segment['감지 기준 [분]'],s17_segment[s17_col],
                    color=s17_series_color[s17_col],lw=1.5)
    s17_ax.axvline(0,color='black',ls='--',lw=1.25)
    s17_ax.hlines(s17_value['전평균'],-s17_width,0,color=s17_gray,ls='--',lw=1.1)
    s17_ax.hlines(s17_value['후평균'],0,s17_width,color=s17_red,ls='--',lw=1.1)
    s17_ax.scatter([-s17_width/2],[s17_value['전평균']],facecolors='white',edgecolors=s17_gray,s=42,zorder=5)
    s17_ax.scatter([s17_width/2],[s17_value['후평균']],color=s17_red,s=34,zorder=5)
    s17_ax.set_xlim(-20,20);s17_ax.margins(y=.14)
    s17_ax.tick_params(axis='x',pad=8)
    s17_ax.set_ylabel('온도 [℃]' if s17_col.startswith('TC') else '제어출력 [%]' if s17_col.startswith('OP') else 'CP-Z4 [무차원]',fontsize=11)
    s17_ax.grid(alpha=.13);s17_ax.spines[['top','right']].set_visible(False)
    if s17_i<2:s17_ax.tick_params(axis='x',labelbottom=False)
    s17_title={'TC-Z3':'② TC-Z3 — 다른 세 TC도 함께 오르거나 바뀌는가?',
               'OP-Z3':'③ OP-Z3 — 같은 시각의 제어출력은 어떻게 변하는가?',
               'CP-Z4':'④ CP-Z4 — 평균 수준과 std는 어떻게 달라지는가?'}[s17_col]
    s17_head.text(0,.87,s17_title,ha='left',va='center',fontsize=13,fontweight='bold')
    s17_head.text(0,.60,
        f"○ 감지 전 {s17_width:.0f}분 {s17_value['전평균']:.{s17_precision}f}{s17_unit}"
        f" → ● 감지 후 {s17_width:.0f}분 {s17_value['후평균']:.{s17_precision}f}{s17_unit}"
        f" · 차이 {s17_value['차이 (후-전)']:+.{s17_precision}f}{s17_diff_unit}",
        ha='left',va='center',fontsize=10.5,color='#425466')
    s17_legend=[Line2D([],[],color=s17_series_color[s17_col],lw=1.5,label=f'실선: {s17_col} 유효 30초 평균')]
    if s17_col=='TC-Z3':
        s17_legend.append(Line2D([],[],color='#A4ADB6',lw=1,label='회색 얇은 선: TC-Z4·TC-Z5·TC-Z6'))
    s17_legend+=[
        Line2D([],[],color=s17_gray,ls='--',marker='o',mfc='white',label='회색 가로선: 감지 전 평균'),
        Line2D([],[],color=s17_red,ls='--',marker='o',mfc=s17_red,label='붉은 가로선: 감지 후 평균')]
    s17_head.legend(handles=s17_legend,loc='lower left',bbox_to_anchor=(0,0),ncol=2,
        frameon=False,fontsize=9.7,columnspacing=1.8,handlelength=2.4,borderaxespad=0,borderpad=0)
    s17_focus=(s17_is_warmup and s17_col=='TC-Z3') or (not s17_is_warmup and s17_col=='CP-Z4')
    s17_focus_color=s17_red if s17_focus else '#263238'
    s17_interpretation=(
        {'TC-Z3':'다른 세 TC도 상승\n공통 승온을 먼저 분리',
         'OP-Z3':'승온 중 출력 변화\n목표값·가동상태와 대조',
         'CP-Z4':'승온·가동 전환과 대조\n분위기 변화 원인은 미확인'}
        if s17_is_warmup else
        {'TC-Z3':'다른 TC의 동반 변화 대조\n감지 시각을 온도 급변으로 확정하지 않음',
         'OP-Z3':'온도와 같은 시각으로 대조\n출력 감소의 제어 맥락 확인',
         'CP-Z4':'평균은 낮아지고 std는 커짐\n교체 효과 판단은 관측 없음'})[s17_col]
    s17_note_items=[
        (.93,'이 패널에서 알 수 있는 것','black',12.5,True),
        (.74,f"평균 차이 {s17_value['차이 (후-전)']:+.{s17_precision}f}{s17_diff_unit}",s17_focus_color,12,True),
        (.56,f"std {s17_value['전 std']:.{s17_precision}f}\n → {s17_value['후 std']:.{s17_precision}f}{s17_unit}",s17_gray,11,False),
        (.34,f"std 후/전 {s17_value['std 후/전']:.2f}배",s17_gray,11,False),
        (.15,s17_interpretation,s17_focus_color,11,True)]
    for s17_y,s17_text,s17_color,s17_size,s17_bold in s17_note_items:
        s17_note.text(0,s17_y,s17_text,ha='left',va='top',fontsize=s17_size,color=s17_color,
            fontweight='bold' if s17_bold else 'normal',linespacing=1.5)

s17_axes[-1].set_xlabel('기록된 감지 시각 기준 [분] · 0 = 감지 · +는 감지 후',fontsize=11,labelpad=8)
s17_fig.suptitle(f"그림 17-감지-{1 if s17_is_warmup else 2}. {s17_grade} {s17_lot} — {s17_heading}",
    x=.075,y=.982,ha='left',fontsize=17,fontweight='bold')
s17_result=(
    f"TBM 01859 · 전후 5분의 TC-Z3 차이 {s17_metrics.loc['TC-Z3','차이 (후-전)']:+.3f}℃ · 다른 세 TC 평균도 +{s17_info['다른 세 TC 평균 차이']:.3f}℃ · 모두 정비 착수 전"
    if s17_is_warmup else
    f"CBM 02105 · CP-Z4 평균 {s17_metrics.loc['CP-Z4','차이 (후-전)']:+.6f} / std {s17_metrics.loc['CP-Z4','std 후/전']:.2f}배 · 실제 교체 완료 후 센서 없음")
s17_fig.text(.075,.946,s17_result,fontsize=11.5,color='#425466')
s17_fig.legend(handles=[
    Patch(fc=s17_pre_color,label=f'회색 비교창: 감지 전 {s17_width:.0f}분'),
    Patch(fc=s17_post_color,label=f'옅은 붉은 비교창: 감지 후 {s17_width:.0f}분 · 해당 작업 착수 전'),
    Patch(fc='#E7EBEF',ec='#B1BAC3',hatch='///',label='빗금: 해당 LOT 관측 밖')],
    loc='upper left',bbox_to_anchor=(.07,.921),ncol=3,frameon=False,fontsize=10)
s17_fig.text(.075,.046,
    '평균·std는 1초 원자료로 계산 · std = 표준편차 · 선은 유효 원본 15개 이상인 30초 평균 · 세로 점선은 감지, 가로 점선은 비교창 평균 · 공백은 연결하지 않음',
    fontsize=10,color=s17_gray)
s17_fig.text(.075,.020,
    ('다음 확인: 실제 착수 전·완료 후는 그림 17-A·B / 19-C · 승온과 CBM 01854 작업 중첩을 함께 확인'
     if s17_is_warmup else
     '다음 확인: CP-Z4를 SPCE 240037·240121과 대조 · 분석계 셀–CP-Z4 태그 관계와 교체 후 센서 확보 · 관측 공백은 그림 17-C'),
    fontsize=10,color=s17_gray)
save_and_show_figures()


#### 그림 17-감지-1 — 감지 후 상승을 왜 정비 효과로 해석하지 않는가?

**확인한 결과:** **SPCD 240117 / TBM 01859**의 감지 직후 TC-Z3는 상승하지만, **TC-Z4·Z5·Z6도 함께 상승**합니다. 비교한 두 창은 모두 **07:40 정비 착수 전**이므로, 우선 **공통 승온**으로 분리해서 봅니다.

| 비교 항목 · 전후 각각 5분 | 감지 전 07:08~07:13 → 감지 후 07:13~07:18 | 무엇을 알게 되었는가? |
|---|---|---|
| **TC-Z3 평균** | **820.151 → 824.647℃ / +4.496℃** | 감지 후 상승을 관측. 다른 세 TC 평균도 **+5.620℃**여서 TC-Z3에만 나타난 상승은 아님 |
| **OP-Z3 평균·std** | 평균 **77.228 → 88.544%**, std **17.768 → 10.861%** | 승온 중 출력 수준·흔들림도 달라짐. 목표값·제어 모드와 함께 볼 항목 |
| **CP-Z4 평균·std** | 평균 **0.449477 → 0.450474**, std **0.001020 → 0.001313** | CP 평균과 변동은 증가. 모든 신호가 안정됐다는 결과는 아님 |
| **정비 시각·관측** | 감지 **07:13**, 착수 **07:40**, 완료 **09:35**. LOT 관측 **07:08~14:32:19** | 감지 후 5분과 실제 완료 후 구간을 서로 나눠 볼 수 있음 |

**다른 세 TC의 평균:** 세 채널이 모두 기록된 원본 행만 사용합니다(감지 전 **289행**, 후 **290행**). 감지 전 TC-Z5의 결측 1행은 공통 평균에서 제외했습니다.

**비교창을 바꾼 이유:** 이전 그림은 감지 전 **5분**과 후 **10분**을 비교해 TC-Z3 **+7.067℃**를 표시했습니다. 새 주 비교는 **5분 ↔ 5분**으로 맞춰 **+4.496℃**입니다. 사용한 시간 범위가 달라진 결과이며, 서로 다른 길이의 승온 구간을 같은 기준처럼 비교하지 않습니다.

**이 그림으로 알 수 있는 것:** 감지 시각을 포함한 승온·출력·CP의 흐름과 실제 작업 시각의 위치입니다. TBM의 DETC_DT는 기록된 감지 시각이며 고장 발생을 확인한 시각으로 바꾸어 해석하지 않습니다.

| 다음 비교 | 무엇을 확인할 것인가? | 노트북 위치·필요 자료 |
|---|---|---|
| **01859 착수 전 07:30~07:40 ↔ 완료 후 09:35~09:45** | 실제 작업 전후의 TC-Z3·OP-Z3·CP-Z4 평균·std. 전 창은 승온 중이고 후 창은 CBM 01854 작업 중 | **그림 17-A·B**, 승온·작업 상세·목표값 |
| **CBM 01854의 실제 작업 전후** | 01859와 **08:54~09:35** 작업이 겹치는 영향, 완료 후 변화의 지속성 | **그림 17-A·B·19-C** |
| 같은 상태의 추가 SPCD LOT | 정비가 없는 승온 구간에서도 유사한 변화가 나타나는가? | 현재 SPCD n=1. 같은 강종·목표값·속도·두께의 추가 관측 |


In [ ]:

# 이 셀에서 작업 하나의 시간축과 세 센서 곡선을 직접 생성합니다. 이미지 파일을 사용하지 않습니다.
s17_work_no='WO-2024-02105'
s17_work=cal[cal.WO_NO.eq(s17_work_no)].iloc[0]
s17_info=s17_detect_windows.loc[s17_work_no]
s17_lot=int(s17_info.LOT);s17_grade=s17_info['강종'];s17_width=float(s17_info['비교창 [분]'])
s17_metrics=before_after[before_after['작업'].eq(s17_work_no)].set_index('신호')
s17_raw=temp[temp.LOT_NO.eq(s17_lot)].copy()
s17_trace=analysis_ts[analysis_ts.LOT_NO.eq(s17_lot)].copy()
s17_trace['감지 기준 [분]']=(s17_trace.MEAS_DT-s17_work.DETC_DT).dt.total_seconds()/60
s17_trace=s17_trace[s17_trace['감지 기준 [분]'].between(-20,20)]
s17_first=(s17_info['센서 시작']-s17_work.DETC_DT).total_seconds()/60
s17_last=(s17_info['센서 종료']-s17_work.DETC_DT).total_seconds()/60
s17_red='#B4232F';s17_gray='#66717D';s17_pre_color='#F1F3F5';s17_post_color='#FDF0F0'
s17_series_color={'TC-Z3':'#263238','OP-Z3':'#2874A6','CP-Z4':'#7D3C98'}
s17_is_warmup=s17_work_no=='WO-2024-01859'
s17_heading=('감지 직후 온도 상승은 정비 전 승온인가?' if s17_is_warmup
             else '감지 주변 CP-Z4의 수준과 변동은 어떻게 달라졌는가?')
s17_fig=plt.figure(figsize=(20,14.5))
s17_grid=s17_fig.add_gridspec(8,2,height_ratios=[.96,1.04]+[.96,2.0]*3,
    width_ratios=[3.1,1],left=.075,right=.975,top=.865,bottom=.12,hspace=.23,wspace=.13)
s17_stage_header=s17_fig.add_subplot(s17_grid[0,:]);s17_stage_header.set_axis_off()
s17_stage=s17_fig.add_subplot(s17_grid[1,:])
s17_headers=[];s17_axes=[];s17_notes=[]

# 실제 시각에서 센서 관측과 착수~완료를 다른 행으로 표시합니다.
for _,s17_segment in s17_raw.groupby('SEG'):
    s17_stage.hlines(1,s17_segment.MEAS_DT.min(),s17_segment.MEAS_DT.max(),color='#7E8790',lw=8)
s17_stage.hlines(0,s17_work.STRT_DT,s17_work.END_DT,color='black',lw=7)
s17_stage.scatter([s17_work.STRT_DT],[0],facecolors='white',edgecolors='black',s=45,zorder=5)
s17_stage.scatter([s17_work.END_DT],[0],marker='s',color='black',s=40,zorder=5)
s17_stage.axvline(s17_work.DETC_DT,color='black',ls='--',lw=1.3)
s17_timeline_left=min(s17_info['센서 시작'],s17_work.DETC_DT)-pd.Timedelta(minutes=15)
s17_timeline_right=max(s17_info['센서 종료'],s17_work.END_DT)+pd.Timedelta(minutes=15)
s17_stage.set_xlim(s17_timeline_left,s17_timeline_right);s17_stage.set_ylim(-.45,1.45)
s17_stage.set_yticks([1,0],['센서 관측 LOT','실제 작업'])
s17_stage.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=4,maxticks=7))
s17_stage.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
s17_stage.spines[['top','right','left']].set_visible(False);s17_stage.grid(axis='x',alpha=.14)
s17_stage_header.text(0,.87,'① 먼저 시각을 대조: 감지 후인가, 실제 작업 완료 후인가?',
    ha='left',va='center',fontsize=13,fontweight='bold')
s17_stage_header.text(0,.56,
    f"{s17_work.DETC_DT:%Y-%m-%d} · 감지 {s17_work.DETC_DT:%H:%M} / 착수 {s17_work.STRT_DT:%H:%M} / 완료 {s17_work.END_DT:%H:%M}"
    f" · 센서 관측 {s17_info['센서 시작']:%H:%M:%S}~{s17_info['센서 종료']:%H:%M:%S}",
    ha='left',va='center',fontsize=11,color=s17_gray)
s17_stage_header.legend(handles=[
    Line2D([],[],color='#7E8790',lw=5,label='회색 가로선: 실제 LOT 센서 관측'),
    Line2D([],[],color='black',lw=5,label='검정 가로선: 착수~완료 · ○ 착수 / ■ 완료'),
    Line2D([],[],color='black',ls='--',label='검정 세로 점선: 기록된 감지 시각')],
    loc='lower left',bbox_to_anchor=(0,0),ncol=3,frameon=False,fontsize=10,borderaxespad=0)

for s17_i,s17_col in enumerate(['TC-Z3','OP-Z3','CP-Z4']):
    s17_head=s17_fig.add_subplot(s17_grid[2+2*s17_i,0]);s17_head.set_axis_off()
    s17_ax=s17_fig.add_subplot(s17_grid[3+2*s17_i,0],sharex=s17_axes[0] if s17_axes else None)
    s17_note=s17_fig.add_subplot(s17_grid[2+2*s17_i:4+2*s17_i,1]);s17_note.set_axis_off()
    s17_headers.append(s17_head);s17_axes.append(s17_ax);s17_notes.append(s17_note)
    s17_value=s17_metrics.loc[s17_col]
    s17_precision=6 if s17_col=='CP-Z4' else 3
    s17_unit='℃' if s17_col.startswith('TC') else '%' if s17_col.startswith('OP') else ''
    s17_diff_unit='%p' if s17_col.startswith('OP') else s17_unit
    s17_ax.axvspan(-s17_width,0,color=s17_pre_color,zorder=0,lw=0)
    s17_ax.axvspan(0,s17_width,color=s17_post_color,zorder=0,lw=0)
    # 시작 전·종료 후를 센서 결측값과 혼동하지 않게 LOT 관측 밖이라고 표시합니다.
    for s17_a,s17_b in [(-20,min(20,s17_first)),(max(-20,s17_last),20)]:
        if s17_b>s17_a:
            s17_ax.axvspan(s17_a,s17_b,fc='#E7EBEF',ec='#B1BAC3',hatch='///',lw=0,zorder=0)
    for _,s17_segment in s17_trace.groupby('SEG'):
        if s17_col=='TC-Z3':
            for s17_peer,s17_ls in zip(['TC-Z4','TC-Z5','TC-Z6'],['-','--',':']):
                s17_ax.plot(s17_segment['감지 기준 [분]'],s17_segment[s17_peer],
                    color='#A4ADB6',ls=s17_ls,lw=.9,alpha=.8)
        s17_ax.plot(s17_segment['감지 기준 [분]'],s17_segment[s17_col],
                    color=s17_series_color[s17_col],lw=1.5)
    s17_ax.axvline(0,color='black',ls='--',lw=1.25)
    s17_ax.hlines(s17_value['전평균'],-s17_width,0,color=s17_gray,ls='--',lw=1.1)
    s17_ax.hlines(s17_value['후평균'],0,s17_width,color=s17_red,ls='--',lw=1.1)
    s17_ax.scatter([-s17_width/2],[s17_value['전평균']],facecolors='white',edgecolors=s17_gray,s=42,zorder=5)
    s17_ax.scatter([s17_width/2],[s17_value['후평균']],color=s17_red,s=34,zorder=5)
    s17_ax.set_xlim(-20,20);s17_ax.margins(y=.14)
    s17_ax.tick_params(axis='x',pad=8)
    s17_ax.set_ylabel('온도 [℃]' if s17_col.startswith('TC') else '제어출력 [%]' if s17_col.startswith('OP') else 'CP-Z4 [무차원]',fontsize=11)
    s17_ax.grid(alpha=.13);s17_ax.spines[['top','right']].set_visible(False)
    if s17_i<2:s17_ax.tick_params(axis='x',labelbottom=False)
    s17_title={'TC-Z3':'② TC-Z3 — 다른 세 TC도 함께 오르거나 바뀌는가?',
               'OP-Z3':'③ OP-Z3 — 같은 시각의 제어출력은 어떻게 변하는가?',
               'CP-Z4':'④ CP-Z4 — 평균 수준과 std는 어떻게 달라지는가?'}[s17_col]
    s17_head.text(0,.87,s17_title,ha='left',va='center',fontsize=13,fontweight='bold')
    s17_head.text(0,.60,
        f"○ 감지 전 {s17_width:.0f}분 {s17_value['전평균']:.{s17_precision}f}{s17_unit}"
        f" → ● 감지 후 {s17_width:.0f}분 {s17_value['후평균']:.{s17_precision}f}{s17_unit}"
        f" · 차이 {s17_value['차이 (후-전)']:+.{s17_precision}f}{s17_diff_unit}",
        ha='left',va='center',fontsize=10.5,color='#425466')
    s17_legend=[Line2D([],[],color=s17_series_color[s17_col],lw=1.5,label=f'실선: {s17_col} 유효 30초 평균')]
    if s17_col=='TC-Z3':
        s17_legend.append(Line2D([],[],color='#A4ADB6',lw=1,label='회색 얇은 선: TC-Z4·TC-Z5·TC-Z6'))
    s17_legend+=[
        Line2D([],[],color=s17_gray,ls='--',marker='o',mfc='white',label='회색 가로선: 감지 전 평균'),
        Line2D([],[],color=s17_red,ls='--',marker='o',mfc=s17_red,label='붉은 가로선: 감지 후 평균')]
    s17_head.legend(handles=s17_legend,loc='lower left',bbox_to_anchor=(0,0),ncol=2,
        frameon=False,fontsize=9.7,columnspacing=1.8,handlelength=2.4,borderaxespad=0,borderpad=0)
    s17_focus=(s17_is_warmup and s17_col=='TC-Z3') or (not s17_is_warmup and s17_col=='CP-Z4')
    s17_focus_color=s17_red if s17_focus else '#263238'
    s17_interpretation=(
        {'TC-Z3':'다른 세 TC도 상승\n공통 승온을 먼저 분리',
         'OP-Z3':'승온 중 출력 변화\n목표값·가동상태와 대조',
         'CP-Z4':'승온·가동 전환과 대조\n분위기 변화 원인은 미확인'}
        if s17_is_warmup else
        {'TC-Z3':'다른 TC의 동반 변화 대조\n감지 시각을 온도 급변으로 확정하지 않음',
         'OP-Z3':'온도와 같은 시각으로 대조\n출력 감소의 제어 맥락 확인',
         'CP-Z4':'평균은 낮아지고 std는 커짐\n교체 효과 판단은 관측 없음'})[s17_col]
    s17_note_items=[
        (.93,'이 패널에서 알 수 있는 것','black',12.5,True),
        (.74,f"평균 차이 {s17_value['차이 (후-전)']:+.{s17_precision}f}{s17_diff_unit}",s17_focus_color,12,True),
        (.56,f"std {s17_value['전 std']:.{s17_precision}f}\n → {s17_value['후 std']:.{s17_precision}f}{s17_unit}",s17_gray,11,False),
        (.34,f"std 후/전 {s17_value['std 후/전']:.2f}배",s17_gray,11,False),
        (.15,s17_interpretation,s17_focus_color,11,True)]
    for s17_y,s17_text,s17_color,s17_size,s17_bold in s17_note_items:
        s17_note.text(0,s17_y,s17_text,ha='left',va='top',fontsize=s17_size,color=s17_color,
            fontweight='bold' if s17_bold else 'normal',linespacing=1.5)

s17_axes[-1].set_xlabel('기록된 감지 시각 기준 [분] · 0 = 감지 · +는 감지 후',fontsize=11,labelpad=8)
s17_fig.suptitle(f"그림 17-감지-{1 if s17_is_warmup else 2}. {s17_grade} {s17_lot} — {s17_heading}",
    x=.075,y=.982,ha='left',fontsize=17,fontweight='bold')
s17_result=(
    f"TBM 01859 · 전후 5분의 TC-Z3 차이 {s17_metrics.loc['TC-Z3','차이 (후-전)']:+.3f}℃ · 다른 세 TC 평균도 +{s17_info['다른 세 TC 평균 차이']:.3f}℃ · 모두 정비 착수 전"
    if s17_is_warmup else
    f"CBM 02105 · CP-Z4 평균 {s17_metrics.loc['CP-Z4','차이 (후-전)']:+.6f} / std {s17_metrics.loc['CP-Z4','std 후/전']:.2f}배 · 실제 교체 완료 후 센서 없음")
s17_fig.text(.075,.946,s17_result,fontsize=11.5,color='#425466')
s17_fig.legend(handles=[
    Patch(fc=s17_pre_color,label=f'회색 비교창: 감지 전 {s17_width:.0f}분'),
    Patch(fc=s17_post_color,label=f'옅은 붉은 비교창: 감지 후 {s17_width:.0f}분 · 해당 작업 착수 전'),
    Patch(fc='#E7EBEF',ec='#B1BAC3',hatch='///',label='빗금: 해당 LOT 관측 밖')],
    loc='upper left',bbox_to_anchor=(.07,.921),ncol=3,frameon=False,fontsize=10)
s17_fig.text(.075,.046,
    '평균·std는 1초 원자료로 계산 · std = 표준편차 · 선은 유효 원본 15개 이상인 30초 평균 · 세로 점선은 감지, 가로 점선은 비교창 평균 · 공백은 연결하지 않음',
    fontsize=10,color=s17_gray)
s17_fig.text(.075,.020,
    ('다음 확인: 실제 착수 전·완료 후는 그림 17-A·B / 19-C · 승온과 CBM 01854 작업 중첩을 함께 확인'
     if s17_is_warmup else
     '다음 확인: CP-Z4를 SPCE 240037·240121과 대조 · 분석계 셀–CP-Z4 태그 관계와 교체 후 센서 확보 · 관측 공백은 그림 17-C'),
    fontsize=10,color=s17_gray)
save_and_show_figures()


#### 그림 17-감지-2 — CP-Z4는 어떻게 바뀌었고, 교체 효과는 볼 수 있는가?

**확인한 결과:** **SPCE 240133 / CBM 02105**의 감지 전후 10분에서 CP-Z4는 **평균이 낮아지고 std는 커졌습니다.** 이 두 창은 정비 착수 전이며, **가스 분석계 셀 교체 후 변화는 현재 센서 관측으로 확인할 수 없습니다.**

| 비교 항목 · 전후 각각 10분 | 감지 전 15:44~15:54 → 감지 후 15:54~16:04 | 무엇을 알게 되었는가? |
|---|---|---|
| **CP-Z4 평균** | **0.450761 → 0.450187 / -0.000574** | 감지 주변 평균 수준이 낮아짐. 작은 차이의 의미는 허용범위·같은 강종 비교로 확인 |
| **CP-Z4 std** | **0.001000 → 0.001273 / 1.27배** | 평균 하강과 변동 감소는 같은 결과가 아님. 이 창에서는 변동이 커짐 |
| **TC-Z3 평균·OP-Z3 평균** | TC-Z3 **+0.291℃**, OP-Z3 **-4.326%p** | 같은 시각의 온도·출력 조건도 대조할 수 있음. 단순 전후 평균만으로 감지 원인·급변은 확정되지 않음 |
| **실제 작업과 관측 종료** | 감지 **15:54** → 센서 종료 **16:44:24** → 착수 **16:54** → 완료 **23:33** | 감지 후 곡선도 실제 작업 전 관측. 완료 후 효과 분석에는 추가 센서가 필요 |

**빗금·빈 공간의 의미:** 빗금은 해당 LOT의 관측 범위 밖입니다. 센서 개별 채널의 결측값이나 설비 정지를 자동으로 뜻하지 않습니다. 위 시간축의 센서 관측과 실제 작업 구간이 떨어진 이유는 원본 수집 범위로 확인됩니다.

| 다음 비교 | 무엇을 확인할 것인가? | 노트북 위치·필요 자료 |
|---|---|---|
| **240133 ↔ SPCE 240037**, 보조 **240121** | 같은 상태·같은 길이 창의 CP-Z4 수준·std·피크. 중간 20~80% 요약과 감지 창은 나눠 비교 | **그림 10-B-CP·10-B-OP**, 운전조건을 맞춘 원자료 비교 |
| **WO-2024-02105 → CAL01·INS-006** | 가스 분석계 셀 교체 사실은 연결됨. **INS-006 → CP-Z4**의 직접 대응은 현재 7개 CSV에 없음 | **그림 17-C**, PLC/I/O 태그표·장착·교정·작업 상세 |
| **실제 교체 전 ↔ 완료·교정 후** | 교체 후 CP 수준과 std가 달라졌는가? | 완료 후 연속 계측 확보. 현재 그림에는 해당 관측 없음 |

**활용:** 이 그림은 **감지 주변의 CP 수준·변동을 점검할 근거**를 제공합니다. 다음 단계는 같은 강종 대조와 작업·태그 기록 확인입니다. 교체 효과의 변화 방향은 관측을 확보한 뒤 계산합니다.


### 정비 유형과 실제 조치를 어떻게 읽을 것인가?

| 유형 | 기록의 의미 | 이 자료에서 연결할 조치 | 전후 비교의 기준 |
|---|---|---|---|
| **TBM** | 시간·일정 기반 정비 | 계획 점검·수리기간 작업·기록된 부품교체 | 감지가 아니라 **착수 직전 → 완료 직후**. 승온·정지 상태를 별도 확인 |
| **CBM** | 상태 기반 정비 | 마모량 초과 등 상태 기록, 부품교체·교체 사유 | 감지에서 작업까지 지연될 수 있어 **감지·착수·완료**를 모두 구분 |
| **BM** | 사후정비 | 동작 불량·절연 저하 등 고장모드·비고, 기록된 교체 | 고장 전조와 수리 후 변화를 보려면 실제 직전·완료 후 센서가 필요 |

**조치는 기록된 범위까지만 연결합니다.** `REMARK`는 증상·점검 내용일 수 있고, `PART_CHG`는 교체 여부입니다. 부품교체이력의 **PART_CD·PART_NM·QTY·CHG_RSN**을 별도로 대조합니다. 예를 들어 '마모량 초과'만 있으면 어떤 조정을 했는지 알 수 없고, '교체 없음'이 '아무 조치도 없음'을 뜻하지도 않습니다.

**추가 비교:** 감지 시점에 센서가 있는 2건과 실제 착수 직전·완료 후 10분을 같은 LOT에서 확보한 2건은 서로 다른 작업 조합입니다. 아래 실제 작업 전후 비교는 **TBM 01859와 CBM 01854**, 모두 **SPCD 240117** 안입니다.


In [ ]:
# 감지 주변 값과 실제 작업 전후를 구분합니다. 원본 CSV에서 작업·부품·센서 시각을 연결합니다.
def mnt_lot_at(value):
    matches=lot_map[(lot_map.start<=value)&(lot_map.end>=value)]
    return int(matches.index[0]) if len(matches) else None
mnt_part_actions={}
for work_no,records in cal_parts.groupby('WO_NO'):
    mnt_part_actions[work_no]='; '.join(
        f'{r.PART_CD} {r.PART_NM} · 기록 QTY {r.QTY} · {r.CHG_RSN}'
        for r in records.itertuples())
mnt_work_rows=[]; mnt_actual_rows=[]
for work in cal.itertuples():
    start_lot=mnt_lot_at(work.STRT_DT); end_lot=mnt_lot_at(work.END_DT)
    same_lot=start_lot is not None and start_lot==end_lot
    full_windows=bool(same_lot and lot_map.loc[start_lot,'start']<=work.STRT_DT-pd.Timedelta(minutes=10)
                      and lot_map.loc[start_lot,'end']>=work.END_DT+pd.Timedelta(minutes=10))
    mnt_work_rows.append({
        'WO_NO':work.WO_NO,'MNT_TYPE':work.MNT_TYPE,'감지 관측':mnt_lot_at(work.DETC_DT) is not None,
        '착수 관측':start_lot is not None,'완료 관측':end_lot is not None,
        '실제 전후 10분 비교':full_windows,'비교 LOT':start_lot if full_windows else np.nan,
        '기록 내용':str(work.REMARK).strip() if pd.notna(work.REMARK) else '구체 조치 미기록',
        '고장모드':work.FAIL_MODE if pd.notna(work.FAIL_MODE) else '미기록',
        '부품교체 기록':mnt_part_actions.get(work.WO_NO,'교체 내역 없음'),
        '교체 작업':work.WO_NO in mnt_part_actions})
    if not full_windows: continue
    sample=temp[temp.LOT_NO.eq(start_lot)]
    before=sample[sample.MEAS_DT.ge(work.STRT_DT-pd.Timedelta(minutes=10))&sample.MEAS_DT.lt(work.STRT_DT)]
    after=sample[sample.MEAS_DT.ge(work.END_DT)&sample.MEAS_DT.lt(work.END_DT+pd.Timedelta(minutes=10))]
    for channel in ['TC-Z3','OP-Z3','CP-Z4']:
        mnt_actual_rows.append({
            '작업':work.WO_NO,'유형':work.MNT_TYPE,'LOT':start_lot,'컬럼':channel,
            '작업 전 평균':before[channel].mean(),'완료 후 평균':after[channel].mean(),
            '평균 차이':after[channel].mean()-before[channel].mean(),
            '작업 전 std':before[channel].std(),'완료 후 std':after[channel].std(),
            '전 유효행':int(before[channel].count()),'후 유효행':int(after[channel].count()),
            '전 시작':work.STRT_DT-pd.Timedelta(minutes=10),'전 끝':work.STRT_DT,
            '후 시작':work.END_DT,'후 끝':work.END_DT+pd.Timedelta(minutes=10)})
mnt_work_audit=pd.DataFrame(mnt_work_rows)
mnt_actual_pairs=pd.DataFrame(mnt_actual_rows)
mnt_type_coverage=mnt_work_audit.groupby('MNT_TYPE').agg(
    작업수=('WO_NO','size'),감지관측=('감지 관측','sum'),착수관측=('착수 관측','sum'),
    완료관측=('완료 관측','sum'),실제전후비교=('실제 전후 10분 비교','sum'),부품교체작업=('교체 작업','sum')).reindex(['TBM','CBM','BM'])
display(mnt_type_coverage)
display(mnt_actual_pairs.drop(columns=['전 시작','전 끝','후 시작','후 끝']).round(6))
mnt_selected_actions=mnt_work_audit[mnt_work_audit.WO_NO.isin([
    'WO-2024-01859','WO-2024-01854','WO-2024-02105','WO-2024-00717','WO-2024-01094','WO-2024-00916'
])][['WO_NO','MNT_TYPE','기록 내용','고장모드','부품교체 기록','실제 전후 10분 비교']]
display(mnt_selected_actions)


In [ ]:
# 그림 17-A: 실제 작업 시각과 센서를 같은 시간축에서 직접 생성합니다.
mnt_lot=240117
mnt_sample=analysis_ts[analysis_ts.LOT_NO.eq(mnt_lot)]
mnt_jobs=cal[cal.WO_NO.isin(['WO-2024-01859','WO-2024-01854'])].sort_values('STRT_DT')
mnt_fig,mnt_axes=plt.subplots(4,1,figsize=(17,13.6),sharex=True,
                            gridspec_kw={'height_ratios':[.7,2,2,2]})
mnt_fig.subplots_adjust(left=.10,right=.98,top=.85,bottom=.055,hspace=.65)
mnt_fig.suptitle('그림 17-A. SPCD 240117 내부 비교 — TBM 01859·CBM 01854의 작업 전후',
                 x=.10,y=.992,ha='left',fontsize=15,fontweight='bold')
mnt_fig.text(.10,.95,'01859 TBM: 계획 점검 수행·교체 없음  |  01854 CBM: 마모량 초과·교체 없음·상세 조치 미기록',
              fontsize=10,color='#6B7280')
mnt_fig.text(.10,.920,'비교: 각 작업의 착수 직전 10분 ↔ 완료 직후 10분 · 두 작업 모두 같은 LOT 240117', fontsize=10,color='#425466')
mnt_fig.text(.10,.895,'TC-Z3·OP-Z3: 변동이 큰 같은 존의 대표 짝 · TC-Z4~Z6: 공통 승온 대조 · CP-Z4: 분위기 수준·std 확인', fontsize=10,color='#6B7280')
mnt_timeline=mnt_axes[0]
for j,work in enumerate(mnt_jobs.itertuples()):
    mnt_timeline.hlines(j,work.STRT_DT,work.END_DT,color=mnt_colors[work.MNT_TYPE],lw=9)
    mnt_timeline.scatter([work.STRT_DT],[j],marker='o',facecolors='white',edgecolors='black',zorder=3)
    mnt_timeline.scatter([work.END_DT],[j],marker='s',color='black',zorder=3)
    if lot_map.loc[mnt_lot,'start']<=work.DETC_DT<=lot_map.loc[mnt_lot,'end']:
        mnt_timeline.scatter([work.DETC_DT],[j],marker='D',color='#B4232F',zorder=3)
mnt_timeline.set_yticks(range(len(mnt_jobs)),[f'{r.WO_NO[-5:]} · {r.MNT_TYPE}' for r in mnt_jobs.itertuples()])
mnt_timeline.set_ylim(-.6,len(mnt_jobs)-.4);mnt_timeline.grid(alpha=.15)
mnt_timeline.set_title('작업 구간 · ○ 착수 / ■ 완료 / ◆ 감지',loc='left',fontweight='bold',pad=18)
mnt_timeline.invert_yaxis()
mnt_cols=['TC-Z3','OP-Z3','CP-Z4']
for ax,col in zip(mnt_axes[1:],mnt_cols):
    for work in mnt_jobs.itertuples():
        ax.axvspan(work.STRT_DT,work.END_DT,color=mnt_colors[work.MNT_TYPE],alpha=.055,zorder=0)
        ax.axvline(work.STRT_DT,color=mnt_colors[work.MNT_TYPE],ls=':',lw=.9)
        ax.axvline(work.END_DT,color=mnt_colors[work.MNT_TYPE],ls='--',lw=.9)
    if col=='TC-Z3':
        for peer_col in ['TC-Z4','TC-Z5','TC-Z6']:
            for _,segment in mnt_sample.groupby('SEG'):
                ax.plot(segment.MEAS_DT,segment[peer_col],color='#9AA3AD',ls='--',lw=.8,alpha=.6)
    for _,segment in mnt_sample.groupby('SEG'):
        ax.plot(segment.MEAS_DT,segment[col],color={'TC-Z3':'#B4232F','OP-Z3':'#2874A6','CP-Z4':'#7D3C98'}[col],lw=1)
    ax.set_title(col+(' · 회색 비교선 = TC-Z4·TC-Z5·TC-Z6' if col=='TC-Z3' else ''),
                 loc='left',fontweight='bold',pad=20)
    ax.set_ylabel(UNIT[col]);ax.set_xlabel('2024-11-13 실제 시각')
    ax.tick_params(axis='x',labelbottom=True);ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.grid(alpha=.15);ax.spines[['top','right']].set_visible(False)
    ax.margins(y=.12)
mnt_axes[-1].set_xlim(lot_map.loc[mnt_lot,'start'],lot_map.loc[mnt_lot,'end'])
mnt_fig.text(.10,.014,'TBM 07:40~09:35와 CBM 08:54~12:44가 겹칩니다. CBM 감지는 전날 15:00으로 이 LOT 밖입니다. 시간상 변화와 정비 효과를 구분합니다.',
              fontsize=9,color='#6B7280')
save_and_show_figures()


#### 그림 17-A — 작업과 승온을 같은 시간축에서 확인한 결과

**비교 대상:** 이 그림은 **240117 / SPCD 한 LOT** 안의 두 작업을 봅니다. **01859 / TBM 전후**, **01854 / CBM 전후**를 각각 계산합니다. 240133 / SPCE는 별도의 CBM 02105 사례입니다. TC-Z3·OP-Z3는 앞선 14 LOT에서 변동이 큰 대표 짝으로 선택했고, 회색 TC-Z4~Z6는 공통 승온을 대조합니다. 해당 작업이 Z3에 직접 조치한 기록은 현재 자료에 없습니다.

**① 읽는 순서:** 맨 위에서 TBM·CBM 착수~완료 구간을 찾고, 아래에서 **TC-Z3(회색 TC-Z4·Z5·Z6 비교), OP-Z3, CP-Z4**의 같은 시각 변화를 봅니다.

**② 확인한 결과**

| 대상 | 실제 기록·관측 | 알게 된 점 |
|---|---|---|
| **TBM WO-2024-01859** | 11/13 **07:40~09:35**, '계획 점검 수행', 부품교체 N | 승온 중 착수했고 완료 전까지 온도 상승이 포함 |
| **CBM WO-2024-01854** | 11/12 15:00 감지 → 11/13 **08:54~12:44**, WER·'마모량 초과', 부품교체 N | 감지는 관측 밖이지만 작업 전후는 240117 안에서 비교 가능. 구체 조정·수리 내용은 미기록 |
| **두 작업의 중첩** | **08:54~09:35** 겹침; 같은 LOT에 초반 공통 승온도 존재 | 두 작업의 개별 효과를 분리하기 어려움 |
| TC-Z3·OP-Z3·CP-Z4 | 온도 상승과 출력·CP의 국소 변화가 실제 시간축에 함께 있음 | 정비 시각과 변화의 위치는 확인할 수 있음. 겹쳤다는 사실만으로 원인은 확정되지 않음 |

**결론:** **감지 주변 변화와 실제 작업 전후 변화가 다른 질문**임을 확인했습니다. 01854는 감지 그림만으로는 놓치던 실제 작업 비교 사례입니다.

**③ 후속 작업**

| 대상·비교군 | 다음 확인 | 필요한 기록 |
|---|---|---|
| 01859·01854 / SPCD 240117 | 작업별 점검·조정 내용, 실재가동·목표온도와 겹침 여부 | 작업 상세·운전상태·목표값 |
| TC-Z3·OP-Z3·CP-Z4 | 동일 운전상태의 작업 전후 평균·std를 비교 | 다음 그림 17-B; 충분한 연속 센서 |
| 추가 SPCD LOT | 작업 없는 같은 상태 구간과 비교 | 같은 강종·속도·두께·목표값의 대조 LOT |


In [ ]:
# 그림 17-B: 감지 직후가 아니라 착수 직전 10분과 완료 후 10분의 실제 관측을 비교합니다.
mnt_metrics=[('TC-Z3','평균'),('TC-Z3','std'),('OP-Z3','평균'),('OP-Z3','std'),('CP-Z4','평균'),('CP-Z4','std')]
mnt_work_order=['WO-2024-01859','WO-2024-01854']
mnt_fig,mnt_axes=plt.subplots(6,1,figsize=(17,15.9))
mnt_fig.subplots_adjust(left=.22,right=.975,top=.87,bottom=.045,hspace=.97)
mnt_fig.suptitle('그림 17-B. TBM·CBM 작업 전후 — 평균 수준과 std를 따로 비교',
                 x=.22,y=.992,ha='left',fontsize=15,fontweight='bold')
mnt_fig.text(.22,.953,'작업 전 = 착수 직전 10분 · 작업 후 = 완료 직후 10분 · 둘 다 SPCD 240117의 실제 원본 · ○ 전 / ● 후',
              fontsize=10,color='#6B7280')
mnt_fig.text(.22,.921,'조치 내용·겹친 작업·승온을 함께 읽습니다. 값이 달라졌다는 사실만으로 정비 효과를 확정하지 않습니다.',
              fontsize=9,color='#6B7280')
for ax,(col,metric) in zip(mnt_axes,mnt_metrics):
    rows=mnt_actual_pairs[mnt_actual_pairs['컬럼'].eq(col)].set_index('작업').loc[mnt_work_order]
    pre_name='작업 전 '+metric;post_name='완료 후 '+metric
    values=np.r_[rows[pre_name].to_numpy(),rows[post_name].to_numpy()]
    span=max(float(values.max()-values.min()),float(np.abs(values).max())*.08,1e-8)
    ax.set_xlim(values.min()-span*.45,values.max()+span*.45)
    precision=6 if col=='CP-Z4' else 3
    for j,(work,r) in enumerate(rows.iterrows()):
        pre,post=r[pre_name],r[post_name]
        ax.hlines(j,pre,post,color='#CBD2D9',lw=1.5,zorder=1)
        ax.scatter([pre],[j],facecolors='white',edgecolors='#7C8792',s=65,zorder=3)
        ax.scatter([post],[j],color='#263238',s=45,zorder=4)
        ax.annotate(f'{pre:.{precision}f}',(pre,j),xytext=(0,9),textcoords='offset points',
                    ha='center',va='bottom',color='#7C8792',fontsize=9)
        ax.annotate(f'{post:.{precision}f}',(post,j),xytext=(0,-9),textcoords='offset points',
                    ha='center',va='top',color='#263238',fontsize=9,fontweight='bold')
    labels=[f'{rows.loc[w,"유형"]} · {w[-5:]}\n'+('계획 점검 · 미교체' if w.endswith('01859') else '마모량 초과 · 상세 조치 미기록') for w in mnt_work_order]
    ax.set_yticks(range(len(rows)),labels,fontsize=9)
    ax.set_ylim(len(rows)-.45,-.55)
    ax.set_title(f'{col} {metric} · 작업 전 → 완료 후',loc='left',fontweight='bold',pad=23)
    ax.set_xlabel('온도 [℃]' if col.startswith('TC') else '제어출력 [%]' if col.startswith('OP') else 'CP-Z4 [무차원]')
    ax.grid(axis='x',alpha=.12);ax.spines[['top','right','left']].set_visible(False)
    ax.tick_params(axis='y',length=0)
mnt_fig.text(.22,.008,'std는 10분 원본의 표준편차입니다. 주기 위상·운전상태·두 작업의 중첩도 전후 값에 영향을 줄 수 있습니다.',
              fontsize=9,color='#6B7280')
save_and_show_figures()


#### 그림 17-B — 실제 작업 전후의 평균·std 결과

**① 읽는 순서:** 각 컬럼의 **평균 → std**를 따로 봅니다. ○는 착수 직전 10분, ●는 완료 직후 10분입니다. 두 창 모두 같은 **SPCD 240117**의 원본 관측입니다.

**② 확인한 결과**

| 작업·기록된 조치 | 컬럼 | 작업 전 → 완료 후 | 무엇을 알게 되었는가? |
|---|---|---|---|
| **01859 · TBM / 계획 점검, 미교체** | TC-Z3 평균 | **837.398 → 858.227℃ (+20.829℃)** | 작업 뒤 높은 온도이지만 작업 전은 승온 중. 점검 때문에 상승했다고 볼 수 없음 |
| 01859 · TBM | OP-Z3 평균·std | 평균 **75.196 → 76.416%**, std **7.860 → 13.226%** | 출력 평균은 비슷하지만 흔들림은 감소하지 않음. 'TBM 후 모든 신호 개선'이라는 해석은 맞지 않음 |
| 01859 · TBM | CP-Z4 평균·std | 평균 **0.449392 → 0.449673**, std **0.001553 → 0.002946** | CP 변동도 증가. 피크·상태 차이와 구분 필요 |
| **01854 · CBM / 마모량 초과, 미교체·상세 조치 미기록** | TC-Z3 평균·std | 평균 **859.686 → 860.089℃ (+0.404℃)**; std **2.573 → 0.512℃** | 온도 수준 변화는 작고, 완료 후 10분의 흔들림은 작아짐 |
| 01854 · CBM | OP-Z3 평균·std | 평균 **64.161 → 78.132% (+13.971%p)**; std **33.199 → 7.934%** | 완료 후 평균 출력은 높고 변동은 작아짐 |
| 01854 · CBM | CP-Z4 평균·std | 평균 **0.450338 → 0.447893**; std **0.005262 → 0.003041** | CP 수준과 변동성 모두 달라짐 |

**결론:** **CBM 01854 완료 후 10분 창에서 TC-Z3·OP-Z3·CP-Z4 std 감소를 관측**했습니다. **[그림 19-C의 창 길이 점검](#interval-review)**에서는 5·10분의 감소가 20분에서 증가로 바뀝니다. 따라서 **일시적 안정은 확인되지만 지속 개선의 근거는 부족**합니다. 작업 중첩·운전상태·진동 위상과 구체 조치의 미기록도 있어 **정비의 인과 효과로 입증한 것은 아닙니다.** TBM은 온도 상승과 출력·CP std 증가가 함께 있어 전후 변화를 한 가지 '개선' 점수로 합칠 수 없습니다.

**③ 후속 작업**

| 대상·비교군 | 다음 확인 | 필요한 기록 |
|---|---|---|
| 01854 전후 / TC-Z3·OP-Z3·CP-Z4 | 20분 창의 재변동은 그림 19-C에서 확인. 12:55:30~13:00:30 저출력과 목표값·모드 변경을 대조 | 긴 완료 후 관측·출력 목표·분위기 설정 |
| 01859·01854 | 실제 작업 범위와 계측 태그의 대응 관계 확인 | 정비 상세·작업 대상 부위·조정 내역 |
| 두 작업 / 동일 상태 SPCD LOT | 비정비 구간의 같은 길이 창에서도 비슷한 std 차이가 나는지 | 반복 LOT·동일 상태 대조군. 현재 SPCD n=1 |


In [ ]:
# 실제 관측과 작업 구간의 색을 통일합니다. 보라/빨강 강종 막대 대신 강종을 글자로 표시합니다.
s17_gap_specs=[
    ('WO-2024-00717',240044,'BM · 동작 불량 · 부품교체 없음 · 상세 조치 미기록'),
    ('WO-2024-01094',240069,'BM · 절연 저하 · O링 세트 QTY 3 · 예방 교체'),
    ('WO-2024-01859',240117,'TBM · 계획 점검 · 부품교체 없음'),
    ('WO-2024-01854',240117,'CBM · 마모량 초과 · 부품교체 없음 · 상세 조치 미기록'),
    ('WO-2024-02105',240133,'CBM · 마모량 초과 · 가스 분석계 셀 QTY 3 · 예방 교체')]
s17_gap_fig=plt.figure(figsize=(20,13))
s17_gap_grid=s17_gap_fig.add_gridspec(10,1,height_ratios=[.72,1.00]*5,
    left=.09,right=.975,top=.84,bottom=.12,hspace=.27)
s17_gap_headers=[];s17_gap_axes=[];s17_gap_evidence=[]
for s17_j,(s17_wo,s17_lot,s17_action) in enumerate(s17_gap_specs):
    s17_work=cal[cal.WO_NO.eq(s17_wo)].iloc[0]
    s17_raw=temp[temp.LOT_NO.eq(s17_lot)]
    s17_reference=s17_work.DETC_DT
    s17_sensor_a=(lot_map.loc[s17_lot,'start']-s17_reference).total_seconds()/3600
    s17_sensor_b=(lot_map.loc[s17_lot,'end']-s17_reference).total_seconds()/3600
    s17_job_a=(s17_work.STRT_DT-s17_reference).total_seconds()/3600
    s17_job_b=(s17_work.END_DT-s17_reference).total_seconds()/3600
    s17_pre10=s17_raw[s17_raw.MEAS_DT.ge(s17_work.STRT_DT-pd.Timedelta(minutes=10))
                       &s17_raw.MEAS_DT.lt(s17_work.STRT_DT)]
    s17_post10=s17_raw[s17_raw.MEAS_DT.ge(s17_work.END_DT)
                        &s17_raw.MEAS_DT.lt(s17_work.END_DT+pd.Timedelta(minutes=10))]
    s17_pre_full=bool(lot_map.loc[s17_lot,'start']<=s17_work.STRT_DT-pd.Timedelta(minutes=10)
                     and lot_map.loc[s17_lot,'end']>=s17_work.STRT_DT)
    s17_post_full=bool(lot_map.loc[s17_lot,'start']<=s17_work.END_DT
                      and lot_map.loc[s17_lot,'end']>=s17_work.END_DT+pd.Timedelta(minutes=10))
    s17_detect_present=bool(lot_map.loc[s17_lot,'start']<=s17_reference<=lot_map.loc[s17_lot,'end'])
    s17_gap_evidence.append({'작업':s17_wo,'LOT':s17_lot,'강종':lot_map.loc[s17_lot,'grade'],
        '감지 시각 관측':s17_detect_present,'착수 전 10분 범위 관측':s17_pre_full,
        '완료 후 10분 범위 관측':s17_post_full,'착수 전 원본행':len(s17_pre10),
        '완료 후 원본행':len(s17_post10)})
    s17_head=s17_gap_fig.add_subplot(s17_gap_grid[2*s17_j]);s17_head.set_axis_off()
    s17_ax=s17_gap_fig.add_subplot(s17_gap_grid[2*s17_j+1],sharex=s17_gap_axes[0] if s17_gap_axes else None)
    s17_gap_headers.append(s17_head);s17_gap_axes.append(s17_ax)
    for _,s17_segment in s17_raw.groupby('SEG'):
        s17_a=(s17_segment.MEAS_DT.min()-s17_reference).total_seconds()/3600
        s17_b=(s17_segment.MEAS_DT.max()-s17_reference).total_seconds()/3600
        s17_ax.hlines(1,s17_a,s17_b,color='#7E8790',lw=8,zorder=2)
    s17_ax.hlines(0,s17_job_a,s17_job_b,color='black',lw=7,zorder=2)
    s17_ax.scatter([s17_job_a],[0],facecolors='white',edgecolors='black',s=45,zorder=4)
    s17_ax.scatter([s17_job_b],[0],marker='s',color='black',s=40,zorder=4)
    s17_ax.axvline(0,color='#B4232F',ls='--',lw=1.2)
    s17_ax.set_yticks([1,0],['센서 관측','실제 작업'],fontsize=10)
    s17_ax.set_ylim(-.5,1.5);s17_ax.set_xlim(-18,24)
    s17_ax.grid(axis='x',alpha=.14);s17_ax.spines[['top','right','left']].set_visible(False)
    s17_ax.set_xticks(np.arange(-18,25,6))
    if s17_j<4:s17_ax.tick_params(axis='x',labelbottom=False)
    s17_head.text(0,.85,
        f"{lot_map.loc[s17_lot,'grade']} LOT {s17_lot} / {s17_wo} / {s17_work.MNT_TYPE}",
        ha='left',va='center',fontsize=12.5,fontweight='bold')
    s17_head.text(0,.49,
        f"감지 {s17_work.DETC_DT:%m/%d %H:%M} · 착수 {s17_work.STRT_DT:%m/%d %H:%M} · 완료 {s17_work.END_DT:%m/%d %H:%M}",
        ha='left',va='center',fontsize=10,color='#66717D')
    s17_head.text(0,.13,s17_action,ha='left',va='center',fontsize=9.6,color='#66717D')
    if s17_pre_full and s17_post_full:
        s17_status='착수 전·완료 후 비교 가능 → 그림 17-A·B / 19-C'
    elif s17_detect_present:
        s17_status='감지 주변은 관측 있음 · 완료 후 없음 → 추가 센서 필요'
    else:
        s17_status=f'LOT 종료 후 {abs(s17_sensor_b):.1f}시간에 감지 · 완료 후 없음'
    s17_head.text(1,.85,s17_status,ha='right',va='center',fontsize=10.5,fontweight='bold',color='#B4232F')
s17_gap_evidence=pd.DataFrame(s17_gap_evidence)
s17_gap_axes[-1].set_xlabel('각 작업의 기록된 감지 시각 기준 [시간] · 0 = 감지 · 음수는 전 / 양수는 후',fontsize=11,labelpad=10)
s17_gap_fig.suptitle('그림 17-C. 어떤 LOT에서 정비 전후를 연결할 수 있는가?',
    x=.09,y=.984,ha='left',fontsize=17,fontweight='bold')
s17_gap_fig.text(.09,.944,
    '확인 결과: 240117의 TBM·CBM은 실제 작업 전후 관측 있음 / 240044·240069·240133은 완료 후 관측 없음',
    fontsize=11.5,color='#425466')
s17_gap_fig.legend(handles=[
    Line2D([],[],color='#7E8790',lw=5,label='회색 가로선: 해당 LOT 센서 관측 기간'),
    Line2D([],[],color='black',lw=5,label='검정 가로선: 정비 착수~완료'),
    Line2D([],[],color='#B4232F',ls='--',label='빨간 세로 점선: 감지 시각 0'),
    Line2D([],[],ls='',marker='o',mfc='white',mec='black',label='착수'),
    Line2D([],[],ls='',marker='s',color='black',label='완료')],
    loc='upper left',bbox_to_anchor=(.085,.915),ncol=3,frameon=False,fontsize=10.5,handlelength=2.4)
s17_gap_fig.text(.09,.042,
    '연결 목적: 센서 변화 → 기록된 작업·부품 조치 → 실제 완료 후 변화 · 관측이 없으면 추가 수집 구간을 지정 · 공백을 설비 정지·센서 이상으로 해석하지 않음',
    fontsize=10,color='#66717D')
s17_gap_fig.text(.09,.018,
    'SPHC 240061·240091도 같은 강종 비교에 포함합니다. 개별 BM 인접 사례와 역할이 달라 위 비교 안내표 및 그림 10-G·H에서 확인합니다.',
    fontsize=10,color='#66717D')
save_and_show_figures()


#### 그림 17-C — 공백을 알아서 무엇을 연결할 수 있는가?

**이 그림의 질문은 “정비로 센서가 달라졌는가를 실제 관측으로 비교할 수 있는가?”입니다.** 센서 관측 시각과 실제 작업 시각을 대조해서, **지금 계산 가능한 비교와 추가 수집이 필요한 비교**를 구분합니다.

| 표시 | 의미 |
|---|---|
| **회색 가로선** | 해당 LOT의 실제 센서 관측 기간. 강종은 행 제목의 글자로 표시 |
| **검정 가로선 / ○ / ■** | 기록된 정비 착수~완료 / ○ 착수 / ■ 완료 |
| **빨간 세로 점선 · 0** | 각 작업의 기록된 감지 시각. 고장 확률이나 상태를 나타내는 색이 아님 |
| **관측 없는 시간** | 이 LOT의 데이터로 변화량을 계산할 수 없는 시간. 실제 정지·수집 중단의 원인은 별도 로그가 필요 |

**확인한 결과 — 어떤 연결까지 가능한가?**

| LOT · 강종 / 작업 | 관측·기록 근거 | 연결 가능한 분석 | 다음에 연결할 것 |
|---|---|---|---|
| **240044 · SPHC / BM 00717** | LOT 종료 **3.7시간 뒤 감지**, 동작 불량. 부품교체 없음·상세 조치 미기록 | LOT의 초반 OP-Z3 결측과 BM의 시간 위치. 두 현상의 인과 관계는 미확인 | **240061·240069·240091**의 결측·같은 상태 대조, 작업 대상·통신·정비 로그, 완료 후 관측 |
| **240069 · SPHC / BM 01094** | LOT 종료 **2.3시간 뒤 감지**, 절연 저하. **SEL-009 O링 세트 QTY 3·예방 교체** 기록 | LOT 안 CP 피크·OP 저출력 → 이후 BM·교체 기록의 시간순 연결 | **240044·240061·240091**과 피크·변동 비교. CP·전기 증상·O링 조치의 실제 부위 관계 및 완료 후 관측 |
| **240117 · SPCD / TBM 01859** | 감지·착수·완료가 LOT 안. 착수 전·완료 후 각각 10분 관측 가능 | **같은 LOT의 실제 작업 전후** TC-Z3·OP-Z3·CP-Z4 평균·std 계산 | **그림 17-A·B·19-C**. 승온·다른 작업 중첩과 비교 창 영향을 분리 |
| **240117 · SPCD / CBM 01854** | 감지는 전날로 LOT 밖이지만 **실제 착수·완료 전후는 LOT 안** | 감지 관측이 없어도 실제 작업 전후는 비교 가능. 후 10분의 std 감소와 20분 창의 재변동 확인 | **그림 17-A·B·19-C**, 작업 상세·동일 상태의 추가 SPCD LOT |
| **240133 · SPCE / CBM 02105** | 감지는 LOT 안, 착수·완료는 관측 종료 이후. **INS-006 가스 분석계 셀 QTY 3·예방 교체** | 감지 주변 CP-Z4·TC-Z3·OP-Z3 변화와 교체 기록 연결. 교체 후 변화는 현재 계산 불가 | **240037·240121** 같은 강종 CP 비교, INS-006→CP-Z4 태그 대응, 완료·교정 후 연속 계측 |

| 대상 | 무엇을 확인하는가? | 비교 대상·구간 |
|---|---|---|
| **240044 · SPHC** | OP-Z3 결측과 이후 BM의 시간 관계 | 240061·240069·240091의 수집 시작 구간 |
| **240061 · SPHC** | 계획수리 후 결측·승온과 안정 이후 변동 | 240044·240069·240091, 승온 전후를 분리 |
| **240069 · SPHC** | CP 피크·OP 저출력과 이후 BM의 시간 관계 | 240044·240061·240091의 같은 운전 상태 |
| **240091 · SPHC** | 상대 흐름과 TC-Z7·Z8의 지속 차이 | 나머지 SPHC 세 LOT. 모든 센서의 정상 기준으로 지정하지 않음 |
| **240117 · SPCD** | TBM 01859·CBM 01854의 **실제 작업 전후 변화** | 같은 LOT에서 각 작업의 착수 전 10분 ↔ 완료 후 10분 |
| **240133 · SPCE** | CBM 02105의 **감지 주변 변화** | 감지 전후 각 10분, 같은 강종 240037·240121과 추가 대조 |

**SPHC의 나머지 두 LOT도 봅니다.**

| LOT | 왜 함께 보는가? | 비교 대상·컬럼 |
|---|---|---|
| **240061 · SPHC** | 계획수리 후 첫 관측. 초반 OP-Z3 결측 **179행**, 공통 승온 **0~73.5분**을 분리할 사례 | **240044·240069·240091**. TC-Z3↔OP-Z3, 다른 가열 TC, 수집 로그. 그림 **10-D·11·12·15·19-A** |
| **240091 · SPHC** | 같은 강종의 상대 흐름을 대조. TC-Z7·TC-Z8에는 지속 수준 차이도 있어 모든 컬럼의 정상 기준은 아님 | **240044·240061·240069**. TC-Z3↔OP-Z3 및 별도의 TC-Z7·Z8↔OP-Z7·Z8. 그림 **10-E-추가·10-G·H** |

**TC-Z3을 보는 이유:** 앞선 14 LOT 비교에서 단기 변동이 큰 온도였고, OP-Z3는 같은 존의 출력이므로 대표 짝으로 선택했습니다. 다른 TC-Z4~Z6는 공통 승온을 대조합니다. 이것은 분석할 채널을 고른 근거이며, BM·CBM의 수리 부위가 Z3라는 근거는 아닙니다. 실제 작업 대상·태그 기록이 있어야 해당 존의 정비 효과로 연결할 수 있습니다.

**이 결과의 활용:** **240117은 실제 작업 전후 비교를 이어갈 대상**, **240133은 감지 주변 분석과 교체 후 데이터 확보 대상**, **SPHC 네 LOT는 결측·승온·CP 피크·상대 변동을 나누는 동일 강종 비교군**입니다. TBM·CBM·BM을 서로 다른 강종에서 한 건씩 골라 효과 순위로 비교하지 않습니다.


<a id="parts"></a>
## 12. 부품교체 — 표준수명과 조달 기간으로 확보 준비를 판단

**분석 질문:** 조달을 기다리는 동안 소모될 운전시간을 부품의 남은 운전시간과 비교하려면 무엇이 필요한가?

**두 기준의 단위:** 마스터 **STD_LIFE_H는 표준수명 기준 시간**, **LEAD_TIME_D는 조달 기간 기준 일수**입니다. 그림 18에서 나란히 보고, 하루 운전시간을 가정한 예시로 연결합니다.

**관계식:** 하루 운전시간 d, 리드타임 L이면 조달 중 운전시간은 **d × L**입니다. 표준수명 H와 현재 부품 누적 운전시간 A를 확보했을 때 **H - A**를 이 값 및 준비 여유와 대조합니다. 표준수명은 개체별 고장 시각이나 실제 잔여수명 예측값으로 쓰지 않습니다.

**그림 읽는 법:** 왼쪽 표준수명[운전시간] → 가운데 리드타임[일] → 오른쪽 **하루 24시간 운전 예시의 조달 기간/표준수명 비율**을 읽습니다. 예시에는 안전 여유·재고를 반영하지 않습니다. 아래 교체 이력 날짜는 실제 교체시각 대신 **정비 완료시각**을 사용합니다.

In [ ]:
# 마스터 표준수명과 조달 리드타임을 직접 비교합니다. 운전시간 가정은 실제 가동 기록과 구분합니다.
part_meta=cal_parts[['PART_CD','PART_NM','STD_LIFE_H','LEAD_TIME_D']].drop_duplicates().sort_values('PART_CD')
reason_color={'손상':'#c0392b','수명도래':'#e67e22','예방':'#2874a6'}
part_summary=cal_parts.groupby(['PART_CD','PART_NM']).agg(교체내역행=('WO_NO','size'),
    작업수=('WO_NO','nunique'),기록수량합=('QTY','sum'),표준수명시간=('STD_LIFE_H','first'),
    리드타임일=('LEAD_TIME_D','first')).reset_index()
reason_counts=pd.crosstab([cal_parts.PART_CD,cal_parts.PART_NM],cal_parts.CHG_RSN).reset_index()
part_summary=part_summary.merge(reason_counts,on=['PART_CD','PART_NM'],validate='one_to_one')
intervals=cal_parts[['PART_CD','PART_NM','WO_NO','END_DT']].drop_duplicates().sort_values(['PART_CD','END_DT'])
intervals['완료기록간격 [일]']=intervals.groupby('PART_CD').END_DT.diff().dt.total_seconds()/86400
PART_EXAMPLE_HOURS_PER_DAY=24
part_planning=part_meta.sort_values('LEAD_TIME_D',ascending=False).copy()
part_planning['조달 중 운전시간 예시 [h]']=part_planning.LEAD_TIME_D*PART_EXAMPLE_HOURS_PER_DAY
part_planning['표준수명 대비 조달 중 운전 [%]']=100*part_planning['조달 중 운전시간 예시 [h]']/part_planning.STD_LIFE_H
part_planning['확보 검토 누적시간 예시 [h]']=part_planning.STD_LIFE_H-part_planning['조달 중 운전시간 예시 [h]']
part_life_fig=plt.figure(figsize=(20,7))
part_life_grid=part_life_fig.add_gridspec(2,3,height_ratios=[.85,3.7],
    left=.22,right=.975,top=.77,bottom=.18,wspace=.28,hspace=.10)
part_life_axes=[];part_life_headers=[]
part_life_specs=[
 ('STD_LIFE_H · 표준수명','운전시간 기준 · 실제 고장시각 아님','STD_LIFE_H','표준수명 기준 [운전 h]','h'),
 ('LEAD_TIME_D · 조달 리드타임','달력 일수 기준 · 현재 재고 미반영','LEAD_TIME_D','조달 기간 기준 [일]','일'),
 ('두 값의 관계 · 24h/일 예시','조달 중 운전시간 ÷ 표준수명','표준수명 대비 조달 중 운전 [%]','표준수명 대비 조달 중 운전 [%]','%')]
for j,(title,desc,col,label,unit) in enumerate(part_life_specs):
    head=part_life_fig.add_subplot(part_life_grid[0,j]);head.set_axis_off()
    ax=part_life_fig.add_subplot(part_life_grid[1,j],sharey=part_life_axes[0] if part_life_axes else None)
    part_life_headers.append(head);part_life_axes.append(ax)
    head.text(0,.77,title,fontsize=12,fontweight='bold',va='center')
    head.text(0,.26,desc,fontsize=10,color='#66717D',va='center')
    vals=part_planning[col].to_numpy()
    colors_life=['#B4232F' if j>0 and code=='INS-006' else '#8C969F' for code in part_planning.PART_CD]
    ax.barh(range(len(vals)),vals,height=.42,color=colors_life,alpha=.9)
    ax.set_xlim(0,max(vals)*1.36);ax.set_ylim(len(vals)-.4,-.6)
    for k,(v,r) in enumerate(zip(vals,part_planning.itertuples())):
        value=f'{v:,.0f}{unit}' if unit!='%' else f'{v:.1f}% · {r.LEAD_TIME_D*PART_EXAMPLE_HOURS_PER_DAY:.0f}h'
        ax.text(v+max(vals)*.035,k,value,va='center',ha='left',fontsize=11,fontweight='bold',color=colors_life[k] if j>0 else '#263238')
    ax.set_yticks(range(len(vals)))
    if j==0:ax.set_yticklabels([f'{r.PART_CD} · {r.PART_NM}' for r in part_planning.itertuples()],fontsize=11)
    else:ax.tick_params(axis='y',labelleft=False,left=False)
    ax.set_xlabel(label,fontsize=11,labelpad=10);ax.grid(axis='x',alpha=.13)
    ax.spines[['top','right','left']].set_visible(False)
part_life_fig.suptitle('그림 18. 표준수명·조달 리드타임 — 발주 준비 시점을 판단할 기준',
    x=.07,y=.983,ha='left',fontsize=17,fontweight='bold')
part_life_fig.text(.07,.918,'확인 목적: 조달을 기다리는 동안 사용될 운전시간을 남은 기준 운전시간과 대조',fontsize=11,color='#425466')
part_life_fig.text(.07,.882,'붉은 막대: 조달 기간·비율이 가장 큰 가스 분석계 셀 · 회색: 다른 마스터 값 · 실제 발주 우선순위는 누적 운전시간·상태·재고를 확인한 뒤 판단',fontsize=10.5,color='#66717D')
part_life_fig.text(.07,.078,'24h/일 예시: 조달 중 운전시간 = 하루 운전시간 × 리드타임 · 확보 검토 누적시간 = 표준수명 - 조달 중 운전시간 - 준비 여유',fontsize=10.5,color='#425466')
part_life_fig.text(.07,.032,'현재 개체별 누적 운전시간·가동률·재고가 없어 실제 잔여수명·발주일을 계산하지 않음 · 교체 완료 기록의 달력 간격을 사용수명으로 대체하지 않음',fontsize=10,color='#66717D')
save_and_show_figures()
display(part_summary)

#### 그림 18 — 표준수명과 리드타임을 어떻게 연결하는가?

**핵심:** 리드타임이 길다는 사실만으로 교체가 급하다고 판단하지 않습니다. **현재 부품의 누적 운전시간**을 알아야 조달을 기다릴 여유가 있는지 계산할 수 있습니다.

| 부품 | 표준수명 기준 | 리드타임 | 조달 중 운전시간 · 24h/일 예시 | 표준수명 대비 비율 | 확보 검토 누적시간 예시 |
|---|---:|---:|---:|---:|---:|
| **INS-006 · 가스 분석계 셀** | **9,500h** | **31일** | **744h** | **7.8%** | **8,756h** |
| **SEL-006 · 오일씰** | **8,000h** | **16일** | **384h** | **4.8%** | **7,616h** |
| **SEL-009 · O링 세트** | **4,500h** | **5일** | **120h** | **2.7%** | **4,380h** |

**예시 해석:** 가스 분석계 셀을 하루 24시간 사용하고 재고·준비 여유를 반영하지 않으면, 조달 31일 동안 **744h**를 사용합니다. 표준수명 기준 **9,500h에 도달하기 전에 확보하려면 누적 8,756h 시점부터 확보를 검토**하는 계산 예시입니다. 실제 장착 부품이 지금 몇 시간을 사용했는지는 자료에 없습니다. 하루 12시간 운전이면 조달 중 운전시간도 절반인 **372h**로 달라집니다.

**판단식:** 현재 기준 잔여 운전시간 **표준수명 - 개체 누적 운전시간**을 **하루 운전시간 × 리드타임 + 준비 여유**와 비교합니다. 이 식은 마스터 기준의 보전 준비 계산이며 실제 고장시간·예측 잔여수명은 아닙니다.

**이번에 알게 된 점:** 가스 분석계 셀은 세 부품 중 조달 기간도 가장 길고, 같은 24h/일 가정에서 표준수명 대비 조달 중 운전 비율도 가장 큽니다. **31일 조달 조건을 미리 확인할 대상**입니다. 실제 발주 순서는 현재 사용량·상태·재고가 결정합니다.

| 후속 작업 | 필요한 자료 | 확인하려는 것 |
|---|---|---|
| **현재 장착 부품별 사용량** | 개체 ID·장착시각·실가동 적산시간 | 표준수명 기준 잔여 운전시간과 조달 기간을 대조 |
| **조달·재고 준비** | 재고·입고 예정·실제 리드타임·일일 운전시간·준비 여유 | 지금 확보 준비가 필요한 부품과 확보 예상일 |
| **센서·정비 연결** | INS-006 장착·교정·CP-Z4 태그 대응, SEL-006·009 작업 부위·압력·누설 기록 | 센서 변화와 해당 부품 상태가 실제로 연결되는가? |

In [ ]:
# 교체 이력은 표준수명·조달 비교 아래에서 실제 작업·부품 연결의 근거로 보여줍니다.
part_positions=dict(zip(part_planning.PART_CD,range(len(part_planning))))
part_history_fig=plt.figure(figsize=(20,6.6))
part_history_ax=part_history_fig.add_axes([.22,.20,.755,.53])
for r in plans.itertuples():
    part_history_ax.axvspan(r.STRT_DT,r.end_exclusive,color='#D6EAF8',alpha=.6,zorder=0)
for reason,d in cal_parts.groupby('CHG_RSN'):
    part_history_ax.scatter(d.END_DT,d.PART_CD.map(part_positions),s=d.QTY*55,
        color=reason_color.get(reason,'gray'),label=reason,alpha=.9,zorder=3)
for r in cal_parts.itertuples():
    part_history_ax.annotate(f'{r.WO_NO[-5:]} · QTY {r.QTY}',(r.END_DT,part_positions[r.PART_CD]),
        xytext=(0,13),textcoords='offset points',fontsize=9,ha='center',color='#425466')
part_history_ax.set_yticks(range(len(part_planning)),
    [f'{r.PART_CD} · {r.PART_NM}' for r in part_planning.itertuples()],fontsize=11)
part_history_ax.set_ylim(len(part_planning)-.5,-.6)
part_history_ax.set_xlim(pd.Timestamp('2024-01-01'),pd.Timestamp('2024-12-31 23:59:59'))
part_history_ax.xaxis.set_major_locator(mdates.MonthLocator())
part_history_ax.xaxis.set_major_formatter(mdates.DateFormatter('%m월'))
part_history_ax.set_xlabel('정비 완료 기록 날짜 [2024년] · 실제 교체시각의 대리값',fontsize=11,labelpad=12)
part_history_ax.grid(axis='x',alpha=.14);part_history_ax.spines[['top','right']].set_visible(False)
handles=[Line2D([],[],ls='',marker='o',color=reason_color[k],label=k) for k in ['손상','수명도래','예방']]
handles.append(Patch(fc='#D6EAF8',label='CR1 계획수리'))
part_history_fig.legend(handles=handles,loc='upper left',bbox_to_anchor=(.065,.875),ncol=4,frameon=False,fontsize=11)
part_history_fig.suptitle('그림 18-이력. 어떤 부품을 어떤 작업·사유로 교체했는가?',
    x=.07,y=.981,ha='left',fontsize=17,fontweight='bold')
part_history_fig.text(.07,.908,f'CAL01 {len(cal_parts)}교체행 / {cal_parts.WO_NO.nunique()}작업 · 점 크기 = 기록 수량 QTY · 글자 = 작업번호 끝 5자리·QTY',
    fontsize=11,color='#425466')
part_history_fig.text(.07,.087,'완료 기록 간 달력 간격은 개체의 실제 사용수명이 아님 · 점 하나와 표준수명 1회 소모를 대응시키지 않음',fontsize=10.5,color='#66717D')
save_and_show_figures()
display(intervals.dropna(subset=['완료기록간격 [일]']).round(2))

#### 그림 18-이력 — 교체 기록을 수명·발주 준비와 연결하기 위한 조건

| 기록에서 확인한 것 | 현재 해석할 수 있는 범위 | 다음에 필요한 연결 |
|---|---|---|
| **9교체행 / 8작업 / 3부품종** | 작업번호·부품·수량·교체 사유·완료시각을 연결 | 장착 개체·실제 교체/교정 시각 |
| **INS-006 가스 분석계 셀** | 손상·수명도래·예방 각 1행 | **240133 / CBM 02105**의 교체 기록과 CP-Z4 태그·완료 후 관측 대조 |
| **SEL-006 오일씰 / SEL-009 O링** | 씰 손상 2행, O링 손상·수명도래·예방 각 1행 | 실제 작업 부위·누설·압력·실가동시간 |
| **같은 부품의 완료 기록 간격** | 달력상 기록 간격 | 개체가 같다는 기록과 운전시간이 있어야 사용수명 비교 가능 |

**후속 작업:** 작업번호 → 부품 개체 → 장착·누적 운전시간 → 센서 태그 → 완료 후 관측을 연결합니다. 그 뒤 그림 18의 조달 기간과 대조해 확보 준비 시점을 계산합니다.

[↑ 목차](#toc)

<a id="conclusion"></a>
## 13. 종합 해석 — 시각화로 무엇을 알게 되었는가?

| 분석 질문 | 시각화 근거 | 이번에 알 수 있는 것 | 판단이 남은 부분 |
|---|---|---|---|
| 연속 시계열인가? | 그림 1 | 14 LOT의 일부 시간만 관측. 정비·이벤트 대부분의 실제 전후 비교 불가 | 공백 원인·수집 및 가동 로그 |
| 수집 품질은 어떤가? | 그림 2·3 | 1초가 대표 간격, OP-Z3 결측은 두 LOT 초반 집중 | 2초 간격의 수집 사양·개별 채널 결측 원인 |
| 어디에 변화가 집중되는가? | 그림 4~9 | TC-Z3·OP-Z3 단기 변화가 상대적으로 큼. 고온 공정의 시작 상승·끝 하강 구분 필요 | 목표값·운전상태·소재규격 |
| 강종을 맞춰도 LOT 차이가 남는가? | 그림 10-A~G | 자기 제외 Δ, 비교 전후 신호군·진행률 지속성 지도·수리 위치로 후보 좁히기 | n=2 참고·n=1 보류, 운전조건과 수리 효과 미확정 |
| 사례 LOT의 낮은 온도는 무엇인가? | 그림 11 | TC-Z3만이 아니라 가열·균열존의 공통 상승이 포함됨 | 실제 재가동·운전조건·목표값 |
| 반복 온도 하강에서 무엇을 점검할 것인가? | 그림 13·13-근거 | TC-OUT1·TC-OUT2의 반복 후보; TC-Z9 상한 피크는 주기 판단 보류 | 같은 SPHC 240069↔240091의 주기·변동폭·운전조건 비교; 정비 효과 미확인 |
| 온도와 출력은 함께 움직이는가? | 그림 14 | 동일 존의 음의 상관, 수준과 차분의 시차 차이 검토 가능 | 인과 방향·실제 제어 지연·정상 범위 |
| 승온 이후 어느 TC-Z3 구간을 확대할 것인가? | 그림 15 | 0~73.5분 승온과 상대차 33개 관측값·연속 4구간을 분리. 129~135.5분 우선 점검 | 같은 시각 OP-Z3·목표값, 다른 SPHC의 승온 제외 구간 |
| 정비와 부품을 센서로 설명할 수 있는가? | 그림 16~18 | TBM 01859·CBM 01854 실제 착수 전/완료 후 비교; CBM std는 5·10분 창 감소, 20분 창 증가; BM·분석계 교체 후 센서 없음 | 승온·작업 중첩·구체 조치 미기록으로 인과 효과 미확정 |

### 시계열 해석의 큰 흐름

**관측 범위·수집 품질 확인 → 전체 후보 → 자기 제외 동일 강종 비교 → LOT 차이·시간 지속성 확인 → 계획수리 위치 연계 → 반복 주기·제어 관계·정비·부품 점검**입니다. 평균이 다르다는 이유만으로 이상을 붙이지 않고, 변화가 언제·어느 공정·어느 채널에서 나타났는지 실제 시각으로 검토합니다.

### 후속 작업 — 이미 확인한 결과를 무엇으로 이어갈 것인가?

**노트북에서 이미 한 비교와 추가 자료가 필요한 점검을 구분합니다.** 같은 강종 차이·진행률 지속성·수리 위치는 위 그림에서 확인했고, 아래 원인 점검은 운전·수집 기록이 있어야 수행할 수 있습니다.

| 순서 | 대상·비교군 | 확인할 컬럼·시점 | 후속 작업 | 확인 후 구분할 것 |
|---|---|---|---|---|
| **1. 수집 품질** | **240044 ↔ 240061**; 대조 240069·240091 (SPHC) | OP-Z3 결측 0~81초 / 0~183초; 같은 시각 TC-Z3·OP-Z4·Z5·Z6 | 태그·PLC·수집·저장 로그로 최초 유효화 조건 대조 | 채널 수집 문제와 실제 제어출력 상태 |
| **2. 운전 전환** | **240061·240117 시작 / 240058·240114 끝** | TC-Z3·Z4·Z5·Z6 및 OP-Z1~Z8의 동시 변화 | 실제 정지·재가동·출력 목표를 대조; 강종이 다른 LOT는 전환 형태만 참고 | 공통 가동 상태 변화와 승온 후 남는 개별 변동 |
| **3. CP 상세** | **240069 ↔ 240044·240061·240091** (SPHC) | CP-Z4 574.5분 피크·std(5분); 동시각 TC-Z3·OP-Z3 | CP 원자료·분위기 설정·분석계 교정·통신 이력 확인 | 일시적 피크·제어 변화·계측 문제 |
| **4. 지속 온도** | **240121 ↔ 240037·240133** (SPCE); **240091 ↔ 나머지 SPHC** | TC-Z7·TC-Z8 각각; 중간 20~80%의 지속 상승 | 존별 목표온도·속도·두께를 맞춰 재비교 | 운전조건으로 설명되는 수준과 설명되지 않는 LOT 차이 |
| **5. 반복 조건** | 240024 ↔ 240108 (SPFH590); 240038 ↔ 240114 (SPCEN) | TC-Z7·TC-Z8·TC-OUT1·TC-OUT2; TC-Z3~Z6·OP-Z3 변동폭 | 동일 조건의 추가 LOT 확보; n=2 차이는 참고로 유지 | 강종의 공통 조건과 개별 LOT 차이 |
| **6. 보류·대조군** | 240001 (SGCC), 240058 (DP590), 240117 (SPCD); 240037·240133 (SPCE) | n=1 표본 부족; SPCE 기준 구성 영향 | 같은 강종 추가 표본 확보; 240037 ↔ 240133 직접 대조 | 비교 불가와 기준 미만의 이유 |
| **7. 정비·부품 연결** | SPHC 240044 → 240061·240069; WO-2024-01859·01854·02105; INS-006 등 | 감지·착수·완료, 실제 수리 후 센서, 부품-태그 관계 | 01859·01854의 작업 중첩·승온을 구분하고, 02105 분석계 교체·BM 완료 후 연속 계측 확보 | 수리 인근 발생 위치와 실제 수리 후 변화 |

**구간 선정 근거는 [아래 점검표·그림 19-A~C](#interval-review)에 반영했습니다.** `LOT·강종 / 비교 LOT / 컬럼 / 실제 시각 / 관측 근거 / 판단 범위`를 연결하고, 피크 제거와 5·10·20분 정비 창의 민감도를 계산했습니다. **다음 작업은 운전조건·수집·정비 로그로 원인을 대조하는 것입니다.** 우선 **240044·240061의 결측 원인**, **240069의 CP 피크**, **240121·240091의 TC-Z7·TC-Z8 목표값**을 확인합니다.


**현재 결과의 용도:** 데이터 구조와 공정 패턴을 이해하고 현장 점검 대상을 좁히는 탐색 분석입니다. 실제 고장·품질·잔여수명 정답이 없어 고장 예측 정확도, 미탐률, RUL 성능은 산출하지 않았습니다. 자동 경보의 기준은 정상 범위와 운영 상태가 확인된 뒤 별도 평가가 필요합니다.

<a id="lot-selection-final"></a>
### 마지막 LOT 선정 보고 — 관측 근거와 채택 이유

**주 상세 사례는 240061(SPHC)입니다.** 결측·승온·저출력·안정 구간의 온도 변동을 한 시간축에서 함께 확인할 수 있습니다. 전체 14 LOT의 검토 결과를 바탕으로 같은 강종 비교 LOT와 다른 강종의 참고 사례도 함께 남깁니다.

**집계 기준:** 운전 전환은 TC-Z3~TC-Z6 평균이 처음 855℃ 이상이 된 시각부터 마지막 855℃ 이상 시각까지의 바깥 구간입니다. 안정 구간의 `TC-Z3 차분 std`는 두 관측이 모두 이 구간 안이고, 동일 LOT·연속구간에서 30초 간격인 변화량만 집계합니다. `OP-Z3 <1% [분]`은 30초 평균의 저출력 빈 수 × 0.5분입니다. 결측은 1초 원자료로 확인합니다.


In [ ]:
# 마지막 LOT 채택표의 근거를 원본과 30초 시계열에서 다시 계산합니다.
# 855℃는 앞선 05 노트북과 비교하기 위한 운전 전환 탐색 기준입니다.
# OP-Z3 <1% 시간은 해당 30초 빈 수 × 0.5분이며 실제 0% 유지시간과 구분합니다.
s13_ts = ts.copy()
s13_heat = s13_ts[HEAT_TC].mean(axis=1).where(s13_ts[HEAT_TC].notna().all(axis=1))
s13_above = s13_heat.ge(855)
s13_st_a = s13_ts.loc[s13_above].groupby('LOT_NO')['min'].min()
s13_st_b = s13_ts.loc[s13_above].groupby('LOT_NO')['min'].max()
s13_stable = s13_ts['min'].between(s13_ts.LOT_NO.map(s13_st_a), s13_ts.LOT_NO.map(s13_st_b))
s13_prev_stable = s13_stable.groupby([s13_ts.LOT_NO, s13_ts.seg]).shift().eq(True)
s13_step_ok = s13_ts.groupby(['LOT_NO','seg']).MEAS_DT.diff().dt.total_seconds().eq(30)
s13_change = s13_ts.groupby(['LOT_NO','seg'])[['TC-Z3','OP-Z3']].diff()
s13_valid_change = s13_step_ok & s13_stable & s13_prev_stable
s13_stable_std = s13_change.loc[s13_valid_change].groupby(s13_ts.LOT_NO[s13_valid_change]).std()
s13_grade_counts = lot_map.grade.value_counts()

# 정기수리 경계와 가장 가까운 앞/뒤 관측 LOT를 연결합니다.
# 종료일은 당일 전체를 포함하므로 END_INCL(다음날 00:00)을 후측 경계로 사용합니다.
s13_plan_position = {}
for _, plan in cr1_repair.sort_values('STRT_DT').iterrows():
    before = lot_map[lot_map.end.lt(plan.STRT_DT)].sort_values('end')
    after = lot_map[lot_map.start.ge(plan.END_INCL)].sort_values('start')
    if len(before):
        lot = int(before.index[-1])
        s13_plan_position[lot] = ('수리 전 마지막 관측', plan.STRT_DT, plan.END_DT,
                                 (plan.STRT_DT-before.end.iloc[-1]).total_seconds()/3600)
    if len(after):
        lot = int(after.index[0])
        s13_plan_position[lot] = ('수리 후 첫 관측', plan.STRT_DT, plan.END_DT,
                                 (after.start.iloc[0]-plan.END_INCL).total_seconds()/3600)

s13_rows = []
for lot, r in lot_map.sort_values('start').iterrows():
    d = s13_ts[s13_ts.LOT_NO.eq(lot)]
    raw = temp[temp.LOT_NO.eq(lot)]
    absent = raw[raw['OP-Z3'].isna()]
    next_maint = cal_maint[cal_maint.DETC_DT.gt(r.end)].head(1)
    last_bin = float(d['min'].max())
    start_transition = s13_st_a[lot] > d['min'].min()
    end_transition = s13_st_b[lot] < last_bin
    transition = ('초반 승온' if start_transition else '') + ('종료 하강' if end_transition else '')
    if start_transition:
        transition_window = f"0~{s13_st_a[lot]:.1f}분"
    elif end_transition:
        first_low = d.loc[d['min'].gt(s13_st_b[lot]), 'min'].min()
        transition_window = f"{first_low:.1f}~{last_bin:.1f}분"
    else:
        transition_window = '없음'
    relation = s13_plan_position.get(int(lot))
    corr_rows = s13_step_ok & s13_ts.LOT_NO.eq(lot)
    s13_rows.append({
        'LOT':int(lot), '강종':r.grade, '동일 강종 LOT수':int(s13_grade_counts[r.grade]),
        '시작':r.start, '종료':r.end, '관측 [분]':r.minutes,
        '운전 전환':transition or '없음', '전환 구간':transition_window,
        '안정 시작 [분]':s13_st_a[lot], '안정 종료 [분]':s13_st_b[lot],
        'OP-Z3 결측 [행]':len(absent),
        '결측 시작 [초]':absent.sec.min() if len(absent) else np.nan,
        '결측 끝 [초]':absent.sec.max() if len(absent) else np.nan,
        'OP-Z3 <1% [분]':int(d['OP-Z3'].lt(1).sum())*.5,
        'OP-Z3 =0% [분]':int(d['OP-Z3'].eq(0).sum())*.5,
        '안정 TC-Z3 차분 std [℃]':s13_stable_std.loc[lot,'TC-Z3'],
        'TC-Z3↔OP-Z3 차분 r':s13_change.loc[corr_rows,'TC-Z3'].corr(s13_change.loc[corr_rows,'OP-Z3']),
        '정기수리 위치':relation[0] if relation else '앞뒤 첫 관측 LOT 아님',
        '수리 경계 거리 [시간]':relation[3] if relation else np.nan,
        '다음 정비 감지':next_maint.DETC_DT.iloc[0] if len(next_maint) else pd.NaT,
        '끝→다음 감지 [시간]':(next_maint.DETC_DT.iloc[0]-r.end).total_seconds()/3600 if len(next_maint) else np.nan,
        '다음 정비':next_maint.MNT_TYPE.iloc[0] if len(next_maint) else '기록 없음',
        '고장모드':next_maint.FAIL_MODE.fillna('').iloc[0] if len(next_maint) else '',
        '정비 비고':next_maint.REMARK.fillna('').iloc[0] if len(next_maint) else '',
    })
s13_lot_evidence = pd.DataFrame(s13_rows).set_index('LOT')
s13_lot_evidence['안정 TC-Z3 차분 std 순위'] = s13_lot_evidence['안정 TC-Z3 차분 std [℃]'].rank(ascending=False, method='min').astype(int)

# 주 사례는 표본이 가장 많은 강종에서 결측·승온·저출력이 함께 관측된 LOT입니다.
# 정비 시점은 이 선별 뒤에 붙인 맥락이며 고장·수리 효과의 판정 기준이 아닙니다.
s13_focus_grade = s13_grade_counts.index[0]
s13_case_candidates = s13_lot_evidence[
    s13_lot_evidence['강종'].eq(s13_focus_grade)
    & s13_lot_evidence['운전 전환'].eq('초반 승온')
    & s13_lot_evidence['OP-Z3 결측 [행]'].gt(0)
    & s13_lot_evidence['OP-Z3 <1% [분]'].gt(0)
]
s13_case_lot = int(s13_case_candidates['안정 TC-Z3 차분 std [℃]'].idxmax())
s13_sphc = s13_lot_evidence[s13_lot_evidence['강종'].eq(s13_focus_grade)].copy()
s13_role = {
    240001:'추가 표본 확보', 240024:'강종 조건 참고', 240037:'동일 강종 대조',
    240038:'출력 변동폭 참고', 240044:'결측·BM 인접 비교', 240058:'종료 하강 참고',
    240061:'주 상세 사례', 240069:'CP·BM 인접 상세 비교', 240091:'TC-Z3 상대 기준',
    240108:'강종 조건 참고', 240114:'종료 하강·동일 강종 참고',
    240117:'초반 승온 참고', 240121:'TC-Z7·TC-Z8 상세 비교', 240133:'동일 강종 대조',
}
s13_lot_evidence['채택 역할'] = pd.Series(s13_role)
s13_sphc['채택 역할'] = pd.Series(s13_role)

# 표의 수치와 뒤의 마크다운 결과를 함께 확인합니다.
display(s13_lot_evidence[['강종','동일 강종 LOT수','관측 [분]','운전 전환','전환 구간',
    'OP-Z3 결측 [행]','OP-Z3 <1% [분]','안정 TC-Z3 차분 std [℃]',
    '안정 TC-Z3 차분 std 순위','TC-Z3↔OP-Z3 차분 r',
    '끝→다음 감지 [시간]','다음 정비','고장모드','채택 역할']].round(3))
print(f'주 상세 사례: {s13_case_lot} · {s13_focus_grade} / 비교 LOT: '+
      ', '.join(str(lot) for lot in s13_sphc.index if lot != s13_case_lot))


### ① 14 LOT에서 관측한 현상

| 현상 | 해당 LOT | 근거와 의미 |
|---|---|---|
| **종료 하강 — 정지형 전환 후보** | **240058(DP590): 173.0~216.5분**, **240114(SPCEN): 274.5~319.0분** | 두 LOT 모두 정기수리 시작 전날 관측. TC-Z3만이 아니라 TC-Z3·TC-Z4·TC-Z5·TC-Z6와 OP-Z1~OP-Z8가 함께 하강하므로 운전 종료 상태를 우선 확인 |
| **초반 승온 — 재가동형 전환 후보** | **240061(SPHC): 0~73.5분**, **240117(SPCD): 0~73.0분** | 정기수리 후 첫 관측 LOT. 수리 종료일 다음 날 00:00부터 각각 **1.1시간**, **2.3일** 뒤 시작. TC-Z3~TC-Z6·TC-Z7·TC-Z8 온도가 함께 상승 |
| 위 기준의 시작·종료 전환 없음 | 나머지 **10 LOT** | 855℃ 기준의 첫~마지막 도달 구간이 전체 관측 범위를 덮음. 개별 TC-Z3 피크·출력 하강은 별도로 점검 |
| **OP-Z3 초반 결측** | **240061: 179행 (0~183초)**, **240044: 75행 (0~81초)** | 같은 시각 TC-Z3는 관측됨. 전체 수집 공백과 특정 출력 채널 결측을 구분 |
| **OP-Z3 저출력(<1%)** | **240069 19.5분 · 240061 10.0분 · 240091 7.0분 · 240117 7.0분**; 240058·240114 각 0.5분 | 30초 평균 기준의 합계. 정확히 0%인 빈의 합계는 240069 14.0분, 240061 9.0분, 240091 5.5분, 240117 0.0분으로 다름 |
| **TC-Z3↔OP-Z3 변화량의 음의 상관** | **14 LOT 모두**, r **-0.68~-0.53** | 온도가 오르는 변화와 출력이 내리는 변화가 함께 나타남. 30초 차분 간 상관이며 온도 수준의 상관·고장 원인으로 해석하지 않음 |

**여기서 알게 된 점:** 끝의 큰 하강과 시작의 승온은 여러 채널이 함께 움직이는 전환 현상입니다. OP-Z3 결측, 저출력, 안정 구간의 흔들림은 서로 다른 현상이므로 분리해서 같은 시간축에 놓아야 합니다. 855℃ 경계는 그림 4~8의 5분 중앙값 하강 탐색 기준과 달라 시작 시각이 다를 수 있습니다.


### ② 선별 과정과 SPHC 4 LOT의 채택 이유

| 순서 | 적용 기준 | 결과 |
|---|---|---|
| **1. 전체 후보 확인** | 전체 LOT → 같은 강종 차이 → 중간 20~80% 시간 지속성 순으로 그림 10-A~H 확인 | CP·온도·출력·결측·전환을 각각 남겨 후보를 한 종류로 합치지 않음 |
| **2. 강종을 고정해 사례 비교** | 동일 강종 표본 수, 결측·저출력의 분포 확인 | **SPHC 4 · SPCE 3 · SPFH590 2 · SPCEN 2**, 나머지 1개씩. **SPHC**가 최대 표본이며 OP-Z3 결측 2 LOT가 모두 포함되고 저출력 ≥7분인 4 LOT 중 3개가 포함됨 |
| **3. 주 상세 사례 채택** | 한 LOT에 결측·초반 승온·저출력·안정 구간 단기 변동이 함께 있는가? | **240061(SPHC)**. 안정 TC-Z3 차분 std **0.709℃, 14 LOT 중 1위** |
| **4. 정비 맥락 연결** | 선정 후 수리 경계, LOT 종료→다음 CAL01 정비 감지 간격, 고장모드·비고 확인 | 240061은 수리 후 승온, 240044·240069는 BM 감지 인접, 240091은 비교용 상대 기준으로 역할을 구분 |

| SPHC LOT | 역할 | 채택 이유 | 함께 확인할 것 |
|---|---|---|---|
| **240061** | **주 상세 사례 — 수리 후 첫 관측·승온** | **OP-Z3 결측 179행 + 0~73.5분 승온 + 저출력 10.0분 + 안정 TC-Z3 차분 std 0.709℃(1위)**. 네 현상을 한 시간축에서 비교할 수 있어 그림 11의 주 사례로 채택 | 초반 결측과 승온을 분리. 중간 TC-Z3~TC-Z6 평균 변동폭 Δ +8.82℃도 확인하되, 결측 점검을 먼저 수행 |
| **240044** | **결측·BM 감지 인접 비교** | 두 번째 OP-Z3 결측 LOT(**75행**). 종료 **3.7시간** 뒤 BM·MAL 감지. 관측 **75분**으로 전체에서 가장 짧아 주 사례보다 결측·정비 맥락 비교에 적합 | 정비 비고는 **동작 불량**. 결측 원인과 BM 감지 전 운전 기록을 대조하며 전조 여부는 별도로 확인 |
| **240069** | **CP·저출력·BM 감지 인접 상세 비교** | 종료 **2.3시간** 뒤 BM·ELE 감지. **813분**으로 가장 길고 저출력 **19.5분**으로 최대. 같은 강종에서도 중간 CP-Z4 std **13.5배**가 남아 CP 상세 비교에 채택 | CP-Z4 피크의 원자료·분위기 제어 기록, 저출력 발생 시각, BM 비고(절연 저하) 대조. CP-Z4 변동의 중간 지속성은 4/12칸(33%). 피크 ±10분 제거 시 std 비율은 1.98배로 감소([구간 근거 점검](#interval-review)) |
| **240091** | **TC-Z3 비교용 상대 기준** | 시작·종료 전환과 OP-Z3 결측이 없고 안정 TC-Z3 차분 std **0.426℃**로 SPHC 중 가장 작음. 다음 감지는 종료 **52.2시간** 뒤로 SPHC 중 가장 멂 | **완전한 정상 기준으로 쓰지 않음.** 이후 BM 기록이 있고 TC-Z7·TC-Z8 평균온도는 같은 강종 대비 +3.14℃가 중간 12/12칸에 지속됨 |

**채택의 의미:** 240061은 여러 현상을 설명하기 위한 주 상세 사례이며 6절의 C(결측 점검 우선) 분류를 유지합니다. 240069의 CP, 240121의 TC-Z7·TC-Z8 평균온도는 별도 지표의 상세 분석 대상으로 유지합니다. BM 감지까지의 간격은 시간적 맥락이고 고장 전조나 수리 효과의 증거는 아닙니다.


### ③ 전체 14 LOT — 역할과 채택·보류 이유

**LOT 번호와 강종 외에도 관측 길이, 변화 공정·채널, 시작/종료 위치, 결측, 같은 강종 차이, 시간 지속성, 정비 간격을 함께 봅니다.** 아래의 역할은 주 사례·비교·참고·추가 표본 확보를 구분한 것이며 정상/고장 판정이 아닙니다.

| LOT | 강종(n) | 최종 역할 | 채택·보류 이유 | 다음 확인 |
|---|---|---|---|---|
| **240001** | SGCC (1) | 보류·추가 표본 | 다중신호 후보 기준 미만이며 SGCC n=1. 정상 기준으로 고정할 근거가 부족 | 추가 SGCC LOT, 목표온도·속도·두께 |
| **240024** | SPFH590 (2) | 강종 조건 참고 | 240108과 TC-Z7·TC-Z8와 TC-OUT1·TC-OUT2의 낮은 수준이 반복. 개별 LOT 이상보다 SPFH590 조건 비교에 적합 | 240108과 운전조건 대조; n=2 참고 |
| **240037** | SPCE (3) | SPCE 비교 LOT | 전체 후보였으나 중간의 강종 내 차이 기준 미만. 240121·240133의 대조 LOT로 남김 | 전체 구간 CP-Z4 피크와 240133의 공통 패턴 |
| **240038** | SPCEN (2) | SPCEN 참고 LOT | 240114와 TC-Z7·TC-Z8 평균온도·OP-Z3 변동폭이 다름. 표본 2개여서 어느 LOT가 특이한지는 확정하기 어려움 | 240114와 목표값·출력 변동폭 비교 |
| **240044** | SPHC (4) | 결측·BM 인접 비교 | OP-Z3 초반 결측 75행, 종료→BM 3.7시간. 짧은 LOT로 결측 비교와 정비 맥락에 채택 | 수집·통신 로그, BM MAL 동작 불량 기록 |
| **240058** | DP590 (1) | 종료 하강 참고·강종 내 보류 | 수리 전날 끝에서 여러 온도·출력이 함께 하강. DP590 n=1이라 강종 특성과 LOT 차이를 분리할 수 없음 | 종료 운전상태, 240114 하강과 비교, DP590 추가 표본 |
| **240061** | SPHC (4) | 주 상세 사례 | 결측 179행·초반 승온·저출력 10.0분·안정 TC-Z3 차분 std 0.709℃가 함께 있어 공정 흐름 상세 설명에 채택 | 결측 없는 구간의 TC-Z3~TC-Z6 평균 변동, 재가동·목표값 기록 |
| **240069** | SPHC (4) | CP 상세 비교 | CP-Z4 std가 같은 강종 대비 13.5배. 저출력 19.5분, 종료→BM 2.3시간도 시간축에 함께 확인 | CP-Z4 피크 원자료, 분위기 제어, BM ELE 기록 |
| **240091** | SPHC (4) | TC-Z3 상대 기준·TC-Z7·TC-Z8 참고 | SPHC 중 안정 TC-Z3 변동이 가장 작아 TC-Z3 비교에 채택. TC-Z7·TC-Z8 평균은 +3.14℃가 지속되어 별도 참고 | 과시효 목표값·속도·두께, 이후 BM 52.2시간 |
| **240108** | SPFH590 (2) | 강종 조건 참고 | 240024와 공통 저온 수준이 반복. 같은 강종의 TC-Z3~TC-Z6 평균 변동폭 차이는 n=2 참고 비교로 남김 | 240024와 소재·운전조건, 종료→CBM 4.0시간 |
| **240114** | SPCEN (2) | 종료 하강·SPCEN 참고 | 수리 전날 종료 하강은 전환 사례로 채택. 240038과의 온도·출력 차이는 n=2라 참고 | 종료 상태, 240038과 조건 비교, 정지계획 |
| **240117** | SPCD (1) | 초반 승온 참고·강종 내 보류 | 수리 후 첫 관측에서 승온과 저출력 7.0분. SPCD n=1이라 LOT 특이성은 비교 불가 | 240061과 전환 형태 비교, SPCD 추가 표본 |
| **240121** | SPCE (3) | TC-Z7·TC-Z8 상세 비교 | 같은 SPCE 대비 중간 TC-Z7·TC-Z8 평균온도 +3.08℃가 12/12칸 지속. 일시적 피크보다 지속 수준 차이를 확인하는 사례 | 240037·240133과 목표값·속도·두께 대조 |
| **240133** | SPCE (3) | SPCE 비교 LOT | 동일 강종 차이 기준 미만. 240037과 함께 높은 240121의 비교 기준이 되므로 보류 이유를 남기고 대조에 사용 | 240037과 공통 수준, TC-Z3 단기 피크 |

**이 비교에서 확인한 것:** 시작 승온·종료 하강, 특정 채널 결측, 제어출력 저하, 안정 구간 변동, 강종 내 지속 수준 차이는 서로 다른 점검 대상입니다. **240061을 중심으로 SPHC 4 LOT를 같은 시간축·강종에서 비교하고, 240121의 지속 온도 차이와 다른 강종의 전환·공통 조건 사례를 함께 검토합니다.**

**아래에서는 각 LOT의 실제 확대 구간과 그 구간을 고른 근거를 연결하고, 핵심 세 사례의 해석을 추가 점검합니다.**


<a id="interval-review"></a>
### ④ 어느 구간을 확대할 것인가 — 선정 이유와 근거 점검

**LOT·강종만으로 점검 대상을 정하지 않습니다.** 아래는 **컬럼 → 실제 구간 → 비교군 → 선정 근거 → 확인 가능한 결론**을 연결한 점검표입니다. 빨간 표시의 타당성은 '현상이 관측되는가'와 '원인까지 설명되는가'를 나눠 평가합니다.

**구간 읽기:** 분은 각 LOT의 원본 수집 시작 기준입니다. 중간 20~80%는 LOT별 길이가 달라 실제 분도 다릅니다. 결측은 원본 시각, 저출력·피크·855℃ 경계는 30초 관측 칸의 시각입니다. 확대 범위는 전후 맥락을 볼 창이며, 이상 상태의 정확한 시작·끝을 뜻하지 않습니다.

**12/12칸의 뜻:** 중간 20~80%를 진행률 5%씩 나눈 12구간의 중앙값 비교입니다. 모든 1초 값에서 차이가 났다는 뜻은 아닙니다. 상하위 순위·855℃·변동성 배율·지속성 기준은 이 자료의 탐색 규칙이며, 검증된 고장 판정 기준은 아닙니다.

| 확인 대상 · 컬럼 | 자세히 볼 구간 | 비교 대상 · 구간을 고른 이유 | 근거의 타당성 · 남은 판단 |
|---|---|---|---|
| **240044·240061 · SPHC / OP-Z3 결측** | **시작 0~5분**. 결측은 각각 **0~81초 / 0~183초**, 유효 기록은 82초 / 184초부터 | 서로의 시작 구간 및 240069·240091. **같은 시각 TC-Z3·OP-Z4·Z5·Z6는 기록됨** | **특정 채널 결측의 현상 근거는 충분.** 채널 초기화·통신·저장 원인은 로그 없이는 구분 불가 |
| **240061 / TC-Z3·Z4·Z5·Z6, OP-Z3** | **0~73.5분 승온**을 따로 보고, 승온 후 **73.5~180.3분** 재확인. 확대는 **123~139.5분**, 저출력 칸은 **128~134.5분** | 같은 LOT의 TC-Z4·Z5·Z6를 동시각 대조하고, 240044·240069·240091의 승온 제외 구간과 비교. **131분 TC-Z3 상대 상승과 OP-Z3=0%가 겹침** | **공통 승온과 TC-Z3 상대 이탈을 나눌 근거가 있음.** 중간 20~80%만 잘라도 45.1~73.5분 승온이 남음. 출력 감소는 정상 제어 반응일 수도 있어 고장으로 확정하지 않음 |
| **240069 · SPHC / CP-Z4** | 중간 전체 **162.5~650.1분**과 **피크 ±10분: 564.5~584.5분**. 피크는 **574.5분, 7/8 02:31:30** | 240044·240061·240091의 CP 중앙값·std. 피크 주변 제거 전후에도 전체 std 차이가 유지되는지 확인 | **피크와 큰 std의 관측 근거는 충분.** ±10분 제거 시 비교 std의 **13.48배 → 1.98배**로 낮아져 피크 주변 영향이 큼. 지속적인 CP 상승·정비 전조 근거는 부족 |
| **240069 / TC-Z3↔OP-Z3** | 저출력 칸 **393~397분**, **713.5~718.5분** 및 각각 전후 5분 | 같은 시각 TC-Z3와 OP-Z3를 대조. 긴 저출력 에피소드부터 확인 | **누적 저출력 19.5분은 한 번에 19.5분 지속했다는 뜻이 아님.** CP 피크와도 시각이 다름. 제어 목표·가동상태 확인 필요 |
| **240121 · SPCE / TC-Z7↔OP-Z7, TC-Z8↔OP-Z8** | **중간 전체 130.2~521.0분**. 관측 칸은 **11/24 09:41:30~16:11:30** | 240037·240133의 같은 진행률 구간. TC-Z7·Z8 평균의 높은 차이가 **12/12칸**에 반복 | **지속적인 수준 차이의 근거가 있음.** 개별 TC 차이는 **+3.14 / +3.01℃**. 목표온도·속도·두께가 없으므로 이상 원인·조업조건 효과는 미분리 |
| **240091 · SPHC / TC-Z7↔OP-Z7, TC-Z8↔OP-Z8** | **중간 전체 128.2~513.0분**. 관측 칸은 **9/9 00:11:30~06:35:30** | 240044·240061·240069의 같은 진행률. 평균 높은 차이 **12/12칸**, 개별 TC **+3.21 / +3.06℃** | **단일군이라도 지속 차이는 확인할 이유가 있음.** TC-Z3의 상대 기준으로 썼다고 모든 컬럼이 정상 기준인 것은 아님 |
| **240058·240114 / TC-Z3~Z6, OP-Z1~Z8, CP-Z4** | **240058: 173.0~216.5분**, **240114: 274.5~319.0분**, 하강 전 10분도 포함 | 여러 컬럼이 함께 하강하는지, 각 LOT의 종료 전 값과 비교. 서로 다른 강종은 하강 형태만 참고 | **공통 종료 전환의 근거가 있음.** 855℃는 탐색용 참고값. 실제 정지 원인·수리 효과는 운전 로그가 필요. DP590 n=1, SPCEN n=2 |
| **240117 · SPCD / TC-Z3~Z6, OP-Z3, CP-Z4** | **시작 0~73.0분** 승온과 아래 정비 창을 분리. CBM 후 저출력 칸 **347.5~352.5분 / 12:55:30~13:00:30**도 확인 | 같은 LOT의 전환 전후와 작업 시각. 240061은 승온 형태 참고 | **승온·작업 중첩·완료 후 재변동을 확인 가능.** n=1이며 정비의 개별 인과 효과는 분리 불가 |

**참고 비교 구간도 빠뜨리지 않습니다.**

| 비교 LOT · 강종 | 사용할 구간 · 컬럼 | 무엇을 확인할 것인가? |
|---|---|---|
| **240024 ↔ 240108 · SPFH590** | 각각 **109.6~438.2분 / 93.2~372.7분**. TC-Z7·Z8, TC-OUT1·OUT2 | 공통 수준과 냉각의 진폭·주기. 진행률을 맞춰도 진동 위상은 다르므로 순간값 차이를 이상으로 세지 않음. n=2 참고 |
| **240038 ↔ 240114 · SPCEN** | 각각 **25.0~99.9분 / 63.8~255.2분**. TC-Z3~Z6, OP-Z3, TC-Z7·Z8 | 종료 하강을 제외한 같은 강종 차이. OP 변동폭 차이가 나타난 진행률 칸만 다시 확대. n=2 참고 |
| **240037 ↔ 240133 · SPCE** | 각각 **122.3~489.1분 / 21.5~85.9분**과 각 LOT 시작·끝 20%. CP-Z4, TC-Z7·Z8, OP-Z3 | 높은 240121을 비교 기준에서 뺐을 때 두 LOT의 공통 수준이 유지되는가? 전체 구간에서만 선정된 변화가 시작·끝에 집중되는가? |
| **240001 · SGCC** | 현재 기준으로 우선 확대할 구간 미선정 | 기준 미만을 정상 판정으로 바꾸지 않음. 동일 강종·동일 조건의 추가 LOT 확보 |

**정비 전후는 감지 시각보다 실제 착수·완료 시각으로 정합니다.** 아래 두 작업 모두 **11/13 · SPCD 240117**입니다.

| 작업 · 유형 | 비교할 실제 구간 | 이 구간을 고른 이유 · 주의점 |
|---|---|---|
| **01859 · TBM** | 전 **07:30~07:40**, 후 **09:35~09:45** (LOT **22~32분 / 147~157분**) | 착수 직전·완료 직후의 동일 길이 관측. 전은 승온 중이며 후는 CBM 작업 중 |
| **01854 · CBM** | 전 **08:44~08:54**, 후 **12:44~12:54** (LOT **96~106분 / 336~346분**). 후 **12:44~13:04**도 확인 | 전은 TBM 작업 중. 완료 후 10분은 안정적으로 보이지만 20분 창에는 **12:55:30~13:00:30 저출력**도 포함 |
| **00717·01094 · BM, 02105 · CBM 부품교체** | 실제 완료 후 센서 관측 **없음** | 고장/교체 기록과 LOT 위치만 연결. 관측 없는 시간의 개선·악화는 판정 불가 |

**근거 점검 순서:** ① 원본·동시각 타 채널 확인 → ② 같은 강종·자기 제외 비교 → ③ 승온/종료 전환 제외 → ④ 단일 피크와 지속 차이 분리 → ⑤ 비교 창 길이를 바꿔 결론이 유지되는지 확인. 아래 세 그림은 이 점검을 실제로 수행한 결과입니다.


In [ ]:
# 확대 구간은 실제 관측에서 선정합니다. 직접 그린 결과를 그림 저장 셀의 설정대로 PNG로 저장합니다.
s14_middle_windows = pd.DataFrame([
    {'LOT': int(lot), '강종': r.grade,
     '중간 시작 [분]': .2*r.minutes, '중간 끝 [분]': .8*r.minutes,
     '첫 관측 칸': s6_analysis.loc[s6_analysis.LOT_NO.eq(lot)&s6_analysis.progress_pct.between(20,80),'MEAS_DT'].min(),
     '마지막 관측 칸': s6_analysis.loc[s6_analysis.LOT_NO.eq(lot)&s6_analysis.progress_pct.between(20,80),'MEAS_DT'].max()}
    for lot,r in s6_lots.iterrows()]).set_index('LOT')

# 저출력은 <1%인 연속 30초 칸만 묶습니다. LOT·SEG·실제 시간 공백을 넘지 않습니다.
s14_low_rows=[]
for s14_lot in [240061,240069,240091,240117]:
    s14_d=s6_analysis[s6_analysis.LOT_NO.eq(s14_lot)].sort_values('MEAS_DT').copy()
    s14_low=s14_d['OP-Z3'].lt(1)
    s14_new=(~s14_low.shift(fill_value=False))|s14_d.SEG.ne(s14_d.SEG.shift())|s14_d.MEAS_DT.diff().dt.total_seconds().ne(30)
    s14_d['low_run']=s14_new.cumsum()
    for _,s14_g in s14_d[s14_low].groupby('low_run'):
        s14_low_rows.append({'LOT':s14_lot,'시작 [분]':s14_g.elapsed_min.min(),
            '마지막 칸 [분]':s14_g.elapsed_min.max(),'시작 시각':s14_g.MEAS_DT.min(),
            '마지막 칸 시각':s14_g.MEAS_DT.max(),'관측 칸':len(s14_g),
            '칸수 환산 [분]':.5*len(s14_g),'OP-Z3 최솟값 [%]':s14_g['OP-Z3'].min()})
s14_low_episodes=pd.DataFrame(s14_low_rows).sort_values(['LOT','관측 칸'],ascending=[True,False])
display(s14_middle_windows.round({'중간 시작 [분]':2,'중간 끝 [분]':2}))
display(s14_low_episodes.groupby('LOT',sort=False).head(2).round({
    '시작 [분]':3,'마지막 칸 [분]':3,'칸수 환산 [분]':3,'OP-Z3 최솟값 [%]':3}))

# 240061: 승온 이후 가장 긴 저출력 에피소드 전후 5분을 확대합니다.
s14_run=s14_low_episodes[s14_low_episodes.LOT.eq(240061)].iloc[0]
s14_zoom=(float(s14_run['시작 [분]'])-5,float(s14_run['마지막 칸 [분]'])+5)
s14_case=analysis_ts[analysis_ts.LOT_NO.eq(240061)&analysis_ts.elapsed_min.between(*s14_zoom)].copy()
s14_tc_peak=s14_case.loc[s14_case.relative_z.abs().idxmax()]
s14_fig,s14_axes=plt.subplots(2,1,figsize=(14,7.8),sharex=True)
s14_fig.suptitle('그림 19-A. SPHC 240061 — 승온 후 TC-Z3 상대 상승과 OP-Z3 저출력',
    x=.09,y=.98,ha='left',fontweight='bold',fontsize=15)
s14_fig.text(.09,.915,
    f'확대 {s14_zoom[0]:.1f}~{s14_zoom[1]:.1f}분 · 실제 시각 '
    f'{s14_case.MEAS_DT.min():%m/%d %H:%M:%S}~{s14_case.MEAS_DT.max():%H:%M:%S} · 관측 공백 연결 없음',
    color='#66717D',fontsize=10)
for s14_c,s14_ls in zip(['TC-Z3','TC-Z4','TC-Z5','TC-Z6'],['-','-','--',':']):
    for _,s14_g in s14_case.groupby('SEG'):
        s14_axes[0].plot(s14_g.elapsed_min,s14_g[s14_c],ls=s14_ls,
            color=colors['SPHC'] if s14_c=='TC-Z3' else '#8C969F',
            lw=1.7 if s14_c=='TC-Z3' else 1,label=s14_c if _==s14_case.SEG.iloc[0] else None)
for _,s14_g in s14_case.groupby('SEG'):
    s14_axes[1].plot(s14_g.elapsed_min,s14_g['OP-Z3'],color=colors['SPHC'],lw=1.5)
s14_axes[0].set_title('TC-Z3 · 같은 시각의 TC-Z4·TC-Z5·TC-Z6와 대조',loc='left',fontweight='bold',pad=12)
s14_axes[1].set_title('OP-Z3 · TC-Z3과 같은 존의 제어출력',loc='left',fontweight='bold',pad=12)
s14_axes[0].legend(frameon=False,ncol=4,loc='upper right',fontsize=9)
s14_axes[1].axhline(1,color='#9099A2',ls='--',lw=.8)
for s14_ax,s14_channel,s14_unit in zip(s14_axes,['TC-Z3','OP-Z3'],['℃','%']):
    s14_ax.scatter([s14_tc_peak.elapsed_min],[s14_tc_peak[s14_channel]],s=55,
        facecolors='none',edgecolors='#B4232F',lw=1.6,zorder=5)
    s14_ax.annotate(f'240061 · {s14_tc_peak.elapsed_min:.1f}분 · {s14_tc_peak[s14_channel]:.2f}{s14_unit}',
        (s14_tc_peak.elapsed_min,s14_tc_peak[s14_channel]),xytext=(15,12),textcoords='offset points',
        color='#B4232F',fontsize=10,fontweight='bold',
        arrowprops={'arrowstyle':'-','color':'#B4232F','lw':.8})
    s14_ax.axvspan(s14_run['시작 [분]'],s14_run['마지막 칸 [분]'],color='#B4232F',alpha=.045)
    s14_ax.spines[['top','right']].set_visible(False)
    s14_ax.grid(alpha=.14)
    s14_ax.set_xlim(*s14_zoom)
s14_axes[0].set_ylabel('온도 [℃]')
s14_axes[1].set_ylabel('제어출력 [%]')
s14_axes[1].set_ylim(-8,110)
s14_axes[1].set_xlabel('LOT 시작 후 경과 [분]')
s14_fig.text(.09,.024,'옅은 붉은 구간 = 저출력 관측 칸의 처음~마지막 시각 · <1%와 정확히 0%는 구분 · 같은 시각 동반 변화는 인과 증명이 아님',
    fontsize=9,color='#66717D')
s14_fig.subplots_adjust(left=.09,right=.98,top=.82,bottom=.11,hspace=.48)
save_and_show_figures()


#### 그림 19-A — 승온 이후에도 자세히 볼 이유가 남는가?

**① 읽는 순서:** 위에서 TC-Z3만 다른 TC보다 높아지는지 보고, 아래에서 같은 시각 OP-Z3가 어떻게 움직이는지 확인합니다. 동그라미 옆 수치는 실제 30초 관측값입니다.

**② 확인한 결과**

| 구간·컬럼 | 관측 근거 | 무엇을 알게 되었는가? |
|---|---|---|
| **128~134.5분 / OP-Z3** | 연속 **14칸, 환산 7.0분**의 <1% 관측. 전체 저출력 누적 10분 중 가장 긴 에피소드 | 어느 시간부터 원자료를 확대할지 지정할 수 있음. 30초 평균 <1%이지 1초 원본의 연속 0% 시간은 아님 |
| **131분 / TC-Z3↔OP-Z3** | **03:15:00**, TC-Z3 **870.536℃**, 다른 TC-Z4~Z6 평균 대비 **+10.366℃**, OP-Z3 **0%** | 공통 승온과 다른, 같은 시각의 TC-Z3 상대 상승·출력 억제가 존재 |
| 승온 제외 참고 구간 | TC-Z3 30초 차분 std **0.709℃**, 14 LOT 중 가장 큼 | 초반 승온만으로 전체 TC-Z3 단기 변동을 설명하기 어려움. 855℃는 정상 규격이 아닌 참고 구간 정의 |

**결론:** **승온 이후 128~134.5분이 TC-Z3↔OP-Z3 상세 점검의 우선 구간**입니다. 온도 상승 때 출력을 줄이는 것은 정상적인 피드백 반응일 수 있습니다. 목표온도 대비 이탈과 제어 입력·외란을 확인하기 전에는 출력 0%를 고장으로 판정하지 않습니다. LOT별 robust z 기준은 이 자료에서 사후 계산한 탐색 기준이며 검증된 경보 한계가 아닙니다.

**③ 후속 작업**

| 대상·시각 | 비교할 것 | 필요한 자료 |
|---|---|---|
| **240061 / 03:07~03:23:30** | TC-Z3 실제값·목표값·OP-Z3, 인접 TC-Z4~Z6. 다른 SPHC의 유사 운전상태·저출력 에피소드와 대조 | 존별 목표값, 제어 모드, 소재 속도·두께, 가동상태 |
| **240044·240061 / 시작 0~5분** | 결측 발생·유효화와 같은 시각 다른 출력 채널 기록 | 태그·통신·수집·저장 로그. 결측 원인과 위 제어 반응의 원인은 별도 확인 |


In [ ]:
# 240069: 중간 구간을 먼저 보고 피크 주변의 확대와 제거 민감도를 확인합니다.
s14_cp=s6_analysis[s6_analysis.LOT_NO.eq(240069)&s6_analysis.progress_pct.between(20,80)].copy()
s14_cp_peak=s14_cp.loc[s14_cp['CP-Z4'].idxmax()]
s14_cp_reference_std=float(s6_peer_ref['중간 20~80%'].loc[240069,'CP_sd'])
s14_cp_reference_level=float(s6_peer_ref['중간 20~80%'].loc[240069,'CP-Z4'])
s14_cp_sensitivity_rows=[]
for s14_exclusion in [0,2.5,5,10]:
    s14_remaining=(s14_cp if s14_exclusion==0 else
        s14_cp[(s14_cp.elapsed_min-s14_cp_peak.elapsed_min).abs().gt(s14_exclusion)])
    s14_cp_sensitivity_rows.append({'피크 주변 제외': '제외 없음' if s14_exclusion==0 else f'±{s14_exclusion:g}분',
        '제거 칸':len(s14_cp)-len(s14_remaining),'유효 CP 칸':int(s14_remaining['CP-Z4'].count()),
        '남은 CP-Z4 std':s14_remaining['CP-Z4'].std(),
        '고정 비교 std':s14_cp_reference_std,
        '비교 std 대비 [배]':s14_remaining['CP-Z4'].std()/s14_cp_reference_std,
        '남은 CP 중앙값':s14_remaining['CP-Z4'].median()})
s14_cp_sensitivity=pd.DataFrame(s14_cp_sensitivity_rows)
s14_cp_zoom=(s14_cp_peak.elapsed_min-10,s14_cp_peak.elapsed_min+10)
s14_fig,s14_axes=plt.subplots(2,1,figsize=(14,7.8))
s14_fig.suptitle('그림 19-B. SPHC 240069 — CP-Z4 피크 주변이 큰 std를 설명하는가?',
    x=.09,y=.98,ha='left',fontweight='bold',fontsize=15)
s14_fig.text(.09,.914,
    f'피크 {s14_cp_peak.elapsed_min:.1f}분 · {s14_cp_peak.MEAS_DT:%m/%d %H:%M:%S} · '
    '검정 점선: 자기 제외 SPHC 중간 중앙값 (시간 정렬 선·정상 한계 아님)',
    fontsize=10,color='#66717D')
for s14_i,s14_ax in enumerate(s14_axes):
    s14_view=s14_cp if s14_i==0 else s14_cp[s14_cp.elapsed_min.between(*s14_cp_zoom)]
    for _,s14_g in s14_view.groupby('SEG'):
        s14_ax.plot(s14_g.elapsed_min,s14_g['CP-Z4'],color=colors['SPHC'],lw=1.3)
    s14_ax.axhline(s14_cp_reference_level,color='#263238',ls='--',lw=1)
    s14_ax.scatter([s14_cp_peak.elapsed_min],[s14_cp_peak['CP-Z4']],s=55,
        facecolors='none',edgecolors='#B4232F',lw=1.6,zorder=5)
    s14_ax.annotate(f'240069 · {s14_cp_peak.elapsed_min:.1f}분 · {s14_cp_peak["CP-Z4"]:.4f}',
        (s14_cp_peak.elapsed_min,s14_cp_peak['CP-Z4']),xytext=(15,12),
        textcoords='offset points',color='#B4232F',fontweight='bold',fontsize=10,
        arrowprops={'arrowstyle':'-','color':'#B4232F','lw':.8})
    s14_ax.set_title('CP-Z4 · 중간 20~80% 전체에서 위치 확인' if s14_i==0 else
        'CP-Z4 · 피크 전후 10분 확대',loc='left',fontweight='bold',pad=12)
    s14_ax.set_xlabel('LOT 시작 후 경과 [분]')
    s14_ax.set_ylabel('CP-Z4 [무차원]')
    s14_ax.spines[['top','right']].set_visible(False)
    s14_ax.grid(alpha=.14)
    s14_ax.set_ylim(.39,.98)
s14_axes[0].axvspan(*s14_cp_zoom,color='#B4232F',alpha=.05)
s14_axes[1].set_xlim(*s14_cp_zoom)
s14_fig.text(.09,.025,
    '제거 민감도는 동일 SPHC의 원래 중간 std 기준을 고정한 참고 계산 · 확대 폭은 사후 점검용 · 새로운 판정 기준으로 사용하지 않음',
    fontsize=9,color='#66717D')
s14_fig.subplots_adjust(left=.09,right=.98,top=.82,bottom=.13,hspace=.55)
save_and_show_figures()
display(s14_cp_sensitivity.round(6))


#### 그림 19-B — CP의 '지속 차이'와 '피크의 영향'은 다른가?

**① 읽는 순서:** 위에서 중간 전체 중 피크가 차지하는 위치를 보고, 아래에서 **7/8 02:21:30~02:41:30**의 전후 흐름을 봅니다. 이어서 피크 주변을 뺀 뒤 std가 얼마나 바뀌는지 확인합니다.

**② 확인한 결과**

| 240069 중간 구간에서 제외한 범위 | 남은 CP-Z4 std | 동일 SPHC 비교 std 대비 | 알게 된 점 |
|---|---:|---:|---|
| 제외 없음 | **0.036325** | **13.48배** | 같은 강종에서도 큰 변동 후보 |
| 피크 ±2.5분 | 0.016441 | 6.10배 | 최고점 하나만이 아니라 주변 변동도 영향을 줌 |
| 피크 ±5분 | 0.007559 | 2.81배 | 주변을 더 제외할수록 차이 감소 |
| **피크 ±10분** | **0.005331** | **1.98배** | 큰 전체 std의 상당 부분은 이 국소 구간에 의존 |

**결론:** **564.5~584.5분을 우선 확대할 이유는 타당합니다.** CP 피크 자체는 명확하지만, LOT 내내 CP가 크게 흔들린다는 해석은 맞지 않습니다. 제외 범위는 사후에 정한 점검 창이고 비교군 std는 기존 중간 구간 값으로 고정했으므로, 이 계산은 피크 기여를 확인하는 민감도 분석입니다. 제외 후 값을 새로운 고장 판정이나 독립 검증으로 사용하지 않습니다.

**③ 후속 작업**

| 대상·구간 | 다음 확인 | 필요한 자료 |
|---|---|---|
| **240069 / 7/8 02:21:30~02:41:30** | 1초 CP-Z4, 분위기 설정·교정·통신 이벤트, 동시각 TC-Z3·OP-Z3 | 원본 태그 정의·분석계/제어 로그. 단일 채널 계측 피크와 실제 분위기 변화를 구분 |
| **240069 / 393~397분, 713.5~718.5분** | CP 피크와 시각이 다른 긴 OP-Z3 저출력 에피소드를 TC-Z3와 별도로 비교 | 존별 목표값·운전 모드. 두 현상을 같은 원인으로 합치지 않음 |


In [ ]:
# 실제 착수 직전/완료 직후 5·10·20분 창을 같은 규칙으로 재계산합니다.
# 창 길이 선택에 따른 해석을 점검합니다. 반복 독립 실험이나 통계적 유의성 검정은 아닙니다.
s14_mnt_rows=[]
for s14_wo in mnt_actual_pairs['작업'].drop_duplicates():
    s14_work=cal[cal.WO_NO.eq(s14_wo)].iloc[0]
    s14_work_lot=mnt_lot_at(s14_work.STRT_DT)
    s14_sample=temp[temp.LOT_NO.eq(s14_work_lot)]
    for s14_width in [5,10,20]:
        s14_before_start=s14_work.STRT_DT-pd.Timedelta(minutes=s14_width)
        s14_after_end=s14_work.END_DT+pd.Timedelta(minutes=s14_width)
        if not (lot_map.loc[s14_work_lot,'start']<=s14_before_start and
                lot_map.loc[s14_work_lot,'end']>=s14_after_end):
            continue  # 전체 창이 관측되지 않으면 전후 비교에 사용하지 않습니다.
        s14_before=s14_sample[s14_sample.MEAS_DT.ge(s14_before_start)&s14_sample.MEAS_DT.lt(s14_work.STRT_DT)]
        s14_after=s14_sample[s14_sample.MEAS_DT.ge(s14_work.END_DT)&s14_sample.MEAS_DT.lt(s14_after_end)]
        for s14_col in ['TC-Z3','OP-Z3','CP-Z4']:
            s14_pre_std=s14_before[s14_col].std()
            s14_post_std=s14_after[s14_col].std()
            s14_mnt_rows.append({'작업':s14_wo,'유형':s14_work.MNT_TYPE,'LOT':s14_work_lot,'창 [분]':s14_width,
                '컬럼':s14_col,'전 시작':s14_before_start,'전 끝':s14_work.STRT_DT,
                '후 시작':s14_work.END_DT,'후 끝':s14_after_end,
                '전 유효행':int(s14_before[s14_col].count()),'후 유효행':int(s14_after[s14_col].count()),
                '평균 차이':s14_after[s14_col].mean()-s14_before[s14_col].mean(),
                '전 std':s14_pre_std,'후 std':s14_post_std,
                'std 후/전 [배]':s14_post_std/s14_pre_std if s14_pre_std>0 else np.nan})
s14_mnt_sensitivity=pd.DataFrame(s14_mnt_rows)
s14_mnt_wide=s14_mnt_sensitivity.pivot(index=['작업','유형','컬럼'],columns='창 [분]',values='std 후/전 [배]')
display(s14_mnt_wide.round(3))
s14_fig,s14_axes=plt.subplots(3,1,figsize=(14,10.4),sharex=True)
s14_fig.suptitle('그림 19-C. 정비 전후 — 비교 창을 넓혀도 std 감소가 유지되는가?',
    x=.09,y=.98,ha='left',fontweight='bold',fontsize=15)
s14_fig.text(.09,.925,
    '같은 SPCD 240117 · 착수 직전/완료 직후 각각 5·10·20분 · 세 창은 겹치는 관측이며 독립 표본이 아님',
    fontsize=10,color='#66717D')
for s14_ax,s14_col in zip(s14_axes,['TC-Z3','OP-Z3','CP-Z4']):
    s14_ax.set_title(f'{s14_col} std · 완료 후 / 착수 전',loc='left',fontweight='bold',pad=11)
    for s14_wo,s14_label,s14_color,s14_style in [
        ('WO-2024-01859','TBM 01859 · 계획 점검','#727C85','--'),
        ('WO-2024-01854','CBM 01854 · 상세 조치 미기록','#B4232F','-')]:
        s14_points=s14_mnt_sensitivity[
            s14_mnt_sensitivity['작업'].eq(s14_wo)&s14_mnt_sensitivity['컬럼'].eq(s14_col)].sort_values('창 [분]')
        s14_ax.plot(s14_points['창 [분]'],s14_points['std 후/전 [배]'],
            marker='o',ms=6,mfc='white',color=s14_color,ls=s14_style,lw=1.4,label=s14_label)
        for s14_point in s14_points.itertuples(index=False,name=None):
            s14_min=s14_point[3];s14_ratio=s14_point[-1]
            s14_ax.annotate(f'{s14_ratio:.2f}배',(s14_min,s14_ratio),
                xytext=(8,8) if s14_wo.endswith('01854') else (-8,8),
                textcoords='offset points',ha='left' if s14_wo.endswith('01854') else 'right',
                fontsize=10,color=s14_color,fontweight='bold' if s14_wo.endswith('01854') else 'normal')
    s14_ax.axhline(1,color='#9099A2',ls=':',lw=1)
    s14_ax.set_xlim(2.5,23)
    s14_max=s14_mnt_sensitivity.loc[s14_mnt_sensitivity['컬럼'].eq(s14_col),'std 후/전 [배]'].max()
    s14_ax.set_ylim(-.12,max(1.5,s14_max*1.3))
    s14_ax.set_ylabel('std 후/전 [배]')
    s14_ax.grid(alpha=.14)
    s14_ax.spines[['top','right']].set_visible(False)
s14_axes[0].legend(loc='upper right',frameon=False,fontsize=9)
s14_axes[-1].set_xticks([5,10,20])
s14_axes[-1].set_xlabel('착수 전과 완료 후에 각각 사용하는 관측 창 [분]')
s14_fig.text(.09,.025,'1배 미만: 해당 창의 std 감소 · 1배 초과: 증가 · 정비 효능/고장 확률을 뜻하지 않음 · 승온·작업 중첩·제어 진동은 별도 확인',
    color='#66717D',fontsize=9)
s14_fig.subplots_adjust(left=.09,right=.98,top=.84,bottom=.1,hspace=.49)
save_and_show_figures()

# 기존 10분 결과와 같은 데이터·같은 구간인지 검증합니다.
for s14_channel in ['TC-Z3','OP-Z3','CP-Z4']:
    for s14_work_no in mnt_actual_pairs['작업'].unique():
        s14_expected=mnt_actual_pairs[
            mnt_actual_pairs['작업'].eq(s14_work_no)&mnt_actual_pairs['컬럼'].eq(s14_channel)].iloc[0]
        s14_actual=s14_mnt_sensitivity[
            s14_mnt_sensitivity['작업'].eq(s14_work_no)&s14_mnt_sensitivity['컬럼'].eq(s14_channel)&
            s14_mnt_sensitivity['창 [분]'].eq(10)].iloc[0]
        np.testing.assert_allclose(
            [s14_actual['평균 차이'],s14_actual['전 std'],s14_actual['후 std']],
            [s14_expected['평균 차이'],s14_expected['작업 전 std'],s14_expected['완료 후 std']])
assert s14_cp_peak.LOT_NO==240069 and s14_tc_peak.LOT_NO==240061
assert s14_low_episodes['칸수 환산 [분]'].eq(s14_low_episodes['관측 칸']*.5).all()
assert s14_middle_windows.index.is_unique and set(s14_middle_windows.index)==set(lot_map.index)
print('구간 점검 완료: LOT별 실제 시간, 연속 저출력, 피크 제거 민감도, 정비 5/10/20분 창과 기존 10분 결과 일치')


#### 그림 19-C — 정비 후 '잠깐 안정'과 '지속 개선'을 구분한 결과

**① 읽는 순서:** 1배 선을 기준으로 봅니다. **std 후/전 <1이면 해당 창에서 감소, >1이면 증가**입니다. x축은 완료 후 경과시간이 아니라 **전·후에 각각 사용할 창 길이**입니다.

**② 확인한 결과 — 앞의 10분 비교 해석을 점검했습니다.**

| 작업 · 컬럼 | 5분 창 후/전 | 10분 창 후/전 | 20분 창 후/전 | 알게 된 점 |
|---|---:|---:|---:|---|
| **CBM 01854 / TC-Z3 std** | **0.36배** | **0.20배** | **2.00배** | 5·10분에서는 감소하나 20분에서는 증가 |
| **CBM 01854 / OP-Z3 std** | **0.43배** | **0.24배** | **1.34배** | 출력 변동 감소도 창을 넓히면 유지되지 않음 |
| **CBM 01854 / CP-Z4 std** | **0.17배** | **0.58배** | **3.23배** | CP도 완료 후 긴 창에서 다시 큰 변동을 포함 |
| TBM 01859 / TC-Z3 std | 2.47배 | 0.85배 | 0.60배 | 전 창의 승온 길이에 따라 방향이 바뀜 |
| TBM 01859 / OP-Z3 std | 5.12배 | 1.68배 | 1.74배 | 세 창 모두 증가. 온도 상승과 출력 안정은 같은 결과가 아님 |
| TBM 01859 / CP-Z4 std | 2.32배 | 1.90배 | 1.75배 | 세 창 모두 증가 |

**결론:** **CBM 완료 후 10분의 일시적 안정은 관측되지만, 지속적인 안정화·정비 효과라는 근거는 약합니다.** 완료 후 20분 **12:44~13:04**에는 **12:55:30~13:00:30 OP-Z3 저출력 에피소드**가 추가됩니다. 전 창도 함께 넓힌 계산이므로 뒤의 재변동만이 차이의 유일한 원인이라고 단정하지 않습니다. 두 작업 중첩, 초반 승온, 제어 주기, SPCD n=1, 상세 조치 미기록도 남아 있습니다.

**③ 후속 작업 — 무엇이 있어야 원인을 설명할 수 있는가?**

| 우선 대상·구간 | 확인할 내용 | 근거를 보강할 자료 |
|---|---|---|
| **240117 / 11/13 12:44~13:04** | 완료 후 재변동, OP-Z3<1%와 TC-Z3·CP-Z4의 동시 변화; 목표값·제어 모드가 바뀌었는가? | 1초 계측, 목표값·가동상태·소재 조건 |
| **TBM 01859·CBM 01854 / 08:54~09:35 중첩** | 어떤 부위를 언제 조정·점검했고 어느 태그와 대응하는가? | 작업 상세·부위·조정값·정확한 적용 시각 |
| 동일 상태·동일 길이의 비정비 구간 | 정비가 없어도 이 정도 전후 std 차이가 발생하는가? | 반복 SPCD LOT, 동일 운전조건 대조군 |
| BM 00717·01094, CBM 02105 | 실제 완료 직후 및 이후 센서가 있는가? | 완료 후 연속 계측. 현재 관측 공백에서는 변화 방향 자체가 미확인 |

**현재 알 수 있는 범위:** **구간 선정과 관측 현상의 근거는 제시할 수 있습니다.** 특정 고장 원인·정비 인과 효과·정상 운전 한계는 운전조건과 정비·수집 로그 없이 확정하지 않습니다.

[↑ 목차](#toc)


In [ ]:
# 결과 재현 및 경계 처리 검증: 원본 데이터는 유지하며 생성한 그림만 지정 폴더에 PNG로 저장합니다.
assert sum(len(cols) for cols in all_groups.values()) == 22
assert len(joined) == len(replacement)
assert feature.index.is_unique and len(feature) == len(lots)
assert lots.EVT_ID.notna().all(), '강종 미매칭: 해석 갱신 필요'
assert temp.groupby('SEG').LOT_NO.nunique().eq(1).all()
assert temp.groupby('SEG').MEAS_DT.apply(lambda s:s.diff().dt.total_seconds().dropna().le(10).all()).all()
assert analysis_ts.groupby('SEG').LOT_NO.nunique().eq(1).all()
assert not analysis_ts.duplicated(['LOT_NO','SEG','MEAS_DT']).any()
assert analysis_ts.heat_rate[delta_t.ne(.5)].isna().all()
print('검증 완료: 7개 CSV 로딩, 조인 행수 유지, 22개 신호 포함, LOT·SEG 경계 보호, 시차·변화율 유효 관측 처리')



# 6절 의미 검증: 자기 제외 기준·신호군 수·시간 창·CP 경계·계획수리 범위.
assert set(s6_full.index)==set(lot_map.index)
assert s6_screen['전체 신호군 수'].le(3).all() and s6_screen['중간 신호군 수'].le(3).all()
assert s6_windows['중간 20~80%'].progress_pct.between(20,80).all()
for lot,peers in s6_peer_ids.items():
    assert lot not in peers and len(peers)==s6_lots.loc[lot,'grade_n']-1
    assert all(s6_lots.loc[p,'grade']==s6_lots.loc[lot,'grade'] for p in peers)
    for window,values in s6_metrics_by_window.items():
        if peers: pd.testing.assert_series_equal(s6_peer_ref[window].loc[lot],values.loc[peers].median(),check_names=False)
        else: assert s6_delta[window].loc[lot].isna().all()
for _,d in s6_analysis.groupby(['LOT_NO','SEG']):
    expected=d.set_index('MEAS_DT')['CP-Z4'].rolling('5min',min_periods=5).std()
    np.testing.assert_allclose(d.CP_roll_sd.to_numpy(),expected.to_numpy(),equal_nan=True)
assert s6_repair_link['경계 거리 [일]'].between(0,REMOTE_DAYS).all()
assert s6_quality_middle.loc[[240044,240061]].eq(0).all()
assert s6_review.loc[240058,'분류'].startswith('D') and s6_review.loc[240061,'분류'].startswith('C')
assert s6_persistence['유효 중간 칸'].between(0,12).all()
print('6절 검증 완료: 자기 제외 강종 기준, 신호군 3표 한도, 20~80% 창, 무보간 비교, CP 이동 SD 경계, 수리 ±60일 탐색')

# 단계 요약 시각화의 값이 기존 계산과 일치하는지 검증합니다.
assert list(s6_summary_after.columns)==list(S6_FAMILIES)
assert s6_summary_before.sum(axis=1).eq(s6_screen.loc[s6_summary_ids,['전체 신호군 수','중간 신호군 수']].max(axis=1)).all()
assert s6_summary_after.loc[s6_lots.loc[s6_summary_ids,'grade_n'].eq(1)].isna().all().all()
assert set(np.unique(s6_time_state[np.isfinite(s6_time_state)]))<={-1.,0.,1.}
assert s6_time_state.shape==(len(s6_time_rows),20)
for row,(lot,c) in zip(s6_time_state,s6_time_rows):
    p=s6_persistence[s6_persistence.LOT.eq(lot)&s6_persistence['채널'].eq(c)].iloc[0]
    middle=row[4:16]
    assert int(np.isfinite(middle).sum())==int(p['유효 중간 칸'])
    assert max(int((middle==1).sum()),int((middle==-1).sum()))==int(p['같은 방향 차이 칸'])
print('단계 요약 검증 완료: 신호군 중복 집계 방지, n=1 비교 불가, 중간 지속성 지도와 원래 집계 일치')

# 마지막 선정 보고와 앞선 상세 사례·전체 LOT 검토가 같은 데이터인지 확인합니다.
assert s13_case_lot == CASE_LOT, '주 상세 사례가 앞선 그림과 다릅니다.'
assert set(s13_lot_evidence.index) == set(s6_audit.index)
assert s13_lot_evidence['채택 역할'].notna().all()
assert s13_lot_evidence['OP-Z3 =0% [분]'].le(s13_lot_evidence['OP-Z3 <1% [분]']).all()
assert s13_lot_evidence.loc[s13_case_lot, '안정 TC-Z3 차분 std 순위'] == 1
print('마지막 보고 검증 완료: 14 LOT 근거·역할, 상세 사례 일치, 0%와 1% 미만 구분, 안정 구간 차분 순위')

# 결측 대상을 신호값의 순위 후보와 별도로 유지하고 원인 비교 근거를 확인합니다.
assert set(s6_quality_lots)==set(s6_screen.index[s6_screen['OP-Z3 원본 결측행'].gt(0)])
assert s6_screen.loc[s6_quality_lots,'값·품질 확인 대상'].all()
assert s6_screen.loc[240044,'결측 확인 대상'] and not s6_screen.loc[240044,'탐색 후보']
for lot,r in s6_quality_pair.iterrows():
    assert r['최초 유효값 [초]']>r['마지막 결측 [초]']
    for channel in ['TC-Z3','OP-Z4','OP-Z5','OP-Z6']:
        assert r[f'결측 동안 {channel} 관측 [행]']==r['OP-Z3 결측 [행]']
    assert r['유효값 시작 후 추가 결측 [행]']==0
    assert r['중간 20~80% 결측 [행]']==0
print('결측 비교 검증 완료: 값·품질 후보 분리, 최초 유효값 시각, 동시 타채널 기록, 초반 결측과 중간 유효 관측 구분')

# 컬럼 표시와 평균 지표가 실제 사용 컬럼에 일치하는지 확인합니다.
for s6_source_group,s6_source_columns in {
    '가열·균열존':['TC-Z3','TC-Z4','TC-Z5','TC-Z6'],
    '과시효대':['TC-Z7','TC-Z8'],'냉각대':['TC-OUT1','TC-OUT2']}.items():
    s6_expected_mean=analysis_ts[s6_source_columns].mean(axis=1).where(analysis_ts[s6_source_columns].notna().all(axis=1))
    np.testing.assert_allclose(analysis_ts[s6_source_group],s6_expected_mean,equal_nan=True)
for s6_tc_row in s6_tc_case_table.itertuples(index=False,name=None):
    s6_check_lot,s6_check_grade,_,s6_check_col,s6_check_metric,s6_check_value,s6_check_reference,_=s6_tc_row
    s6_check_matrix=s6_tc_level if s6_check_metric=='중앙값' else s6_tc_width
    assert s6_check_reference==s6_check_matrix.loc[s6_peer_ids[s6_check_lot],s6_check_col].median()
assert len(s6_tc_details)==14*8*2 and len(s6_tc_case_table)==12
print('컬럼 표시 검증 완료: TC-Z3~TC-Z6, TC-Z7·TC-Z8, TC-OUT1·TC-OUT2 평균 및 개별 컬럼 자기 제외 비교')
